<div style="background: linear-gradient(135deg, #0b1021, #14213d, #1b2a4a); border-radius: 16px; padding: 36px 40px; margin-bottom: 8px;">
  <h1 style="color: #8ecae6; font-size: 2.3em; font-weight: 800; margin: 0 0 10px 0; letter-spacing: 0.5px;">
    &#9889; iTransformer &middot; Walk-Forward BTCUSDT 1 jam
  </h1>
  <p style="color: #ffb703; font-size: 1.12em; margin: 0 0 18px 0; font-weight: 500;">
    Variat nominal atau dimensionalitas efektif? &mdash; 15 origin &middot; 1.620 run &middot; 2 &times; T4
  </p>
  <hr style="border: none; border-top: 1px solid #2a4365; margin: 16px 0;">
  <p style="color: #a8c0dd; font-size: 0.97em; margin: 0 0 12px 0;">
    <strong>Dibaca dari atas ke bawah, sekali jalan.</strong> Notebook ini disusun menurut
    <em>tahap pipeline</em>, bukan menurut nama berkas: konfigurasi &rarr; muat data &rarr;
    pra-proses &rarr; split &rarr; algoritma &rarr; latih &rarr; metrik &rarr; baseline &rarr;
    grid &rarr; evaluasi &rarr; laporan. Definisi satu tahap dan eksekusinya duduk berdampingan,
    jadi tidak ada lagi tumpukan modul di tengah yang harus dilewati untuk sampai ke pekerjaannya
    (<code>D73</code>). Nama modul turun satu tingkat, ke banner <code>###</code> dan ke
    <code>cell.metadata</code>, karena badan tiap sel adalah potongan byte-exact berkas itu dan
    pembaca yang membandingkan keduanya tetap butuh namanya.
  </p>
  <p style="color: #a8c0dd; font-size: 0.97em; margin: 0 0 12px 0;">
    <strong>Mandiri.</strong> Yang dibutuhkan sesi Kaggle hanya dua: notebook ini, dan
    <code>BTCUSDT_1h.parquet</code> terpasang sebagai Kaggle Dataset. Setiap definisi yang dipakai
    ada <em>di dalamnya</em> &mdash; sel-sel di bawah adalah <code>def</code>, <code>class</code>,
    dan konstanta biasa yang dijalankan berurutan. Tidak ada yang ditulis ke disk untuk diimpor
    balik, tidak ada paket <code>itransformer_btc</code> di mesin ini, dan tidak ada
    <code>src/</code> di <code>sys.path</code>.
  </p>
  <p style="color: #7f9cc0; font-size: 0.93em; margin: 0;">
    Menjawab <strong>RQ1</strong> (manfaat mengikuti K atau K<sub>eff</sub>?),
    <strong>RQ2</strong> (apakah gap multivariat menyempit seiring umur model?),
    <strong>RQ3</strong> (cadence retraining berapa?) &mdash; ketiganya dipra-registrasi sebelum
    satu model pun berjalan, dan tak satu pun bisa diubah sekarang tanpa mendeklarasikan
    eksperimen baru.
  </p>
</div>

<div style="background: linear-gradient(135deg, #0b1021, #14213d); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #8ecae633;">
  <h2 style="color: #8ecae6; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🔧 0 &middot; Persiapan
  </h2>
  <p style="color: #a8c7d8; margin: 0; font-size: 1.02em;">Temukan artefak imutabel lewat glob, tidak pernah lewat slug Dataset. Pasang hanya yang belum ada di image Kaggle.</p>
  <ul style="color: #a8c7d8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Kaggle membawa <code>torch</code> dan <code>pyarrow</code>-nya sendiri; mem-pin keduanya terhadap venv lokal terlarang. <code>/kaggle/input</code> hanya-baca, semua tulisan jatuh ke <code>/kaggle/working</code>.</li>
    <li style="margin-bottom: 6px;">Kandidat parquet dicocokkan lewat <strong>isi</strong>, bukan nama: <code>PAR1</code> di kedua ujung, diperiksa di titik pemilihan (<code>D72</code>). Pencariannya turun ke kedalaman berapa pun lewat <code>rglob</code>, karena path yang UI Kaggle berikan tiga tingkat dalam — satu tingkat lebih dalam dari pola terdalam yang lama (<code>D71</code>).</li>
    <li style="margin-bottom: 6px;">Parquet <strong>tidak</strong> diunduh ulang di sini meski Stage 1 sanggup: unduhan baru adalah vintage baru, dan root &sect;12 melarang angka dari dua vintage berbagi satu tabel.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 4px solid #f48c06; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffba08; margin: 0 0 6px 0; font-size: 1.22em;">🗺️ Peta artefak — sel mana memproduksi apa</h3>
  <p style="color: #ffd08a; margin: 0; font-size: 0.94em;">Notebook ini punya 350-an sel dan hanya belasan di antaranya menghasilkan sesuatu. Sel ini mencetak indeksnya: nomor sel, apa yang ditulis, apa yang dibaca — sehingga <em>di mana figure dibuat</em> dan <em>di mana panel metrik ditulis</em> terjawab sebelum satu sel pun dibuka. Indeksnya dibangun dari metadata sel yang benar-benar dipancarkan, jadi ia tidak bisa menyebut sel yang tidak ada (<code>D87</code>).</p>
</div>

In [1]:
# PETA ARTEFAK — sel mana memproduksi apa (`D87`).
#
# Dibangun dari metadata sel yang benar-benar dipancarkan generator,
# bukan dari daftar yang ditulis tangan, jadi ia tidak bisa mengklaim
# sel yang tidak ada. Nomor sel nol-indeks, sama seperti yang dipakai
# Jupyter di sisi kiri.
_ARTIFACT_MAP = [
    (3, 'artifact_map', [], []),
    (5, 'setup', [], ['data/raw/BTCUSDT_1h.parquet']),
    (29, 'provenance_sources', [], []),
    (59, 'data', [], ['data/raw/BTCUSDT_1h.parquet']),
    (69, 'features', [], []),
    (115, 'keff', ['artifacts/keff_table.parquet'], []),
    (340, 'module_names', [], []),
    (342, 'code_digest', [], []),
    (345, 'invariants', [], []),
    (348, 'pilot', ['artifacts/preds/*.parquet', 'artifacts/meta/*.json'], []),
    (350, 'tune', ['artifacts/meta/tuning_selection.json'], []),
    (353, 'grid', ['artifacts/preds/*.parquet', 'artifacts/meta/*.json', 'artifacts/attn/*.parquet'], ['data/raw/BTCUSDT_1h.parquet']),
    (356, 'rq1', [], ['artifacts/preds/*.parquet', 'artifacts/meta/*.json']),
    (358, 'rq2', [], ['artifacts/preds/*.parquet', 'artifacts/meta/*.json']),
    (360, 'rq3', [], ['artifacts/preds/*.parquet', 'artifacts/meta/*.json']),
    (363, 'save', ['artifacts/paper_numbers.json', 'artifacts/run_block_metrics.parquet', 'artifacts/seed_averaged_cells.parquet', 'artifacts/amplification_panel.parquet', 'artifacts/decay_panel.parquet'], ['artifacts/preds/*.parquet', 'artifacts/meta/*.json']),
    (366, 'report', ['paper/paper_numbers.json', 'paper/tables/*.tex', 'paper/figures/*.pdf', 'paper/figures/*.png', 'paper/panels/*.parquet'], ['artifacts/paper_numbers.json', 'artifacts/preds/*.parquet', 'artifacts/attn/*.parquet']),
    (369, 'sync_back', [], []),
]

_produsen = [r for r in _ARTIFACT_MAP if r[2]]
print("PETA ARTEFAK — " + str(len(_produsen)) + " sel produsen dari "
      + str(len(_ARTIFACT_MAP)) + " sel orkestrasi")
print("=" * 78)
for _i, _slug, _w, _r in _ARTIFACT_MAP:
    if not _w:
        continue
    print()
    print("  sel " + str(_i).rjust(3) + "  [" + _slug + "]")
    for _p in _w:
        print("        menulis  " + _p)
    for _p in _r:
        print("        membaca  " + _p)

print()
print("=" * 78)
print("Sel yang hanya membaca dan mencetak, tanpa menulis artefak:")
print("  " + ", ".join(r[1] for r in _ARTIFACT_MAP if not r[2]))


PETA ARTEFAK — 6 sel produsen dari 18 sel orkestrasi

  sel 115  [keff]
        menulis  artifacts/keff_table.parquet

  sel 348  [pilot]
        menulis  artifacts/preds/*.parquet
        menulis  artifacts/meta/*.json

  sel 350  [tune]
        menulis  artifacts/meta/tuning_selection.json

  sel 353  [grid]
        menulis  artifacts/preds/*.parquet
        menulis  artifacts/meta/*.json
        menulis  artifacts/attn/*.parquet
        membaca  data/raw/BTCUSDT_1h.parquet

  sel 363  [save]
        menulis  artifacts/paper_numbers.json
        menulis  artifacts/run_block_metrics.parquet
        menulis  artifacts/seed_averaged_cells.parquet
        menulis  artifacts/amplification_panel.parquet
        menulis  artifacts/decay_panel.parquet
        membaca  artifacts/preds/*.parquet
        membaca  artifacts/meta/*.json

  sel 366  [report]
        menulis  paper/paper_numbers.json
        menulis  paper/tables/*.tex
        menulis  paper/figures/*.pdf
        menulis  paper/fig

<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">🧰 Siapkan sesi</h3>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.94em;">Cap <code>SESSION_T0</code>, laporkan perangkat yang terlihat, dan temukan parquet input di kedalaman berapa pun lalu <strong>periksa ia benar-benar parquet</strong> — mencocokkan nama tidak memverifikasi apa pun (<code>D71</code>, <code>D72</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>data/raw/BTCUSDT_1h.parquet</code></span></div>
</div>

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

# Kaggle's 12 h wall runs from HERE, not from the moment the grid starts. The
# budget guard counts from whatever it is handed, so the prelude — data, K_eff,
# invariants, the twelve pilot runs — would sit outside the budget entirely and
# the two clocks would drift apart by however long it took. The grid cell
# subtracts this. Losing /kaggle/working to the wall costs the whole session's
# runs, so the margin is not somewhere to be approximate.
SESSION_T0 = time.perf_counter()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()).resolve()
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Relative paths inside the definitions below resolve against the process working
# directory, so artifacts have to land beside it whichever machine this is.
# Kaggle already starts in /kaggle/working; a local run started from notebooks/
# does not. Nothing is added to sys.path — there is no package to import, and an
# entry there could only serve to shadow these cells with someone else's copy.
if Path.cwd().resolve() != WORK:
    os.chdir(WORK)


def ensure(module: str, pip_name: str | None = None) -> None:
    """Install only what is genuinely missing.

    Kaggle ships torch, numpy, pyarrow and usually polars. Pinning any of them
    against a local venv is forbidden by root section 16 — the image wins.
    """
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or module]
        )


# torch and the data plane run the grid; the rest are the reporting pass, which
# root section 16 confines to one named stats boundary. Kaggle ships scipy,
# statsmodels and matplotlib and usually not `arch`, so this list is the whole
# difference between a session that renders Table 2 and one that raises on it.
for _mod in ("polars", "pyarrow", "numpy", "torch",
             "scipy", "statsmodels", "arch", "matplotlib"):
    ensure(_mod)


def looks_like_parquet(path: Path) -> bool:
    """True only for a file that begins and ends with the parquet magic.

    Four bytes at each end. A parquet file opens with ``PAR1`` and closes with
    ``PAR1`` after its footer, so this catches a truncated upload, a Git LFS
    pointer, an HTML error page saved under the right name, and anything else
    that merely occupies the path.

    It exists because the alternative is what happened: something that was not a
    parquet was accepted here, and the failure surfaced three cells later as
    ``ComputeError: File out of specification`` from inside polars, naming
    neither the file nor the reason it was chosen. A check at the point of
    selection can say both.
    """
    try:
        with path.open("rb") as handle:
            if handle.read(4) != b"PAR1":
                return False
            handle.seek(-4, 2)
            return handle.read(4) == b"PAR1"
    except OSError:
        return False


def find_parquet() -> Path:
    """Locate BTCUSDT_1h.parquet by globbing — never by Kaggle dataset slug.

    Root section 10.5: discovery is by glob so the Dataset can be renamed without
    editing anything. The shallow patterns come first because they express a
    *preference* -- data/raw/ over a flat upload -- and because they are cheap.

    The recursive pass exists because a fixed depth is a hard-coded assumption
    wearing a glob. Kaggle mounts a dataset at /kaggle/input/<slug>/, but the
    path a user copies out of the web UI can carry the owner and the datasets/
    segment too, and a repository uploaded whole nests data/raw/ one level
    deeper again. Any of those is three or four levels down, and the previous
    patterns stopped at two -- so the file was there and discovery said it was
    not. Depth is not a thing to enumerate.
    """
    patterns = (
        "data/raw/BTCUSDT_1h.parquet",
        "*/data/raw/BTCUSDT_1h.parquet",
        "BTCUSDT_1h.parquet",
        "*/BTCUSDT_1h.parquet",
        "*/*/BTCUSDT_1h.parquet",
    )
    roots = [WORK, Path("/kaggle/input")] if ON_KAGGLE else [WORK, WORK.parent]
    rejected: list[str] = []

    def accept(candidate: Path):
        """The candidate, or None once it has failed the magic check."""
        if candidate.is_file() and looks_like_parquet(candidate):
            return candidate.resolve()
        rejected.append(str(candidate))
        return None

    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for hit in sorted(root.glob(pattern)):
                if (found := accept(hit)) is not None:
                    return found
    # Depth-independent fallback. Preferring data/raw/ still, then anything.
    for root in roots:
        if not root.exists():
            continue
        hits = [h for h in sorted(root.rglob("BTCUSDT_1h.parquet")) if h.is_file()]
        valid = [h for h in hits if looks_like_parquet(h)]
        rejected += [str(h) for h in hits if h not in valid]
        if valid:
            preferred = [h for h in valid if h.parent.name == "raw"]
            chosen = (preferred or valid)[0]
            if len(valid) > 1:
                print(f"note: {len(valid)} copies of BTCUSDT_1h.parquet under "
                      f"{root}; using {chosen}. Section 12 forbids two vintages "
                      f"in one table -- check they are the same file.")
            return chosen.resolve()

    if rejected:
        raise FileNotFoundError(
            "found candidates but none is a parquet file -- each failed the "
            f"PAR1 magic check at one or both ends: {rejected}. A truncated "
            "upload, a Git LFS pointer, or the wrong file under the right name."
        )
    raise FileNotFoundError(
        "BTCUSDT_1h.parquet not found under "
        f"{[str(r) for r in roots]}. Attach data/raw/ as a Kaggle Dataset. "
        "It is NOT re-downloaded here on purpose: a fresh download is a new "
        "vintage, and section 12 forbids numbers from two vintages sharing a table."
    )


PARQUET = find_parquet()
# Every meta/*.json records the digest of the artifact its run consumed (section
# 12), and can only find it if told.
os.environ["ITBTC_PARQUET"] = str(PARQUET)

import numpy as np
import polars as pl
import torch

print(f"work      {WORK}")
print(f"parquet   {PARQUET}  ({PARQUET.stat().st_size / 1e6:.1f} MB)")
print(f"artifacts {ARTIFACTS}")
print(f"polars {pl.__version__} | torch {torch.__version__} | numpy {np.__version__}")
print(f"CUDA devices: {torch.cuda.device_count()}")
for _i in range(torch.cuda.device_count()):
    _cap = torch.cuda.get_device_capability(_i)
    print(f"  cuda:{_i}  {torch.cuda.get_device_name(_i)}  sm_{_cap[0]}{_cap[1]}")

# Root section 10.3: never gate precision on torch.cuda.is_bf16_supported(). It
# defaults to including_emulation=True and returns True on a T4 (sm_75),
# selecting an emulated bf16 path *slower than fp32*. Gate on capability.
if torch.cuda.is_available():
    print(f"native bf16: {torch.cuda.get_device_capability(0)[0] >= 8}  "
          f"(is_bf16_supported() says {torch.cuda.is_bf16_supported()} "
          f"and is not to be trusted here)")


<div style="background: linear-gradient(135deg, #101010, #1c1c1c); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #9e9e9e33;">
  <h2 style="color: #d0d0d0; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📚 1 &middot; Pustaka
  </h2>
  <p style="color: #a8a8a8; margin: 0; font-size: 1.02em;">Setiap impor yang paket ini pakai, dimuat sekali di satu tempat.</p>
  <ul style="color: #a8a8a8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Sel-sel definisi di bawah <strong>tidak membawa satu impor pun</strong> — termasuk <code>from __future__ import annotations</code>, karena IPython mengakumulasi flag <code>__future__</code> lintas sel dan direktif di sini berlaku untuk seluruh sesi (<code>D67</code>).</li>
    <li style="margin-bottom: 6px;">Daftarnya <strong>digenerate dari modul-modulnya sendiri</strong>, jadi dependensi yang muncul di paket tidak bisa hilang di sini.</li>
    <li style="margin-bottom: 6px;">Satu-satunya sel lain yang mengimpor adalah Persiapan di atas, dan itu tak terhindarkan: ia yang <em>memasang</em> paket yang sel ini impor.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 4px solid #9e9e9e; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #d0d0d0; margin: 0 0 6px 0; font-size: 1.22em;">📚 Library — satu-satunya sel yang mengimpor</h3>
  <p style="color: #a8a8a8; margin: 0; font-size: 0.94em;">Setiap impor yang paket ini pakai, dimuat sekali. Sel definisi di bawahnya tidak membawa satu impor pun, termasuk <code>from __future__ import annotations</code>: IPython mengakumulasi flag <code>__future__</code> lintas sel (<code>D66</code>, <code>D67</code>).</p>
</div>

In [ ]:
from __future__ import annotations
import argparse
import gc
import hashlib
import json
import math
import numpy as np
import os
import polars as pl
import random
import re
import subprocess
import sys
import threading
import time
import torch
import warnings
from collections import OrderedDict
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timedelta, timezone
from pathlib import Path
from torch import Tensor, nn
from typing import Final, Literal, Protocol


<div style="background: linear-gradient(135deg, #0b1021, #14213d); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #8ecae633;">
  <h2 style="color: #8ecae6; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    ⚙️ 2 &middot; Konfigurasi & permukaan paket
  </h2>
  <p style="color: #a8c7d8; margin: 0; font-size: 1.02em;">Setiap angka protokol yang root &sect;8.1 kunci, sebagai konstanta bernama — lalu daftar nama yang paket ini ekspor.</p>
  <ul style="color: #a8c7d8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Definisi, bukan berkas.</strong> Sel-sel di bawah dieksekusi, tidak ditulis ke disk: dekorator berjalan, tipe field dataclass diselesaikan, konstanta modul dievaluasi. Sel yang menyebut sesuatu yang baru didefinisikan belakangan gagal seketika, bukan saat dipanggil.</li>
    <li style="margin-bottom: 6px;"><strong>Dua suntingan, dan hanya dua.</strong> Impor intra-paket dibuang — namanya sudah hidup di namespace ini, dan impornya akan gagal karena tak ada paketnya. Dan guard <code>if __name__ == "__main__":</code> milik <code>runner</code> dibuang: di dalam sel, <code>__name__</code> <em>adalah</em> <code>"__main__"</code>, jadi ia akan meluncurkan seluruh grid begitu sel definisinya jalan. Sisanya adalah paket itu sendiri, karakter demi karakter.</li>
    <li style="margin-bottom: 6px;"><em>Save Version &rarr; Save &amp; Run All</em> menjalankannya berurutan, dan hanya urutan itu yang bekerja.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 4px solid #8ecae6; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #8ecae6; margin: 0 0 6px 0; font-size: 1.22em;">📘 config.py</h3>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.94em;">Setiap angka protokol yang root §8.1 kunci, sebagai konstanta bernama. Tidak ada magic number di modul lain.</p>
</div>

<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">Docstring modul. Impornya hidup di sel Library di atas.</p>
</div>

In [ ]:
"""Design constants and the walk-forward origin grid.

Every number here is fixed by ``CLAUDE.md`` and none of it may be tuned. The
module exists so that no magic number is buried in pipeline code (root §16) and
so that a change to the design is a one-line diff with a visible blast radius.

The origin grid is *derived*, never transcribed: transcribing it is how the
13-origin figure survived in four documents after `D26` replaced it.
"""


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📐 Kontrak data & jendela</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">Angka terukur dari laporan Stage 1, plus L=96 dan H=24. Root §4.1 dan §6.2.</p>
</div>

In [ ]:

# -- data window (root §4.1) -------------------------------------------------

DATA_START: Final = datetime(2018, 1, 1, tzinfo=timezone.utc)
DATA_END: Final = datetime(2026, 8, 1, tzinfo=timezone.utc)  # EXCLUSIVE

BARS_EXPECTED: Final = 75_216
BARS_ACTUAL: Final = 75_094
MISSING_BARS: Final = 122
GAP_BLOCKS: Final = 27

# -- model geometry (root §6.2) ----------------------------------------------

SEQ_LEN: Final = 96   # L — 4 days of lookback
PRED_LEN: Final = 24  # H — headline horizon

#: A window occupies ``L + H`` consecutive bars, so a break inside a span
#: destroys ``L + H - 1`` *start positions*, not ``L + H``. Root §4.3 turns on
#: this off-by-one: across 30 breaks it is a 30-window difference in the
#: assertion target, the same size as the drift the assertion exists to catch.
WINDOW_SPAN: Final = SEQ_LEN + PRED_LEN         # 120
STARTS_LOST_PER_BREAK: Final = WINDOW_SPAN - 1  # 119


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🗓️ Protokol walk-forward</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">24 bulan latih, 21/3 latih-validasi, enam blok 30 hari, spasi lima bulan (<code>D26</code>).</p>
</div>

In [ ]:

# -- walk-forward protocol (root §8.1) ---------------------------------------

TRAIN_MONTHS: Final = 24      # fixed rolling window, never expanding
VAL_MONTHS: Final = 3         # final 3 months of the training window
TRAIN_SUB_MONTHS: Final = TRAIN_MONTHS - VAL_MONTHS  # 21 — where the scaler is fit
TEST_BLOCKS: Final = 6
BLOCK_DAYS: Final = 30
BLOCK_HOURS: Final = BLOCK_DAYS * 24  # 720 window starts per block

#: 5, not 6. The calendar month a block lands on is ``m0 + s*i + (b-1) mod 12``,
#: so for fixed ``b`` the months visited form a coset of size ``12/gcd(s,12)``.
#: At s=6 that is 2 months, and a significant beta1 becomes observationally
#: equivalent to "February and August are harder" — a bias no post-hoc analysis
#: removes. Only s coprime to 12 fully decouples; 5 maximises the origin count
#: among those. Root §8.1 / `D26`.
ORIGIN_SPACING_MONTHS: Final = 5

FIRST_ORIGIN: Final = datetime(2020, 1, 1, tzinfo=timezone.utc)

# -- variate ladder (root §5.2) ----------------------------------------------

K_LADDER: Final = (1, 4, 8, 12)
SEEDS: Final = (42, 43, 44, 45, 46)

#: Horizons swept in root §10.2's 192-run arm. H=24 is the headline.
HORIZONS: Final = (1, 3, 24, 168)

#: Origins the horizon sweep runs at, **named in advance** (`D48`). Choosing
#: them after the main grid would be origin selection.
SWEEP_ORIGIN_INDICES: Final = (1, 5, 10, 15)

#: Offset of the falsification arm's fresh model (root §8.1). Exactly 90 days,
#: not 3 calendar months: test blocks are 30 **days**, so only 90 days lands the
#: fresh origin precisely on the aged model's block-4 boundary, which is what
#: makes "the *same* calendar blocks 4-6" true rather than approximately true.
FRESH_OFFSET_DAYS: Final = 90


def add_months(when: datetime, months: int) -> datetime:
    """Shift ``when`` by whole calendar months, keeping the day of month.

    Every boundary in this study falls on the first of a month, so the
    day-clamping question a general implementation must answer never arises.
    This raises rather than clamping if it ever does: a silently clamped
    boundary moves a split by a day and no assertion downstream would notice.

    Args:
        when: A timezone-aware datetime on the first of some month.
        months: Whole months to add; may be negative.

    Returns:
        The shifted datetime, same tzinfo and time of day.

    Raises:
        ValueError: If ``when`` is not on the first of a month.
    """
    if when.day != 1:
        raise ValueError(
            f"add_months is only used on month boundaries in this study; got "
            f"day={when.day}. Clamping rules would silently move a split."
        )
    total = when.month - 1 + months
    return when.replace(year=when.year + total // 12, month=total % 12 + 1)


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📍 Origin</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">Satu titik latih-ulang: batas latih, validasi, uji, seluruhnya epoch-based UTC.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class Origin:
    """One walk-forward origin and every boundary derived from it.

    All boundaries are half-open ``[start, end)``. The origin is both the end of
    validation and the start of testing: a forecaster standing at ``o`` has seen
    everything before ``o`` and nothing after it.
    """

    index: int
    origin: datetime

    @property
    def train_start(self) -> datetime:
        """Start of the 24-month rolling training window."""
        return add_months(self.origin, -TRAIN_MONTHS)

    @property
    def train_sub_end(self) -> datetime:
        """End of the 21-month sub-block; equivalently ``val_start``.

        The scaler is fit on this sub-block and nothing else, and training
        windows are enumerated over it and nothing else (`D25`). The 24-month
        count — ~17,400 windows, ~80 MB — is what you get by training on the
        validation months too, which is `D24`'s leak wearing a sample-count
        disguise.
        """
        return add_months(self.origin, -VAL_MONTHS)

    @property
    def val_start(self) -> datetime:
        return self.train_sub_end

    @property
    def val_end(self) -> datetime:
        return self.origin

    @property
    def test_start(self) -> datetime:
        return self.origin

    @property
    def test_end(self) -> datetime:
        return self.origin + timedelta(days=BLOCK_DAYS * TEST_BLOCKS)

    def block(self, b: int) -> tuple[datetime, datetime]:
        """Half-open bounds of test block ``b``, one-indexed as in the paper."""
        if not 1 <= b <= TEST_BLOCKS:
            raise ValueError(f"block index must be in 1..{TEST_BLOCKS}, got {b}")
        start = self.origin + timedelta(days=BLOCK_DAYS * (b - 1))
        return start, start + timedelta(days=BLOCK_DAYS)

    def blocks(self) -> list[tuple[int, datetime, datetime]]:
        """Every test block as ``(label, start, end)``, label one-indexed.

        The label is carried rather than inferred from position because the
        falsification arm evaluates blocks 4-6 and nothing else: there, the
        first tensor in the tuple is block **4**, and writing it out as block 1
        would silently re-index the arm the comparison depends on.
        """
        return [(b, *self.block(b)) for b in range(1, TEST_BLOCKS + 1)]

    @property
    def label(self) -> str:
        """``YYYY-MM`` — the form used in every table and figure."""
        return self.origin.strftime("%Y-%m")


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧪 Origin falsifikasi</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">Model fresh di <code>o + 90 hari</code>, dievaluasi pada blok kalender yang sama (root §8.1).</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class FalsificationOrigin:
    """A model trained fresh at ``o_i + 90 days``, scored on blocks 4-6.

    Root §8.1's pre-registered falsification arm, and **the only design in the
    study that identifies decay directly**. If the aged-minus-fresh gap is zero
    while beta1 < 0, then beta1 is calendar, not age — the aged model is not
    decaying, the market simply got harder in months 4-6, and RQ2's headline
    would be an artefact.

    Every training boundary is the base origin's, shifted by the same 90 days,
    so the fresh model trains on a window of **identical duration** to the aged
    one. Re-deriving the window from a 24-month subtraction instead would land
    on 2020-03-31-style dates, where the day-of-month clamping question
    :func:`add_months` refuses to answer silently would arise for the first time
    in this study — and a clamped boundary moves a split by a day with no
    assertion downstream to notice.

    Its validation sub-block overlaps the aged model's test blocks 1-3. That is
    not a leak: this is a *different* model, standing at a later origin, and a
    forecaster there has legitimately seen everything before ``o_i + 90 days``.
    """

    base: Origin
    offset_days: int = FRESH_OFFSET_DAYS

    @property
    def index(self) -> int:
        return self.base.index

    @property
    def _shift(self) -> timedelta:
        return timedelta(days=self.offset_days)

    @property
    def origin(self) -> datetime:
        return self.base.origin + self._shift

    @property
    def train_start(self) -> datetime:
        return self.base.train_start + self._shift

    @property
    def train_sub_end(self) -> datetime:
        return self.base.train_sub_end + self._shift

    @property
    def val_start(self) -> datetime:
        return self.train_sub_end

    @property
    def val_end(self) -> datetime:
        return self.origin

    @property
    def test_start(self) -> datetime:
        return self.origin

    @property
    def test_end(self) -> datetime:
        return self.base.test_end

    def blocks(self) -> list[tuple[int, datetime, datetime]]:
        """The **base** origin's blocks 4-6, keeping their original labels.

        ``o_i + 90 days`` is exactly where base block 4 opens, so these are the
        same calendar hours the aged model was scored on — which is the entire
        content of the comparison.
        """
        return [(b, *self.base.block(b)) for b in (4, 5, 6)]

    @property
    def label(self) -> str:
        return f"{self.base.label}+{self.offset_days}d"


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧭 Grid lima belas origin</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">2020-01 hingga 2025-11. Spasi lima bulan koprima terhadap 12, jadi indeks blok tidak kolinear dengan bulan kalender.</p>
</div>

In [ ]:


#: Anything :func:`itransformer_btc.splits.build_origin_tensors` accepts. The
#: two share an interface rather than an inheritance chain because they share no
#: implementation: one derives its boundaries from calendar months, the other by
#: shifting another origin's.
OriginLike = Origin | FalsificationOrigin


def origin_grid(
    first: datetime = FIRST_ORIGIN,
    spacing_months: int = ORIGIN_SPACING_MONTHS,
    data_start: datetime = DATA_START,
    data_end: datetime = DATA_END,
) -> list[Origin]:
    """Derive every origin that fits inside the data window.

    An origin is admissible when its training window starts no earlier than the
    data and its sixth test block ends no later than the data:
    ``o - 24 months >= data_start`` and ``o + 180 days <= data_end``.

    Under the committed constants this yields **15** origins, 2020-01 … 2025-11.
    The count is derived rather than written down so that changing the spacing
    changes the grid instead of leaving a stale integer behind — which is
    exactly what happened to the superseded 13.

    Returns:
        Origins in chronological order, ``index`` one-based.

    Raises:
        ValueError: If the first origin would need data from before the window.
    """
    grid: list[Origin] = []
    candidate = first
    while True:
        origin = Origin(index=len(grid) + 1, origin=candidate)
        if origin.train_start < data_start:
            raise ValueError(
                f"origin {origin.label} needs training data from "
                f"{origin.train_start.date()}, before the data window opens at "
                f"{data_start.date()}"
            )
        if origin.test_end > data_end:
            break
        grid.append(origin)
        candidate = add_months(candidate, spacing_months)
    return grid


#: Materialised once. Import this rather than rebuilding it — a second call with
#: different arguments silently produces a different study.
ORIGINS: Final = origin_grid()


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📚 Provenance source code</h4>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.9em;">Enam belas algoritma, masing-masing dengan repo resmi, lisensi, tanggal akses, dan <strong>daftar penyimpangannya</strong>. Author-year tidak menjawab pertanyaan <em>kodenya dari mana</em> (<code>D16</code>).</p>
</div>

In [ ]:


# -- provenance of every algorithm this study runs (root §12, §13.3) ----------


@dataclass(frozen=True, slots=True)
class Upstream:
    """Where one algorithm in this repository came from, and what was changed.

    Root §1's deliverable is a manuscript, and a manuscript that runs a named
    architecture is asked where its code came from. Author-year alone does not
    answer that: `D16` shows this project already carries mis-dated references
    assembled from memory, and an examiner comparing a cell against an upstream
    release needs the *repository*, the *licence* and the list of deliberate
    departures, not a surname.

    The convention is the one the LTSF literature uses when it reports a
    baseline it did not write: cite the paper by venue, footnote the official
    code URL with an access date, and state in one clause whether the code was
    **used**, **adapted** or **reimplemented**. That third distinction is the
    load-bearing one here, because nothing in ``src/`` is a copy of an upstream
    file --- every model is written from the published description against this
    study's own tensor contract, which is a stronger claim than a fork and a
    checkable one.

    Attributes:
        component: The names in this package the row accounts for.
        module: The ``src/itransformer_btc`` file they live in. The binding test
            reads this, so a row can never describe a module it is not in.
        status: ``reimplemented`` (written here from the published description),
            ``library`` (imported and called), or ``own`` (no upstream code
            exists --- the algorithm is defined in a paper and implemented here).
        reference: IEEE-style, so the row lifts straight into the bibliography.
        repo: Official implementation, empty when the algorithm has none.
        licence: Upstream licence. Empty when there is no upstream code.
        accessed: ISO date ``repo`` was last opened and confirmed.
        adapted: Every deliberate departure, with the divergence ID that forced
            it. Empty only where nothing was changed.
        verified: True when the repository page or an indexing service was read
            and the fields above were taken from it. False means root §13.3
            still owes this row a check --- recorded rather than assumed,
            because `D16` is what silence buys.
    """

    component: str
    module: str
    status: str
    reference: str
    repo: str = ""
    licence: str = ""
    accessed: str = ""
    adapted: str = ""
    verified: bool = False


#: Read as a table by the notebook's provenance cell and asserted against the
#: module docstrings by ``tests/test_provenance.py`` --- two copies that must
#: agree, with something checking that they do (`D54a`, `D69`).
SOURCE_PROVENANCE: Final[tuple[Upstream, ...]] = (
    Upstream(
        component="ITransformer, InvertedEmbedding, VariateAttention, EncoderLayer",
        module="model.py",
        status="reimplemented",
        reference=(
            "Y. Liu, T. Hu, H. Zhang, H. Wu, S. Wang, L. Ma, and M. Long, "
            '"iTransformer: Inverted transformers are effective for time series '
            'forecasting," in Proc. 12th Int. Conf. Learn. Represent. (ICLR), '
            "2024. arXiv:2310.06625."
        ),
        repo="https://github.com/thuml/iTransformer",
        licence="MIT",
        accessed="2026-09-03",
        adapted=(
            "d_model 512 -> 128, because attention here runs over N <= 12 variate "
            "tokens rather than L timesteps and 512 over-parameterises ~14k "
            "training windows (`D25`); loss on the target channel only, where the "
            "reference defaults to all channels, which would make K itself vary "
            "the number of supervised tasks (`D39`); lr halved every 4 epochs "
            "rather than every epoch (`D47`); a runtime `capture` attribute for "
            "the attention maps, deliberately not a config field so a captured "
            "run stays bit-identical (`D62d`). Every other hyperparameter is "
            "adopted unchanged and never tuned (`D38`)."
        ),
        verified=True,
    ),
    Upstream(
        component="DLinear, SeriesDecomposition",
        module="baselines.py",
        status="reimplemented",
        reference=(
            "A. Zeng, M. Chen, L. Zhang, and Q. Xu, "
            '"Are transformers effective for time series forecasting?," in Proc. '
            "37th AAAI Conf. Artif. Intell., 2023, pp. 11121-11128. "
            "arXiv:2205.13504."
        ),
        repo="https://github.com/cure-lab/LTSF-Linear",
        licence="Apache-2.0",
        accessed="2026-09-03",
        adapted=(
            "The published all-channel objective and channel-shared weights are "
            "kept deliberately: trained on the target channel alone this model "
            "would be K=1 wearing a K=8 label (`D40`, `D56`). Its centred moving "
            "average is retained as published and is confined to the 96-bar "
            "lookback, so root §8.3's no-embargo argument is untouched (`D56`)."
        ),
        verified=True,
    ),
    Upstream(
        component="PatchTST",
        module="baselines.py",
        status="reimplemented",
        reference=(
            "Y. Nie, N. H. Nguyen, P. Sinthong, and J. Kalagnanam, "
            '"A time series is worth 64 words: Long-term forecasting with '
            'transformers," in Proc. 11th Int. Conf. Learn. Represent. (ICLR), '
            "2023. arXiv:2211.14730."
        ),
        repo="https://github.com/yuqinie98/PatchTST",
        licence="Apache-2.0",
        accessed="2026-09-03",
        adapted=(
            "Reuses this study's own EncoderLayer with iTransformer's d_model, "
            "d_ff, e_layers, n_heads and dropout, so the two models differ in "
            "what a token is and in nothing else --- the cleanest form of the "
            "contrast, and it extends `D38`'s no-tuning posture to the baselines "
            "instead of quietly exempting them. Patch 16 / stride 8 as published."
        ),
        verified=True,
    ),
    Upstream(
        component="use_norm instance normalisation (ITransformer, PatchTST)",
        module="model.py",
        status="reimplemented",
        reference=(
            "T. Kim, J. Kim, Y. Tae, C. Park, J.-H. Choi, and J. Choo, "
            '"Reversible instance normalization for accurate time-series '
            'forecasting against distribution shift," in Proc. 10th Int. Conf. '
            "Learn. Represent. (ICLR), 2022."
        ),
        repo="https://github.com/ts-kim/RevIN",
        licence="MIT",
        accessed="2026-09-03",
        adapted=(
            "Normalisation and denormalisation only; the learnable affine "
            "transform is not used, matching iTransformer's own use_norm. Held "
            "True at every rung as a fixed property of the design rather than a "
            "tuning knob (root §6.2), which is what makes the outer "
            "StandardScaler algebraically inert (root §6.3, `D03`). The upstream "
            "README dates the paper ICLR 2021; dblp records conf/iclr/KimKTPCC22, "
            "so 2022 is used here."
        ),
        verified=True,
    ),
    Upstream(
        component="LSTMForecaster",
        module="baselines.py",
        status="library",
        reference=(
            "S. Hochreiter and J. Schmidhuber, "
            '"Long short-term memory," Neural Computation, vol. 9, no. 8, '
            "pp. 1735-1780, 1997."
        ),
        repo="https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html",
        licence="BSD-3-Clause (PyTorch)",
        accessed="2026-09-03",
        adapted=(
            "torch.nn.LSTM is called directly; only the forecasting head around "
            "it is written here. Genuinely multivariate with a target-channel "
            "loss, so unlike DLinear and PatchTST its K=8 means what the ladder's "
            "K means and its best_val_mse is comparable to theirs (`D64`)."
        ),
        verified=False,
    ),
    Upstream(
        component="RidgeForecaster",
        module="baselines.py",
        status="own",
        reference=(
            "A. E. Hoerl and R. W. Kennard, "
            '"Ridge regression: Biased estimation for nonorthogonal problems," '
            "Technometrics, vol. 12, no. 1, pp. 55-67, 1970."
        ),
        adapted=(
            "NOT scikit-learn. The normal equations are solved once in float64 "
            "and the Gram matrix reused across the alpha grid, because at "
            "L*K = 1152 a float32 Gram makes the smallest alphas report noise. "
            "The intercept is fitted by centring and left unpenalised: shrinking "
            "it would shrink the forecast toward r = mu_g, the constant-drift "
            "model `D31` spent a section removing from Naive-RW. alpha is the "
            "only hyperparameter selected anywhere in this study."
        ),
        verified=False,
    ),
    Upstream(
        component="NaiveForecaster (naive-persist, seasonal-naive), Naive-RW",
        module="baselines.py",
        status="own",
        reference=(
            "R. J. Hyndman and G. Athanasopoulos, Forecasting: Principles and "
            "Practice, 3rd ed. Melbourne, Australia: OTexts, 2021."
        ),
        adapted=(
            "Closed forms; nothing is trained. Naive-RW is not the last return "
            "but y_raw = 0, mapped into scaler space as y_z = -mu_g/sigma_g so "
            "the EMH baseline does not quietly become a constant-drift model "
            "(`D31`), and it needs no run at all."
        ),
        verified=False,
    ),
    Upstream(
        component="walk-forward with purging (Origin, build_origin_tensors)",
        module="splits.py",
        status="own",
        reference=(
            "M. Lopez de Prado, Advances in Financial Machine Learning. "
            "Hoboken, NJ: Wiley, 2018, ch. 7; L. J. Tashman, "
            '"Out-of-sample tests of forecasting accuracy: An analysis and '
            'review," Int. J. Forecast., vol. 16, no. 4, pp. 437-450, 2000; '
            "C. Bergmeir and J. M. Benitez, "
            '"On the use of cross-validation for time series predictor '
            'evaluation," Information Sciences, vol. 191, pp. 192-213, 2012.'
        ),
        adapted=(
            "Purging adopted; embargo and CPCV deliberately NOT, with the "
            "written arguments in root §8.3 and §8.4 rather than a protocol "
            "element left silently absent (`D15`). The purge runs at BOTH "
            "boundaries, train/validation as well as train/test, which the "
            "source is not read as requiring and which is the one that governs "
            "model selection (`D24`). Origin spacing is 5 months, not 6, "
            "because only a spacing coprime to 12 decouples the block index "
            "from the calendar month (`D26`)."
        ),
        verified=False,
    ),
    Upstream(
        component="Scaler",
        module="splits.py",
        status="own",
        reference=(
            "No upstream publication: the z-score is arithmetic. The row exists "
            "because a reader is entitled to know this is NOT "
            "sklearn.preprocessing.StandardScaler."
        ),
        adapted=(
            "Fitted on the 21-month training sub-block only, at every origin. "
            "Under use_norm=True it cancels algebraically (root §6.3, `D03`), "
            "so what it controls is the reporting scale and the baselines with "
            "no internal normalisation. RobustScaler and MinMaxScaler are "
            "rejected on correctness rather than preference (root §6.3)."
        ),
        verified=False,
    ),
    Upstream(
        component="Adam optimiser and StepLR schedule (train_one)",
        module="train.py",
        status="library",
        reference=(
            "D. P. Kingma and J. Ba, "
            '"Adam: A method for stochastic optimization," in Proc. 3rd Int. '
            "Conf. Learn. Represent. (ICLR), 2015. arXiv:1412.6980."
        ),
        repo="https://docs.pytorch.org/docs/stable/optim.html",
        licence="BSD-3-Clause (PyTorch)",
        accessed="2026-09-03",
        adapted=(
            "torch.optim.Adam at lr = 1e-4, adopted unchanged (`D38`). StepLR "
            "halves every FOUR epochs, not every epoch: per-epoch halving "
            "reaches ~4e-7 by epoch 9, so the 30-epoch budget could never bind "
            "(`D47`). The loop itself takes nothing from a reference "
            "implementation --- there is no Dataset and no DataLoader, the "
            "split is GPU-resident and batching is index-slicing, which is what "
            "puts the grid inside the weekly quota at all (`D19`, `D57`)."
        ),
        verified=False,
    ),
    Upstream(
        component="dm_test, clark_west_test (HLN correction, rectangular LRV)",
        module="metrics.py",
        status="own",
        reference=(
            "F. X. Diebold and R. S. Mariano, "
            '"Comparing predictive accuracy," J. Bus. Econ. Statist., vol. 13, '
            "no. 3, pp. 253-263, 1995; D. Harvey, S. Leybourne, and P. Newbold, "
            '"Testing the equality of prediction mean squared errors," Int. J. '
            "Forecast., vol. 13, no. 2, pp. 281-291, 1997; T. E. Clark and "
            'K. D. West, "Approximately normal tests for equal predictive '
            'accuracy in nested models," J. Econometrics, vol. 138, no. 1, '
            "pp. 291-311, 2007."
        ),
        repo="https://pkg.robjhyndman.com/forecast/reference/dm.test.html",
        licence="GPL-3 (R forecast --- validation target, not a dependency)",
        accessed="2026-09-03",
        adapted=(
            "Written on numpy, not taken from a package. The long-run variance is "
            "the truncated rectangular estimator at lag h-1, NOT Newey-West "
            "Bartlett, which would shrink the lag-22 autocovariance by ~92% and "
            "manufacture optimistic p-values (`D34`) --- statsmodels' cov_hac is "
            "Bartlett by default, which is why it is not used here. Nested pairs "
            "take Clark-West, non-nested pairs DM with the HLN correction "
            "(`D29`)."
        ),
        verified=False,
    ),
    Upstream(
        component="wild cluster restricted bootstrap (WCR) for beta1",
        module="metrics.py",
        status="own",
        reference=(
            "A. C. Cameron, J. B. Gelbach, and D. L. Miller, "
            '"Bootstrap-based improvements for inference with clustered errors," '
            "Rev. Econ. Statist., vol. 90, no. 3, pp. 414-427, 2008; "
            "J. G. MacKinnon, M. O. Nielsen, and M. D. Webb, "
            '"Cluster-robust inference: A guide to empirical practice," '
            "J. Econometrics, vol. 232, no. 2, pp. 272-299, 2023."
        ),
        adapted=(
            "NOT the wildboottest package, which root §9.2 names as the reference "
            "implementation but which this package does not import. Restricted "
            "(the null is imposed when generating samples), bootstrapping the "
            "cluster-robust t rather than beta-hat, B = 99,999, Rademacher and "
            "Webb weights both reported, and p computed as (1 + count)/(1 + B) "
            "because the observed statistic belongs to its own reference "
            "distribution and mean(t* <= t_obs) returned a literal p = 0 "
            "(`D42`, `D53d`)."
        ),
        verified=False,
    ),
    Upstream(
        component="romano_wolf",
        module="comparisons.py",
        status="own",
        reference=(
            "J. P. Romano and M. Wolf, "
            '"Stepwise multiple testing as formalized data snooping," '
            "Econometrica, vol. 73, no. 4, pp. 1237-1282, 2005."
        ),
        adapted=(
            "Written here; it did not exist in this package until `D62a`. Applied "
            "to the all-pairs matrix, where White's Reality Check and Hansen's SPA "
            "do not apply because they test a one-against-many null (`D35`). "
            "Reported in two columns after `D79`: the all-pairs family, and the "
            "declared claim family."
        ),
        verified=False,
    ),
    Upstream(
        component="model_confidence_set, mcs_table",
        module="comparisons.py",
        status="own",
        reference=(
            "P. R. Hansen, A. Lunde, and J. M. Nason, "
            '"The model confidence set," Econometrica, vol. 79, no. 2, '
            "pp. 453-497, 2011."
        ),
        adapted=(
            "Written here (`D62a`). Reported at 90% and 75% as a membership "
            "column inside Table 6 rather than as a table of its own."
        ),
        verified=False,
    ),
    Upstream(
        component="deflated_sharpe",
        module="economics.py",
        status="own",
        reference=(
            "D. H. Bailey and M. Lopez de Prado, "
            '"The deflated Sharpe ratio: Correcting for selection bias, backtest '
            'overfitting, and non-normality," J. Portfolio Manage., vol. 40, '
            "no. 5, pp. 94-107, 2014."
        ),
        adapted=(
            "Computed per origin from that origin's non-overlapping 24-hour "
            "strategy returns and their per-period Sharpe, never the annualised "
            "one, which would inflate it by sqrt(periods per year). N is the "
            "number of configurations evaluated on that origin's own test span, "
            "not the full run total, because DSR counts candidates selected on "
            "the SAME return series; the full total is reported separately as the "
            "development trial count (`D46`)."
        ),
        verified=False,
    ),
    Upstream(
        component="variance_ratio and adf inside efficiency_table",
        module="efficiency.py",
        status="library",
        reference=(
            "A. W. Lo and A. C. MacKinlay, "
            '"Stock market prices do not follow random walks: Evidence from a '
            'simple specification test," Rev. Financial Stud., vol. 1, no. 1, '
            "pp. 41-66, 1988."
        ),
        repo="https://github.com/bashtage/arch",
        licence="NCSA (arch), BSD-3-Clause (statsmodels)",
        accessed="2026-09-03",
        adapted=(
            "arch.unitroot.VarianceRatio and statsmodels.tsa.stattools.adfuller "
            "are called directly. These two and scipy.stats are the ONLY "
            "third-party statistics anywhere in the package, and they sit at root "
            "§16's named boundary --- imported inside the function that needs "
            "them, never at module level."
        ),
        verified=False,
    ),
    Upstream(
        component="hurst_rs",
        module="efficiency.py",
        status="own",
        reference=(
            "H. E. Hurst, "
            '"Long-term storage capacity of reservoirs," Trans. Amer. Soc. Civil '
            "Eng., vol. 116, no. 1, pp. 770-799, 1951."
        ),
        adapted=(
            "Rescaled-range implementation written here; no package provides one "
            "under a licence and an API this project already depends on."
        ),
        verified=False,
    ),
    Upstream(
        component="participation_ratio, stable_rank, lookback_correlation_pr",
        module="keff.py",
        status="own",
        reference=(
            "L. Laloux, P. Cizeau, J.-P. Bouchaud, and M. Potters, "
            '"Noise dressing of financial correlation matrices," Phys. Rev. '
            "Lett., vol. 83, no. 7, pp. 1467-1470, 1999; V. Plerou, "
            "P. Gopikrishnan, B. Rosenow, L. A. N. Amaral, T. Guhr, and "
            'H. E. Stanley, "Random matrix approach to cross correlations in '
            'financial data," Phys. Rev. E, vol. 65, no. 6, 066126, 2002.'
        ),
        adapted=(
            "PR is taken on the CORRELATION matrix, never the covariance one: the "
            "covariance spectrum is not monotone in K and its ordering is a "
            "statement about units, since log_quote_volume's deviations sit two "
            "orders of magnitude above r's (`D53a`, `D53b`, `D81`). Measured per "
            "origin on that origin's own 21-month training sub-block, so RQ1's "
            "regressor never reads the test period (`D44`)."
        ),
        verified=False,
    ),
    Upstream(
        component="parkinson, garman_klass, rogers_satchell (family F2)",
        module="features.py",
        status="own",
        reference=(
            "M. Parkinson, "
            '"The extreme value method for estimating the variance of the rate '
            'of return," J. Business, vol. 53, no. 1, pp. 61-65, 1980; '
            'M. B. Garman and M. J. Klass, "On the estimation of security price '
            'volatilities from historical data," J. Business, vol. 53, no. 1, '
            "pp. 67-78, 1980; L. C. G. Rogers and S. E. Satchell, "
            '"Estimating variance from high, low and closing prices," Ann. Appl. '
            "Probab., vol. 1, no. 4, pp. 504-512, 1991."
        ),
        adapted=(
            "Per-bar, with NO trailing average --- which is what makes the "
            "center=True leakage class structurally unrepresentable and licenses "
            "root §8.3's no-embargo argument (`D13`, `D15`). Rogers-Satchell "
            "vanishes on the 33 shadowless bars in the sample, so it is taken as "
            "log(RS + 1e-9); Parkinson and Garman-Klass need no floor (`D52a`)."
        ),
        verified=False,
    ),
)


<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 4px solid #9e9e9e; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #d0d0d0; margin: 0 0 6px 0; font-size: 1.22em;">📘 __init__.py</h3>
  <p style="color: #a8a8a8; margin: 0; font-size: 0.94em;">Permukaan paket. Di notebook ia tidak mengimpor apa pun — seluruh nama sudah hidup di namespace kernel yang sama.</p>
</div>

<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 3px solid #9e9e9e88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #d0d0d0; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📦 Permukaan paket</h4>
  <p style="color: #a8a8a8; margin: 0; font-size: 0.9em;">Di checkout ia mengekspor nama; di notebook impornya dibuang dan <code>__all__</code> tinggal dokumentasi.</p>
</div>

In [ ]:
"""Data plane and experiment scaffolding for the spot-only iTransformer study.

Root ``CLAUDE.md`` is the project law, and it is the *only* one: the
subdirectory files were deleted on 2026-08-06 because a rule that loads solely
when a file in its subtree is opened is absent exactly when an agent reasons
about the area without opening one. Two of those rules shape every module here:

* **polars only.** Its rolling API is backward-closed by construction, so the
  ``center=True`` leak class is unrepresentable. Stage 1 ingest
  (``spot_klines_btc.py``, at the repository root) is the one documented
  exemption and lives outside this package.
* **Fail loudly.** A schema mismatch, a window-count mismatch or a hash mismatch
  raises. The anti-leakage checklist is ``assert``s wherever it can be, because
  a checklist that lives only in prose is a checklist nobody runs.
"""

__all__ = [
    "ORIGINS",
    "Origin",
    "PRED_LEN",
    "SEQ_LEN",
    "STARTS_LOST_PER_BREAK",
    "WINDOW_SPAN",
    "origin_grid",
]


<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 4px solid #8ecae6; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #8ecae6; margin: 0 0 6px 0; font-size: 1.22em;">📚 Provenance source code — dari mana tiap algoritma berasal</h3>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.94em;">Enam belas algoritma, tercetak sebagai <em>output</em> dan bukan hanya sebagai teks, jadi notebook yang tersimpan membawa jawabannya sebagai bukti (root &sect;15). Format sitasinya IEEE — paper, venue, repo resmi, lisensi, tanggal akses — supaya tiap baris bisa langsung diangkat ke bibliografi. <strong>Tidak satu pun berkas di sini salinan berkas upstream</strong>: yang <code>reimplemented</code> ditulis ulang dari deskripsi terbitannya, dan penyimpangannya didaftar dengan ID divergensi yang memaksanya.</p>
</div>

In [ ]:
# The answer to "where did this code come from", printed rather than implied.
#
# Author-year is what every docstring in this package carried before, and it does
# not answer the question an examiner actually asks. `D16` is the precedent: this
# project already carries mis-dated references assembled from memory, so a table
# that names the repository, the licence and the access date is the only form of
# the claim that can be checked. `tests/test_provenance.py` binds every URL here
# to the module docstring that repeats it, because two copies with nothing
# checking them is `D54a` and `D69` over again.
_STATUS_NOTE = {
    "reimplemented": "written here from the published description",
    "library": "imported and called",
    "own": "no upstream code exists; implemented here from the paper",
}

print(f"SOURCE PROVENANCE — {len(SOURCE_PROVENANCE)} algorithms")
print("=" * 78)
# ``_prov``, never ``_row``: in a flattened notebook every module shares one
# kernel namespace, and ``efficiency._row`` is a function three phases below.
# `D65` is that collision having already cost a twelve-hour session once.
for _prov in SOURCE_PROVENANCE:
    print()
    print(f"{_prov.component}")
    print(f"  in         {_prov.module}")
    print(f"  status     {_prov.status} ({_STATUS_NOTE[_prov.status]})")
    print(f"  reference  {_prov.reference}")
    if _prov.repo:
        _seen = "verified" if _prov.verified else "NOT yet verified (root 13.3)"
        print(f"  code       {_prov.repo}")
        print(f"             {_prov.licence}, accessed {_prov.accessed} — {_seen}")
    else:
        print("  code       no upstream implementation")
    if _prov.adapted:
        print(f"  adapted    {_prov.adapted}")

_by_status = {s: sum(1 for r in SOURCE_PROVENANCE if r.status == s)
              for s in _STATUS_NOTE}
print()
print("=" * 78)
print(f"{len(SOURCE_PROVENANCE)} rows: " + ", ".join(
    f"{n} {s}" for s, n in _by_status.items()))
print(f"repository pages read in session: "
      f"{sum(1 for r in SOURCE_PROVENANCE if r.verified)} — the rest are "
      f"paper-only citations root 13.3 still owes a check")


<div style="background: linear-gradient(135deg, #1a1200, #2b1d00); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #ffb70333;">
  <h2 style="color: #ffd60a; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📥 3 &middot; Muat data & hukum segmen
  </h2>
  <p style="color: #ffca7a; margin: 0; font-size: 1.02em;">Muat artefak imutabel, potong deret di tiap gap, lalu asersi anggaran jendela per origin dengan kesamaan persis.</p>
  <ul style="color: #ffca7a; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Tidak ada imputasi di mana pun.</strong> Saat bursa mati tidak ada harga yang terbentuk, jadi imputasi <em>tak terdefinisi</em>, bukan sekadar berisiko — taksonomi Rubin berlaku untuk nilai yang ada tapi tak teramati.</li>
    <li style="margin-bottom: 6px;"><strong>Per origin, kesamaan persis</strong> (<code>D45</code>). Diasersi terhadap angka gabungan 4,9% ia menyala palsu di empat belas dari lima belas origin, dilonggarkan sampai lolos, dan setelah itu tidak lagi bisa membedakan drift indeks posisional dari variasi antar-origin biasa.</li>
    <li style="margin-bottom: 6px;">Blok uji memuat <strong>720</strong> origin ramalan, bukan 601 (<code>D51b</code>): lookback jendela uji boleh menyeberang ke belakang, target jendela latih tidak boleh menyeberang ke depan.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 4px solid #ffb703; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffd60a; margin: 0 0 6px 0; font-size: 1.22em;">📘 segments.py</h3>
  <p style="color: #ffca7a; margin: 0; font-size: 0.94em;">Hukum segmen root §4.3: gap memutus deret, dan bar tak-layak-pakai memutusnya juga. Tidak ada imputasi di mana pun.</p>
</div>

<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #ffb70388; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffd60a; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Jam dalam milidetik dan path parquet bawaan.</p>
</div>

In [ ]:
"""The segment law: what breaks the series, and where.

A **segment** is a maximal run of contiguous, usable hourly bars. Root §4.3
breaks the series at two kinds of position:

1. **any missing bar** — 27 downtime blocks, 122 bars. When the exchange is down
   no price forms, so there is nothing to infer and imputation is *undefined*,
   not merely risky (root §4.2);
2. **any zero-volume or ``high == low`` bar** — it carries no trade information,
   exactly like downtime, so it gets the same treatment. This is what makes
   ``(VWAP - C)/(H - L)`` and ``log(volume)`` total rather than partial
   functions (`D14`), and why the F2 estimators are strictly positive and their
   logs total (root §5.1).

The ``high == low`` count has never been measured — root §4.3 and
``docs/ORIGIN_WINDOW_BUDGET.md`` both flag it as assumed. :func:`break_summary`
measures it, which is why this module exists before any model does.
"""

#: One hour in the integer domain every timestamp comparison uses. Root §2:
#: "Every timestamp is epoch-based and compared as an integer."
HOUR_MS: Final = 3_600_000

DEFAULT_PARQUET: Final = Path("data/raw/BTCUSDT_1h.parquet")


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #ffb70388; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffd60a; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧱 Segmen</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Deretan maksimal bar per-jam yang bersambung dan layak pakai.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class Segment:
    """A maximal run of contiguous usable bars, as half-open row indices.

    Attributes:
        start_row: First row index into the *usable* frame, inclusive.
        end_row: One past the last row index, exclusive.
        start_ts: Epoch ms of the first bar.
        end_ts: Epoch ms of the last bar — inclusive, because this is a bar and
            not a bound.
    """

    start_row: int
    end_row: int
    start_ts: int
    end_ts: int

    @property
    def n_bars(self) -> int:
        return self.end_row - self.start_row

    def window_starts(self, span: int) -> int:
        """How many ``span``-bar windows start inside this segment.

        A segment shorter than ``span`` contributes *zero*, never a negative
        number. The closed-form budget arithmetic in root §4.3 quietly assumes
        every segment clears ``span``; where it does not, the two disagree and
        :mod:`itransformer_btc.budget` reports the disagreement rather than
        papering over it.
        """
        return max(0, self.n_bars - span + 1)


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #ffb70388; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffd60a; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📥 Memuat bar & masker layak-pakai</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Bar bervolume nol atau ber-<code>H == L</code> dikecualikan — tiga bar, dan ketiganya bar yang sama (<code>D51c</code>).</p>
</div>

In [ ]:


def load_bars(path: Path | str = DEFAULT_PARQUET) -> pl.DataFrame:
    """Load the immutable Stage 1 artifact, sorted, with an epoch-ms column.

    Args:
        path: Parquet written by ``spot_klines_btc.py``.

    Returns:
        Every original column plus ``ts_ms`` (Int64 epoch milliseconds), sorted
        ascending.

    Raises:
        FileNotFoundError: If the artifact is absent.
        ValueError: If the frame is empty, carries duplicate timestamps, or
            reaches outside the declared half-open data window. That last check
            is the runnable form of `D33`: the boundary bar at
            ``2026-08-01T00:00`` sat one hour past the window and shifted every
            count derived from ``len(df)``.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. The four Stage 1 artifacts live in data/raw/ "
            f"(`D33`); regenerate with "
            f"`python spot_klines_btc.py --rebuild-only --outdir ./data/raw`."
        )

    frame = (
        pl.read_parquet(path)
        .with_columns(pl.col("open_time").dt.epoch("ms").alias("ts_ms"))
        .sort("ts_ms")
    )

    if frame.height == 0:
        raise ValueError(f"{path} is empty")

    n_unique = frame.select(pl.col("ts_ms").n_unique()).item()
    if n_unique != frame.height:
        raise ValueError(
            f"{path} carries {frame.height - n_unique} duplicate timestamps; "
            f"Stage 1 de-duplicates, so this is not Stage 1 output"
        )

    lo, hi = frame.select(
        pl.col("ts_ms").min().alias("lo"), pl.col("ts_ms").max().alias("hi")
    ).row(0)
    window_lo = int(DATA_START.timestamp() * 1000)
    window_hi = int(DATA_END.timestamp() * 1000)
    if lo < window_lo or hi >= window_hi:
        raise ValueError(
            f"{path} reaches outside the half-open data window "
            f"[{DATA_START.isoformat()}, {DATA_END.isoformat()}): "
            f"first={lo} last={hi}. `D33` — re-emit with --rebuild-only, which "
            f"applies clip_to_window()."
        )
    return frame


def usable_mask(frame: pl.DataFrame) -> pl.DataFrame:
    """Flag each bar usable or not, with the reason attached.

    A bar is unusable when it carries no trade information: zero volume, or
    ``high == low`` (no intrabar range). Both are treated exactly like downtime
    by root §4.3 — excluded, and the series splits there.

    ``zero_trades`` is measured but does **not** by itself mark a bar unusable:
    root §4.3 names only zero-volume and ``H == L``. It is carried so the open
    question in ``docs/ORIGIN_WINDOW_BUDGET.md`` — whether the 3 zero-volume and
    3 zero-trade bars are the same 3 bars — can be answered rather than assumed.

    Returns:
        The input frame plus boolean ``zero_volume``, ``flat_bar``,
        ``zero_trades`` and ``usable``.
    """
    return frame.with_columns(
        (pl.col("volume") <= 0).alias("zero_volume"),
        (pl.col("high") <= pl.col("low")).alias("flat_bar"),
        (pl.col("trades") <= 0).alias("zero_trades"),
    ).with_columns(
        (~pl.col("zero_volume") & ~pl.col("flat_bar")).alias("usable")
    )


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #ffb70388; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffd60a; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">✂️ Membangun segmen</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Gap memutus deret. Tidak ada ffill, tidak ada reindex ke grid jam penuh (root §4.2).</p>
</div>

In [ ]:


def build_segments(
    frame: pl.DataFrame,
    start: datetime | None = None,
    end: datetime | None = None,
) -> list[Segment]:
    """Split the usable bars of ``[start, end)`` into contiguous segments.

    A new segment begins wherever the previous usable bar is not exactly one
    hour earlier. That covers downtime and exclusion alike: an excluded bar has
    already been filtered out, so it shows up here as a time jump.

    Args:
        frame: Output of :func:`usable_mask`, or anything carrying ``ts_ms`` and
            ``usable``.
        start: Inclusive lower bound; ``None`` for unbounded.
        end: Exclusive upper bound; ``None`` for unbounded.

    Returns:
        Segments in chronological order; empty if the span holds no usable bar.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    span = frame.filter(pl.col("usable"))
    if start is not None:
        span = span.filter(pl.col("ts_ms") >= int(start.timestamp() * 1000))
    if end is not None:
        span = span.filter(pl.col("ts_ms") < int(end.timestamp() * 1000))
    if span.height == 0:
        return []

    ts = span.get_column("ts_ms").to_list()
    segments: list[Segment] = []
    seg_start = 0
    for i in range(1, len(ts)):
        if ts[i] - ts[i - 1] != HOUR_MS:
            segments.append(Segment(seg_start, i, ts[seg_start], ts[i - 1]))
            seg_start = i
    segments.append(Segment(seg_start, len(ts), ts[seg_start], ts[-1]))
    return segments


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #ffb70388; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffd60a; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📊 Ringkasan break</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Break per origin — target asersi kesamaan-persis yang <code>D45</code> tuntut.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class BreakSummary:
    """Measured break profile of a span. Every field is counted, not assumed."""

    calendar_hours: int
    bars_present: int
    bars_usable: int
    missing_bars: int
    zero_volume_bars: int
    flat_bars: int
    zero_trade_bars: int
    excluded_positions: int
    break_runs: int

    @property
    def segments(self) -> int:
        """Segments the span splits into.

        ``break_runs + 1`` only when every run is interior. A run touching
        either edge of the span produces one fewer segment, so this counts
        segments directly from the run structure rather than assuming.
        """
        return max(1, self.break_runs + 1)

    @property
    def window_starts_lost(self) -> int:
        """``119 x break_runs + excluded_positions`` — root §4.3's cost model."""
        return STARTS_LOST_PER_BREAK * self.break_runs + self.excluded_positions


def break_summary(
    frame: pl.DataFrame,
    start: datetime,
    end: datetime,
) -> BreakSummary:
    """Measure every break-inducing condition in ``[start, end)``.

    A **break run** is a maximal contiguous stretch of excluded calendar
    positions, whether excluded because the bar is missing or because it is
    unusable. Runs, not bars, are what the cost model charges 119 window starts
    to — so a zero-volume bar adjacent to a downtime block joins that block into
    one run instead of adding a second charge. Counting bars here instead of
    runs would overstate the loss by 119 per adjacency.

    This is the function that answers the two quantities
    ``docs/ORIGIN_WINDOW_BUDGET.md`` lists under "Not yet measured".
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    lo = int(start.timestamp() * 1000)
    hi = int(end.timestamp() * 1000)
    span = frame.filter((pl.col("ts_ms") >= lo) & (pl.col("ts_ms") < hi))

    calendar_hours = (hi - lo) // HOUR_MS
    usable = int(span.select(pl.col("usable").sum()).item())
    counts = span.select(
        pl.col("zero_volume").sum().alias("zv"),
        pl.col("flat_bar").sum().alias("fb"),
        pl.col("zero_trades").sum().alias("zt"),
    ).row(0)

    # Walk the calendar, not the rows: a missing bar has no row to inspect, and
    # a run mixing missing with unusable positions must count once.
    usable_ts = set(span.filter(pl.col("usable")).get_column("ts_ms").to_list())
    break_runs = 0
    in_run = False
    for t in range(lo, hi, HOUR_MS):
        if t in usable_ts:
            in_run = False
        else:
            if not in_run:
                break_runs += 1
            in_run = True

    return BreakSummary(
        calendar_hours=calendar_hours,
        bars_present=span.height,
        bars_usable=usable,
        missing_bars=calendar_hours - span.height,
        zero_volume_bars=int(counts[0]),
        flat_bars=int(counts[1]),
        zero_trade_bars=int(counts[2]),
        excluded_positions=calendar_hours - usable,
        break_runs=break_runs,
    )


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 4px solid #fb8500; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffb703; margin: 0 0 6px 0; font-size: 1.22em;">📘 windows.py</h3>
  <p style="color: #ffca7a; margin: 0; font-size: 0.94em;">Jendela divalidasi lewat <strong>timestamp</strong>, tidak pernah lewat indeks posisional — bug paling senyap di pipeline ini.</p>
</div>

<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #fb850088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb703; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Docstring modul.</p>
</div>

In [ ]:
"""Window enumeration, validated by timestamp rather than by position.

Importers: ``itransformer_btc.budget`` and ``tests/test_data_plane.py``.
Reads no file directly — it consumes a frame produced by
:mod:`itransformer_btc.segments` and writes nothing.

Root §4.3 names this the highest-probability silent bug in the pipeline: after
any row drop, positional sliding closes gaps invisibly, and a window that spans
a two-day outage looks identical to one that does not. The rule is therefore

    window [s, s+L+H) is valid  <=>  t[s+L+H-1] - t[s] == (L+H-1) hours

and it is checked on every emitted window, not sampled. The check is cheap; the
failure it prevents is a leak no metric would reveal, because a model trained
across a gap looks *better*, not worse.

**The purge is structural here, not a separate step.** A window occupies
``[s, s+L+H)`` and its target is the final ``H`` bars. Enumerating only windows
that lie wholly inside a span therefore guarantees the last target ends exactly
at the span boundary — which is what root §8.2 asks for at *both* boundaries,
train→validation and train→test (`D24`). There is no separate purge to forget.
"""


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #fb850088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb703; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪟 Enumerasi jendela</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;"><strong>Divalidasi lewat timestamp, tidak pernah lewat indeks posisional.</strong> Setelah baris manapun jatuh, sliding posisional menutup gap tanpa terlihat.</p>
</div>

In [ ]:

def enumerate_windows(
    frame: pl.DataFrame,
    start: datetime | None = None,
    end: datetime | None = None,
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> list[int]:
    """Every valid window start inside ``[start, end)``, as epoch ms.

    Windows are built *inside* segments and never across them, so no window can
    span a break. Enumerating inside ``[start, end)`` also applies the purge:
    the last window's target ends at ``end``, never past it.

    Args:
        frame: Bars carrying ``ts_ms``; ``usable`` is derived if absent.
        start: Inclusive lower bound of the span.
        end: Exclusive upper bound of the span.
        seq_len: Lookback ``L``.
        pred_len: Horizon ``H``.

    Returns:
        Window-start timestamps in ascending order.

    Raises:
        ValueError: If any emitted window fails the timestamp identity. That is
            an assertion about the segment builder, not about the data, so a
            failure means the pipeline is broken rather than the market.
    """
    span = seq_len + pred_len
    segments = build_segments(frame, start, end)
    if not segments:
        return []

    rows = (frame if "usable" in frame.columns else usable_mask(frame)).filter(
        pl.col("usable")
    )
    if start is not None:
        rows = rows.filter(pl.col("ts_ms") >= int(start.timestamp() * 1000))
    if end is not None:
        rows = rows.filter(pl.col("ts_ms") < int(end.timestamp() * 1000))
    ts = rows.get_column("ts_ms").to_list()

    starts: list[int] = []
    for seg in segments:
        for s in range(seg.start_row, seg.end_row - span + 1):
            last = s + span - 1
            if ts[last] - ts[s] != (span - 1) * HOUR_MS:
                raise ValueError(
                    f"window at ts={ts[s]} spans a break: "
                    f"t[{last}] - t[{s}] = {ts[last] - ts[s]} ms, expected "
                    f"{(span - 1) * HOUR_MS} ms. The segment builder is wrong; "
                    f"do not relax this check."
                )
            starts.append(ts[s])
    return starts


<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #fb850088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb703; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔢 Menghitung jendela</h4>
  <p style="color: #ffca7a; margin: 0; font-size: 0.9em;">Bentuk tertutup hanyalah batas atas; hitungan sebenarnya per segmen (<code>D51a</code>).</p>
</div>

In [ ]:


def count_windows(
    segments: list[Segment],
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> int:
    """Total window starts across segments — the measured truth.

    Uses ``max(0, n - span + 1)`` per segment, so a segment shorter than one
    window contributes nothing rather than a negative count. Root §4.3's closed
    form ``(bars - 119) - [119 x breaks + missing]`` is algebraically identical
    to this **only while every segment clears the span**; where one does not,
    this is right and the closed form is not.
    """
    span = seq_len + pred_len
    return sum(seg.window_starts(span) for seg in segments)


<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 4px solid #f48c06; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffba08; margin: 0 0 6px 0; font-size: 1.22em;">📘 budget.py</h3>
  <p style="color: #ffd08a; margin: 0; font-size: 0.94em;">Anggaran jendela per origin, ditegakkan dengan kesamaan persis terhadap <code>docs/ORIGIN_WINDOW_BUDGET.md</code> (<code>D45</code>).</p>
</div>

<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c0688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffba08; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & anggaran terkomit</h4>
  <p style="color: #ffd08a; margin: 0; font-size: 0.9em;">Angka per origin yang <code>docs/ORIGIN_WINDOW_BUDGET.md</code> kunci.</p>
</div>

In [ ]:
"""Per-origin window accounting — the assertion target of root §11.

Importers: ``tests/test_data_plane.py``, and the Stage 2 launcher in
``notebooks/``. Reads ``data/raw/BTCUSDT_1h.parquet`` only through
:func:`itransformer_btc.segments.load_bars`; writes nothing.

``docs/ORIGIN_WINDOW_BUDGET.md`` was committed *before* any run so the pipeline
has something to be checked **against** rather than something to be tuned
**to**. This module measures the same quantities from the artifact and compares.

A divergence is a finding, not a nuisance. The committed table is derived from
the 27 downtime blocks alone, while the segment law (root §4.3) also breaks at
zero-volume and ``high == low`` bars, whose count has never been measured. If
those bars fall inside training spans, measured windows are **lower** than the
table and the table needs regenerating.

Root §11 requires an **exact equality per origin**, never a comparison against
the pooled 4.9% figure — asserted pooled, it fires spuriously at fourteen of
fifteen origins, gets loosened until it passes, and then can no longer
distinguish positional drift from ordinary between-origin variation.
"""

#: Origin label to ``(break_runs, excluded_positions, windows_kept)`` for the
#: 21-month training sub-block. **Measured from the artifact on 2026-08-06**,
#: superseding the derived table `D26` shipped with (`D51`). It is pinned here
#: rather than recomputed inside the test so the test can catch *drift*: with
#: both sides computed the same way, a regression would agree with itself and
#: pass. Regenerate with :func:`format_markdown` and update both this dict and
#: ``docs/ORIGIN_WINDOW_BUDGET.md`` together, never one alone.
#:
#: The derived table it replaces read, for the first four origins,
#: ``(10, 86, 13_917) (12, 62, 13_727) (11, 47, 13_861) (12, 40, 13_797)``.
#: It diverged at twelve of fifteen origins for two reasons, both structural:
#: it never counted the three unusable bars, and its closed form charges a
#: short segment a negative window count. See `D51`.
COMMITTED_TRAIN_BUDGET: Final[dict[str, tuple[int, int, int]]] = {
    "2020-01": (11, 87, 13_934),
    "2020-06": (13, 63, 13_701),
    "2020-11": (12, 48, 13_741),
    "2021-04": (13, 41, 13_716),
    "2021-09": (14, 30, 13_560),
    "2022-02": (14, 32, 13_558),
    "2022-07": (9, 20, 14_165),
    "2022-12": (8, 19, 14_285),
    "2023-05": (2, 6, 15_021),
    "2023-10": (1, 2, 15_072),
    "2024-03": (1, 2, 15_120),
    "2024-08": (1, 2, 15_096),
    "2025-01": (1, 2, 15_096),
    "2025-06": (0, 0, 15_217),
    "2025-11": (0, 0, 15_217),
}


<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c0688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffba08; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">💰 Anggaran per origin</h4>
  <p style="color: #ffd08a; margin: 0; font-size: 0.9em;">Jendela latih yang bertahan, break, dan bar hilang untuk satu origin.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class OriginBudget:
    """Measured window accounting for one origin's training sub-block."""

    origin: Origin
    summary: BreakSummary
    windows_measured: int
    windows_closed_form: int
    test_block_starts: tuple[int, ...]

    @property
    def label(self) -> str:
        return self.origin.label

    @property
    def loss_pct(self) -> float:
        """Window starts lost to breaks, as a percentage of a gap-free span."""
        ceiling = self.summary.calendar_hours - STARTS_LOST_PER_BREAK
        return 100.0 * (1.0 - self.windows_measured / ceiling) if ceiling else 0.0

    @property
    def closed_form_agrees(self) -> bool:
        """Whether root §4.3's arithmetic matches the segment-wise truth.

        Disagreement means some segment is shorter than one window, so the
        closed form has gone negative somewhere and been silently absorbed.
        """
        return self.windows_measured == self.windows_closed_form


<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c0688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffba08; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎯 Start blok uji yang bertahan</h4>
  <p style="color: #ffd08a; margin: 0; font-size: 0.9em;">720 origin ramalan per blok bersih, bukan 601 — akuntansi uji memakai semantik uji (<code>D51b</code>).</p>
</div>

In [ ]:


def surviving_block_starts(frame: pl.DataFrame, origin: Origin, b: int) -> int:
    """Window starts surviving inside test block ``b`` — out of 720.

    **Test blocks do not use the training semantics, and the difference is 119
    windows per block.** A training window must lie wholly inside its span,
    because its target may not cross into validation (root §8.2). A *test*
    window may not: root §8.3 states explicitly that a window's 96-bar input
    reaching back across the boundary is past information legitimately available
    to a forecaster at that moment, and blocking it "would make the evaluation
    unrealistically pessimistic". So every one of the block's 720 hours is an
    admissible forecast origin; what disqualifies one is a break inside the 120
    bars it spans, wherever those bars fall.

    Counting test blocks the training way yields 601 out of 720 even for a
    perfectly clean block — a 16.5% phantom loss that would be read as outage
    damage and would enter §9.2's block-coverage covariate as pure noise.
    """
    lo, hi = origin.block(b)
    lo_ms = int(lo.timestamp() * 1000)
    hi_ms = int(hi.timestamp() * 1000)
    span_ms = (WINDOW_SPAN - 1) * HOUR_MS

    usable = set(
        frame.filter(pl.col("usable")).get_column("ts_ms").to_list()
    )
    survivors = 0
    for start in range(lo_ms, hi_ms, HOUR_MS):
        # The window is contiguous exactly when every hour it spans is usable.
        if all((start + k * HOUR_MS) in usable for k in range(WINDOW_SPAN)):
            survivors += 1
    return survivors


<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c0688; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffba08; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel anggaran</h4>
  <p style="color: #ffd08a; margin: 0; font-size: 0.9em;">Lima belas baris yang Tabel 1 render, diukur bukan disalin.</p>
</div>

In [ ]:


def origin_budget(frame: pl.DataFrame, origin: Origin) -> OriginBudget:
    """Measure one origin's training sub-block and its six test blocks.

    The sub-block is ``[train_start, train_sub_end)`` — 21 months, not 24
    (`D25`). Test-block figures are surviving window *starts* inside each block.
    """
    summary = break_summary(frame, origin.train_start, origin.train_sub_end)
    segments = build_segments(frame, origin.train_start, origin.train_sub_end)

    ceiling = summary.calendar_hours - STARTS_LOST_PER_BREAK
    blocks = [surviving_block_starts(frame, origin, b) for b in range(1, TEST_BLOCKS + 1)]

    return OriginBudget(
        origin=origin,
        summary=summary,
        windows_measured=count_windows(segments, SEQ_LEN, PRED_LEN),
        windows_closed_form=ceiling - summary.window_starts_lost,
        test_block_starts=tuple(blocks),
    )


def budget_table(frame: pl.DataFrame) -> list[OriginBudget]:
    """Measure every origin in the committed grid."""
    return [origin_budget(frame, origin) for origin in ORIGINS]


def format_markdown(budgets: list[OriginBudget]) -> str:
    """Render the measured table in the shape of ``ORIGIN_WINDOW_BUDGET.md``.

    Used to regenerate the document when measurement supersedes derivation —
    never to silently overwrite it. Root §12: a number that cannot be
    regenerated is a documented failure, not a footnote.
    """
    head = (
        "| # | Origin | Training sub-block | Breaks | Excluded | Windows kept "
        "| Loss | Test-block starts B1…B6 |\n"
        "|---:|---|---|---:|---:|---:|---:|---|\n"
    )
    rows = [
        f"| {b.origin.index:>2} | {b.origin.origin:%Y-%m-%d} "
        f"| {b.origin.train_start:%Y-%m-%d} → {b.origin.train_sub_end:%Y-%m-%d} "
        f"| {b.summary.break_runs} | {b.summary.excluded_positions} "
        f"| {b.windows_measured:,} | {b.loss_pct:.1f}% "
        f"| {' / '.join(str(n) for n in b.test_block_starts)} |"
        for b in budgets
    ]
    return head + "\n".join(rows) + "\n"


<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 4px solid #f48c06; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffba08; margin: 0 0 6px 0; font-size: 1.22em;">📊 Jalankan Stage 2 — muat bar & asersi anggaran jendela</h3>
  <p style="color: #ffd08a; margin: 0; font-size: 0.94em;">Artefak imutabel dimuat, bar tak-layak-pakai ditandai, lalu anggaran jendela diasersi <strong>per origin dengan kesamaan persis</strong> terhadap <code>docs/ORIGIN_WINDOW_BUDGET.md</code> (<code>D45</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>data/raw/BTCUSDT_1h.parquet</code></span></div>
</div>

In [22]:
bars = usable_mask(load_bars(PARQUET))
print(f"bars {bars.height:,}  usable {int(bars['usable'].sum()):,}  "
      f"unusable {int((~bars['usable']).sum())}")

# D51c: the same 3 bars are zero-volume, zero-trade and H == L. No volume means
# no trades, and no trades means high and low never separate.
print(bars.filter(~pl.col("usable")).select(
    ["open_time", "zero_volume", "flat_bar", "zero_trades"]))

budgets = budget_table(bars)
drift = [
    (b.label, b.summary.break_runs, b.summary.excluded_positions, b.windows_measured,
     COMMITTED_TRAIN_BUDGET[b.label])
    for b in budgets
    if (b.summary.break_runs, b.summary.excluded_positions, b.windows_measured)
    != COMMITTED_TRAIN_BUDGET[b.label]
]
assert not drift, f"budget drift against the committed table: {drift}"
print(f"\nwindow budget matches docs/ORIGIN_WINDOW_BUDGET.md at all "
      f"{len(budgets)} origins (exact equality, D45)")

coverage = pl.DataFrame([
    {"origin": b.label, "train_windows": b.windows_measured,
     "loss_pct": round(b.loss_pct, 2), "closed_form_agrees": b.closed_form_agrees,
     **{f"B{i}": n for i, n in enumerate(b.test_block_starts, start=1)}}
    for b in budgets
])
print(coverage)
print(f"\ntraining range {coverage['train_windows'].min():,} … "
      f"{coverage['train_windows'].max():,} windows (raw-bar frame)")
print("The closed form disagrees wherever a segment is shorter than one window "
      "(D51a) and is kept only as an upper bound.")


bars 75,094  usable 75,091  unusable 3
shape: (3, 4)
┌─────────────────────────┬─────────────┬──────────┬─────────────┐
│ open_time               ┆ zero_volume ┆ flat_bar ┆ zero_trades │
│ ---                     ┆ ---         ┆ ---      ┆ ---         │
│ datetime[ms, UTC]       ┆ bool        ┆ bool     ┆ bool        │
╞═════════════════════════╪═════════════╪══════════╪═════════════╡
│ 2019-06-07 21:00:00 UTC ┆ true        ┆ true     ┆ true        │
│ 2021-02-11 03:00:00 UTC ┆ true        ┆ true     ┆ true        │
│ 2023-03-24 12:00:00 UTC ┆ true        ┆ true     ┆ true        │
└─────────────────────────┴─────────────┴──────────┴─────────────┘

window budget matches docs/ORIGIN_WINDOW_BUDGET.md at all 15 origins (exact equality, D45)
shape: (15, 10)
┌─────────┬───────────────┬──────────┬────────────────────┬───┬─────┬─────┬─────┬─────┐
│ origin  ┆ train_windows ┆ loss_pct ┆ closed_form_agrees ┆ … ┆ B3  ┆ B4  ┆ B5  ┆ B6  │
│ ---     ┆ ---           ┆ ---      ┆ ---                ┆ 

<div style="background: linear-gradient(135deg, #001a1a, #002b2b); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #48cae433;">
  <h2 style="color: #90e0ef; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🧪 4 &middot; Pra-proses — dua belas variat
  </h2>
  <p style="color: #ade8f4; margin: 0; font-size: 1.02em;">Fungsi per-bar dari bar saat ini, kecuali r yang memakai penutup sekarang dan sebelumnya. Tidak ada rolling window di mana pun.</p>
  <ul style="color: #ade8f4; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Itu properti keamanan <strong>struktural</strong>, bukan pilihan gaya: tanpa satu pun rolling window di pipeline, kelas kebocoran <code>center=True</code> <em>tak terwakili</em>.</li>
    <li style="margin-bottom: 6px;">Urutan kolom adalah urutan tangga, jadi rung K persis K kolom pertama dan <code>r</code> adalah kanal 0 di setiap rung — yang membuat loss kanal-tunggal satu konstanta, bukan sebuah lookup.</li>
    <li style="margin-bottom: 6px;">Rogers&ndash;Satchell <strong>tidak</strong> positif tegas: ia lenyap pada bar tanpa bayangan, dan 33 bar seperti itu ada. <code>log(RS + 1e-9)</code> menaruh <code>log &kappa; = &minus;20,7</code> di dalam support terukur, bukan sebagai 33 lonjakan di luar support (<code>D52a</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 4px solid #48cae4; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #90e0ef; margin: 0 0 6px 0; font-size: 1.22em;">📘 features.py</h3>
  <p style="color: #ade8f4; margin: 0; font-size: 0.94em;">Dua belas variat F1–F5, seluruhnya fungsi per-bar. Tidak satu pun memakai rolling window (root §5.3).</p>
</div>

<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header, tangga variat, konstanta</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Dua belas variat berurutan, target <code>r</code>, dan penstabil κ = 1e-9 untuk Rogers–Satchell (<code>D52a</code>).</p>
</div>

In [ ]:
"""The twelve variates, and the K ladder cut over them.

All twelve are **engineered**; none is a raw kline column. What is excluded is a
class, not a list: technical indicators, multi-bar rolling statistics, calendar
dummies, cross-asset, on-chain, macro. Root §5.3 carries the argument — K is
RQ1's independent variable, and anything outside families F1–F5 breaks the
taxonomy that makes K_eff interpretable.

**No variate uses a rolling window.** Every one is a pure per-bar function of
the current bar, except ``r``, which uses the current and previous close. That
is a structural safety property rather than a style choice: with no rolling
window anywhere, the ``center=True`` leak class is unrepresentable (root §5.3).

**The ladder is cumulative and its order is load-bearing.** Column order is
ladder order, so rung K is exactly the first K columns and ``r`` is channel 0 at
every rung. Root §6.2 requires the loss be MSE on the **target channel only** at
every rung; with ``r`` pinned at index 0 that is one constant, not a lookup.

Upstream
--------
**All twelve variates are written here in polars from their published closed
forms. Nothing is taken from a technical-analysis library, and that exclusion is
a design decision rather than an oversight** -- RSI, MACD and Bollinger belong
to no F1-F5 family, so admitting them would break the taxonomy that makes
``K_eff`` interpretable and render the K=8 versus K=12 contrast meaningless
(`D37`).

The F2 volatility estimators, each per-bar:

- M. Parkinson, "The extreme value method for estimating the variance of the
  rate of return," *J. Business*, vol. 53, no. 1, pp. 61-65, 1980.
- M. B. Garman and M. J. Klass, "On the estimation of security price
  volatilities from historical data," *J. Business*, vol. 53, no. 1,
  pp. 67-78, 1980.
- L. C. G. Rogers and S. E. Satchell, "Estimating variance from high, low and
  closing prices," *Ann. Appl. Probab.*, vol. 1, no. 4, pp. 504-512, 1991.

Two departures worth stating where a reader meets the code. **No estimator is
trailing-averaged** (`D13`): every variate is a pure per-bar function, which is
what makes the ``center=True`` leakage class structurally unrepresentable and
licenses root §8.3's no-embargo argument (`D15`). And Rogers-Satchell **is not
strictly positive** -- it vanishes on the 33 shadowless bars in this sample --
so it is taken as ``log(RS + 1e-9)``, the floor chosen to land inside the
measured support rather than as 33 out-of-support spikes; Parkinson and
Garman-Klass need no floor (`D52a`).
:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries this row in full.
"""

#: Ladder order. Rung K is ``VARIATE_ORDER[:K]``. Root §5.2's unique consistent
#: cut (`D01` — the source specification's K=8 rung summed to nine and
#: double-assigned ``log_mean_trade_size``).
VARIATE_ORDER: Final[tuple[str, ...]] = (
    # F1 price trajectory — K=1 is `r` alone
    "r",
    "upper_shadow",
    "lower_shadow",
    # F3 intensity, first member — completes K=4
    "log_quote_volume",
    # K=8: intensity, order flow, intrabar location
    "log_trade_count",
    "taker_buy_ratio",
    "signed_flow",
    "vwap_location",
    # K=12: the F2 volatility estimators + the dependent intensity member
    "log_parkinson",
    "log_garman_klass",
    "log_rogers_satchell",
    "log_mean_trade_size",
)

TARGET: Final = "r"
TARGET_INDEX: Final = 0

#: Parkinson's normaliser, ``1 / (4 ln 2)``.
_PARKINSON_C: Final = 1.0 / (4.0 * math.log(2.0))
#: Garman–Klass's second-term coefficient, ``2 ln 2 - 1`` ≈ 0.386. Strictly
#: below 0.5, which is what keeps the estimator positive: ``|ln(C/O)| <=
#: ln(H/L)`` because C and O both lie in ``[L, H]``, so GK >= 0.114 (ln H/L)^2.
_GK_C: Final = 2.0 * math.log(2.0) - 1.0

#: Stabiliser for Rogers–Satchell only (`D52`).
#:
#: Root §5.1 claims all three F2 estimators are "strictly positive once H == L
#: bars are excluded". That holds for Parkinson (proportional to ``(ln H/L)^2``)
#: and for Garman–Klass (bounded below by ``0.114 (ln H/L)^2``), but **not** for
#: Rogers–Satchell, which is
#:
#:     ln(H/C) ln(H/O) + ln(L/C) ln(L/O)
#:
#: and vanishes on any bar with no shadows at all — H equal to one of O/C and L
#: equal to the other. Such a bar has H > L, carries real trade information, and
#: passes the segment law; it is a marubozu, not a degenerate bar. Measured: 33
#: of 75,091 usable bars, 0.044%.
#:
#: ``1e-9`` is chosen so ``log(kappa) = -20.7`` lands **inside the measured
#: support** of ``log RS`` — median -10.91, 0.1st percentile -17.57, minimum
#: -23.5 — in the low tail where a shadowless bar belongs. A hard floor far
#: below support (1e-12 gives -27.6, about -11 sigma) would instead create 33
#: spikes that distort the instance normalisation of every window containing
#: one, and would smuggle a categorical marubozu flag into a continuous
#: variate — the convenience-variate failure root §5.2 forbids. The shift it
#: applies to a typical bar is negligible: at the median RS of 1.8e-5, adding
#: 1e-9 moves ``log RS`` by 5e-5.
#:
#: Deliberately **not** applied to Parkinson or Garman–Klass: both are provably
#: positive, their measured minima are 1.16e-8 and 1.48e-8, and adding kappa
#: there would shift the smallest values by roughly 8% for no reason.
_RS_STABILISER: Final = 1e-9


#: Two eight-variate subsets, built to differ in **effective** rank while holding
#: nominal K fixed at 8 (`D70`).
#:
#: RQ1 asks whether the marginal benefit of added variates is governed by nominal
#: count or by effective dimensionality, and the ladder answers it only through a
#: panel: K and K_eff move together there, ``corr(K, K_eff) = 0.828``, so the two
#: explanations are separated by a non-nested test rather than by contrast. These
#: two rungs separate them **directly** — same K, same target, same everything
#: else, and PR is the only thing that moves.
#:
#: - ``redundant`` loads F2 whole. All three volatility estimators carry about one
#:   independent degree of freedom between them (root §5.1), and all three of F3
#:   are present where the third is the difference of the first two. Low PR by
#:   construction.
#: - ``orthogonal`` takes one or two from each of F1-F5 and never doubles up
#:   inside a family. High PR by construction.
#:
#: ``r`` leads both, because :data:`TARGET_INDEX` is 0 and every consumer reads
#: the target there.
MATCHED_K_SUBSETS: dict[str, tuple[str, ...]] = {
    "redundant": (
        "r",
        "log_parkinson",
        "log_garman_klass",
        "log_rogers_satchell",
        "log_quote_volume",
        "log_trade_count",
        "log_mean_trade_size",
        "taker_buy_ratio",
    ),
    "orthogonal": (
        "r",
        "upper_shadow",
        "lower_shadow",
        "log_parkinson",
        "log_quote_volume",
        "log_trade_count",
        "taker_buy_ratio",
        "vwap_location",
    ),
}


<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪜 Kolom per rung</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">K ∈ {1, 4, 8, 12} — satu-satunya potongan konsisten yang <code>D01</code> tetapkan.</p>
</div>

In [ ]:


def ladder_columns(k: int) -> list[str]:
    """The variate names at rung ``k``.

    Raises:
        ValueError: If ``k`` is not one of the pre-registered rungs. Rungs are
            fixed before any model runs (root §3); an ad-hoc K is a new
            experiment and must be declared as one.
    """
    if k not in (1, 4, 8, 12):
        raise ValueError(f"K must be a pre-registered rung 1/4/8/12, got {k}")
    return list(VARIATE_ORDER[:k])


<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔬 Membangun dua belas variat</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Seluruhnya fungsi per-bar. <strong>Tidak satu pun memakai rolling window</strong>, jadi kelas kebocoran <code>center=True</code> tak terwakili (root §5.3).</p>
</div>

In [ ]:


def build_features(frame: pl.DataFrame) -> pl.DataFrame:
    """Compute all twelve variates, per segment, dropping what is undefined.

    ``r`` is computed **within** each segment. Computing it on a concatenated
    series would inject giant cross-gap returns into ``mu_g`` and ``sigma_g``
    before any window is excluded (root §4.3): the 2018-02-08 outage would book
    a 33-hour move as a one-hour return, and the scaler would be fitted on it.

    The first bar of each segment therefore yields a null ``r`` and is dropped.
    That shortens every segment by one bar, which the window enumerator picks up
    for free — the dropped bar leaves a two-hour jump at the segment head, and
    :func:`itransformer_btc.segments.build_segments` splits on any jump.

    Args:
        frame: Bars carrying ``ts_ms`` and the raw kline columns. ``usable`` is
            derived if absent.

    Returns:
        ``ts_ms``, ``usable`` (all True), and the twelve variates in ladder
        order as ``Float64``, with no nulls.

    Raises:
        ValueError: If any variate is null or non-finite. Both are impossible
            once the segment law has excluded zero-volume and ``H == L`` bars,
            so either means the exclusion did not run.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    rows = frame.filter(pl.col("usable")).sort("ts_ms")

    # Segment identity from the timestamp alone. Excluded bars are already gone,
    # so they show up here as jumps, exactly as downtime does.
    rows = rows.with_columns(
        (pl.col("ts_ms").diff().fill_null(HOUR_MS) != HOUR_MS).cum_sum().alias("_seg")
    )

    log_h_l = (pl.col("high") / pl.col("low")).log()
    log_c_o = (pl.col("close") / pl.col("open")).log()
    vwap = pl.col("quote_volume") / pl.col("volume")

    out = rows.with_columns(
        # -- F1 price trajectory, 3 dof -------------------------------------
        (pl.col("close").log() - pl.col("close").log().shift(1).over("_seg")).alias("r"),
        (pl.col("high") / pl.max_horizontal("open", "close")).log().alias("upper_shadow"),
        (pl.min_horizontal("open", "close") / pl.col("low")).log().alias("lower_shadow"),

        # -- F3 intensity, 2 dof — the third is the difference of the first two
        pl.col("quote_volume").log().alias("log_quote_volume"),
        pl.col("trades").log().alias("log_trade_count"),
        (pl.col("quote_volume") / pl.col("trades")).log().alias("log_mean_trade_size"),

        # -- F4 order flow, 1–2 dof -----------------------------------------
        # Base-denominated: the canonical buyer-initiated volume share (`D12`).
        # The quote-denominated variant is a robustness check, not the default.
        (pl.col("taker_buy_base") / pl.col("volume")).alias("taker_buy_ratio"),

        # -- F5 intrabar location, 1 dof ------------------------------------
        # A total function, not a partial one, because H == L bars are segment
        # breaks (`D14`). Without that exclusion this divides by zero.
        ((vwap - pl.col("close")) / (pl.col("high") - pl.col("low"))).alias("vwap_location"),

        # -- F2 volatility estimators, ~1 dof, redundant by construction -----
        # Per-bar with no trailing average (`D13`). Pre-smoothing over 24 bars
        # is strictly less informative: the model can compute that average
        # itself and cannot recover what smoothing destroyed (root §5.3).
        # Parkinson and Garman-Klass are provably positive once H > L, so their
        # logs are total. Rogers-Satchell is NOT — it vanishes on shadowless
        # bars — hence the stabiliser, and only there (`D52`).
        (_PARKINSON_C * log_h_l.pow(2)).log().alias("log_parkinson"),
        (0.5 * log_h_l.pow(2) - _GK_C * log_c_o.pow(2)).log().alias("log_garman_klass"),
        (
            (pl.col("high") / pl.col("close")).log() * (pl.col("high") / pl.col("open")).log()
            + (pl.col("low") / pl.col("close")).log() * (pl.col("low") / pl.col("open")).log()
            + _RS_STABILISER
        ).log().alias("log_rogers_satchell"),
    ).with_columns(
        # A deterministic product of two other K=8 members. Kept, with the
        # dependence disclosed (`D12`): it weakens the claim that K=8 is the
        # rung of maximum effective rank, and the measured participation ratio
        # settles that question rather than the argument doing so.
        (
            (2.0 * pl.col("taker_buy_ratio") - 1.0) * pl.col("log_quote_volume")
        ).alias("signed_flow"),
    )

    # The first bar of each segment has no predecessor inside its segment.
    out = out.filter(pl.col("r").is_not_null())

    out = out.select(["ts_ms", "usable", *VARIATE_ORDER]).with_columns(
        [pl.col(c).cast(pl.Float64) for c in VARIATE_ORDER]
    )

    offenders = {
        name: n
        for name in VARIATE_ORDER
        if (
            n := int(
                out.select(
                    (~pl.col(name).is_finite() | pl.col(name).is_null()).sum()
                ).item()
            )
        )
    }
    if offenders:
        raise ValueError(
            f"non-finite variate values: {offenders}. Every variate is total "
            f"once zero-volume and H == L bars are excluded by the segment law "
            f"(root §4.3 / `D14`), so this means the exclusion did not run."
        )
    return out


<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 4px solid #48cae4; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #90e0ef; margin: 0 0 6px 0; font-size: 1.22em;">🔬 Bangun frame fitur</h3>
  <p style="color: #ade8f4; margin: 0; font-size: 0.94em;">Dua belas variat dari bar layak-pakai. Satu baris per segmen jatuh karena <code>r</code> butuh penutup sebelumnya (<code>D52c</code>), dan tiap variat diasersi finite — kalau tidak, hukum segmen tidak berjalan.</p>
</div>

In [26]:
features = build_features(bars)
print(f"feature frame {features.height:,} rows x {len(VARIATE_ORDER)} variates")
print(f"dropped {int(bars['usable'].sum()) - features.height} rows = one per segment "
      f"(D52c: r is per segment, so each segment's first bar has no predecessor)")

for k in (1, 4, 8, 12):
    print(f"  K={k:>2}: {ladder_columns(k)}")

print(features.select(VARIATE_ORDER).describe().filter(
    pl.col("statistic").is_in(["mean", "std", "min", "max"])))

# D52a: Rogers-Satchell is NOT strictly positive — it vanishes on a shadowless
# (marubozu) bar, of which 33 exist. log(RS + 1e-9) puts log kappa = -20.7 inside
# the measured support rather than leaving 33 out-of-support spikes that would
# distort the instance normalisation of every window containing one.
rs = features["log_rogers_satchell"]
print(f"\nlog_rogers_satchell: min {rs.min():.3f}  q0.1% {rs.quantile(0.001):.3f}  "
      f"median {rs.median():.3f}  at-floor {int((rs <= np.log(1e-9) + 1e-9).sum())}")
assert features.select([pl.col(c).is_finite().all() for c in VARIATE_ORDER]).row(0) \
    == tuple([True] * 12), "a variate is non-finite; the segment law did not run"


feature frame 75,062 rows x 12 variates
dropped 29 rows = one per segment (D52c: r is per segment, so each segment's first bar has no predecessor)
  K= 1: ['r']
  K= 4: ['r', 'upper_shadow', 'lower_shadow', 'log_quote_volume']
  K= 8: ['r', 'upper_shadow', 'lower_shadow', 'log_quote_volume', 'log_trade_count', 'taker_buy_ratio', 'signed_flow', 'vwap_location']
  K=12: ['r', 'upper_shadow', 'lower_shadow', 'log_quote_volume', 'log_trade_count', 'taker_buy_ratio', 'signed_flow', 'vwap_location', 'log_parkinson', 'log_garman_klass', 'log_rogers_satchell', 'log_mean_trade_size']
shape: (4, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ r         ┆ upper_sha ┆ lower_sha ┆ … ┆ log_parki ┆ log_garma ┆ log_roger ┆ log_mean │
│ ---       ┆ ---       ┆ dow       ┆ dow       ┆   ┆ nson      ┆ n_klass   ┆ s_satchel ┆ _trade_s │
│ str       ┆ f64       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ l         ┆ ize      │

<div style="background: linear-gradient(135deg, #002200, #003300); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #7ae58233;">
  <h2 style="color: #95d5b2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🌊 5 &middot; Uji efisiensi pasar
  </h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 1.02em;">ADF, variance ratio Lo&ndash;MacKinlay, dan eksponen Hurst — full sample dan tiap sub-blok latih 21 bulan.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Mengubah &ldquo;pasar efisien&rdquo; dari asumsi menjadi <strong>temuan</strong>. Klaimnya soal <em>variasi</em>, dan satu baris tidak bisa menunjukkannya — karena itu per origin, bukan sekali di seluruh sampel.</li>
    <li style="margin-bottom: 6px;">Jangan mengklaim pasarnya efisien. Nyatakan buktinya campur dan berubah menurut waktu, lalu laporkan angka sendiri (root &sect;4.5).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 4px solid #7ae582; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #95d5b2; margin: 0 0 6px 0; font-size: 1.22em;">📘 efficiency.py</h3>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.94em;">Uji efisiensi pasar root §4.5: ADF, variance ratio, Hurst. Full sample dan per sub-blok latih.</p>
</div>

<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae58288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Lag variance ratio dan ambang minimum Hurst.</p>
</div>

In [ ]:
"""Root §4.5's preliminary market-efficiency tests.

Run once, reported in the Data section. The point is to convert "efficient
market" from an assumption into a finding: §4.5 forbids *claiming* the market is
efficient and requires stating that the evidence is mixed and time-varying
(Urquhart 2016; Nadarajah & Chu 2017; Bariviera 2017; Sensoy 2019), then
reporting our own numbers beside it.

Reported over two spans, because "time-varying" is a claim about variation and
one full-sample row cannot exhibit it: the whole sample, and each origin's
**21-month training sub-block** -- the same span the scaler is fitted on. Those
rows are descriptive and gate nothing, so the span rule `D44` imposes on K_eff
does not bind, but reading a test block merely to describe the data would still
be indefensible when avoiding it costs one filter.

``arch`` and ``statsmodels`` accept numpy arrays, so this module crosses root
§16's stats boundary without pandas ever entering the process.

Upstream
--------
**This is the only module in the package that calls a third-party statistics
implementation, and it does so at root §16's named boundary -- imported inside
the function that needs it, never at module level.**

- Variance ratio -- ``arch.unitroot.VarianceRatio``
  (https://github.com/bashtage/arch, NCSA; accessed 2026-09-03), implementing
  A. W. Lo and A. C. MacKinlay, "Stock market prices do not follow random
  walks: Evidence from a simple specification test," *Rev. Financial Stud.*,
  vol. 1, no. 1, pp. 41-66, 1988.
- ADF -- ``statsmodels.tsa.stattools.adfuller`` (BSD-3-Clause), implementing
  D. A. Dickey and W. A. Fuller, *J. Amer. Statist. Assoc.*, vol. 74, no. 366,
  pp. 427-431, 1979.
- ``hurst_rs`` -- **written here.** H. E. Hurst, "Long-term storage capacity of
  reservoirs," *Trans. Amer. Soc. Civil Eng.*, vol. 116, no. 1, pp. 770-799,
  1951. No package provides a rescaled-range estimator under a licence and an
  API this project already depends on, so taking a dependency to import a
  single function was the worse trade.

:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries these rows in full.
"""

#: Lags for the Lo-MacKinlay variance ratio. Powers of two spanning two hours to
#: two thirds of a day, which brackets the horizons this study forecasts.
VR_LAGS: Final[tuple[int, ...]] = (2, 4, 8, 16)

#: Smallest R/S block. Below ~16 points the rescaled range is dominated by its
#: own small-sample bias and the log-log slope bends upward on white noise.
HURST_MIN_N: Final = 16

#: Blocks needed for the log-log regression to mean anything. Two points define a
#: line exactly and would report a slope with no residual to doubt it.
HURST_MIN_BLOCK_SIZES: Final = 3

#: "c" admits a non-zero drift in the random walk, which BTC plainly has. Passed
#: explicitly rather than left to the library default: root §16 forbids a magic
#: number, and a silent default is worse than one -- it is a magic number nobody
#: can see.
VR_TREND: Final = "c"


<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae58288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧾 Baris hasil</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Bentuk keluaran untuk VR dan ADF.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class VarianceRatioRow:
    """One Lo-MacKinlay variance ratio. ``vr`` near 1 is consistent with a random walk."""

    lag: int
    vr: float
    statistic: float
    p_value: float


@dataclass(frozen=True, slots=True)
class ADFRow:
    """Augmented Dickey-Fuller on log-returns, which should reject a unit root."""

    statistic: float
    p_value: float
    used_lag: int
    n_obs: int


<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae58288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🌊 Eksponen Hurst (R/S)</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">H ≈ 0,5 berarti tanpa memori panjang. Terukur 0,5515 full sample.</p>
</div>

In [ ]:


def hurst_rs(x: np.ndarray, min_n: int = HURST_MIN_N, max_n: int | None = None) -> float:
    """Hurst exponent by rescaled range, read off the log-log plot as an OLS slope.

    ``H ~ 0.5`` on log-returns is the no-long-memory reading §4.5 pins. Block
    sizes are dyadic; at each size the series is cut into non-overlapping blocks
    and the mean R/S over them is the point that enters the regression.

    Args:
        x: The series to measure. Table 2 passes **log-returns**; passing a level
            series is the control that shows the estimator responds to memory at
            all, and returns ``H`` near 1.
        min_n: Smallest block. See :data:`HURST_MIN_N`.
        max_n: Largest block, defaulting to ``len(x) // 4`` so the largest size
            still averages over four blocks rather than reporting one range.

    Returns:
        The estimated Hurst exponent.

    Raises:
        ValueError: If fewer than :data:`HURST_MIN_BLOCK_SIZES` usable block
            sizes fit inside the series.
    """
    x = np.asarray(x, dtype=np.float64)
    n_total = len(x)
    if max_n is None:
        max_n = n_total // 4
    sizes = [n for n in (min_n * 2**i for i in range(64)) if n <= max_n]
    if len(sizes) < HURST_MIN_BLOCK_SIZES:
        raise ValueError(
            f"R/S needs at least {HURST_MIN_BLOCK_SIZES} block sizes between "
            f"{min_n} and {max_n}; got {len(sizes)} at n={n_total}"
        )

    logs_n: list[float] = []
    logs_rs: list[float] = []
    for n in sizes:
        blocks = x[: (n_total // n) * n].reshape(-1, n)
        deviate = np.cumsum(blocks - blocks.mean(axis=1, keepdims=True), axis=1)
        spread = deviate.max(axis=1) - deviate.min(axis=1)
        sd = blocks.std(axis=1, ddof=1)
        # A constant block has no scale to rescale by. Dropping it is not
        # imputation -- nothing is invented, the block simply carries no R/S.
        keep = sd > 0
        if not keep.any():
            continue
        logs_n.append(float(np.log(n)))
        logs_rs.append(float(np.log(float((spread[keep] / sd[keep]).mean()))))

    if len(logs_n) < HURST_MIN_BLOCK_SIZES:
        raise ValueError(
            f"only {len(logs_n)} block sizes carried a non-zero standard deviation"
        )
    slope, _ = np.polyfit(np.asarray(logs_n), np.asarray(logs_rs), 1)
    return float(slope)


<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae58288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📏 Variance ratio Lo–MacKinlay</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">VR ≈ 1 konsisten dengan random walk.</p>
</div>

In [ ]:


def variance_ratios(
    r: np.ndarray, lags: tuple[int, ...] = VR_LAGS
) -> list[VarianceRatioRow]:
    """Lo-MacKinlay variance ratio at each lag, from a **return** series.

    ``arch.unitroot.VarianceRatio`` consumes a **level** series and differences
    it itself, so the returns are cumulated back to a level here rather than
    handed over raw. Fed the returns directly it reports ``VR = 1/lag`` -- 0.49,
    0.25, 0.12, 0.06 at lags 2, 4, 8, 16 on white noise -- the signature of
    over-differencing, and a p-value of 0.0000 that would read as decisive
    evidence against a random walk while being evidence of nothing at all. The
    unit tests pin both directions.

    The cumulation is safe across gaps and does not smuggle one in. ``r`` is
    computed **per segment** with each segment's first bar dropped (root §4.3),
    so the series contains no cross-gap return; the cumulative sum merely glues
    the segments into a pseudo-level whose first differences are exactly the
    returns the estimator then recovers. No value is fabricated at a boundary,
    which is what §2's no-imputation rule is about.

    Args:
        r: Log-returns.
        lags: Multi-period horizons for the numerator variance.

    Returns:
        One row per lag, in the order given.
    """
    from arch.unitroot import VarianceRatio  # root §16's named stats boundary

    level = np.cumsum(np.asarray(r, dtype=np.float64))
    rows: list[VarianceRatioRow] = []
    for lag in lags:
        ratio = VarianceRatio(level, lags=lag, trend=VR_TREND, overlap=True)
        rows.append(
            VarianceRatioRow(
                lag=lag,
                vr=float(ratio.vr),
                statistic=float(ratio.stat),
                p_value=float(ratio.pvalue),
            )
        )
    return rows


<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae58288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧪 ADF</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Log-return stasioner. Terukur −38,43.</p>
</div>

In [ ]:


def adf(r: np.ndarray) -> ADFRow:
    """Augmented Dickey-Fuller with AIC lag selection, on log-returns."""
    from statsmodels.tsa.stattools import adfuller  # root §16's named stats boundary

    stat, p_value, used_lag, n_obs, *_ = adfuller(
        np.asarray(r, dtype=np.float64), autolag="AIC"
    )
    return ADFRow(
        statistic=float(stat),
        p_value=float(p_value),
        used_lag=int(used_lag),
        n_obs=int(n_obs),
    )


<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae58288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel efisiensi</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Full sample <strong>dan</strong> tiap sub-blok latih — klaimnya soal <em>variasi</em>, dan satu baris tidak bisa menunjukkannya.</p>
</div>

In [ ]:


def _row(span: str, r: np.ndarray) -> dict[str, float | str | int]:
    """One Table 2 row: ADF, Hurst, and the variance ratio at every lag."""
    unit_root = adf(r)
    row: dict[str, float | str | int] = {
        "span": span,
        "n": int(len(r)),
        "adf_stat": unit_root.statistic,
        "adf_p": unit_root.p_value,
        "hurst": hurst_rs(r),
    }
    for ratio in variance_ratios(r):
        row[f"vr_{ratio.lag}"] = ratio.vr
        row[f"vr_p_{ratio.lag}"] = ratio.p_value
    return row


def _training_returns(features: pl.DataFrame, origin: OriginLike) -> np.ndarray:
    """Log-returns of one origin's 21-month training sub-block.

    ``[train_start, train_sub_end)``, half-open, matching
    :func:`itransformer_btc.keff._training_windows` exactly -- the two must cut
    the same span or Table 2 and Table 2b describe different data.
    """
    lo = int(origin.train_start.timestamp() * 1000)
    hi = int(origin.train_sub_end.timestamp() * 1000)
    return (
        features.filter((pl.col("ts_ms") >= lo) & (pl.col("ts_ms") < hi))
        .get_column("r")
        .to_numpy()
    )


#: Shortest sub-block the R/S regression can describe: enough rows for
#: :data:`HURST_MIN_BLOCK_SIZES` dyadic sizes, each averaged over four blocks.
MIN_SPAN_ROWS: Final = HURST_MIN_N * 2 ** (HURST_MIN_BLOCK_SIZES - 1) * 4


def efficiency_table(
    features: pl.DataFrame, origins: list[OriginLike] | None = None
) -> pl.DataFrame:
    """Table 2 -- one row for the whole sample, one per origin's training sub-block.

    Args:
        features: The frame :func:`itransformer_btc.features.build_features`
            returns, carrying ``ts_ms`` and ``r``.
        origins: Defaults to the full walk-forward grid.

    Returns:
        ``span, n, adf_stat, adf_p, hurst`` plus a ``vr_{lag}`` / ``vr_p_{lag}``
        pair per lag. The ``"full"`` row comes first; origin rows follow in
        walk-forward order.

    An origin whose sub-block is shorter than :data:`MIN_SPAN_ROWS` is skipped.
    On the real artifact that never happens -- the shortest sub-block holds
    roughly 15,000 bars -- so the branch exists for synthetic and truncated
    frames, and a caller can tell it fired by comparing the row count against
    ``len(origins) + 1``.
    """
    grid = list(origins if origins is not None else ORIGINS)
    rows = [_row("full", features.get_column("r").to_numpy())]
    for origin in grid:
        returns = _training_returns(features, origin)
        if len(returns) < MIN_SPAN_ROWS:
            continue
        rows.append(_row(origin.label, returns))
    return pl.DataFrame(rows)


<div style="background: linear-gradient(135deg, #001233, #001845); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #4cc9f033;">
  <h2 style="color: #8ecae6; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🪟 6 &middot; Split walk-forward & scaler
  </h2>
  <p style="color: #a9d6e5; margin: 0; font-size: 1.02em;">Purge H langkah di kedua batas, dan StandardScaler yang dipasang hanya pada sub-blok latih 21 bulan.</p>
  <ul style="color: #a9d6e5; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Dua batas, bukan satu</strong> (<code>D24</code>). Tanpa purge latih&rarr;validasi, jendela latih yang targetnya menembus batas 21 bulan membawa observasi validasi ke dalam latihan — dan validasi itulah yang memutuskan early stopping dan &alpha; ridge, jadi split yang terkontaminasi adalah yang mengatur <em>pemilihan model</em>.</li>
    <li style="margin-bottom: 6px;">Asimetrinya disengaja: <em>input</em> 96 bar jendela validasi boleh menjangkau mundur ke periode latih. Itu informasi masa lalu yang sah bagi peramal di saat itu, dan memblokirnya membuat evaluasi pesimistis secara tidak realistis. Hanya <strong>target</strong> yang dipurge.</li>
    <li style="margin-bottom: 6px;">Memindahkan <code>train_end</code> adalah kebocoran, bukan ketidakcocokan.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 4px solid #4cc9f0; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #8ecae6; margin: 0 0 6px 0; font-size: 1.22em;">📘 splits.py</h3>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.94em;">Purge H langkah di <strong>kedua</strong> batas, dan scaler yang dipasang hanya pada sub-blok 21 bulan (<code>D24</code>).</p>
</div>

<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 3px solid #4cc9f088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & semantik</h4>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.9em;">Semantik latih lawan uji — yang membedakan 601 dari 720.</p>
</div>

In [ ]:
"""Per-origin splits, the scaler, and the tensors the training loop slices.

Three things live here because they are one decision: which windows exist, what
standardises them, and how they reach the device.

**Window semantics differ by split, and the difference is 119 windows (`D51`).**
A *training* or *validation* window must lie wholly inside its span — its H-step
target may not cross the boundary, which is the purge at both boundaries (root
§8.2 / `D24`). A *test* window may not: root §8.3 states that its 96-bar
lookback reaching back across the boundary is past information legitimately
available to a forecaster, and that blocking it would make the evaluation
unrealistically pessimistic. Every hour of a test block is an admissible
forecast origin.

**The scaler is fitted on the 21-month sub-block and nothing else**, at every
origin. Moving ``train_end`` is a leak, not a mismatch (root §8.2).

Upstream
--------
**Both the evaluation protocol and the scaler are written here. Neither is a
library call, and the scaler in particular is not scikit-learn's.**

- Rolling-origin evaluation -- L. J. Tashman, "Out-of-sample tests of
  forecasting accuracy: An analysis and review," *Int. J. Forecast.*, vol. 16,
  no. 4, pp. 437-450, 2000; C. Bergmeir and J. M. Benitez, "On the use of
  cross-validation for time series predictor evaluation," *Information
  Sciences*, vol. 191, pp. 192-213, 2012 -- the primary justification for
  rolling-origin over a fixed origin.
- Purging -- M. Lopez de Prado, *Advances in Financial Machine Learning*.
  Hoboken, NJ: Wiley, 2018, ch. 7. Adopted; **embargo and CPCV deliberately are
  not**, and root §8.3 and §8.4 carry the written arguments rather than leaving
  a protocol element silently absent (`D15`). The purge runs at **both**
  boundaries -- train/validation as well as train/test -- which the source is
  not read as requiring and which `D24` shows is the one that governs model
  selection.
- ``Scaler`` -- **not** ``sklearn.preprocessing.StandardScaler``, though it
  computes the same thing. Under ``use_norm=True`` the outer affine scaler
  cancels algebraically (root §6.3), so what it actually controls is the
  reporting scale and the baselines that have no internal normalisation;
  writing it here keeps that fitted object inside the per-origin tensor build
  rather than taking a dependency for two lines of arithmetic.

:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries these rows in full.
"""

Semantics = Literal["contained", "origin"]


<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 3px solid #4cc9f088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪟 Start jendela per semantik</h4>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.9em;">Enumerasi hingga <code>val_start − L − H</code>: purge H langkah di batas latih→validasi (<code>D24</code>).</p>
</div>

In [ ]:


def window_starts(
    ts: np.ndarray,
    start: datetime,
    end: datetime,
    semantics: Semantics,
    span: int = WINDOW_SPAN,
) -> np.ndarray:
    """Absolute row indices of valid window starts in ``[start, end)``.

    Args:
        ts: Epoch-ms timestamps of the full feature frame, ascending.
        start: Inclusive lower bound.
        end: Exclusive upper bound.
        semantics: ``"contained"`` requires the whole window inside the span —
            training and validation, where the target may not cross the
            boundary. ``"origin"`` requires only the *start* inside — test
            blocks, where the lookback may reach back across it.
        span: ``L + H``.

    Returns:
        Row indices, ascending; empty if the span admits no window.
    """
    lo = int(start.timestamp() * 1000)
    hi = int(end.timestamp() * 1000)

    first = np.arange(len(ts) - span + 1)
    if len(first) == 0:
        return np.empty(0, dtype=np.int64)

    # Contiguity: the window covers `span` consecutive hours with no break.
    contiguous = (ts[first + span - 1] - ts[first]) == (span - 1) * HOUR_MS

    if semantics == "contained":
        inside = (ts[first] >= lo) & (ts[first + span - 1] < hi)
    else:
        inside = (ts[first] >= lo) & (ts[first] < hi)

    return first[contiguous & inside]


<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 3px solid #4cc9f088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⚖️ Scaler</h4>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.9em;">StandardScaler dipasang <strong>hanya</strong> pada sub-blok 21 bulan. Memindahkan <code>train_end</code> adalah kebocoran, bukan ketidakcocokan.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class Scaler:
    """Per-channel standardiser fitted on the training sub-block only.

    Root §6.3: the outer affine scaler **cancels algebraically** under instance
    normalisation, because ``(z - m)/s`` recovers ``(x - mean_t)/std_t`` with
    ``mu_g`` and ``sigma_g`` dropping out. What it still controls is the
    *reporting scale* of every metric, and learning for the baselines that have
    no internal normalisation. StandardScaler is chosen for literature
    comparability, inertness under ``use_norm=True``, and cross-model
    consistency — not because it changes what the transformer learns.
    """

    mean: np.ndarray
    std: np.ndarray
    columns: tuple[str, ...]

    @classmethod
    def fit(cls, values: np.ndarray, columns: list[str]) -> "Scaler":
        std = values.std(axis=0, ddof=0)
        if not np.all(np.isfinite(std)) or np.any(std <= 0):
            raise ValueError(
                f"degenerate channel std in the training sub-block: "
                f"{dict(zip(columns, std))}"
            )
        return cls(values.mean(axis=0), std, tuple(columns))

    def transform(self, values: np.ndarray) -> np.ndarray:
        return (values - self.mean) / self.std

    @property
    def target_mu_over_sigma(self) -> float:
        """``mu_g / sigma_g`` on the target channel — the Naive-RW offset.

        Root §7 / `D31`: a random walk in price implies ``y_raw = 0``, but the
        metrics live on standardised returns, so ``y_z = 0`` would silently mean
        ``r_hat = mu_g``, the training-window mean hourly return — a constant
        drift model wearing the EMH baseline's name. The baseline is mapped as
        ``y_z = -mu_g / sigma_g`` instead, and this value is logged per origin
        so the size of the tilt is auditable.
        """
        return float(self.mean[TARGET_INDEX] / self.std[TARGET_INDEX])


<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 3px solid #4cc9f088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧮 Tensor split</h4>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.9em;">Tensor GPU-resident per split, plus <code>mu_g</code>/<code>sigma_g</code> yang Naive-RW butuh (<code>D31</code>).</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class SplitTensors:
    """One split's inputs, targets, and the timestamps they were issued at.

    ``y`` and ``y_all`` are the same block of future bars read at two widths, and
    both exist because the study now trains two kinds of model (`D56`). Root §6.2
    / `D39` fixes the iTransformer loss on the **target channel only** at every
    rung, so the ladder trains against ``y``. The channel-independent baselines —
    DLinear, PatchTST — carry their published all-channel objective, and that is
    the only thing making their ``K = 8`` label true: a channel-independent
    forecast for one channel depends on that channel's history alone, so
    supervision on all eight through shared weights is the sole route by which
    the other seven reach the target's forecast at all. Trained against ``y``
    they would be K=1 wearing a K=8 label — exactly the collapse `D40` exists to
    prevent.

    ``preds/*.parquet`` holds the target channel for **every** model regardless
    (root §10.4), so nothing downstream needs to know which width was trained on.
    """

    x: np.ndarray      # (n, L, K) float32, standardised
    y: np.ndarray      # (n, H)    float32, standardised target channel
    y_all: np.ndarray  # (n, H, K) float32, every channel's H-step target
    ts: np.ndarray     # (n,)      int64, window start — for traceability

    def __len__(self) -> int:
        return len(self.ts)


@dataclass(frozen=True, slots=True)
class OriginTensors:
    """Everything one training run consumes, already standardised."""

    origin: OriginLike
    k: int
    scaler: Scaler
    train: SplitTensors
    val: SplitTensors
    test_blocks: tuple[SplitTensors, ...]
    #: One-indexed block label per entry of ``test_blocks``. ``(1,…,6)`` for a
    #: normal origin, ``(4, 5, 6)`` for the falsification arm — which is why the
    #: label is stored rather than recovered from position.
    block_labels: tuple[int, ...]

    @property
    def naive_rw_z(self) -> float:
        """The Naive-RW prediction in scaler space (`D31`)."""
        return -self.scaler.target_mu_over_sigma


<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 3px solid #4cc9f088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #8ecae6; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🏗️ Merakit tensor origin</h4>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.9em;">Satu origin, tiga split, tanpa <code>DataLoader</code> di mana pun (root §10.3).</p>
</div>

In [ ]:


def _gather(
    values: np.ndarray, starts: np.ndarray, ts: np.ndarray, seq_len: int, pred_len: int
) -> SplitTensors:
    """Slice windows out of a standardised array by index arithmetic.

    No ``Dataset``, no ``DataLoader``, no per-item Python. Root §10.3: at ~280k
    parameters the run is dominated by data movement and interpreter overhead,
    which a per-item loader maximises — the naive path costs roughly 10x and
    puts the grid outside the weekly GPU quota outright.
    """
    if len(starts) == 0:
        return SplitTensors(
            x=np.empty((0, seq_len, values.shape[1]), np.float32),
            y=np.empty((0, pred_len), np.float32),
            y_all=np.empty((0, pred_len, values.shape[1]), np.float32),
            ts=np.empty(0, np.int64),
        )
    rows = starts[:, None] + np.arange(seq_len)[None, :]
    tgt = starts[:, None] + seq_len + np.arange(pred_len)[None, :]
    targets = values[tgt].astype(np.float32, copy=False)
    return SplitTensors(
        x=values[rows].astype(np.float32, copy=False),
        # Copied out rather than left as a strided view of `targets`: `y` is what
        # the whole pipeline reads, and a non-contiguous array of it would make
        # every downstream `from_numpy` and `reshape` behave differently for a
        # reason nobody would think to look for.
        y=np.ascontiguousarray(targets[:, :, TARGET_INDEX]),
        y_all=targets,
        ts=ts[starts],
    )


def build_origin_tensors(
    features: pl.DataFrame,
    origin: OriginLike,
    k: int,
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
    columns: tuple[str, ...] | None = None,
) -> OriginTensors:
    """Build every split for one (origin, K) cell.

    The scaler is fitted on the **rows** of the 21-month sub-block, before any
    window is cut, and then applied to every split. Fitting it on validation or
    test rows is the leak root §11 calls fatal.

    Raises:
        ValueError: If the training split is empty, or if the last training
            window's target reaches at or past ``val_start`` — the purge
            assertion (`D24`), checked here rather than trusted.
    """
    # Named columns override the rung, so a matched-K arm can hold K fixed and
    # move only the effective rank (`D70`). ``k`` still has to agree with the set:
    # it is what the model is built with, and a mismatch would be silent.
    columns = tuple(columns) if columns else ladder_columns(k)
    if len(columns) != k:
        raise ValueError(f"{len(columns)} columns for K={k}")
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()
    span = seq_len + pred_len

    train_idx = window_starts(ts, origin.train_start, origin.train_sub_end,
                              "contained", span)
    val_idx = window_starts(ts, origin.val_start, origin.val_end, "contained", span)
    if len(train_idx) == 0:
        raise ValueError(f"origin {origin.label}: empty training split")

    last_train_target = ts[train_idx[-1] + span - 1]
    if last_train_target >= int(origin.val_start.timestamp() * 1000):
        raise ValueError(
            f"origin {origin.label}: a training target reaches into validation "
            f"({last_train_target}); the purge did not hold"
        )

    scaler = Scaler.fit(values[train_idx[0] : train_idx[-1] + span], columns)
    scaled = scaler.transform(values)

    blocks = origin.blocks()
    return OriginTensors(
        origin=origin,
        k=k,
        scaler=scaler,
        train=_gather(scaled, train_idx, ts, seq_len, pred_len),
        val=_gather(scaled, val_idx, ts, seq_len, pred_len),
        test_blocks=tuple(
            _gather(scaled, window_starts(ts, lo, hi, "origin", span),
                    ts, seq_len, pred_len)
            for _, lo, hi in blocks
        ),
        block_labels=tuple(label for label, _, _ in blocks),
    )


<div style="background: linear-gradient(135deg, #150029, #240046); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #9d4edd33;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📐 7 &middot; Dimensionalitas efektif — Stage 3b
  </h2>
  <p style="color: #c8a2e0; margin: 0; font-size: 1.02em;">K_eff adalah variabel bebas RQ1 dan ia diukur sebelum satu epoch pun berjalan.</p>
  <ul style="color: #c8a2e0; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Per origin, pada sub-blok latih 21 bulan origin itu sendiri</strong> (<code>D44</code>). PR full-sample akan diestimasi dari data yang sama dengan outcome-nya, membuat RQ1 sebagian sirkular — satu-satunya jalur kebocoran yang lolos dari tiap butir checklist, karena root &sect;11 hanya mengaudit gerbangnya.</li>
    <li style="margin-bottom: 6px;"><strong>Gerbang membaca rentang pra-origin-pertama saja</strong> (<code>D02</code>), pemicu dipra-registrasi di PR &lt; 5,0. Aksinya <em>disklosur, bukan re-cut</em> (<code>D48</code>): <code>D01</code> tidak menyisakan potongan konsisten kedua atas F1&ndash;F5.</li>
    <li style="margin-bottom: 6px;">Dilaporkan juga pada fitur <strong>ternormalisasi-jendela</strong> (<code>D04</code>) — <code>use_norm</code> mengupas <em>level</em> volatilitas, jadi rung 8&rarr;12 bisa mendatar karena sebab yang tak ada hubungannya dengan redundansi.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 4px solid #9d4edd; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">📘 keff.py</h3>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.94em;">Dimensionalitas efektif — regresor RQ1. Dihitung hanya pada rentang latih, per origin (<code>D02</code>, <code>D44</code>).</p>
</div>

<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;">Lantai gerbang PR = 5,0, dipra-registrasi sebelum apa pun berjalan (<code>D02</code>).</p>
</div>

In [ ]:
"""Effective dimensionality — RQ1's independent variable, measured before training.

Root §5.4. These statistics run **before any model does**, and two of the study's
three claimed contributions rest on them: the separation of nominal K from
effective dimensionality, and the K-versus-K_eff horse race in §9.1.

Three constraints shape every function here, and each closes a leak the earlier
design left open:

* **`D44` — span.** Every reported K_eff declares its span, and the one that
  feeds RQ1's regression is computed **per origin on that origin's own 21-month
  training sub-block**. Nothing previously forbade computing it over 2018-2026,
  a span containing every origin's test blocks; the regressor would then be
  estimated on the same data as the outcome and RQ1's claim would be partly
  circular, while §11's fatal checklist item still passed because it audits only
  the *gate*. That was the one leakage path surviving every checklist item by
  construction.
* **`D44` — construct validity.** PR on the K x K *contemporaneous* correlation
  matrix is blind to cross-lag structure, while the model consumes a K x 96
  block and embeds each variate's entire lookback. Two variates can be
  near-uncorrelated contemporaneously yet near-redundant to a model with a
  96-hour lookback. A lookback-aware measure is therefore reported on the same
  rungs, and the divergence between them is reported whatever it turns out to
  be. If the construct does not correspond to what the architecture consumes,
  the second contribution is a measurement-validity failure rather than a
  finding — which is what a methods referee will spend the review on.
* **`D04` — the instance-normalisation confound.** ``use_norm=True`` divides each
  window by its own per-variate sigma over L, so the F2 estimators contribute
  *shape*, not *level*. The 8->12 rung can flatten for a reason that has nothing
  to do with redundancy. PR is therefore measured on window-normalised features
  as well as raw, both reported, and the confound disclosed in Limitations
  whatever RQ1 returns.

Upstream
--------
**Written here on numpy; the participation ratio has no reference
implementation to vendor.** It is a random-matrix statistic, not a library
function, and it enters this study as RQ1's independent variable rather than as
a diagnostic -- which is why its span and its matrix are pinned rather than
left to a default.

- L. Laloux, P. Cizeau, J.-P. Bouchaud, and M. Potters, "Noise dressing of
  financial correlation matrices," *Phys. Rev. Lett.*, vol. 83, no. 7,
  pp. 1467-1470, 1999.
- V. Plerou, P. Gopikrishnan, B. Rosenow, L. A. N. Amaral, T. Guhr, and
  H. E. Stanley, "Random matrix approach to cross correlations in financial
  data," *Phys. Rev. E*, vol. 65, no. 6, 066126, 2002.

Two departures from the naive reading, both load-bearing. PR is taken on the
**correlation** matrix and never the covariance one: the covariance spectrum is
not monotone in K, and its ordering is a statement about units, since
``log_quote_volume``'s deviations sit two orders of magnitude above ``r``'s
(`D53a`, `D53b`, `D81`). And every reported ``K_eff`` -- the RQ1 regressor
included -- is measured on a **training-only** span, per origin, so the
regressor never reads the test period (`D02`, `D44`).
:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries this row in full.
"""

#: Root §8.5's Stage 3b trigger, pre-registered numerically **before** measuring:
#: a gate without a number stated in advance is not a gate (`D02`).
GATE_PR_FLOOR: float = 5.0

#: Windows sampled for the lookback-aware measures. The stable rank is a per
#: window SVD, so the cost is linear in this, and the sample is drawn by a fixed
#: stride — deterministic, not random, so the number is regenerable under root
#: §12 without carrying a seed.
LOOKBACK_SAMPLE: int = 2_000


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📐 Participation ratio</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;"><code>PR = (Σλ)² / Σλ²</code> pada matriks korelasi. Diukur juga pada fitur ternormalisasi-jendela — confound instance-norm <code>D04</code>.</p>
</div>

In [ ]:


def participation_ratio(eigenvalues: np.ndarray) -> float:
    """``PR = (sum lambda)^2 / sum lambda^2``, bounded in ``[1, K]``.

    PR is 1 when one eigenvalue carries everything and K when the spectrum is
    flat, so it reads directly as "how many independent directions are actually
    here". Negative eigenvalues from floating-point noise on a
    positive-semidefinite matrix are clipped to zero rather than dropped, since
    dropping them would change the trace and so the numerator.

    Raises:
        ValueError: If the spectrum sums to zero — a degenerate block that no
            downstream number could be computed from.
    """
    lam = np.clip(np.asarray(eigenvalues, dtype=np.float64), 0.0, None)
    total = lam.sum()
    if total <= 0:
        raise ValueError("degenerate spectrum: eigenvalues sum to zero")
    return float(total * total / np.square(lam).sum())


def contemporaneous_pr(values: np.ndarray) -> float:
    """PR of the ``K x K`` correlation matrix — Table 2b's first column.

    Args:
        values: ``(n, K)`` observations, one row per bar.

    The correlation matrix rather than the covariance, so the statistic is
    invariant to the arbitrary units the twelve variates carry: log-returns,
    log-volumes and a bounded ratio do not share a scale, and on the covariance
    the log-volume channel would dominate the spectrum for that reason alone.
    """
    corr = np.atleast_2d(
        np.corrcoef(np.asarray(values, dtype=np.float64), rowvar=False)
    )
    return participation_ratio(np.linalg.eigvalsh(corr))


def window_normalised_pr(windows: np.ndarray) -> float:
    """PR after per-window standardisation over L — `D04`'s required companion.

    Args:
        windows: ``(n, L, K)``.

    This reproduces exactly what ``use_norm=True`` hands the embedding: each
    channel of each window centred and divided by its own sigma over the
    lookback. Level information is gone by construction, so if the raw and
    normalised PR disagree at the K=12 rung, the F2 estimators' apparent
    redundancy is an artefact of the normalisation rather than a property of the
    data — and RQ1's axis is confounded in a way no post-hoc analysis removes.
    """
    x = np.asarray(windows, dtype=np.float64)
    mean = x.mean(axis=1, keepdims=True)
    std = np.sqrt(x.var(axis=1, keepdims=True) + 1e-12)
    return contemporaneous_pr(((x - mean) / std).reshape(-1, x.shape[2]))


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📊 Stable rank & spektrum sadar-lookback</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;">Standardisasi dalam jendela lebih dulu; tanpa itu satu baris mendominasi kedua norma dan angkanya jadi artefak satuan (<code>D53a</code>, <code>D53b</code>).</p>
</div>

In [ ]:


def stable_rank(matrix: np.ndarray) -> float:
    """``||M||_F^2 / ||M||_2^2`` — bounded in ``[1, min(rows, cols)]``.

    The lookback-aware measure that is *commensurable* with the contemporaneous
    PR: applied to a ``K x L`` block with ``K <= 12 < 96`` it lives in the same
    ``[1, K]`` interval, so the two sit in one table and can be compared rung by
    rung. The ``K*L x K*L`` covariance PR below cannot — its ceiling is ``K*L``,
    which is 1,152 at K=12.
    """
    singular = np.linalg.svd(np.asarray(matrix, dtype=np.float64), compute_uv=False)
    if singular[0] <= 0:
        raise ValueError("degenerate window block: largest singular value is 0")
    return float(np.square(singular).sum() / (singular[0] ** 2))


def lookback_stable_rank(windows: np.ndarray, sample: int = LOOKBACK_SAMPLE) -> float:
    """Mean stable rank of each window's ``K x L`` block, **as the model sees it**.

    Args:
        windows: ``(n, L, K)``.
        sample: Windows to evaluate, taken by a fixed stride across the whole
            span so the sample spreads over the sub-block rather than
            concentrating at its head.

    Each channel is standardised **within its window** first — exactly what
    ``use_norm=True`` does before the embedding. Centring alone is not enough and
    the difference is not cosmetic: measured on origin 1, the merely-centred
    version returns 1.00 / 1.00 / 1.16 / 1.65 across the four rungs, because
    ``log_quote_volume`` deviations are two orders of magnitude larger than
    ``r`` deviations in absolute terms, so one row dominates both the Frobenius
    and the spectral norm and the statistic reports "one effective direction" at
    every rung. That is a units artefact, not a finding about the data.

    With standardisation the quantity has a closed form worth stating: the block
    is ``K x L`` with unit-variance rows, so ``||M||_F^2 = K L`` and
    ``sigma_1^2 = L lambda_1`` where ``lambda_1`` is the leading eigenvalue of
    the **within-window** correlation matrix of the K channels. The stable rank
    is therefore ``K / lambda_1`` — the reciprocal of the dominant direction's
    share, bounded in ``[1, K]`` and directly comparable to the contemporaneous
    PR beside it.
    """
    x = np.asarray(windows, dtype=np.float64)
    if len(x) == 0:
        raise ValueError("no windows to measure")
    stride = max(1, len(x) // sample)
    blocks = np.transpose(x[::stride][:sample], (0, 2, 1))   # (m, K, L)
    blocks = blocks - blocks.mean(axis=2, keepdims=True)
    blocks = blocks / np.sqrt(np.square(blocks).mean(axis=2, keepdims=True) + 1e-12)
    return float(np.mean([stable_rank(b) for b in blocks]))


def lookback_correlation_pr(windows: np.ndarray) -> float:
    """PR of the ``K*L x K*L`` **correlation** spectrum — §5.4's first alternative.

    The correlation matrix, not the covariance §5.4 names literally. On the raw
    covariance the statistic is dominated by whichever channel happens to carry
    the largest variance, and it stops being monotone in K: measured on origin 1
    it returned 92.1 / 3.0 / 44.0 / 8.8 across the four rungs, where the drop
    from K=1 to K=4 is entirely the arrival of ``log_quote_volume`` and says
    nothing about dimensionality. Standardising the ``K*L`` columns first makes
    it scale-free, exactly as :func:`contemporaneous_pr` uses the correlation
    matrix for the same reason.

    This is the only measure here that sees genuine **cross-lag** structure — the
    stable rank above sees cross-*variate* structure inside a window. Its ceiling
    is ``K*L``, so it is **not** on the contemporaneous PR's scale; report it as
    a fraction of that ceiling (:attr:`KeffRow.pr_lookback_ratio`) when comparing
    rungs.

    **The name was wrong until `D81`, and the correction is not only cosmetic.**
    This function has computed the correlation spectrum since `D53b`, but it,
    its field and its parquet column were all called ``...covariance...``, so a
    reader checking whether `D53b`'s fix had landed found the word it replaced.
    Worse, `D53b` justified the fix by monotonicity in K and the correlation
    spectrum **is not monotone in K either** --- measured 92.1 / 21.9 / 37.3 /
    15.5 across the rungs, against the covariance's 92.1 / 3.0 / 44.0 / 8.8. What
    the change actually buys is scale-freeness, which is a real reason and the
    only one that survives; the ratio to the ``K*L`` ceiling then falls
    0.980 / 0.078 / 0.047 / 0.020, i.e. **anti**-monotone in K while the
    contemporaneous PR rises. The two K_eff constructs are ordinally opposed and
    root §4.1b has to say so (`D44`).
    """
    x = np.asarray(windows, dtype=np.float64)
    flat = x.reshape(len(x), -1)
    flat = flat - flat.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.square(flat).mean(axis=0, keepdims=True) + 1e-24)
    flat = flat / scale
    gram = (flat.T @ flat) / max(1, len(flat) - 1)
    return participation_ratio(np.linalg.eigvalsh(gram))


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧾 Baris K_eff per origin</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;">Regresor RQ1, dihitung <strong>hanya pada sub-blok latih 21 bulan</strong> origin itu (<code>D44</code>).</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class KeffRow:
    """One (origin, rung) cell of Table 2b."""

    origin: str
    origin_index: int
    k: int
    n_rows: int
    n_windows: int
    pr_raw: float
    pr_window_norm: float
    stable_rank_lookback: float
    pr_lookback_corr: float

    @property
    def pr_lookback_ratio(self) -> float:
        """``pr_lookback_corr / (K * L)`` — the cross-lag PR as a share of its ceiling.

        The raw value lives in ``[1, K*L]`` and so cannot be compared rung to
        rung; this can.
        """
        return self.pr_lookback_corr / (self.k * SEQ_LEN)

    @property
    def divergence(self) -> float:
        """``stable_rank - pr_raw`` — §5.4 requires this be reported as such.

        Positive means the lookback carries structure the contemporaneous
        correlation cannot see; negative means variates that look independent
        bar-to-bar turn redundant once 96 hours of each are in view. Either way
        it goes in §4.1b: if the effective-dimensionality construct does not
        correspond to what the architecture consumes, the study's second claimed
        contribution is a measurement-validity failure rather than a finding.
        """
        return self.stable_rank_lookback - self.pr_raw


def _training_windows(
    features: pl.DataFrame, origin: OriginLike, k: int, seq_len: int = SEQ_LEN
) -> tuple[np.ndarray, np.ndarray]:
    """Rows and windows of one origin's 21-month training sub-block.

    Both are cut from ``[train_start, train_sub_end)`` and nothing else — the
    span the scaler is fitted on. This is `D44`'s closure: RQ1's regressor may
    not see a single bar its outcome is measured on.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()

    lo = int(origin.train_start.timestamp() * 1000)
    hi = int(origin.train_sub_end.timestamp() * 1000)
    rows = values[(ts >= lo) & (ts < hi)]

    starts = window_starts(
        ts, origin.train_start, origin.train_sub_end, "contained", WINDOW_SPAN
    )
    if len(starts) == 0:
        raise ValueError(f"origin {origin.label}: no training window to measure")
    idx = starts[:, None] + np.arange(seq_len)[None, :]
    return rows, values[idx]


def keff_row(features: pl.DataFrame, origin: OriginLike, k: int) -> KeffRow:
    """Measure every K_eff variant for one (origin, rung) cell."""
    rows, windows = _training_windows(features, origin, k)
    return KeffRow(
        origin=origin.label,
        origin_index=origin.index,
        k=k,
        n_rows=len(rows),
        n_windows=len(windows),
        pr_raw=contemporaneous_pr(rows),
        pr_window_norm=window_normalised_pr(windows),
        stable_rank_lookback=lookback_stable_rank(windows),
        pr_lookback_corr=lookback_correlation_pr(windows),
    )


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔗 Tabel K_eff & korelasi</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;"><code>corr(K, K_eff) = 0,828</code>, bukan ≈0,97 yang diantisipasi — pacuannya jadi <em>lebih</em> teridentifikasi.</p>
</div>

In [ ]:


def keff_table(
    features: pl.DataFrame,
    origins: list[OriginLike] | None = None,
    rungs: tuple[int, ...] = K_LADDER,
) -> pl.DataFrame:
    """Table 2b — every rung at every origin, on training spans only.

    Returns:
        One row per (origin, rung): the raw, window-normalised and two
        lookback-aware measures side by side, plus their divergence.

    K=1 is included even though its PR is identically 1. §9.1's horse race
    regresses on the rung's K_eff, and dropping the rung whose value is known in
    advance would unbalance the panel for no gain.
    """
    grid = list(origins if origins is not None else ORIGINS)
    return pl.DataFrame(
        [
            {
                "origin": row.origin,
                "origin_index": row.origin_index,
                "k": row.k,
                "n_rows": row.n_rows,
                "n_windows": row.n_windows,
                "pr_raw": row.pr_raw,
                "pr_window_norm": row.pr_window_norm,
                "stable_rank_lookback": row.stable_rank_lookback,
                "pr_lookback_corr": row.pr_lookback_corr,
                "pr_lookback_ratio": row.pr_lookback_ratio,
                "divergence": row.divergence,
            }
            for origin in grid
            for k in rungs
            for row in (keff_row(features, origin, k),)
        ]
    )


def corr_k_keff(table: pl.DataFrame, column: str = "pr_raw") -> float:
    """``corr(K, K_eff)`` across the rungs — §9.1 requires it in Table 2b.

    A reader is entitled to this before reading the non-nested comparison: if it
    sits near 1, the two theories are close to collinear and the horse race has
    little to separate them, whatever the reported p-value says.
    """
    means = table.group_by("k").agg(pl.col(column).mean().alias("keff")).sort("k")
    k = means.get_column("k").to_numpy().astype(np.float64)
    keff = means.get_column("keff").to_numpy()
    return float(np.corrcoef(k, keff)[0, 1])


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🚧 Gerbang Stage 3b</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;">PR di K=8 terukur 4,393 &lt; 5,0. Aksinya disklosur, bukan re-cut (<code>D48</code>).</p>
</div>

In [ ]:


def gate_pr(features: pl.DataFrame, k: int = 8) -> float:
    """Stage 3b's gate value — **pre-first-origin span only** (`D02`).

    Computed on ``[2018-01, 2020-01)``, which contains no origin's test block.
    The full-sample rolling PR is descriptive and may inform no design decision:
    every origin's test block lies inside it, so a ladder re-cut driven by it
    would be a design choice made with the answers already in hand.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    lo = int(DATA_START.timestamp() * 1000)
    hi = int(FIRST_ORIGIN.timestamp() * 1000)
    rows = features.select(columns).to_numpy()[(ts >= lo) & (ts < hi)]
    if len(rows) == 0:
        raise ValueError("the pre-first-origin span holds no usable bar")
    return contemporaneous_pr(rows)


def gate_verdict(measured: float, floor: float = GATE_PR_FLOOR) -> str:
    """Stage 3b's action, which is **disclosure, not a re-cut** (`D48`).

    §8.5 originally said a PR below the floor should re-cut the ladder, but
    `D01` establishes that exactly one consistent cut exists over F1-F5, so
    "re-cut" named no reachable alternative. The gate therefore reports, the
    grid proceeds unchanged, and the divergence from §5.2's expected values is
    disclosed in §4.1b. A gate whose only action is unreachable is not a gate.
    """
    if measured >= floor:
        return (
            f"PASS: measured PR at K=8 is {measured:.3f} >= {floor:.1f}. "
            f"Proceed; report the value in Table 2b."
        )
    return (
        f"DISCLOSE: measured PR at K=8 is {measured:.3f} < {floor:.1f}. "
        f"Root §8.5 / `D48` — proceed unchanged and disclose the divergence "
        f"from §5.2's expected K_eff in §4.1b. Do not re-cut the ladder: `D01` "
        f"leaves no second consistent cut over F1-F5."
    )


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🌀 PR bergulir — deskriptif saja</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;">Jendela 90 hari, 2018–2026. <strong>Tidak boleh menginformasikan satu pun keputusan desain</strong> (root §5.4).</p>
</div>

In [ ]:


# -- Figure 2b: the descriptive rolling statistics (root §5.4) ---------------
#
# Both functions below are **descriptive only**, and the constraint is not
# decorative. Every origin's test block lies inside the full sample, so a
# full-sample rolling statistic **may inform no design decision** (`D02`): the
# statistic that *gates* the ladder is :func:`gate_pr` on the pre-first-origin
# span, and substituting one for the other would be the leak §5.4 exists to
# forbid. What these are for is establishing H2's premise --- that the
# microstructure-to-return mapping is regime-specific --- **before a single
# epoch runs**, which is why Figure 2b sits in the paper ahead of every result.

#: Root §5.4's window. Ninety days is long enough for an 8 x 8 correlation to be
#: estimated from ~2,160 bars and short enough to resolve a regime change.
ROLLING_WINDOW_DAYS: Final = 90

#: Step between consecutive windows. One day, so the curve is readable without
#: emitting one point per bar. Deterministic, so root §12 can regenerate it.
ROLLING_STEP_DAYS: Final = 1


def _rolling_spans(
    ts: np.ndarray, window_days: int, step_days: int
) -> list[tuple[int, int, int]]:
    """``(window_end_ms, lo, hi)`` for each window, sliced **by time, not position**.

    Position slicing would silently shorten a window wherever the series has a
    gap, and gaps are not uniform: 26 of 27 downtime blocks fall in 2018-2021 and
    none after 2023-03 (`D45`). A position-sliced "90-day" window would therefore
    span more calendar time early in the sample than late --- a trend in the
    estimator that a reader would read as a trend in the market.
    """
    window_ms = window_days * 24 * HOUR_MS
    step_ms = step_days * 24 * HOUR_MS
    spans: list[tuple[int, int, int]] = []
    for end in range(int(ts[0]) + window_ms, int(ts[-1]) + 1, step_ms):
        lo = int(np.searchsorted(ts, end - window_ms, side="left"))
        hi = int(np.searchsorted(ts, end, side="left"))
        spans.append((end, lo, hi))
    return spans


def rolling_pr(
    features: pl.DataFrame,
    k: int = 8,
    window_days: int = ROLLING_WINDOW_DAYS,
    step_days: int = ROLLING_STEP_DAYS,
) -> pl.DataFrame:
    """Rolling participation ratio over the full sample --- **descriptive only**.

    Args:
        features: The frame :func:`itransformer_btc.features.build_features`
            returns, carrying ``ts_ms`` and the twelve variates.
        k: Rung to measure. Eight is the rung RQ2 contrasts against K=1.

    Returns:
        ``window_end_ms, n_rows, pr`` --- one row per window.

    Never pass this to a regression and never compare it against
    :data:`GATE_PR_FLOOR`. It reads the test period by construction.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()

    rows = []
    for end, lo, hi in _rolling_spans(ts, window_days, step_days):
        block = values[lo:hi]
        if len(block) <= len(columns):
            continue        # a correlation needs more rows than columns
        rows.append(
            {"window_end_ms": end, "n_rows": len(block), "pr": contemporaneous_pr(block)}
        )
    return pl.DataFrame(rows)


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📉 OLS R² bergulir</h4>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.9em;">Premis H2 diuji sebelum satu epoch pun berjalan — Figure 2b.</p>
</div>

In [ ]:


def rolling_ols_r2(
    features: pl.DataFrame,
    k: int = 8,
    window_days: int = ROLLING_WINDOW_DAYS,
    step_days: int = ROLLING_STEP_DAYS,
) -> pl.DataFrame:
    """In-window R^2 of ``r_{t+1} ~ (K features at t)`` --- **descriptive only**.

    Root §5.4: if this is unstable, H2's premise --- that the
    microstructure-to-return mapping is regime-specific --- is established before
    a single epoch runs. In-window rather than out-of-sample, deliberately: the
    question is whether the *relationship* moves, not whether it forecasts, and
    the second question is what the entire rest of the study answers.

    The target is formed **within a segment**: a row whose successor is not
    exactly one hour later is dropped, because ``r`` at the next row would then
    be the first return after an outage and the pair would straddle a gap the
    segment law (§4.3) exists to keep apart.

    Returns:
        ``window_end_ms, n_pairs, r2`` --- one row per window.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()
    target = features.get_column("r").to_numpy()

    contiguous = np.zeros(len(ts), dtype=bool)
    contiguous[:-1] = np.diff(ts) == HOUR_MS

    rows = []
    for end, lo, hi in _rolling_spans(ts, window_days, step_days):
        # Clamped so the last window cannot ask for a successor that does not
        # exist; the mask and both slices then have one length by construction
        # rather than by a length check that fires after an IndexError would.
        n = min(hi, len(ts) - 1) - lo
        if n <= 0:
            continue
        keep = contiguous[lo : lo + n]
        x = values[lo : lo + n][keep]
        y = target[lo + 1 : lo + 1 + n][keep]
        if len(x) <= len(columns) + 1:
            continue
        design = np.column_stack([np.ones(len(x)), x])
        coefficients, *_ = np.linalg.lstsq(design, y, rcond=None)
        residual = y - design @ coefficients
        total = float(((y - y.mean()) ** 2).sum())
        rows.append({
            "window_end_ms": end,
            "n_pairs": len(y),
            "r2": float(1.0 - (residual ** 2).sum() / total) if total > 0 else 0.0,
        })
    return pl.DataFrame(rows)


<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 4px solid #9d4edd; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">📐 Jalankan Stage 3b — ukur K_eff</h3>
  <p style="color: #c8a2e0; margin: 0; font-size: 0.94em;">Gerbang dibaca pada rentang pra-origin-pertama, lalu tabel K_eff diukur per origin pada sub-blok latih 21 bulan masing-masing. Semuanya sebelum satu epoch pun berjalan.</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 8px;"><span style="color:#ffd166; font-weight:600; font-size:0.84em;">MENULIS</span> <span style="color:#dcdcdc; font-size:0.84em;">🧊 <code>artifacts/keff_table.parquet</code></span></div>
</div>

In [46]:
gate = gate_pr(features, k=8)
print(gate_verdict(gate))

t0 = time.perf_counter()
keff_tbl = keff_table(features)          # 15 origins x 4 rungs, training spans only
print(f"\nmeasured in {time.perf_counter() - t0:.0f}s")

rung_view = (
    keff_tbl.group_by("k")
    .agg(
        pl.col("pr_raw").mean().alias("PR_raw"),
        pl.col("pr_raw").std().alias("PR_raw_sd"),
        pl.col("pr_window_norm").mean().alias("PR_windownorm"),
        pl.col("stable_rank_lookback").mean().alias("stable_rank"),
        pl.col("pr_lookback_ratio").mean().alias("crosslag_share"),
        pl.col("divergence").mean().alias("divergence"),
    )
    .sort("k")
)
print(rung_view)
print(f"\ncorr(K, K_eff) = {corr_k_keff(keff_tbl):.4f}")
print("A reader is entitled to that before reading the K-vs-K_eff horse race: near "
      "1 means the two theories are close to collinear and there is little to "
      "separate, whatever the p-value says.")
print("Section 5.2 expected 1 / ~3.5 / ~6.5 / ~7, reasoned from family structure "
      "and not measured. Fix the hypothesis to the measurement, never the reverse.")

keff_tbl.write_parquet(ARTIFACTS / "keff_table.parquet")


DISCLOSE: measured PR at K=8 is 4.393 < 5.0. Root §8.5 / `D48` — proceed unchanged and disclose the divergence from §5.2's expected K_eff in §4.1b. Do not re-cut the ladder: `D01` leaves no second consistent cut over F1-F5.

measured in 35s
shape: (4, 7)
┌─────┬──────────┬───────────┬───────────────┬─────────────┬────────────────┬────────────┐
│ k   ┆ PR_raw   ┆ PR_raw_sd ┆ PR_windownorm ┆ stable_rank ┆ crosslag_share ┆ divergence │
│ --- ┆ ---      ┆ ---       ┆ ---           ┆ ---         ┆ ---            ┆ ---        │
│ i64 ┆ f64      ┆ f64       ┆ f64           ┆ f64         ┆ f64            ┆ f64        │
╞═════╪══════════╪═══════════╪═══════════════╪═════════════╪════════════════╪════════════╡
│ 1   ┆ 1.0      ┆ 0.0       ┆ 1.0           ┆ 1.0         ┆ 0.979891       ┆ 0.0        │
│ 4   ┆ 3.327497 ┆ 0.10339   ┆ 3.316502      ┆ 2.355108    ┆ 0.078479       ┆ -0.972388  │
│ 8   ┆ 4.269219 ┆ 0.116815  ┆ 4.013051      ┆ 2.70075     ┆ 0.04731        ┆ -1.568469  │
│ 12  ┆ 3.983529 

<div style="background: linear-gradient(135deg, #150029, #22003d); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #c77dff33;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🧠 8 &middot; Algoritma — arsitektur iTransformer
  </h2>
  <p style="color: #cbb2e8; margin: 0; font-size: 1.02em;">Encoder-only. Tiap variat adalah satu token, dan attention berjalan lintas variat, bukan lintas waktu.</p>
  <ul style="color: #cbb2e8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Tanpa causal mask.</strong> Masking berlaku pada sumbu waktu; attention di sini berjalan pada sumbu variat, tempat seluruh token sezaman. Kausalitas ditegakkan di hulu — pada fitur dan windowing.</li>
    <li style="margin-bottom: 6px;"><code>d_model = 128</code>, bukan 512: panjang urutan attention adalah N &le; 12, dan 512 akan over-parameterise terhadap ~14.000 sampel (<code>D25</code>).</li>
    <li style="margin-bottom: 6px;"><strong>Tidak ada yang di-tune di sini</strong> (<code>D38</code>). Tiap nilai kecuali <code>d_model</code> diadopsi apa adanya dari Liu et al. (2024) dan identik di setiap rung — menahan kapasitas tetaplah yang membuat rung-rungnya sebanding.</li>
    <li style="margin-bottom: 6px;">K=1 mendegenerasi menjadi <em>kendali</em>, bukan bug (<code>D50</code>): pada N=1 softmax memberi bobot 1, tapi proyeksi value dan output serta residualnya <strong>tetap ada</strong> — ia bukan identitas telanjang.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 4px solid #c77dff; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">📘 model.py</h3>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.94em;">iTransformer encoder-only. Attention berjalan lintas variat, bukan lintas waktu; kausalitas ditegakkan di hulu.</p>
</div>

<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Hanya torch yang dipakai modul ini. TensorFlow, Keras, dan JAX terlarang (root §2).</p>
</div>

In [ ]:
"""Encoder-only iTransformer: each variate is a token, attention runs across them.

Root §6.1. The inversion is the whole point — attention operates over the
**variate** axis, not the time axis, so the sequence the attention sees is
``N <= 12`` long rather than ``L = 96``. That is why ``d_model = 128`` and not
the reference implementation's 512: at ~14,000 training samples per origin, 512
over-parameterises badly, and the usual justification for a wide model — a long
attention sequence — does not apply here (`D25`).

**No causal mask.** Masking applies to the time axis; this attention runs over
the variate axis, where all tokens are contemporaneous. Causality is enforced
upstream, in the features and the windowing.

**K=1 is a designed control, not a degenerate bug** (`D50`). At ``N = 1``,
softmax over a single token returns weight 1, so attention reduces to
``W_O W_V x + x`` — the value and output projections and the residual **remain**;
it is not a bare identity. Parameter count is identical at every rung, because K
changes the token count and not a single weight shape. Say so in the
methodology: unexplained, an examiner reads it as an implementation error.

Upstream
--------
**Reimplemented, not vendored.** No file here is a copy of an upstream file:
the architecture is written from the published description against this study's
own tensor contract. That is a narrower claim than a fork and a stronger one to
defend, because an examiner can check it by reading both sides.

- Y. Liu, T. Hu, H. Zhang, H. Wu, S. Wang, L. Ma, and M. Long, "iTransformer:
  Inverted transformers are effective for time series forecasting," in *Proc.
  12th Int. Conf. Learn. Represent. (ICLR)*, 2024. arXiv:2310.06625. Official
  code: https://github.com/thuml/iTransformer (THUML, Tsinghua University;
  MIT; accessed 2026-09-03).
- T. Kim, J. Kim, Y. Tae, C. Park, J.-H. Choi, and J. Choo, "Reversible
  instance normalization for accurate time-series forecasting against
  distribution shift," in *Proc. 10th Int. Conf. Learn. Represent. (ICLR)*,
  2022 — the operation ``use_norm=True`` performs. Official code:
  https://github.com/ts-kim/RevIN (MIT; accessed 2026-09-03). The upstream
  README dates the paper 2021; dblp records ``conf/iclr/KimKTPCC22``, and 2022
  is used here.

Deliberate departures, each with the ID that forced it: ``d_model`` 512 -> 128
(`D25`); target-channel loss against the reference's all-channel default
(`D39`); ``lr`` halved every four epochs rather than every epoch (`D47`); and a
runtime ``capture`` attribute for the attention maps that consumes no RNG, so a
captured run stays bit-identical (`D62d`). Everything else is adopted unchanged
and never tuned (`D38`).
:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries this row and the
fifteen others in the same form.
"""


<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎛️ Konfigurasi arsitektur</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;"><code>d_model=128</code>, bukan 512 — panjang urutan attention adalah N ≤ 12, dan 512 akan over-parameterise terhadap ~14.000 sampel (<code>D25</code>).</p>
</div>

In [ ]:

@dataclass(frozen=True, slots=True)
class ITransformerConfig:
    """Hyperparameters, adopted unchanged from Liu et al. (2024) bar ``d_model``.

    **Nothing here is tuned, deliberately** (`D38`). Holding capacity fixed is
    what makes the rungs comparable; per-rung tuning would confound the ladder
    with model selection. Root §11's checklist item on validation-based
    hyperparameter selection therefore applies to ARIMA order and ridge alpha
    only — those are the two models where selection actually happens.
    """

    seq_len: int = 96
    pred_len: int = 24
    d_model: int = 128
    d_ff: int = 256
    e_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.1
    use_norm: bool = True
    #: Force attention weights uniform — the third main-grid arm (`D50`).
    #: K=1 vs K=8 differs in *information* and in *whether attention is active*
    #: at the same time, so a decaying A(b) is equally consistent with a
    #: capacity story as with the information story RQ2 claims. This separates
    #: them, at runs Figure 5 needs anyway.
    uniform_attention: bool = False

    # -- the Architecture protocol (`D56`) ----------------------------------
    #
    # Methods, not fields. ``write_artifacts`` records ``asdict(cfg)``, so
    # anything added here as a *field* would enter every iTransformer
    # ``meta/*.json`` and change bytes the 534-run grid has already produced.

    def build(self) -> "ITransformer":
        """A fresh module for this configuration."""
        return ITransformer(self)

    def loss_target(self) -> str:
        """``"target"`` — MSE on the target channel only, at every rung (`D39`).

        A constant rather than a field, deliberately. Standard iTransformer
        implementations compute the loss over all N channels, which would make
        K=12 a 12-task problem and K=1 a 1-task problem: auxiliary supervision
        varying with the study's own independent variable, and K=1 no longer the
        stated control but a different learning problem. Root §11 carries this as
        a verifiable assertion, and a field would be one edit away from failing
        it silently.
        """
        return "target"

    def schedule(self) -> "TrainSchedule":
        """Root §6.2's optimisation budget: 30 epochs, patience 5, LR halved every 4.

        A method rather than a field, and that is load-bearing for the same reason
        ``build`` and ``loss_target`` are: ``write_artifacts`` records
        ``asdict(cfg)``, so a field added here would enter every iTransformer
        ``meta/*.json`` and change bytes the 684-run grid has already produced.
        The schedule is logged under its own ``meta`` key instead.
        """

        return TrainSchedule()

    def fit(
        self,
        tensors: "OriginTensors",
        spec: "RunSpec",
        *,
        device: "torch.device | None" = None,
    ) -> tuple["ITransformer", "ITransformerConfig", "TrainOutcome"]:
        """Train one cell and hand back this config **unchanged**.

        Nothing is selected here: every hyperparameter is fixed a priori and
        identical at every rung (`D38`), which is what makes the rungs
        comparable. Ridge is the contrast — its alpha is chosen on the validation
        sub-block, so its ``fit`` returns a different config than it received.

        The import is deferred because ``train`` owns the protocol this
        satisfies, and importing it at module scope would be a cycle. In the
        flattened notebook there are no modules at all and ``train_one`` is
        simply a name a later cell defines, bound by the time this is called.
        """

        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⏱️ Jadwal panjang</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Arm <code>longsched</code>: <code>lr_halve_every=8</code>, 60 epoch, patience 10 (<code>D62c</code>).</p>
</div>

In [ ]:


class LongScheduleConfig(ITransformerConfig):
    """`D62c` --- the exploratory arm that answers "you under-trained".

    The most obvious attack on a null result, and until now it had no answer. The
    document implied the 30-epoch cap was the budget that bound; measured, **0 of
    444** iTransformer runs reached it, mean 10.49, maximum 26. What bound was the
    learning-rate schedule, so widening the cap alone would have changed nothing.
    This widens the schedule: LR halved every 8 epochs instead of 4, 60 epochs
    instead of 30, patience 10 instead of 5.

    A **plain subclass adding no field**. ``asdict`` therefore returns exactly the
    parent's dict, ``meta['config']`` is unchanged, and the difference lives in
    the ``meta['schedule']`` key and in the ``run_id`` tag ``itrl``. Nothing about
    the ladder's confirmatory numbers moves: this arm is **exploratory**, declared
    as such under §13.2's confirmatory-versus-exploratory rule, and reported in
    its own table because §12 forbids two code vintages sharing one.
    """

    def schedule(self) -> "TrainSchedule":

        return TrainSchedule(max_epochs=60, patience=10, lr_halve_every=8)


<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎯 Konfigurasi ter-tuning</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Arm <code>tuned</code>: satu-satunya konfigurasi di studi ini yang <code>lr</code>-nya sebuah field, karena pencarian validasi memilihnya dan run harus menjalankannya (<code>D76</code>).</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class TunedConfig(ITransformerConfig):
    """`D70`'s tuned arm --- the winning configuration, **learning rate included**.

    `D76`: the arm shipped without this class, and that was the defect. The search
    space is ``d_model x e_layers x lr`` and
    :func:`~itransformer_btc.runner.tune_on_validation` ranked all eighteen points
    on origin 1's validation --- but it returned a bare
    :class:`ITransformerConfig`, which carries no learning rate. The winner's
    ``lr = 1e-3`` was therefore *selected and then discarded*, and the arm ran the
    winner's architecture under the default ``1e-4``: a point the search had also
    evaluated and had **not** ranked first. Root §12 requires a number to resolve
    to the decision that produced it, the documented decision and the executed run
    disagreed, and the arm did not answer the referee's question it exists for.

    ``lr`` is a **field**, unlike every other member of the Architecture protocol,
    and the exception is deliberate. ``write_artifacts`` records ``asdict(cfg)``,
    so a field on :class:`ITransformerConfig` would enter every iTransformer
    ``meta/*.json``; a field on a subclass one arm uses enters only that arm's,
    where recording the rate that ran is exactly what §12 asks for. It reaches the
    trainer through :meth:`schedule`, the same route `D62c`'s ``itrl`` takes.
    """

    #: The learning rate the validation search selected. The default is root
    #: §6.2's, so an un-tuned construction of this class is the main arm.
    lr: float = 1e-4

    def schedule(self) -> "TrainSchedule":

        return TrainSchedule(lr=self.lr)


<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧠 Blok encoder</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Embedding terbalik <code>Linear(L → d_model)</code>, attention lintas variat, FFN. Tanpa causal mask — semua token sezaman.</p>
</div>

In [ ]:


class InvertedEmbedding(nn.Module):
    """Embed each variate's entire lookback: ``Linear(L -> d_model)``.

    With ``d_model = 128 > L = 96`` this projection is generically injective, so
    the whole lookback survives it. Root §5.3 leans on that: the reason for
    excluding moving averages is not that they are unrecoverable — a linear
    function of the lookback is recoverable in principle — but that adding one
    raises nominal K without raising information, which is precisely the
    phenomenon RQ1 exists to measure.
    """

    def __init__(self, seq_len: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.projection = nn.Linear(seq_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor) -> Tensor:
        """``(B, L, N) -> (B, N, d_model)``."""
        return self.dropout(self.projection(x.permute(0, 2, 1)))


class VariateAttention(nn.Module):
    """Multi-head attention over the variate axis, optionally forced uniform."""

    def __init__(self, d_model: int, n_heads: int, dropout: float, uniform: bool) -> None:
        super().__init__()
        if d_model % n_heads:
            raise ValueError(f"d_model {d_model} not divisible by n_heads {n_heads}")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.uniform = uniform
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        #: Runtime capture of the attention weights -- Figure 5's only input
        #: (`D62d`). A plain module attribute, **never** an
        #: :class:`ITransformerConfig` field: ``write_artifacts`` records
        #: ``asdict(cfg)``, so a field here would appear in every iTransformer
        #: ``meta/*.json`` and change bytes the 684-run grid has already
        #: produced. Off by default, and the branch consumes no RNG, so a run
        #: with capture on is bit-identical to one without --- which is what
        #: makes the maps describe the model whose numbers the paper reports
        #: rather than a second model that merely resembles it.
        self.capture: bool = False
        self.last_weights: Tensor | None = None

    def forward(self, x: Tensor) -> Tensor:
        b, n, d = x.shape
        shape = (b, n, self.n_heads, self.head_dim)
        v = self.v(x).view(shape).transpose(1, 2)

        if self.uniform:
            # Every variate attends equally to every variate. W_V, W_O and the
            # parameter count stay intact, so the arm isolates *what attention
            # selects* rather than how much capacity the model has.
            context = v.mean(dim=2, keepdim=True).expand(-1, -1, n, -1)
        else:
            q = self.q(x).view(shape).transpose(1, 2)
            k = self.k(x).view(shape).transpose(1, 2)
            scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
            weights = torch.softmax(scores, dim=-1)
            if self.capture:
                # Averaged over heads: Figure 5 shows variate-level reliance, and
                # eight per-head panels per layer would be a different figure.
                self.last_weights = weights.detach().mean(dim=1)
            context = self.dropout(weights) @ v

        return self.out(context.transpose(1, 2).reshape(b, n, d))


class EncoderLayer(nn.Module):
    """Attention over variates, then a position-wise FFN. Post-norm, as in the paper."""

    def __init__(self, cfg: ITransformerConfig) -> None:
        super().__init__()
        self.attention = VariateAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, cfg.uniform_attention
        )
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_ff, cfg.d_model),
        )
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: Tensor) -> Tensor:
        x = self.norm1(x + self.dropout(self.attention(x)))
        return self.norm2(x + self.dropout(self.ffn(x)))


<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔁 ITransformer</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Encoder-only. Loss dihitung hanya pada kanal target, di setiap rung (<code>D39</code>).</p>
</div>

In [ ]:


class ITransformer(nn.Module):
    """``(B, L, N) -> (B, H)`` on the target channel.

    The output is the target channel alone even though the projection produces
    all N, because the **loss is single-channel** (`D39`). Standard
    implementations compute it over all N, which would make K=12 a 12-task
    problem and K=1 a 1-task problem — auxiliary supervision varying with the
    study's own independent variable, and K=1 no longer the stated control but a
    different learning problem. The reference implementation defaults to the
    option that breaks the design, so root §11 carries this as an assertion.
    """

    def __init__(self, cfg: ITransformerConfig, target_index: int = 0) -> None:
        super().__init__()
        self.cfg = cfg
        self.target_index = target_index
        self.embedding = InvertedEmbedding(cfg.seq_len, cfg.d_model, cfg.dropout)
        self.layers = nn.ModuleList(EncoderLayer(cfg) for _ in range(cfg.e_layers))
        self.projection = nn.Linear(cfg.d_model, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        """Args: ``x`` of shape ``(B, L, N)``. Returns ``(B, H)``."""
        mean = std = None
        if self.cfg.use_norm:
            # Per-channel instance normalisation over time. Root §6.3: this is
            # what makes the outer StandardScaler cancel algebraically — and it
            # is itself a nonlinearity, which is why the F2 estimators
            # contribute *shape* and not *level*, the confound `D04` requires be
            # disclosed in Limitations whatever RQ1 returns.
            mean = x.mean(dim=1, keepdim=True)
            x = x - mean
            std = torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-5)
            x = x / std

        h = self.embedding(x)
        for layer in self.layers:
            h = layer(h)
        out = self.projection(h).permute(0, 2, 1)  # (B, H, N)

        if self.cfg.use_norm:
            out = out * std[:, 0, :].unsqueeze(1) + mean[:, 0, :].unsqueeze(1)

        return out[:, :, self.target_index]

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, N) -> (B, H)`` — here, identical to ``forward``.

        The method exists because ``preds/*.parquet`` holds the target channel
        for **every** model in the study, and a channel-independent baseline's
        ``forward`` returns all N (`D56`). Declaring the projection on the model
        that wrote the file beats sniffing the rank of an output tensor: the
        prediction file's meaning then rests on something a model said, not on a
        shape a reader has to reverse-engineer.
        """
        return self(x)

    def n_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


<div style="background: linear-gradient(135deg, #1b0033, #2d0052); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #bf5af233;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🔥 9 &middot; Loop latih & kontrak keterlacakan
  </h2>
  <p style="color: #cbb2e8; margin: 0; font-size: 1.02em;">Loop latih GPU-resident tanpa DataLoader, plus kontrak keterlacakan root &sect;12 yang setiap run tulis.</p>
  <ul style="color: #cbb2e8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Loss MSE pada kanal target saja</strong>, di setiap rung (<code>D39</code>). Di bawah loss all-channel, K=12 menjadi masalah 12-tugas dan K=1 masalah 1-tugas, jadi supervisi tambahan ikut berubah bersama variabel bebas studi ini sendiri.</li>
    <li style="margin-bottom: 6px;">Data tidak bergerak setelah muat awal: seluruh split masuk GPU sekali, lalu batch dibentuk dengan mengiris indeks tensor itu. Bentuk <code>DataLoader</code> per-item ~10&times; lebih buruk dan menaruh grid di luar kuota mingguan.</li>
    <li style="margin-bottom: 6px;">LR dibelah tiap 4 epoch, bukan tiap epoch (<code>D47</code>) — pembelahan per-epoch mencapai ~4e-7 di epoch 9, jadi anggaran epoch tidak akan pernah mengikat. Terukur: <strong>0 dari 444 run</strong> menyentuh cap 30 epoch (<code>D62c</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">📘 train.py</h3>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.94em;">Loop latih GPU-resident, tanpa <code>DataLoader</code>, plus kontrak keterlacakan root §12 yang setiap run tulis.</p>
</div>

<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Path artefak dan <code>CODE_SHA256_OVERRIDE</code> — digest yang dipin generator (root §12).</p>
</div>

In [ ]:
"""Training loop, run identity, and the two files every run must leave behind.

Root §10.3's regime is the load-bearing part: **load the whole split to the
device once, then batch by index-slicing that tensor.** No ``Dataset``, no
``DataLoader``, no workers. At ~280k parameters the compute is trivial and the
run is dominated entirely by data movement and Python overhead, which a per-item
loader maximises — the naive path costs roughly 10x and puts the 837-run grid
outside the 30 h weekly quota outright.

Root §10.4: **persist raw predictions, always.** They are required for the
Diebold-Mariano test, the per-regime analysis and the economic evaluation.
Re-running the grid because only metrics were saved is an expensive, avoidable
mistake.

Upstream
--------
The optimiser and the schedule are PyTorch's; the loop around them is this
study's, and the departures from a stock training loop are the point.

- ``torch.optim.Adam`` -- D. P. Kingma and J. Ba, "Adam: A method for
  stochastic optimization," in *Proc. 3rd Int. Conf. Learn. Represent. (ICLR)*,
  2015. arXiv:1412.6980. Called directly at ``lr = 1e-4``, adopted unchanged
  from Liu et al. (2024) and never tuned (`D38`).
- ``torch.optim.lr_scheduler.StepLR`` -- halving every **four** epochs, not
  every epoch. Per-epoch halving reaches ~4e-7 by epoch 9, so the 30-epoch
  budget could never bind and the cap would be decorative (`D47`). PyTorch:
  https://docs.pytorch.org/docs/stable/optim.html (BSD-3-Clause; accessed
  2026-09-03).

What is **not** taken from any reference loop: there is no ``Dataset`` and no
``DataLoader`` anywhere. The whole split is resident on the GPU and batching is
index-slicing into that tensor, which is the difference between ~30 s and
~300 s per run at this model size and is what puts the grid inside the weekly
quota at all (`D19`, `D57`). Shuffling permutes an index tensor **on device**;
moving data after the initial load would undo the entire regime.
:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries this row in full.
"""

ARTIFACTS: Path = Path("artifacts")

DEFAULT_PARQUET: Path = Path("data/raw/BTCUSDT_1h.parquet")

#: Environment variable naming the input artifact actually consumed.
#:
#: Root §12 forbids comparing numbers produced under different input-artifact
#: hashes, so every ``meta/*.json`` must name the vintage it read. The
#: repository-relative default is right locally and **wrong everywhere the grid
#: actually runs**: on Kaggle the artifact arrives as an attached Dataset under
#: ``/kaggle/input/<slug>/``, and root §10.5 forbids hard-coding that slug. With
#: the path unresolvable the digest logged as ``"unknown"`` on every Kaggle run,
#: which is §12 unenforceable at precisely the place the grid executes. The
#: launcher and the worker CLI both set this from the path they were handed.
INPUT_PARQUET_ENV: str = "ITBTC_PARQUET"

#: Digest supplied by a launcher that has no package files to hash.
#:
#: :func:`code_sha256` normally hashes ``*.py`` beside this module, which needs
#: a ``__file__`` — and a notebook that carries the package as **plain
#: definition cells** rather than materialised files has none. The generator
#: computes the identical digest from ``src/itransformer_btc/`` and sets this,
#: so the number in ``meta/*.json`` still names the code that ran (root §12,
#: `D54b`) and still matches the digest a local checkout of the same source
#: produces. Left ``None`` in every file-based context, where the real hash is
#: strictly better because nothing has to be told to keep it honest.
CODE_SHA256_OVERRIDE: str | None = None


#: Serialises the seeded prologue of a run — seeding, then building the module.
#:
#: Only the prologue. Everything after it draws from the **device's own** CUDA
#: generator, which :func:`set_seed` scopes per device, so two workers pinned to
#: two GPUs never touch each other's stream. The prologue is milliseconds against
#: a ~32 s run, so holding a lock across it costs no measurable parallelism and
#: buys back the one thing run-level parallelism would otherwise destroy: the
#: bit-exact reproducibility `D62d` demonstrated and root §12 requires (`D68`).
SEED_LOCK = threading.Lock()


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎲 Seed & pemilihan perangkat</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Presisi digerbangi <code>get_device_capability(0)[0] >= 8</code>, tidak pernah <code>is_bf16_supported()</code>, yang mengembalikan True palsu di T4.</p>
</div>

In [ ]:


def set_seed(seed: int, device: torch.device | None = None) -> None:
    """Seed every source of nondeterminism root §16 names, scoped to one device.

    ``cudnn.deterministic`` costs throughput and is set anyway: a run that
    cannot be reproduced cannot enter the manuscript (root §12).

    **``device`` is what makes two GPUs safe (`D68`).** ``torch.manual_seed``
    reseeds the CPU generator *and every CUDA device*, so with two workers running
    concurrently one worker's seeding would reset the other's stream mid-training
    and neither run would reproduce. Given a device, this seeds the CPU generator
    and **only that device's** generator, leaving the other worker's untouched.

    Single-device behaviour is unchanged, which is what keeps the 894 completed
    runs reproducible: with one device in use, seeding it alone and seeding all of
    them set the same generator to the same value.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    # ``torch.manual_seed`` would fan out to every CUDA device; the default
    # generator is the CPU one and nothing else.
    torch.default_generator.manual_seed(seed)
    if torch.cuda.is_available():
        if device is not None and torch.device(device).type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.manual_seed(seed)
        else:
            torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def pick_device() -> torch.device:
    """Prefer CUDA; fall back to CPU."""
    return torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")


def supports_native_bf16(device: torch.device) -> bool:
    """True only on sm_80+.

    Never gate on ``torch.cuda.is_bf16_supported()``: it defaults to
    ``including_emulation=True`` and returns **True** on a T4 (sm_75), selecting
    an emulated bf16 path slower than fp32 (root §10.3).
    """
    if device.type != "cuda":
        return False
    return torch.cuda.get_device_capability(device.index or 0)[0] >= 8


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📝 Spesifikasi run & hasil latih</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Satu sel grid: model, origin, K, H, seed — komponen <code>run_id</code> root §10.4.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class RunSpec:
    """Deterministic, human-readable run identity (root §10.4).

    Changing any component deliberately **orphans** prior outputs rather than
    silently reusing a mismatched result.
    """

    model: str
    origin_index: int
    k: int
    pred_len: int
    seed: int

    @property
    def run_id(self) -> str:
        return (
            f"{self.model}_o{self.origin_index:02d}_K{self.k:02d}"
            f"_H{self.pred_len:03d}_s{self.seed}"
        )


@dataclass(frozen=True, slots=True)
class TrainOutcome:
    """What one completed run produced, beyond its two artifacts."""

    run_id: str
    epochs_run: int
    best_val_mse: float
    train_loss: float
    wall_time_s: float
    n_parameters: int
    device: str


@dataclass(frozen=True, slots=True)
class TrainSchedule:
    """Root §6.2's optimisation budget, as data rather than as call-site defaults.

    The defaults reproduce the 684-run grid exactly, and that is the point.
    `D47` fixed ``lr_halve_every`` at 4 because per-epoch halving reaches ~4e-7
    by epoch 9, which makes both the 30-epoch budget and the patience-5 stop
    decorative.

    Measured afterwards, the grid early-stopped at a **mean of 10.49 epochs and
    never once reached the cap** across 444 iTransformer runs, maximum 26. So the
    binding constraint was the schedule, not the budget: by epoch 26 the learning
    rate is ~1.6e-6 and by epoch 30 ~7.8e-7, and raising ``max_epochs`` alone
    would have been a no-op. That is why `D62c`'s robustness arm widens the
    schedule instead, and why this exists as an overridable object rather than as
    four numbers frozen into a signature.
    """

    max_epochs: int = 30
    patience: int = 5
    lr: float = 1e-4
    lr_halve_every: int = 4


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🏛️ Arsitektur & protokol Forecaster</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Antarmuka yang iTransformer dan ketiga baseline sama-sama penuhi.</p>
</div>

In [ ]:


class Architecture(Protocol):
    """What the trainer, the runner and the artifact writer need of a config.

    :func:`write_artifacts` is the **only** definition of the ``meta/*.json``
    schema — root §12's traceability contract expressed as code — and it was
    bound to ``ITransformer``/``ITransformerConfig`` until the §7 baselines
    arrived (`D56`). A second writer in ``baselines.py`` would have made two
    definitions of one contract, which is the drift surface `D54d` exists to
    prevent, so the writer was widened to this protocol instead of copied.

    **Everything here is a method, never a field, and that is load-bearing.**
    ``write_artifacts`` records ``asdict(cfg)``, so a field added merely to steer
    dispatch would appear in every iTransformer ``meta/*.json`` and change bytes
    the study has already produced. Methods are invisible to
    :func:`dataclasses.asdict`; fields are not.
    """

    pred_len: int

    def build(self) -> nn.Module:
        """A fresh, untrained module for this configuration."""

    def loss_target(self) -> str:
        """``"target"`` or ``"all"`` — which target tensor the loss reads.

        See :class:`itransformer_btc.splits.SplitTensors`: the ladder is
        single-channel by `D39`, the channel-independent baselines are
        all-channel by their own published objective, and that difference is what
        makes their K label mean anything.
        """

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple[nn.Module, "Architecture", TrainOutcome]:
        """Fit one cell: the model, the **resolved** config, and the outcome.

        The config comes back because selection happens for one model and not the
        others. Every iTransformer hyperparameter is fixed a priori and identical
        at every rung (`D38`), so its ``fit`` returns what it was given; ridge's
        alpha is chosen on the validation sub-block, and with ARIMA outside the
        minimal set that is the **only** hyperparameter selected anywhere in this
        study (root §11). A chosen value that never reached ``meta['config']``
        would be a number the manuscript could not regenerate.
        """


class Forecaster(Protocol):
    """What the artifact writer needs of a fitted model."""

    cfg: Architecture

    def eval(self) -> "Forecaster":
        """Inference mode — dropout off."""

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, K) -> (B, H)`` on the target channel: what ``preds/`` holds."""


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⚡ Helper batch & prediksi</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Batch dengan mengiris indeks tensor yang sudah di GPU. Data tidak bergerak setelah muat awal.</p>
</div>

In [ ]:


def _to_device(
    split: SplitTensors, device: torch.device, *, target: str = "target"
) -> tuple[Tensor, Tensor]:
    """Move one split's inputs and its loss target to the device.

    ``target`` selects the width, not the content: ``"target"`` is the ``r``
    channel the ladder is scored on, ``"all"`` is every channel, which is what a
    channel-independent baseline's published objective supervises.
    """
    y = split.y if target == "target" else split.y_all
    return (
        torch.from_numpy(split.x).to(device, non_blocking=True),
        torch.from_numpy(y).to(device, non_blocking=True),
    )


@torch.no_grad()
def _mean_loss(model: nn.Module, x: Tensor, y: Tensor, batch: int = 512) -> float:
    """Mean MSE over a split, batched to bound peak memory rather than for speed.

    The divisor is every element of one sample's target, so this is the mean over
    ``(H,)`` for a single-channel target and over ``(H, N)`` for an all-channel
    one — the same quantity ``mse_loss`` returns, computed in pieces.
    """
    if len(x) == 0:
        return float("nan")
    model.eval()
    total = 0.0
    for i in range(0, len(x), batch):
        total += nn.functional.mse_loss(
            model(x[i : i + batch]), y[i : i + batch], reduction="sum"
        ).item()
    return total / (len(x) * int(np.prod(y.shape[1:])))


@torch.no_grad()
def predict(model: Forecaster, x: Tensor, batch: int = 512) -> np.ndarray:
    """The target channel's H-step forecasts for every window, batched.

    Routed through ``forecast_target`` rather than ``__call__`` because a
    channel-independent baseline trained on its published all-channel objective
    returns ``(B, H, N)`` from ``forward`` while ``preds/`` holds one channel
    (root §10.4). Sniffing the rank of the output instead would leave the
    prediction file's meaning resting on a shape no model ever declared.
    """
    model.eval()
    if len(x) == 0:
        return np.empty((0, model.cfg.pred_len), np.float32)
    return np.concatenate(
        [
            model.forecast_target(x[i : i + batch]).cpu().numpy()
            for i in range(0, len(x), batch)
        ]
    )


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔥 Loop latih</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Early stopping patience 5 pada MSE validasi; LR dibelah tiap 4 epoch (<code>D47</code>).</p>
</div>

In [ ]:


def train_one(
    tensors: OriginTensors,
    spec: RunSpec,
    cfg: Architecture,
    *,
    device: torch.device | None = None,
    batch_size: int = 32,
    max_epochs: int | None = None,
    patience: int | None = None,
    lr: float | None = None,
    lr_halve_every: int | None = None,
) -> tuple[nn.Module, TrainOutcome]:
    """Train one (origin, K, seed) cell and return the best-validation model.

    The learning rate halves every **four** epochs, not every epoch (`D47`):
    per-epoch halving reaches ~4e-7 by epoch 9, which makes the 30-epoch budget
    and the patience-5 early stop decorative — the model stops moving long
    before either can bind.

    ``epochs_run`` and the final training loss come back because root §6.2
    requires them logged per rung: they are how a reader tells a flat 8->12 rung
    from an under-trained one.

    **One trainer serves every gradient model in the study**, iTransformer and
    the §7 baselines alike (`D56`). The schedule, the patience and the early-stop
    rule are root §6.2's, and duplicating them per model would be a second
    definition of the training protocol — the same drift `D54d` names for the
    artifact schema. The config supplies only what genuinely differs: the module
    to build, and the width of the target its loss reads.
    """
    device = device or pick_device()

    # The schedule comes from the config when it declares one, so an arm can widen
    # it without a new trainer (`D62c`). ``hasattr`` rather than a Protocol member:
    # ridge is a solve with no epochs at all, and DLinear and PatchTST take root
    # §6.2's schedule unchanged, so requiring the method would add a stub to three
    # classes to say "nothing special". Explicit arguments still win, which is what
    # keeps ``stage5_pilot`` and the tests able to shorten a run.
    training_schedule = cfg.schedule() if hasattr(cfg, "schedule") else TrainSchedule()
    max_epochs = training_schedule.max_epochs if max_epochs is None else max_epochs
    patience = training_schedule.patience if patience is None else patience
    lr = training_schedule.lr if lr is None else lr
    lr_halve_every = (
        training_schedule.lr_halve_every if lr_halve_every is None else lr_halve_every
    )

    # Seeding and construction together, under one lock: both draw from the CPU
    # generator, which every worker shares whatever device it owns (`D68`).
    with SEED_LOCK:
        set_seed(spec.seed, device)
        model = cfg.build().to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    schedule = torch.optim.lr_scheduler.StepLR(
        optimiser, step_size=lr_halve_every, gamma=0.5
    )

    # The whole split, resident. Index-slicing it is the entire batching
    # strategy; shuffling permutes an index tensor on device, never the data.
    loss_target = cfg.loss_target()
    x_tr, y_tr = _to_device(tensors.train, device, target=loss_target)
    x_va, y_va = _to_device(tensors.val, device, target=loss_target)

    best_val = float("inf")
    best_state: dict[str, Tensor] | None = None
    epochs_run = 0
    train_loss = float("nan")
    stale = 0
    started = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(x_tr), device=device)
        running = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i : i + batch_size]
            optimiser.zero_grad(set_to_none=True)
            # For the ladder this is MSE on the target channel only, at every
            # rung (`D39`): `ITransformerConfig.loss_target` is the constant
            # "target" and `ITransformer.forward` returns that channel alone, so
            # it cannot drift to the all-channel loss the reference
            # implementation defaults to — which would make K=12 a 12-task
            # problem and K=1 a 1-task one, varying supervision with the study's
            # own independent variable. A channel-independent baseline says
            # "all" and gets the all-channel objective it is published with.
            loss = nn.functional.mse_loss(model(x_tr[idx]), y_tr[idx])
            loss.backward()
            optimiser.step()
            running += loss.item() * len(idx)
        schedule.step()

        epochs_run = epoch
        train_loss = running / len(x_tr)
        val = _mean_loss(model, x_va, y_va)

        if val < best_val - 1e-9:
            best_val, stale = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, TrainOutcome(
        run_id=spec.run_id,
        epochs_run=epochs_run,
        best_val_mse=best_val,
        train_loss=train_loss,
        wall_time_s=time.perf_counter() - started,
        n_parameters=model.n_parameters(),
        device=str(device),
    )


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔍 Uji invarian skala</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;"><code>MSE(c·x)/c² == MSE(x)</code> — bentuk yang benar; versi loss-identik tidak mungkin lolos (<code>D03</code>).</p>
</div>

In [ ]:


def scale_invariance_check(
    model: Forecaster, x: Tensor, y: Tensor, c: float = 100.0
) -> tuple[float, float]:
    """Root §6.3's corrected ``use_norm`` invariant (`D03`).

    The source specification said to multiply the input by 100 and assert
    identical losses. That **cannot pass**: the target is a channel of the same
    array, so it scales too and the loss scales by ``c^2``. The invariant that
    does hold is ``MSE(c x) / c^2 == MSE(x)``.

    Returns:
        ``(MSE(x), MSE(c x) / c^2)`` — equal to floating-point tolerance while
        ``use_norm`` is active, and visibly unequal the moment it is not.
    """
    return _mean_loss(model, x, y), _mean_loss(model, x * c, y * c) / (c * c)


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔐 Provenance kode & input</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;"><code>code_sha256</code> ada karena <code>git_sha</code> berbunyi unknown di Kaggle — persis tempat grid berjalan (<code>D54b</code>).</p>
</div>

In [ ]:


def _git_sha() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return "unknown"


def code_sha256() -> str:
    """Digest of this package's own source — the git sha's stand-in off-repo.

    Root §12 asks a run to name the code that produced it, and names the git sha
    as the way. **There is no git repository on Kaggle**, so the sha logs as
    ``"unknown"`` there and the traceability contract loses its code half exactly
    where the grid runs. Hashing the package source answers the same question and
    answers it better: it identifies the code that ran, not the commit someone
    happened to be standing on with a dirty tree.

    Line endings are normalised, so a CRLF checkout on Windows and the LF copy a
    notebook materialises give the **same** digest for identical logic. Without
    that, every Kaggle run would appear to be a different code vintage from the
    local run of the same commit — a false positive on the one check §12 exists
    to make possible.

    ``CODE_SHA256_OVERRIDE`` short-circuits this where there are no files to
    hash — a notebook carrying the package as plain definition cells. The
    ``__file__`` lookup below sits *after* that check on purpose: in such a
    launcher there is no module file at all, so reaching it would raise rather
    than return the digest the traceability contract asks for.
    """
    if CODE_SHA256_OVERRIDE is not None:
        return CODE_SHA256_OVERRIDE
    root = Path(__file__).resolve().parent
    digest = hashlib.sha256()
    for path in sorted(root.glob("*.py")):
        digest.update(path.name.encode("utf-8"))
        digest.update(path.read_bytes().replace(b"\r\n", b"\n"))
    return digest.hexdigest()


def resolve_input_parquet(parquet: Path | str | None = None) -> Path:
    """The input artifact this process consumed: argument, then env, then default."""
    if parquet is not None:
        return Path(parquet)
    from_env = os.environ.get(INPUT_PARQUET_ENV)
    return Path(from_env) if from_env else DEFAULT_PARQUET


def _input_sha256(parquet: Path | str | None = None) -> tuple[str, str]:
    """``(digest, provenance)`` for the parquet actually read.

    The Stage 1 report sitting beside the artifact is preferred, because that
    digest was written by the ingest script over the bytes it emitted and is the
    figure root §4.1 pins. Where the report did not travel — an attached Kaggle
    Dataset holding the parquet alone — the file is hashed directly, which yields
    the same number by construction.

    Returns ``("unknown", "unresolved")`` rather than raising: a run that cannot
    name its input is a documented failure under §12, and failing the run instead
    would lose 90 s of GPU time to a bookkeeping problem.
    """
    path = resolve_input_parquet(parquet)
    report = path.with_name(f"{path.stem}_report.json")
    try:
        return json.loads(report.read_text())["artifact_sha256"][path.name], "report"
    except Exception:
        pass
    try:
        return hashlib.sha256(path.read_bytes()).hexdigest(), "file-digest"
    except Exception:
        return "unknown", "unresolved"


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">💾 Menulis artefak run</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Prediksi mentah selalu, bukan cuma metrik. Tanpanya DM, analisis rezim, dan evaluasi ekonomi mustahil.</p>
</div>

In [ ]:


def write_artifacts(
    model: Forecaster,
    tensors: OriginTensors,
    spec: RunSpec,
    cfg: Architecture,
    outcome: TrainOutcome,
    device: torch.device,
    root: Path = ARTIFACTS,
    attention: "pl.DataFrame | None" = None,
) -> tuple[Path, Path]:
    """Write ``preds/{run_id}.parquet`` and ``meta/{run_id}.json``.

    A run is complete **only when both files exist and ``status == "complete"``**
    (root §10.5). Anything else is re-run from scratch; intra-run checkpointing
    is deliberately omitted, since at ~30 s per run measured (`D57`) it costs far
    more complexity than it saves.

    **This function is the schema.** Root §12 admits no number into the
    manuscript that does not resolve to a prediction file, a config hash and a
    documented decision, and these two files are where all three live. Every
    model in the study — the ladder, ridge, DLinear, PatchTST — writes through
    here, typed against :class:`Forecaster` and :class:`Architecture` rather than
    against one model, so there is exactly one place the contract can be read and
    exactly one place it can be broken.
    """
    (root / "preds").mkdir(parents=True, exist_ok=True)
    (root / "meta").mkdir(parents=True, exist_ok=True)

    frames = []
    for b, split in zip(tensors.block_labels, tensors.test_blocks):
        # The label, not the position: the falsification arm's first tensor is
        # block 4, and re-indexing it to 1 would silently compare the fresh
        # model against the aged model's wrong blocks.
        if len(split) == 0:
            continue
        x, _ = _to_device(split, device)
        pred = predict(model, x)
        n, h = pred.shape
        frames.append(
            pl.DataFrame(
                {
                    "block": np.full(n * h, b, dtype=np.int8),
                    "step": np.tile(np.arange(1, h + 1, dtype=np.int16), n),
                    "timestamp": np.repeat(split.ts, h),
                    "y_true": split.y.reshape(-1),
                    "y_pred": pred.reshape(-1),
                }
            )
        )
    preds = (
        pl.concat(frames)
        if frames
        else pl.DataFrame(
            schema={
                "block": pl.Int8,
                "step": pl.Int16,
                "timestamp": pl.Int64,
                "y_true": pl.Float32,
                "y_pred": pl.Float32,
            }
        )
    )

    preds_path = root / "preds" / f"{spec.run_id}.parquet"
    meta_path = root / "meta" / f"{spec.run_id}.json"
    preds.write_parquet(preds_path)

    if attention is not None:
        # Figure 5's input (`D62d`). A third directory rather than a column on
        # ``preds``: the maps are one row per (tercile, layer, variate pair) and
        # the forecasts are one row per (block, timestamp, step), so joining them
        # into one file would mean padding one of the two with nulls. Completeness
        # still keys off ``preds`` and ``meta`` alone, so an arm that writes no map
        # is complete without one.
        (root / "attn").mkdir(parents=True, exist_ok=True)
        attention.write_parquet(root / "attn" / f"{spec.run_id}.parquet")

    input_parquet = resolve_input_parquet()
    input_digest, input_provenance = _input_sha256(input_parquet)
    meta = {
        "run_id": spec.run_id,
        "spec": asdict(spec),
        "config": asdict(cfg),
        # Recorded separately because a schedule override is a **method**, and
        # ``asdict`` sees fields only (`D62c`). ``LongScheduleConfig`` adds no
        # field, so its ``config`` block is byte-identical to the main arm's and
        # this key is the only place the difference is visible --- besides the
        # ``run_id`` tag, which is what keeps the two from colliding on disk.
        "schedule": (
            asdict(cfg.schedule()) if hasattr(cfg, "schedule") else None
        ),
        "origin": tensors.origin.label,
        "origin_index": tensors.origin.index,
        "block_labels": list(tensors.block_labels),
        "k": tensors.k,
        "variates": list(tensors.scaler.columns),
        "git_sha": _git_sha(),
        # Root §12's code half. `git_sha` is "unknown" off-repo, which is every
        # Kaggle session; `code_sha256` answers the same question there.
        "code_sha256": code_sha256(),
        "input_parquet": str(input_parquet),
        "input_sha256": input_digest,
        "input_sha256_source": input_provenance,
        "n_train": len(tensors.train),
        "n_val": len(tensors.val),
        "n_test_per_block": [len(s) for s in tensors.test_blocks],
        # Root §7 / `D31`: logged per origin so the drift tilt the Naive-RW
        # baseline carries in scaler space is auditable rather than assumed away.
        "mu_g": float(tensors.scaler.mean[0]),
        "sigma_g": float(tensors.scaler.std[0]),
        "mu_over_sigma": tensors.scaler.target_mu_over_sigma,
        "naive_rw_z": tensors.naive_rw_z,
        "epochs_run": outcome.epochs_run,
        "best_val_mse": outcome.best_val_mse,
        "train_loss": outcome.train_loss,
        "wall_time_s": outcome.wall_time_s,
        "n_parameters": outcome.n_parameters,
        "device": outcome.device,
        "status": "complete",
    }
    meta_path.write_text(json.dumps(meta, indent=2))
    return preds_path, meta_path


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af288; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">✅ Idempotensi</h4>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.9em;">Run lengkap hanya bila kedua berkas ada <strong>dan</strong> <code>meta.status</code> berbunyi complete.</p>
</div>

In [ ]:


def is_complete(run_id: str, root: Path = ARTIFACTS) -> bool:
    """Root §10.5's idempotence rule, as one function."""
    preds = root / "preds" / f"{run_id}.parquet"
    meta = root / "meta" / f"{run_id}.json"
    if not (preds.exists() and meta.exists()):
        return False
    try:
        return json.loads(meta.read_text()).get("status") == "complete"
    except Exception:
        return False


<div style="background: linear-gradient(135deg, #03071e, #370617); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #e9456033;">
  <h2 style="color: #f5a623; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📏 10 &middot; Metrik & uji statistik
  </h2>
  <p style="color: #ffd6a5; margin: 0; font-size: 1.02em;">Seluruh estimator paper: metrik, DM/Clark&ndash;West, survival, dan bootstrap klaster liar untuk &beta;&#8321;.</p>
  <ul style="color: #ffd6a5; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Setiap metrik rasio dibentuk dari <strong>MSE yang sudah dirata-rata seed</strong>, tidak pernah dari rata-rata rasio per-seed (<code>D42</code>): keduanya berbeda oleh Jensen.</li>
    <li style="margin-bottom: 6px;">Pasangan <strong>bersarang</strong> memakai Clark&ndash;West, bukan DM baku — DM baku sistematis undersized justru terhadap alternatif yang studi ini ada untuk menegakkannya (<code>D29</code>).</li>
    <li style="margin-bottom: 6px;">Varians jangka panjang memakai estimator <strong>rektangular terpotong</strong> pada lag h&minus;1, bukan bobot Bartlett: Bartlett menyusutkan &gamma;&#770;&#8322;&#8322; sekitar 92% dan menghasilkan p yang terlalu optimistis (<code>D34</code>).</li>
    <li style="margin-bottom: 6px;"><code>D(i,b)</code> hidup di skala <em>skill</em>, bukan RelMSE (<code>D23</code>) — di skala RelMSE tiap &tau; tak terjangkau dan RQ3 akan mengembalikan &ldquo;tanpa decay&rdquo; karena satuan yang tidak cocok, bukan karena pasarnya.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #e94560; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #f5a623; margin: 0 0 6px 0; font-size: 1.22em;">📘 metrics.py</h3>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.94em;">Seluruh estimator paper: metrik, DM/Clark–West, survival, dan bootstrap klaster liar untuk β₁.</p>
</div>

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">τ headline 5% dan sensitivitasnya, plus <code>B_STAR_SCHEMA</code> — skema dideklarasikan supaya kasus semua-origin-dikecualikan tetap mengembalikan frame berkolom (<code>D55</code>).</p>
</div>

In [ ]:
"""Metrics, tests, and the three RQ estimators.

Root §9. Everything here consumes ``preds/{run_id}.parquet`` and
``meta/{run_id}.json`` and produces numbers that go straight into
``artifacts/paper_numbers.json`` — which is why nothing in this module reads a
model or a tensor. Root §12: a number that cannot be regenerated from a
persisted prediction file plus a config hash is a documented failure, not a
footnote.

Six decisions here are load-bearing, and each reverses something the source
design said:

* **`D31`** — Naive-RW is ``y_z = -mu_g/sigma_g``, never ``y_z = 0``. In scaler
  space zero means ``r_hat = mu_g``, the training-window mean hourly return: a
  constant-drift model wearing the EMH baseline's name. Measured, ``mu_g/sigma_g``
  spans -0.00818 … +0.01733 and **changes sign across origins**, so it is not a
  constant tilt a reader could mentally subtract.
* **`D23`** — ``D(i,b)`` lives on the **skill** scale, not on RelMSE. On RelMSE
  every pre-registered tau is arithmetically unreachable and RQ3 returns "no
  decay detected" by construction, before a single epoch runs.
* **`D05` follow-on** — the decay denominator is the **within-origin mean**, not
  block 1. One 30-day block under heavy tails would sit in the denominator of
  five quantities, making their errors perfectly correlated, and ``b*`` reads a
  threshold crossing straight off the series, so an unlucky block 1 moves the
  crossing by whole blocks.
* **`D42`** — every ratio metric is formed from **seed-averaged MSEs**, never
  from an average of per-seed ratios. The two differ by Jensen, and the second
  additionally requires pairing seed 42 at K=1 with seed 42 at K=8, which are
  independent training runs of different models: any of 5! orderings gives a
  different answer.
* **`D29`** — nested pairs get **Clark-West**, not Diebold-Mariano. Under the
  null with nested models and estimated parameters the loss differential has a
  mean shifted away from zero, so standard DM is systematically undersized
  against the alternative this study exists to establish.
* **`D34`** — the long-run variance estimator is **rectangular**, not Bartlett.
  Under the DM null, h-step optimal forecast errors are MA(h-1), so every
  autocovariance to lag 23 is genuinely nonzero and equally real; Bartlett
  weights shrink the lag-22 term by ~92%, understating the variance and
  producing exactly the over-optimistic p-values this module exists to prevent.

Upstream
--------
**Every estimator here is written on numpy from its published definition. No
statistical package is imported at module level, and none is vendored.** The
algorithms are cited to their papers; the implementations are this study's.

- ``dm_test`` — F. X. Diebold and R. S. Mariano, "Comparing predictive
  accuracy," *J. Bus. Econ. Statist.*, vol. 13, no. 3, pp. 253-263, 1995, with
  the small-sample correction of D. Harvey, S. Leybourne, and P. Newbold,
  "Testing the equality of prediction mean squared errors," *Int. J.
  Forecast.*, vol. 13, no. 2, pp. 281-291, 1997. Validated against R's
  ``forecast::dm.test``
  (https://pkg.robjhyndman.com/forecast/reference/dm.test.html, accessed
  2026-09-03) — a validation target, not a dependency.
- ``clark_west_test`` — T. E. Clark and K. D. West, "Approximately normal tests
  for equal predictive accuracy in nested models," *J. Econometrics*, vol. 138,
  no. 1, pp. 291-311, 2007. Used for every nested pair, where standard DM is
  undersized against the alternative this study exists to establish (`D29`).
- Wild cluster restricted bootstrap — A. C. Cameron, J. B. Gelbach, and
  D. L. Miller, *Rev. Econ. Statist.*, vol. 90, no. 3, pp. 414-427, 2008;
  J. G. MacKinnon, M. O. Nielsen, and M. D. Webb, *J. Econometrics*, vol. 232,
  no. 2, pp. 272-299, 2023; the ``(1 + count)/(1 + B)`` p-value from
  A. C. Davison and D. V. Hinkley, *Bootstrap Methods and their Application*,
  CUP, 1997. **Not the** ``wildboottest`` **package**, which root §9.2 names as
  the reference implementation but which this package does not import
  (`D42`, `D53d`).

The rectangular long-run variance is deliberate, and it is the reason
``statsmodels``' ``cov_hac`` is absent here: that estimator is Bartlett by
default (`D34`). :data:`itransformer_btc.config.SOURCE_PROVENANCE` carries
these rows in full.
"""

#: ``{model}_o{origin:02d}_K{K:02d}_H{H:03d}_s{seed}`` (root §10.4).
RUN_ID_PATTERN = re.compile(
    r"^(?P<model>[a-z0-9]+)_o(?P<origin>\d{2})_K(?P<k>\d{2})"
    r"_H(?P<h>\d{3})_s(?P<seed>\d+)$"
)

HOUR_MS = 3_600_000

#: Root §3's pre-registered thresholds. The headline is 5%; the rest are
#: sensitivities. Choosing tau after seeing the decay curve is p-hacking.
TAU_HEADLINE: float = 0.05
TAU_SENSITIVITY: tuple[float, ...] = (0.025, 0.05, 0.10, 0.50)

#: Declared so `DecayResult.b_star` keeps its columns when every origin is
#: excluded (`D55`). An inferred schema over zero rows yields a frame with no
#: columns at all, and the caller's ``bs["b_star"]`` then raises rather than
#: reporting the pre-registered null.
B_STAR_SCHEMA: dict[str, pl.DataType] = {
    "origin": pl.Utf8,
    "tau": pl.Float64,
    "b_star": pl.Int64,
    "event": pl.Boolean,
}


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📥 Memuat run</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Parse <code>run_id</code> dan muat prediksi/meta dari akar mana pun yang ditemukan lewat glob.</p>
</div>

In [ ]:


# -- artifact I/O ------------------------------------------------------------


def parse_run_id(run_id: str) -> dict[str, int | str]:
    """Decompose a ``run_id`` into its five components.

    Raises:
        ValueError: If it does not match root §10.4's pattern. A run whose id
            cannot be parsed cannot be placed in the grid, and skipping it
            silently would drop a cell from a table without saying so.
    """
    match = RUN_ID_PATTERN.match(run_id)
    if match is None:
        raise ValueError(f"{run_id!r} is not a root §10.4 run_id")
    g = match.groupdict()
    return {
        "model": g["model"],
        "origin_index": int(g["origin"]),
        "k": int(g["k"]),
        "pred_len": int(g["h"]),
        "seed": int(g["seed"]),
    }


def _locate(run_id: str, roots: list[Path], kind: str, suffix: str) -> Path:
    for root in roots:
        candidate = Path(root) / kind / f"{run_id}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{kind}/{run_id}{suffix} in none of {[str(r) for r in roots]}"
    )


def load_predictions(run_id: str, roots: list[Path]) -> pl.DataFrame:
    """Read one run's raw predictions.

    Roots are searched in order, so a working directory listed first shadows an
    older attached dataset — which is what lets a single re-run cell take effect
    without deleting the previous session's output.
    """
    return pl.read_parquet(_locate(run_id, roots, "preds", ".parquet"))


def load_meta(run_id: str, roots: list[Path]) -> dict:
    return json.loads(_locate(run_id, roots, "meta", ".json").read_text())


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📐 Metrik inti</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">MSE, MAE, RelMSE, dan <code>R²_oos = 1 − RelMSE</code> (<code>D20</code>). RMSE mentah dilaporkan berdampingan supaya dua skala dapat direkonsiliasi.</p>
</div>

In [ ]:


# -- point metrics -----------------------------------------------------------


def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.square(np.asarray(y_true) - np.asarray(y_pred))))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def rel_mse(model: float, naive: float) -> float:
    """``MSE_model / MSE_naive`` — controls for period difficulty (root §9.1)."""
    return model / naive


def r2_oos(model: float, naive: float) -> float:
    """``1 - RelMSE`` (`D20`) — the readable form of the same quantity.

    RelMSE near 1.00 is hard to read; ``R2_oos`` reads directly as skill against
    a random walk, and its sign is the whole question.
    """
    return 1.0 - rel_mse(model, naive)


def raw_rmse(mse_z: float, sigma_g: float) -> float:
    """RMSE back in raw log-return units — root §9.1's second reporting scale.

    "RMSE 0.0043 on hourly log-returns" tells a reader far more than "MSE 0.187
    on normalized data", and stating ``sigma_g`` is what lets the two reconcile.
    """
    return math.sqrt(mse_z) * sigma_g


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎯 Jendela non-overlap & Pesaran–Timmermann</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Di H=24 target tumpang-tindih 23 dari 24 jam, jadi PT over-reject parah; versi tumpang-tindih deskriptif saja (<code>D21</code>).</p>
</div>

In [ ]:


def non_overlapping_mask(timestamps: np.ndarray) -> np.ndarray:
    """Window starts whose forecast period opens at **00:00 UTC** (`D46`).

    There are 24 admissible alignments of a non-overlapping daily partition and
    each gives a different Sharpe, MDD and turnover, so the phase is fixed in
    advance rather than chosen after seeing the equity curve.

    ``timestamp`` in the prediction file is the **window start**; the first
    target hour is ``start + L``. With ``L = 96`` a multiple of 24 the phase is
    preserved, so selecting starts at hour 0 selects targets opening at hour 0.
    """
    return (np.asarray(timestamps) // HOUR_MS) % 24 == 0


# -- directional accuracy and its testing regime (`D21`) ---------------------


def pesaran_timmermann(actual: np.ndarray, predicted: np.ndarray) -> tuple[float, float]:
    """Pesaran-Timmermann (1992) test of directional predictability.

    Returns:
        ``(statistic, one-sided p)`` against ``N(0,1)``. Without a null
        hypothesis, directional accuracy is a descriptive number; this supplies
        the null.

    Zero targets are excluded rather than assigned a direction: a zero
    log-return has no sign to predict, and assigning one would inflate the hit
    rate by whatever the model happened to output there.
    """
    a = np.sign(np.asarray(actual, dtype=np.float64))
    f = np.sign(np.asarray(predicted, dtype=np.float64))
    keep = a != 0
    a, f = a[keep], f[keep]
    n = len(a)
    if n < 2:
        return float("nan"), float("nan")

    hit = float(np.mean(a == f))
    py = float(np.mean(a > 0))
    px = float(np.mean(f > 0))
    p_star = py * px + (1 - py) * (1 - px)

    var_hit = p_star * (1 - p_star) / n
    var_star = (
        (2 * py - 1) ** 2 * px * (1 - px)
        + (2 * px - 1) ** 2 * py * (1 - py)
        + 4 * py * px * (1 - py) * (1 - px) / n
    ) / n
    denom = var_hit - var_star
    if denom <= 0:
        return float("nan"), float("nan")

    stat = (hit - p_star) / math.sqrt(denom)
    return stat, 0.5 * math.erfc(stat / math.sqrt(2.0))


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧭 Akurasi arah</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">DA di h=1, h=24, dan pada return kumulatif 24 jam — tiga rezim pengujian berbeda.</p>
</div>

In [ ]:


def _hit_rate(actual: np.ndarray, predicted: np.ndarray) -> float:
    keep = np.sign(actual) != 0
    if not keep.any():
        return float("nan")
    return float(np.mean(np.sign(actual[keep]) == np.sign(predicted[keep])))


@dataclass(frozen=True, slots=True)
class DirectionalAccuracy:
    """DA at the three horizons §9.1 requires, with their testing regimes.

    ``da_h24`` and ``da_cum`` carry p-values **only** on the non-overlapping
    sample. On hourly spacing their targets overlap by 23 of 24 hours, giving
    lag-1 autocorrelation of about 23/24; Pesaran-Timmermann's variance is then
    far too small and the test over-rejects badly. The overlapping figures are
    reported descriptively, without p-values, and the resulting power loss
    (T = 30 per block) is stated rather than recovered by using the invalid
    sample.
    """

    da_h1: float
    p_h1: float
    da_hH_overlapping: float
    da_hH: float
    p_hH: float
    da_cum_overlapping: float
    da_cum: float
    p_cum: float
    n_h1: int
    n_non_overlapping: int


def directional_accuracy(frame: pl.DataFrame) -> DirectionalAccuracy:
    """Compute all three DA variants for one run's predictions."""
    last_step = int(frame.get_column("step").max())

    step1 = frame.filter(pl.col("step") == 1)
    a1 = step1.get_column("y_true").to_numpy()
    f1 = step1.get_column("y_pred").to_numpy()
    _, p1 = pesaran_timmermann(a1, f1)

    step_h = frame.filter(pl.col("step") == last_step)
    ts_h = step_h.get_column("timestamp").to_numpy()
    a_h = step_h.get_column("y_true").to_numpy()
    f_h = step_h.get_column("y_pred").to_numpy()
    keep_h = non_overlapping_mask(ts_h)
    _, p_h = pesaran_timmermann(a_h[keep_h], f_h[keep_h])

    cum = (
        frame.group_by("timestamp")
        .agg(pl.col("y_true").sum(), pl.col("y_pred").sum())
        .sort("timestamp")
    )
    ts_c = cum.get_column("timestamp").to_numpy()
    a_c = cum.get_column("y_true").to_numpy()
    f_c = cum.get_column("y_pred").to_numpy()
    keep_c = non_overlapping_mask(ts_c)
    _, p_c = pesaran_timmermann(a_c[keep_c], f_c[keep_c])

    return DirectionalAccuracy(
        da_h1=_hit_rate(a1, f1),
        p_h1=p1,
        da_hH_overlapping=_hit_rate(a_h, f_h),
        da_hH=_hit_rate(a_h[keep_h], f_h[keep_h]),
        p_hH=p_h,
        da_cum_overlapping=_hit_rate(a_c, f_c),
        da_cum=_hit_rate(a_c[keep_c], f_c[keep_c]),
        p_cum=p_c,
        n_h1=len(a1),
        n_non_overlapping=int(keep_c.sum()),
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧱 Metrik per blok</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Baseline dinilai pada <strong>himpunan jendela yang persis sama</strong> dengan pembandingnya (<code>D45</code>).</p>
</div>

In [ ]:


# -- per-block tables --------------------------------------------------------


def assert_same_windows(left: pl.DataFrame, right: pl.DataFrame, what: str) -> None:
    """`D45` — two models may only be compared on identical evaluated windows.

    Naive-RW needs no 96-bar lookback, so unless it is restricted to the window
    set its comparator actually evaluated, RelMSE is a ratio across two different
    samples. Test-window survival is conditioned on *future* gaps and outages
    cluster on stress, so the two samples would differ precisely in their
    high-volatility content.

    Raises:
        ValueError: If the evaluated ``(block, timestamp)`` sets differ.
    """
    def _key(frame: pl.DataFrame) -> np.ndarray:
        pairs = frame.select(["block", "timestamp"]).unique().sort(["block", "timestamp"])
        return (
            pairs.get_column("block").to_numpy().astype(np.int64) * (1 << 44)
            + pairs.get_column("timestamp").to_numpy().astype(np.int64) // HOUR_MS
        )

    a, b = _key(left), _key(right)
    if len(a) != len(b) or not np.array_equal(a, b):
        raise ValueError(
            f"{what}: evaluated window sets differ ({len(a)} vs {len(b)} "
            f"windows). RelMSE across two samples is not a ratio."
        )


def block_metrics(frame: pl.DataFrame, naive_z: float) -> pl.DataFrame:
    """Per-block MSE, MAE, RelMSE and ``R2_oos`` for one run.

    Args:
        frame: One run's predictions.
        naive_z: ``-mu_g/sigma_g`` from ``meta.json`` (`D31`). Passing 0 here
            silently substitutes a constant-drift model for the EMH baseline.

    The Naive-RW error is computed on exactly the rows the model was scored on,
    so :func:`assert_same_windows` has nothing to check for this pair — the
    sample is shared by construction. It still applies across *models*.
    """
    return (
        frame.with_columns(
            (pl.col("y_true") - pl.col("y_pred")).pow(2).alias("_se"),
            (pl.col("y_true") - pl.col("y_pred")).abs().alias("_ae"),
            (pl.col("y_true") - naive_z).pow(2).alias("_se_naive"),
        )
        .group_by("block")
        .agg(
            pl.col("timestamp").n_unique().alias("n_windows"),
            pl.col("_se").count().alias("n_points"),
            pl.col("_se").mean().alias("mse"),
            pl.col("_ae").mean().alias("mae"),
            pl.col("_se_naive").mean().alias("mse_naive"),
        )
        .with_columns(
            (pl.col("mse") / pl.col("mse_naive")).alias("rel_mse"),
            (1.0 - pl.col("mse") / pl.col("mse_naive")).alias("r2_oos"),
        )
        .sort("block")
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧮 Perakitan grid & rata-rata seed</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Rata-rata seed <strong>lebih dulu</strong>, rasio kedua — dibalik keduanya berbeda karena Jensen (<code>D42</code>).</p>
</div>

In [ ]:


def gather_grid(run_ids: list[str], roots: list[Path]) -> pl.DataFrame:
    """Per (run, block) metrics for many runs — the input to every RQ estimator.

    Returns:
        Long frame with ``run_id, model, origin_index, origin, k, pred_len,
        seed, block, n_windows, mse, mae, mse_naive, rel_mse, r2_oos, sigma_g``.

    Raises:
        FileNotFoundError: If any run is absent. A quietly short grid produces a
            table whose cells came from different arms, which is worse than no
            table.
    """
    rows: list[pl.DataFrame] = []
    missing: list[str] = []
    for run_id in run_ids:
        try:
            preds = load_predictions(run_id, roots)
            meta = load_meta(run_id, roots)
        except FileNotFoundError:
            missing.append(run_id)
            continue
        parts = parse_run_id(run_id)
        rows.append(
            block_metrics(preds, float(meta["naive_rw_z"])).with_columns(
                pl.lit(run_id).alias("run_id"),
                pl.lit(str(parts["model"])).alias("model"),
                pl.lit(int(parts["origin_index"])).cast(pl.Int32).alias("origin_index"),
                pl.lit(str(meta["origin"])).alias("origin"),
                pl.lit(int(parts["k"])).cast(pl.Int32).alias("k"),
                pl.lit(int(parts["pred_len"])).cast(pl.Int32).alias("pred_len"),
                pl.lit(int(parts["seed"])).cast(pl.Int32).alias("seed"),
                pl.lit(float(meta["sigma_g"])).alias("sigma_g"),
            )
        )
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} of {len(run_ids)} runs are absent, first few: "
            f"{missing[:5]}. Complete the grid or pass the subset explicitly — "
            f"a silently short grid mixes arms inside one table."
        )
    if not rows:
        raise ValueError("no runs gathered")
    return pl.concat(rows)


def seed_average(grid: pl.DataFrame) -> pl.DataFrame:
    """Average MSE across seeds **before** any ratio is formed (`D42`).

    Seeds are computational noise, not population draws. Averaging ratios
    instead would differ by Jensen and would additionally require pairing seed
    42 at K=1 with seed 42 at K=8 — independent training runs of different
    models, where any of 5! orderings gives a different answer.

    The cell mean still carries Monte-Carlo error, which enters as measurement
    error in the dependent variable: unbiased for beta1, but inflating residual
    variance. ``n_seeds`` and ``mse_seed_std`` are carried so §9.2's dispersion
    rule can bind the error bar to the aggregation level (`D30`) — seed std is a
    Monte-Carlo diagnostic, never the uncertainty on an origin-aggregated row.
    """
    return (
        grid.group_by(["model", "origin_index", "origin", "k", "pred_len", "block"])
        .agg(
            pl.col("mse").mean().alias("mse"),
            pl.col("mae").mean().alias("mae"),
            pl.col("mse_naive").mean().alias("mse_naive"),
            pl.col("mse").std().alias("mse_seed_std"),
            pl.col("n_windows").first().alias("n_windows"),
            pl.col("sigma_g").first().alias("sigma_g"),
            pl.col("mse").count().alias("n_seeds"),
        )
        .with_columns(
            (pl.col("mse") / pl.col("mse_naive")).alias("rel_mse"),
            (1.0 - pl.col("mse") / pl.col("mse_naive")).alias("r2_oos"),
        )
        .sort(["model", "origin_index", "k", "pred_len", "block"])
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📶 Amplifikasi A dan A_attn</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;"><code>A</code> variabel terikat RQ2, hanya K=1 lawan K=8. <code>A_attn</code> memisahkan <em>attention</em> dari <em>informasi</em> (<code>D50</code>).</p>
</div>

In [ ]:


# -- RQ2: the amplification gap and its decay --------------------------------


def amplification(
    seed_avg: pl.DataFrame,
    k_small: int = 1,
    k_large: int = 8,
    model: str = "itr",
    pred_len: int = 24,
) -> pl.DataFrame:
    """``A(i,b) = [MSE_K1 - MSE_K8] / MSE_K1`` — RQ2's dependent variable.

    **K=8, never K=12.** K=12 carries deliberate redundancy (root §5.2), so
    using it would confound decay with that redundancy — which is why the pair
    is a parameter with a pre-registered default rather than a free choice.

    Both models are evaluated on the same block, so period difficulty cancels in
    the ratio, and it cancels *well*: ``MSE_model`` and ``MSE_naive`` on one
    block correlate near 1. That is the argument the whole ratio-metric design
    rests on.
    """
    base = seed_avg.filter(
        (pl.col("model") == model) & (pl.col("pred_len") == pred_len)
    )
    small = (
        base.filter(pl.col("k") == k_small)
        .select(["origin_index", "origin", "block", "mse", "n_windows"])
        .rename({"mse": "mse_small", "n_windows": "n_small"})
    )
    large = (
        base.filter(pl.col("k") == k_large)
        .select(["origin_index", "block", "mse", "n_windows"])
        .rename({"mse": "mse_large", "n_windows": "n_large"})
    )

    joined = small.join(large, on=["origin_index", "block"], how="inner")
    if joined.height != small.height:
        raise ValueError(
            f"K={k_small} has {small.height} cells but only {joined.height} "
            f"matched K={k_large}; the panel must be balanced before beta1"
        )
    mismatched = joined.filter(pl.col("n_small") != pl.col("n_large"))
    if mismatched.height:
        raise ValueError(
            f"`D45`: {mismatched.height} cells evaluate K={k_small} and "
            f"K={k_large} on different window counts; A would be a ratio across "
            f"two samples"
        )
    return joined.with_columns(
        ((pl.col("mse_small") - pl.col("mse_large")) / pl.col("mse_small")).alias("A")
    ).sort(["origin_index", "block"])


def attention_amplification(
    seed_avg: pl.DataFrame, k: int = 8, pred_len: int = 24
) -> pl.DataFrame:
    """``A_attn(i,b) = [MSE_uniformK8 - MSE_K8] / MSE_uniformK8`` (`D50`).

    K=1 versus K=8 does **not** isolate attention: the two arms differ in
    *information* and in *whether attention is active* simultaneously, so a
    decaying ``A(b)`` is equally consistent with "cross-variate attention
    overfits regime-specific structure" — a capacity story — as with the
    information story RQ2 claims. This contrast holds information fixed and
    varies only what attention selects, at runs Figure 5 needs anyway.
    """
    sel = seed_avg.filter((pl.col("k") == k) & (pl.col("pred_len") == pred_len))
    uniform = (
        sel.filter(pl.col("model") == "itru")
        .select(["origin_index", "origin", "block", "mse"])
        .rename({"mse": "mse_uniform"})
    )
    attended = (
        sel.filter(pl.col("model") == "itr")
        .select(["origin_index", "block", "mse"])
        .rename({"mse": "mse_attended"})
    )
    return (
        uniform.join(attended, on=["origin_index", "block"], how="inner")
        .with_columns(
            (
                (pl.col("mse_uniform") - pl.col("mse_attended"))
                / pl.col("mse_uniform")
            ).alias("A_attn")
        )
        .sort(["origin_index", "block"])
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⏳ Decay dan b*</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;"><code>D(i,b)</code> di skala skill (<code>D23</code>). Origin ber-<code>R²_oos ≤ 0</code> dikecualikan dengan disebut namanya — dan itu <strong>seluruh</strong> lima belasnya.</p>
</div>

In [ ]:


# -- RQ3: skill decay and the retraining cadence -----------------------------


@dataclass(frozen=True, slots=True)
class DecayResult:
    """Per-origin ``D(i,b)`` and the censored ``b*`` it implies."""

    table: pl.DataFrame
    excluded_origins: tuple[str, ...]

    def b_star(self, tau: float) -> pl.DataFrame:
        """``b*(i) = min{b : D(i,b) > tau}``, **right-censored at 6** (`D41`).

        ``min{.}`` does not commute with averaging, so pooling MSEs across
        origins and *then* taking the minimum is a different estimand and is
        forbidden. Each origin contributes one observation, censored or not.

        The schema is declared rather than inferred (`D55`). When `decay`'s
        non-positive-skill guard excludes *every* origin, ``self.table`` is empty
        and an inferred schema yields a frame with no columns, so a caller's
        ``bs["b_star"]`` raises ``ColumnNotFoundError`` — which is what took the
        Kaggle notebook down at the exact moment its grid output was the only
        thing worth keeping. That guard firing is the **expected** outcome under
        non-positive skill, not an edge case: root §10.3's first measured run
        returned ``R2_oos = -0.0183`` and the completed grid returned it at all
        fifteen origins. `D54e` gates the estimators on grid *completeness*,
        which is a different failure and does not cover this path.

        An empty return means the estimand is **undefined** — there is no edge to
        lose a proportion of — which is not the same as every origin being
        censored at 6, where an edge exists and simply never decays past tau.
        Callers must report the two differently; ``excluded_origins`` is what
        tells them apart.
        """
        rows = []
        for key, part in self.table.group_by("origin", maintain_order=True):
            label = key[0] if isinstance(key, tuple) else key
            crossed = part.filter(pl.col("D") > tau).sort("block")
            rows.append(
                {
                    "origin": str(label),
                    "tau": tau,
                    "b_star": int(crossed.get_column("block")[0]) if crossed.height else 6,
                    "event": bool(crossed.height),
                }
            )
        return pl.DataFrame(rows, schema=B_STAR_SCHEMA)


def decay(
    seed_avg: pl.DataFrame,
    k: int = 8,
    model: str = "itr",
    pred_len: int = 24,
) -> DecayResult:
    """``D(i,b)`` on the **skill** scale (`D23`), normalised within origin (`D05`).

    ``D(i,b) = [mean_b' R2_oos(i,b') - R2_oos(i,b)] / mean_b' R2_oos(i,b')``

    On the RelMSE scale the pre-registered thresholds are unreachable: with
    ``RelMSE(1) = 0.996``, even *total* destruction of the model's edge gives
    ``D = 1/0.996 - 1 = 0.402%``, so tau = 2.5% would require the model to become
    2% worse than forecasting zero. RQ3 would return "no decay detected"
    regardless of the data — a result fixed by a units mismatch rather than by
    the market. On the skill scale ``D`` runs from 0 (no decay) through 1 (edge
    fully gone) and the taus are commensurate with it.

    The denominator is the within-origin **mean** rather than block 1's value
    (`D05` follow-on): block 1 is one 30-day block under heavy tails and
    volatility clustering, and it would otherwise sit in the denominator of five
    quantities and make their errors perfectly correlated.

    **Origins with non-positive mean skill are excluded and named**, never
    silently dropped: ``D`` is a proportion of an edge, and an origin with no
    edge has no proportion of one. Root §10.3's first measured run returned
    ``R2_oos = -0.0183``, so this guard may well be the common case rather than
    the edge case §9.1 assumed.
    """
    sel = seed_avg.filter(
        (pl.col("model") == model)
        & (pl.col("k") == k)
        & (pl.col("pred_len") == pred_len)
    ).sort(["origin_index", "block"])

    reference = sel.group_by("origin").agg(pl.col("r2_oos").mean().alias("r2_ref"))
    joined = sel.join(reference, on="origin", how="left")

    excluded = tuple(
        sorted(joined.filter(pl.col("r2_ref") <= 0).get_column("origin").unique().to_list())
    )
    kept = joined.filter(pl.col("r2_ref") > 0).with_columns(
        ((pl.col("r2_ref") - pl.col("r2_oos")) / pl.col("r2_ref")).alias("D")
    )
    return DecayResult(
        table=kept.select(
            ["origin", "origin_index", "block", "n_windows", "r2_oos", "r2_ref", "D"]
        ).sort(["origin_index", "block"]),
        excluded_origins=excluded,
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📈 Kurva survival & kuantil normal</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;"><code>b*</code> right-censored di 6 — data survival berinterval, bukan bilangan bulat telanjang (<code>D41</code>).</p>
</div>

In [ ]:


# -- survival analysis for b* (`D41`) ----------------------------------------


@dataclass(frozen=True, slots=True)
class SurvivalCurve:
    """Kaplan-Meier estimate on the 30-day block grid."""

    times: np.ndarray
    survival: np.ndarray
    lower: np.ndarray
    upper: np.ndarray
    n_events: int
    n_censored: int

    @property
    def median(self) -> float:
        """Smallest block with ``S(t) <= 0.5``; ``inf`` when never reached.

        ``inf`` is the honest answer, and root §3 fixes its wording: *"no decay
        detected within 180 days"* — a right-censored result, not a missing one.
        """
        below = self.times[self.survival <= 0.5]
        return float(below[0]) if len(below) else float("inf")

    @property
    def median_interval(self) -> tuple[float, float]:
        """Confidence set for the median: blocks whose CI band straddles 0.5.

        Table 5 carries this interval, never a bare integer, and the abstract's
        recommended cadence is this interval.
        """
        straddle = self.times[(self.lower <= 0.5) & (self.upper >= 0.5)]
        if not len(straddle):
            return (self.median, self.median)
        return (float(straddle[0]), float(straddle[-1]))


def _normal_quantile(p: float) -> float:
    """Acklam's inverse normal CDF — avoids a scipy import for one number."""
    a = [-3.969683028665376e+01, 2.209460984245205e+02, -2.759285104469687e+02,
         1.383577518672690e+02, -3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02, -1.556989798598866e+02,
         6.680131188771972e+01, -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00, 2.938163982698783e+00]
    d = [7.784695709041462e-03, 3.224671290700398e-01, 2.445134137142996e+00,
         3.754408661907416e+00]
    plow, phigh = 0.02425, 1 - 0.02425
    if p < plow:
        q = math.sqrt(-2 * math.log(p))
        return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
               ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    if p > phigh:
        q = math.sqrt(-2 * math.log(1 - p))
        return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
                ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    q, r = p - 0.5, (p - 0.5) ** 2
    return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q / \
           (((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)


def normal_quantile(p: float) -> float:
    """Standard normal inverse CDF -- the public name of :func:`_normal_quantile`.

    Exists because :mod:`itransformer_btc.economics` needs it for the Deflated
    Sharpe Ratio threshold, and root §15's flattening rule says a module reaches a
    sibling **by name**. Reaching for a private one across modules would work in
    the package and read as an accident in the notebook.
    """
    return _normal_quantile(p)


def normal_cdf(x: float) -> float:
    """Standard normal CDF, via ``erfc`` so no optional dependency is implied."""
    return 0.5 * math.erfc(-x / math.sqrt(2.0))


def _loglog_band(surv: float, var_sum: float, z: float) -> tuple[float, float]:
    """Log-log transformed Greenwood band — stays inside ``[0, 1]`` at small G.

    The plain Greenwood band routinely leaves the unit interval at G = 15, which
    would make the median interval unreadable at exactly the sample size this
    study has.
    """
    if surv <= 0.0 or surv >= 1.0 or var_sum <= 0.0:
        return surv, surv
    se = math.sqrt(var_sum) / abs(math.log(surv))
    lo = surv ** math.exp(z * se)
    hi = surv ** math.exp(-z * se)
    return float(min(lo, hi)), float(max(lo, hi))


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🩺 Kaplan–Meier & log-rank</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Estimator dan uji H3. Tidak tersedia di sini: tidak satu arm pun punya origin bertahan.</p>
</div>

In [ ]:


def kaplan_meier(
    times: np.ndarray, events: np.ndarray, alpha: float = 0.05
) -> SurvivalCurve:
    """Kaplan-Meier with Greenwood log-log confidence bands.

    ``b*`` is right-censored survival data on a six-point grid: an origin that
    never crosses tau is censored at 6, not missing. Reporting a bare mean over
    the crossers would condition on the event and bias the recommended cadence
    downward — which is the number the abstract carries.
    """
    t = np.asarray(times, dtype=np.float64)
    e = np.asarray(events, dtype=bool)
    z = _normal_quantile(1 - alpha / 2)

    surv, var_sum = 1.0, 0.0
    out_t, out_s, out_lo, out_hi = [], [], [], []
    for time in np.unique(t):
        at_risk = int(np.sum(t >= time))
        died = int(np.sum((t == time) & e))
        if at_risk > 0 and died > 0:
            surv *= 1.0 - died / at_risk
            if at_risk > died:
                var_sum += died / (at_risk * (at_risk - died))
        lo, hi = _loglog_band(surv, var_sum, z)
        out_t.append(time)
        out_s.append(surv)
        out_lo.append(lo)
        out_hi.append(hi)

    return SurvivalCurve(
        times=np.array(out_t),
        survival=np.array(out_s),
        lower=np.array(out_lo),
        upper=np.array(out_hi),
        n_events=int(e.sum()),
        n_censored=int(len(e) - e.sum()),
    )


def logrank(
    times_a: np.ndarray, events_a: np.ndarray,
    times_b: np.ndarray, events_b: np.ndarray,
) -> tuple[float, float]:
    """Two-sample log-rank test — H3's "larger K decays faster" (`D41`).

    Returns:
        ``(chi-square statistic on 1 df, two-sided p)``.
    """
    t = np.concatenate([times_a, times_b]).astype(np.float64)
    e = np.concatenate([events_a, events_b]).astype(bool)
    g = np.concatenate([np.zeros(len(times_a)), np.ones(len(times_b))]).astype(bool)

    observed = expected = variance = 0.0
    for time in np.unique(t[e]):
        n_risk = int(np.sum(t >= time))
        n_risk_b = int(np.sum((t >= time) & g))
        d = int(np.sum((t == time) & e))
        d_b = int(np.sum((t == time) & e & g))
        if n_risk < 2 or d == 0:
            continue
        share = n_risk_b / n_risk
        observed += d_b
        expected += d * share
        variance += d * share * (1 - share) * (n_risk - d) / (n_risk - 1)
    if variance <= 0:
        return float("nan"), float("nan")
    chi2 = (observed - expected) ** 2 / variance
    return float(chi2), float(math.erfc(math.sqrt(chi2 / 2.0)))


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📊 Varians jangka panjang</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Estimator <strong>rektangular</strong> terpotong pada lag h−1, bukan bobot Bartlett — Bartlett menyusutkan γ̂₂₂ ~92% dan menghasilkan p terlalu optimistis (<code>D34</code>).</p>
</div>

In [ ]:


# -- Diebold-Mariano and Clark-West (`D29`, `D34`) ---------------------------


def _rectangular_lrv(d: np.ndarray, h: int) -> float:
    """``[gamma_0 + 2 sum_{k=1}^{h-1} gamma_k] / T`` — the variance of ``d_bar``.

    Rectangular, not Bartlett (`D34`). Under the DM null, h-step optimal
    forecast errors are MA(h-1), so all autocovariances to lag ``h-1`` are
    genuinely nonzero and equally real. Bartlett weights shrink the lag-22 term
    by about 92%, understating the long-run variance and producing exactly the
    over-optimistic p-values this estimator exists to prevent. ``statsmodels``'
    ``cov_hac`` is Bartlett by default, so a literal reading of "Newey-West"
    fails validation against R's ``forecast::dm.test``.
    """
    t = len(d)
    dm = d - d.mean()
    total = float(dm @ dm) / t
    for k in range(1, min(h, t)):
        total += 2.0 * float(dm[k:] @ dm[:-k]) / t
    return total / t


def _bartlett_lrv(d: np.ndarray, h: int) -> float:
    """Only ever the fallback, and its use is reported (root §9.2)."""
    t = len(d)
    dm = d - d.mean()
    total = float(dm @ dm) / t
    for k in range(1, min(h, t)):
        total += 2.0 * (1.0 - k / h) * float(dm[k:] @ dm[:-k]) / t
    return total / t


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔧 Koreksi Harvey–Leybourne–Newbold</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Dirujuk ke Student-t dengan T−1 dof, bukan normal baku. Faktornya diasersi positif sebelum dipakai.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class TestResult:
    """One forecast-comparison test, carrying everything needed to redo it."""

    name: str
    statistic: float
    p_value: float
    T: int
    h: int
    one_sided: bool
    fallback_fired: bool

    def __str__(self) -> str:
        tail = "  [Bartlett fallback fired]" if self.fallback_fired else ""
        side = "one-sided" if self.one_sided else "two-sided"
        return (
            f"{self.name}: S*={self.statistic:+.4f}  p={self.p_value:.4g} "
            f"({side})  T={self.T}  h={self.h}{tail}"
        )


def _upper_tail(stat: float, df: int) -> float:
    """``P(T_df > stat)`` — Student-t, falling back to the normal without scipy."""
    try:
        from scipy import stats as _stats  # root §16's named stats boundary

        return float(_stats.t.sf(stat, df=df))
    except ImportError:  # pragma: no cover - scipy ships with the Kaggle image
        return 0.5 * math.erfc(stat / math.sqrt(2.0))


def _hln_and_p(d: np.ndarray, h: int, name: str, one_sided: bool) -> TestResult:
    """Harvey-Leybourne-Newbold correction, referred to ``t(T-1)``.

    ``S* = S sqrt[(T + 1 - 2h + h(h-1)/T) / T]``, compared against Student-t
    with ``T-1`` degrees of freedom — **not** the standard normal. The factor is
    asserted positive before use: at ``h = 24`` it is exactly 0 at ``T = 24`` and
    0.047 at ``T = 30``, precisely the T a non-overlapping 30-day block would
    produce, so a silent negative would yield a complex statistic reported as a
    real one.
    """
    d = np.asarray(d, dtype=np.float64)
    t = len(d)
    if t < 2:
        raise ValueError(f"{name}: T={t} is too small for a loss differential")
    factor = (t + 1 - 2 * h + h * (h - 1) / t) / t
    if factor <= 0:
        raise ValueError(
            f"{name}: the HLN factor is {factor:.4f} <= 0 at T={t}, h={h}. "
            f"Root §9.2 refuses to report where it fails; state T instead."
        )

    variance = _rectangular_lrv(d, h)
    fallback = False
    if variance <= 0:
        # The rectangular estimator is not guaranteed positive in finite
        # samples. Root §9.2: fall back to Bartlett and *report that it fired*.
        variance = _bartlett_lrv(d, h)
        fallback = True
        if variance <= 0:
            raise ValueError(f"{name}: no positive long-run variance at T={t}")

    stat = float(d.mean() / math.sqrt(variance) * math.sqrt(factor))
    upper = _upper_tail(abs(stat), t - 1)
    if one_sided:
        p = upper if stat >= 0 else 1.0 - upper
    else:
        p = 2.0 * upper
    return TestResult(name, stat, float(min(p, 1.0)), t, h, one_sided, fallback)


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⚔️ DM & Clark–West</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Pasangan <strong>bersarang</strong> memakai Clark–West; DM baku undersized justru terhadap alternatif yang studi ini uji (<code>D29</code>).</p>
</div>

In [ ]:


def hln_test(
    d: np.ndarray, h: int, name: str = "HLN", one_sided: bool = False
) -> TestResult:
    """Harvey-Leybourne-Newbold on a **pre-assembled** loss differential.

    :func:`dm_test` and :func:`clark_west_test` build their own differential from
    forecasts. Table 6 assembles one itself --- the Clark-West adjusted series
    aggregated over the 24 forecast steps of each origin, so that ``T`` counts
    window starts and ``h`` stays 24, which is the sample root §9.2 pins. Handing
    that series back through the forecasts would require inventing a pair of
    pseudo-forecasts whose differential happens to equal it.

    Args:
        d: The loss differential, one value per forecast origin.
        h: Forecast horizon, setting the truncation lag at ``h - 1``.
        name: Label carried into the result.
        one_sided: True for a nested pair, where the alternative is directional.
    """
    return _hln_and_p(np.asarray(d, dtype=np.float64), h, name, one_sided)


def dm_test(
    loss_a: np.ndarray, loss_b: np.ndarray, h: int, name: str = "DM"
) -> TestResult:
    """Diebold-Mariano for a **non-nested** pair — iTransformer vs DLinear etc.

    Do not use this on K=1 vs K=8, on anything vs Naive-RW, or on Ridge-K1 vs
    Ridge-K8: those pairs are nested, and there the statistic is not
    asymptotically ``N(0,1)`` (Clark & McCracken 2001; McCracken 2007). Use
    :func:`clark_west_test`.
    """
    return _hln_and_p(
        np.asarray(loss_a) - np.asarray(loss_b), h, name, one_sided=False
    )


def clark_west_test(
    y: np.ndarray,
    pred_small: np.ndarray,
    pred_large: np.ndarray,
    h: int,
    name: str = "Clark-West",
) -> TestResult:
    """Clark-West (2007) for a **nested** pair — the comparisons that carry the paper.

    ``f_t = (y - y_small)^2 - (y - y_large)^2 + (y_small - y_large)^2``

    The third term is the adjustment. Under the null of equal population
    predictive ability the larger model's extra estimation noise makes it look
    worse, so the unadjusted differential has a mean shifted away from zero and
    standard DM is systematically undersized **against the alternative this
    study exists to establish**. The Stage 5 gate turns a title decision on this
    statistic, which is why the choice is not left to the caller.

    One-sided by construction: the alternative is that the larger model helps.
    """
    y = np.asarray(y, dtype=np.float64)
    s = np.asarray(pred_small, dtype=np.float64)
    lg = np.asarray(pred_large, dtype=np.float64)
    f = np.square(y - s) - np.square(y - lg) + np.square(s - lg)
    return _hln_and_p(f, h, name, one_sided=True)


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧾 Rugi per origin</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Rugi per origin per model, masukan matriks pasangan.</p>
</div>

In [ ]:


def per_origin_loss(frame: pl.DataFrame) -> pl.DataFrame:
    """Mean squared error per forecast origin — the ``d_t`` series DM consumes.

    Root §9.2 pins the DM sample: **per (origin, block)**, on the overlapping
    hourly loss differential, ``T ~ 720``, ``h = 24``, truncation lag 23. Block
    level statistics are combined across cells by a stated method and **never**
    by concatenating ``d_t`` across origins: the model changes at each origin, so
    the DM null has no interpretation across that boundary.
    """
    return (
        frame.with_columns((pl.col("y_true") - pl.col("y_pred")).pow(2).alias("_se"))
        .group_by(["block", "timestamp"])
        .agg(pl.col("_se").mean().alias("loss"))
        .sort(["block", "timestamp"])
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎲 Bobot bootstrap & matriks panel</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Rademacher dan Webb 6-titik, keduanya dilaporkan. Panel tak seimbang ditolak keras — reduksi β₁ ke rata-rata within-slope hanya berlaku di panel seimbang.</p>
</div>

In [ ]:


# -- RQ2's core regression: beta1 with origin FE and a wild cluster bootstrap -


@dataclass(frozen=True, slots=True)
class Beta1Result:
    """``A(i,b) = alpha_i + beta1 b + eps`` with clustered inference (`D06`, `D42`)."""

    beta1: float
    t_statistic: float
    cluster_se: float
    p_rademacher: float
    p_webb: float
    n_clusters: int
    n_observations: int
    within_slopes: np.ndarray
    B: int

    @property
    def headline_p(self) -> float:
        """The more conservative of the two weight schemes, as root §9.2 requires."""
        return max(self.p_rademacher, self.p_webb)

    def __str__(self) -> str:
        return (
            f"beta1 = {self.beta1:+.6f}   t = {self.t_statistic:+.3f}   "
            f"G = {self.n_clusters}   N = {self.n_observations}\n"
            f"WCR one-sided p (H1: beta1 < 0): Rademacher {self.p_rademacher:.4f}, "
            f"Webb {self.p_webb:.4f}  ->  headline {self.headline_p:.4f}\n"
            f"Effective independence is bounded near 4 by the training-window "
            f"overlap (root §8.1), well below G = {self.n_clusters}."
        )


def _weights(kind: str, shape: tuple[int, int], rng: np.random.Generator) -> np.ndarray:
    if kind == "rademacher":
        return rng.choice(np.array([-1.0, 1.0]), size=shape)
    if kind == "webb":
        # Webb's 6-point distribution. At G = 15 Rademacher already admits
        # 2^15 = 32,768 distinct draws, a minimum two-sided p of about 6e-5, so
        # the original small-G justification for preferring Webb no longer
        # binds — both are reported and the more conservative is the headline.
        atoms = np.array([
            -math.sqrt(1.5), -1.0, -math.sqrt(0.5),
            math.sqrt(0.5), 1.0, math.sqrt(1.5),
        ])
        return rng.choice(atoms, size=shape)
    raise ValueError(f"unknown weight scheme {kind!r}")


def _balanced_matrix(panel: pl.DataFrame, value: str) -> tuple[np.ndarray, np.ndarray]:
    """``(G x B)`` outcome matrix and the block axis, or a loud failure.

    Built by hand rather than with ``pivot`` so the code does not depend on which
    polars major version the Kaggle image happens to ship.
    """
    origins = sorted(set(panel.get_column("origin").to_list()))
    blocks = sorted(set(int(b) for b in panel.get_column("block").to_list()))
    index = {(o, b): i for i, (o, b) in enumerate([(o, b) for o in origins for b in blocks])}

    out = np.full(len(index), np.nan)
    for origin, block, val in zip(
        panel.get_column("origin").to_list(),
        panel.get_column("block").to_list(),
        panel.get_column(value).to_list(),
    ):
        out[index[(str(origin), int(block))]] = float(val)

    matrix = out.reshape(len(origins), len(blocks))
    if np.isnan(matrix).any():
        raise ValueError(
            "unbalanced panel: beta1's reduction to the mean of within-slopes "
            "holds only when every origin carries every block"
        )
    return matrix, np.array(blocks, dtype=np.float64)


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧬 β₁ dengan bootstrap klaster liar</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">WCR dengan null dipaksakan, mem-bootstrap <em>t</em> bukan β̂, B = 99.999, satu sisi, klaster = origin. p memakai <code>(1 + count)/(1 + B)</code> (<code>D53d</code>).</p>
</div>

In [ ]:


def panel_beta1(
    panel: pl.DataFrame,
    value: str = "A",
    B: int = 99_999,
    seed: int = 42,
) -> Beta1Result:
    """Fit ``A(i,b) = alpha_i + beta1 b + eps`` and test ``H1: beta1 < 0``.

    Args:
        panel: Long frame with ``origin``, ``block`` and ``value``. Must be
            balanced — every origin carries the same block set.
        value: Dependent variable column.
        B: Bootstrap replications. 99,999 as pre-registered.
        seed: Bootstrap seed, recorded so the p-value is regenerable (root §12).

    **Without ``alpha_i``, beta1 absorbs origin-level difficulty** (`D06`). With
    origin fixed effects and a balanced panel, ``beta1`` reduces algebraically to
    the simple mean of the origin-specific within-slopes, so inference on the
    paper's core claim is a one-sample test on **G** numbers. Citing "15 x 6 = 90
    observations" invites the reader to infer power that does not exist; both
    counts are reported, and effective independence is bounded near 4 by the
    training-window overlap (root §8.1).

    The bootstrap is **restricted** (WCR — the null imposed when generating
    samples), bootstraps the **cluster-robust t-statistic** rather than beta-hat,
    and is **one-sided at alpha = 0.05 declared in advance**. WCU is severely
    size-distorted at small G (MacKinnon, Nielsen & Webb 2023), and the
    asymptotic refinement comes from bootstrapping *t* (Cameron, Gelbach &
    Miller 2008). A side chosen after seeing the sign is not pre-registered.
    """
    a, x = _balanced_matrix(panel, value)
    g, n_blocks = a.shape
    xd = x - x.mean()
    sxx = float(xd @ xd)

    within = a - a.mean(axis=1, keepdims=True)
    beta = float((within * xd).sum() / (g * sxx))
    resid = within - beta * xd
    score = resid @ xd
    variance = float((score @ score) / (g * sxx) ** 2)
    se = math.sqrt(variance) if variance > 0 else float("nan")
    t_obs = beta / se if se == se and se > 0 else float("nan")

    # Restricted residuals: with beta1 = 0 imposed the fitted value is the origin
    # mean, so u_tilde is exactly the within-origin demeaned outcome. Because
    # each row of u_tilde already sums to zero, the bootstrap origin means are
    # unchanged and the whole replication collapses to s = u_tilde @ xd.
    s = within @ xd

    def _p(kind: str) -> float:
        rng = np.random.default_rng(seed)
        weights = _weights(kind, (B, g), rng)
        beta_star = (weights @ s) / (g * sxx)
        score_star = weights * s[None, :] - beta_star[:, None] * sxx
        var_star = np.square(score_star).sum(axis=1) / (g * sxx) ** 2
        ok = var_star > 0
        t_star = beta_star[ok] / np.sqrt(var_star[ok])
        # (1 + count) / (1 + B), not count / B (Davison & Hinkley 1997): the
        # observed statistic is one of its own reference distribution, and the
        # naive form returns a literal p = 0, which is not a probability any
        # finite bootstrap can support. At B = 99,999 the floor it reports is
        # 1e-5, and at G = 15 Rademacher's own granularity bounds it at ~3e-5
        # anyway — so the floor is honest rather than conservative padding.
        below = int(np.sum(t_star <= t_obs))  # H1: beta1 < 0, left tail
        return (1.0 + below) / (1.0 + int(ok.sum()))

    return Beta1Result(
        beta1=beta,
        t_statistic=t_obs,
        cluster_se=se,
        p_rademacher=_p("rademacher"),
        p_webb=_p("webb"),
        n_clusters=g,
        n_observations=g * n_blocks,
        within_slopes=(within * xd).sum(axis=1) / sxx,
        B=B,
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⚖️ TOST & uji-J non-nested</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">TOST menguji <em>ekuivalensi</em> rung 8→12; uji-J memacu penjelasan K lawan K_eff (<code>D32</code>, <code>D49</code>).</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class EquivalenceResult:
    """TOST verdict on a rung `D49` pre-registers as flat."""

    mean_delta: float
    margin: float
    p_lower: float
    p_upper: float
    n: int

    @property
    def equivalent(self) -> bool:
        return max(self.p_lower, self.p_upper) < 0.05

    def __str__(self) -> str:
        verdict = "EQUIVALENT (flat)" if self.equivalent else "NOT shown equivalent"
        return (
            f"TOST: mean delta = {self.mean_delta:+.6f}, margin = +/-{self.margin:.6f}, "
            f"p = ({self.p_lower:.4f}, {self.p_upper:.4f}), G = {self.n}  ->  {verdict}"
        )


def tost_equivalence(
    deltas: np.ndarray, margin: float, alpha: float = 0.05
) -> EquivalenceResult:
    """Two one-sided tests — RQ1's pre-registered equivalence check (`D49`).

    RQ1's claim that the 8->12 rung is flat is an assertion of **no effect**, and
    a non-significant ΔMSE is a failure to reject, not evidence of equivalence.
    The margin is fixed in advance at ``0.25 x ΔMSE(4->8)``: choosing it after
    seeing the rung is the same p-hacking root §3 forbids for tau.

    Args:
        deltas: One within-origin ΔMSE per cluster. The inferential unit is the
            origin, never the (origin, block) cell.
        margin: ``Δ_eq``, positive.
    """
    d = np.asarray(deltas, dtype=np.float64)
    n = len(d)
    if n < 2:
        raise ValueError("TOST needs at least two clusters")
    se = float(np.std(d, ddof=1) / math.sqrt(n))
    if se <= 0:
        raise ValueError("zero dispersion across clusters; TOST is undefined")
    mean = float(d.mean())
    return EquivalenceResult(
        mean_delta=mean,
        margin=abs(margin),
        p_lower=_upper_tail((mean + abs(margin)) / se, n - 1),   # H0: mu <= -margin
        p_upper=_upper_tail(-(mean - abs(margin)) / se, n - 1),  # H0: mu >= +margin
        n=n,
    )


def j_test(
    y: np.ndarray, x_a: np.ndarray, x_b: np.ndarray, groups: np.ndarray
) -> tuple[float, float]:
    """Davidson-MacKinnon J-test of model A against model B (`D32`).

    RQ1 is a **non-nested** comparison: "benefit tracks K" and "benefit tracks
    K_eff" are two different regressors for the same outcome, and neither nests
    the other. Fitting both and comparing R-squared answers nothing; the J-test
    augments A with B's fitted values and asks whether they still carry
    information.

    All three inputs are within-transformed by ``groups`` first — the (origin x
    block) fixed effects of §9.1's specification — so the comparison is
    identified from within-cell variation across rungs, which is the only
    variation that distinguishes the two theories.

    Returns:
        ``(t statistic on B's fitted values, two-sided p)``. A significant t
        means A alone is inadequate. Run it both ways: if both reject, neither
        explanation is sufficient; if neither does, the data cannot separate
        them, which at ``corr(K, K_eff) ~ 0.97`` is the outcome to expect and to
        report plainly.
    """
    def _demean(v: np.ndarray) -> np.ndarray:
        v = np.asarray(v, dtype=np.float64)
        out = v.astype(np.float64).copy()
        for g in np.unique(groups):
            mask = groups == g
            out[mask] = v[mask] - v[mask].mean()
        return out

    yd, ad, bd = _demean(y), _demean(x_a), _demean(x_b)
    fitted_b = bd * (float(bd @ yd) / float(bd @ bd))

    design = np.column_stack([ad, fitted_b])
    coef, *_ = np.linalg.lstsq(design, yd, rcond=None)
    resid = yd - design @ coef
    dof = len(yd) - design.shape[1] - len(np.unique(groups))
    if dof <= 0:
        return float("nan"), float("nan")
    sigma2 = float(resid @ resid) / dof
    cov = sigma2 * np.linalg.pinv(design.T @ design)
    se = math.sqrt(max(cov[1, 1], 0.0))
    if se <= 0:
        return float("nan"), float("nan")
    t = float(coef[1] / se)
    return t, 2.0 * _upper_tail(abs(t), dof)


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔬 Efek minimum terdeteksi</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">MDE dipublikasi <strong>sebelum</strong> blok tes dibuka. Tanpa itu null tidak terbedakan dari desain yang memang tak sanggup mendeteksi (root §9.2 syarat 6).</p>
</div>

In [ ]:


def minimum_detectable_beta1(
    within_slopes: np.ndarray, alpha: float = 0.05, power: float = 0.80
) -> float:
    """The MDE root §13.2 calls the most damaging omission on its list.

    Every design choice in this study implies someone reasoned about precision,
    and no number was written down. If the MDE exceeds the plausible magnitude
    of ``A``, RQ2 must be repositioned as descriptive **before** the grid runs —
    otherwise a non-significant beta1 is indistinguishable from a design that
    could never have detected decay.

    Computed from the between-origin dispersion of the within-slope, which is
    exactly what the Stage 5 pilot estimates. Returned negative, since the
    alternative is one-sided and downward.
    """
    g = len(within_slopes)
    if g < 2:
        return float("nan")
    se = float(np.std(within_slopes, ddof=1) / math.sqrt(g))
    return -(_normal_quantile(1 - alpha) + _normal_quantile(power)) * se


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel akurasi arah & skala mentah</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">DA per rung dan rekonsiliasi MSE-scaler dengan RMSE mentah.</p>
</div>

In [ ]:


# -- drivers for the paper's tables (`D62a`) ---------------------------------
#
# Everything below reads what the grid already wrote. None of it trains and none
# of it needs a GPU. Each existed as a function nobody called, or as a number
# that lived only in `CLAUDE.md` prose and therefore fell outside §12's
# regenerability contract.


def directional_accuracy_table(run_ids: list[str], roots: list[Path]) -> pl.DataFrame:
    """DA at all three horizons for many runs (`D21`) -- an input to Table 4.

    :func:`directional_accuracy` has existed and been tested since the model
    plane was built and **was never called**: no DA figure appears in
    ``paper_numbers.json`` or anywhere in the session log. This is the driver.

    The three variants do not share a testing regime and the table keeps that
    visible rather than tidying it away. ``da_h1`` carries a Pesaran-Timmermann
    p-value on hourly spacing. ``da_hH`` and ``da_cum`` carry one **only** on the
    non-overlapping sample; their ``*_overlapping`` twins are descriptive and have
    no p-value at all, because on hourly spacing those targets overlap by 23 of
    24 hours, giving lag-1 autocorrelation near 23/24 -- PT's variance is then far
    too small and the test over-rejects badly. The resulting power loss is
    **stated** as ``n_non_overlapping``, never recovered by using the invalid
    sample.

    Args:
        run_ids: Runs to measure.
        roots: Artifact roots, working directory first.

    Returns:
        One row per run: its identity, all eight DA figures, and both sample sizes.
    """
    rows: list[dict[str, float | str | int]] = []
    for run_id in run_ids:
        parts = parse_run_id(run_id)
        meta = load_meta(run_id, roots)
        da = directional_accuracy(load_predictions(run_id, roots))
        rows.append(
            {
                "run_id": run_id,
                "model": str(parts["model"]),
                "origin": str(meta["origin"]),
                "origin_index": int(parts["origin_index"]),
                "k": int(parts["k"]),
                "pred_len": int(parts["pred_len"]),
                "seed": int(parts["seed"]),
                "da_h1": da.da_h1,
                "p_h1": da.p_h1,
                "da_hH": da.da_hH,
                "p_hH": da.p_hH,
                "da_hH_overlapping": da.da_hH_overlapping,
                "da_cum": da.da_cum,
                "p_cum": da.p_cum,
                "da_cum_overlapping": da.da_cum_overlapping,
                "n_h1": da.n_h1,
                "n_non_overlapping": da.n_non_overlapping,
            }
        )
    return pl.DataFrame(rows)


def raw_scale_table(seed_avg: pl.DataFrame) -> pl.DataFrame:
    """Add RMSE in raw log-return units -- root §9.1's second metric scale.

    "RMSE 0.0043 on hourly log-returns" tells a reader far more than "MSE 0.187
    on normalized data". Both scales are reported and ``sigma_g`` is stated, which
    is what lets the two reconcile. :func:`raw_rmse` has existed all along and,
    like :func:`directional_accuracy`, was never called.
    """
    return seed_avg.with_columns(
        (pl.col("mse").sqrt() * pl.col("sigma_g")).alias("rmse_raw")
    )


<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e9456088; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #f5a623; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧯 RelMSE falsifikasi & β₁ dengan cakupan</h4>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.9em;">Arm falsifikasi dilaporkan pada RelMSE, tidak pernah MSE ruang-scaler — 99,7% angka mentahnya adalah drift scaler (<code>D60i</code>, <code>D62e</code>).</p>
</div>

In [ ]:


def falsification_relmse(seed_avg: pl.DataFrame) -> pl.DataFrame:
    """``aged - fresh`` on **RelMSE**, per (origin, block) -- `D60i`.

    Root §8.1's falsification arm is the only design in the study that identifies
    decay directly, and the number reported for it was a units artefact. The
    notebook printed ``mean(aged - fresh) = -0.053341`` as a raw scaler-space MSE
    difference. The two arms are fitted at origins 90 days apart and therefore
    carry **different sigma_g** -- 0.009151 against 0.007297 at origin 1 -- so
    that difference compares numbers in different units. The matching naive
    baselines differ by -0.053196, i.e. about **99.7% of it is scaler drift**, and
    the sign reads backwards, appearing to say the aged model beat the fresh one.

    Root §9.1 already forbade the comparison by requiring RelMSE "to control for
    period difficulty". The arm was simply never brought under the rule, and the
    corrected figure lived only in prose. **The raw-MSE figure must not appear in
    the manuscript**, and the general rule this defect bought is that any
    cross-origin model comparison is on RelMSE or ``R2_oos``, never on
    scaler-space MSE.

    Returns:
        ``origin_index, origin, block, rel_aged, rel_fresh, gap_rel_mse`` over the
        cells the arm covers -- blocks 4-6 at each origin, which are the same
        calendar hours the aged model was scored on.
    """
    sel = seed_avg.filter(pl.col("pred_len") == 24)
    aged = (
        sel.filter((pl.col("model") == "itr") & (pl.col("k") == 8))
        .select(["origin_index", "origin", "block", "rel_mse"])
        .rename({"rel_mse": "rel_aged"})
    )
    fresh = (
        sel.filter(pl.col("model") == "itrf")
        .select(["origin_index", "block", "rel_mse"])
        .rename({"rel_mse": "rel_fresh"})
    )
    return (
        aged.join(fresh, on=["origin_index", "block"], how="inner")
        .with_columns((pl.col("rel_aged") - pl.col("rel_fresh")).alias("gap_rel_mse"))
        .sort(["origin_index", "block"])
    )


def per_origin_relmse(
    seed_avg: pl.DataFrame, model: str, k: int | None = None
) -> pl.DataFrame:
    """One RelMSE per origin for one arm: window-weighted over its test blocks.

    Seed-averaged MSEs first, ratio second, exactly as `D42` requires --- the two
    orders differ by Jensen, and the per-seed form would additionally have to pair
    seed 42 of one arm with seed 42 of another, which are independent training
    runs of different models.

    Args:
        seed_avg: :func:`seed_average`'s output.
        model: Model tag.
        k: Rung, when the tag carries more than one.

    Returns:
        ``origin, rel_mse, n_windows``, sorted by origin.
    """
    part = seed_avg.filter(
        (pl.col("model") == model) & (pl.col("pred_len") == PRED_LEN)
    )
    if k is not None:
        part = part.filter(pl.col("k") == k)
    return (
        part.group_by("origin")
        .agg(
            (pl.col("mse") * pl.col("n_windows")).sum().alias("_num"),
            (pl.col("mse_naive") * pl.col("n_windows")).sum().alias("_den"),
            pl.col("n_windows").sum().alias("n_windows"),
        )
        .with_columns((pl.col("_num") / pl.col("_den")).alias("rel_mse"))
        .drop(["_num", "_den"])
        .sort("origin")
    )


def paired_contrast(
    seed_avg: pl.DataFrame,
    left: tuple[str, int | None],
    right: tuple[str, int | None],
) -> dict:
    """Paired difference in RelMSE between two arms, across the origins they share.

    **The contrast the marginal columns cannot give you** (`D82`). Table 4 and
    Table 9 print each arm's mean RelMSE with its standard error *across* origins,
    and a reader differencing two such rows is comparing marginal spreads when the
    arms are evaluated on the same fifteen origins with the same naive baselines.
    The paired standard error is roughly half the marginal one here, so overlapping
    error bars in those tables say nothing about whether the arms differ.

    It is what RQ1's matched-K pair needs in particular. ``itro`` and ``itrr`` hold
    K = 8, the target and the seeds fixed and move only the participation ratio, so
    their difference is the direct K-versus-K_eff contrast the ladder can only
    infer through a panel at ``corr(K, K_eff) = 0.828``. Reporting them as two
    separate rows against the ladder leaves that difference uncomputed.

    Sign convention: **positive means the left arm is worse**, since a higher
    RelMSE is a worse forecast.

    The origin is the inferential unit (`D30`) and RelMSE is scale-free, so this
    never compares scaler-space MSEs fitted under different ``sigma_g`` (`D60i`).
    It is a **post-hoc** statistic with no multiplicity control of its own: it
    guides what belongs in the paper, and root §9.2's pre-registered machinery is
    what a confirmatory claim goes through.

    Args:
        seed_avg: :func:`seed_average`'s output.
        left: ``(model_tag, k)``; ``k`` may be None when the tag has one rung.
        right: The arm it is measured against.

    Returns:
        ``left, right, mean_diff, se, t, p_two_sided, ci_low, ci_high,
        n_origins, left_better``. ``p_two_sided`` is referred to ``t(G-1)``.
    """
    a = per_origin_relmse(seed_avg, left[0], left[1])
    b = per_origin_relmse(seed_avg, right[0], right[1])
    joined = a.join(b, on="origin", how="inner", suffix="_right").sort("origin")
    diff = (
        joined.get_column("rel_mse").to_numpy()
        - joined.get_column("rel_mse_right").to_numpy()
    )
    g = int(diff.size)
    label = lambda arm: arm[0] if arm[1] is None else f"{arm[0]}-K{arm[1]}"
    if g < 2:
        return {
            "left": label(left), "right": label(right), "n_origins": g,
            "mean_diff": None, "se": None, "t": None, "p_two_sided": None,
            "ci_low": None, "ci_high": None, "left_better": None,
        }

    mean = float(diff.mean())
    se = float(diff.std(ddof=1) / math.sqrt(g))
    if se > 0:
        t_stat = mean / se
        p = 2.0 * _upper_tail(abs(t_stat), g - 1)
        half = _t_critical(g - 1) * se
    else:
        # Identical at every origin. The attention arm is exactly this: it
        # reproduces the main grid bit for bit, so the contrast is a zero with no
        # spread and a t-statistic would be 0/0 (`D62d`).
        t_stat, p, half = (0.0, 1.0, 0.0) if mean == 0.0 else (math.inf, 0.0, 0.0)

    return {
        "left": label(left),
        "right": label(right),
        "mean_diff": mean,
        "se": se,
        "t": t_stat,
        "p_two_sided": float(p),
        "ci_low": mean - half,
        "ci_high": mean + half,
        "n_origins": g,
        "left_better": int((diff < 0).sum()),
    }


def _t_critical(df: int) -> float:
    """Two-sided 5% Student-t critical value, normal fallback without scipy."""
    try:
        from scipy import stats as _stats  # root §16's named stats boundary

        return float(_stats.t.ppf(0.975, df=df))
    except ImportError:  # pragma: no cover - scipy ships with the Kaggle image
        return 1.959963984540054


def beta1_with_coverage(
    panel: pl.DataFrame,
    min_coverage: float = 0.9,
    B: int = 99_999,
    seed: int = 42,
) -> tuple[Beta1Result, Beta1Result | None]:
    """beta1 on the full panel, and on well-covered blocks only (`D45`).

    Test-window survival is conditioned on **future** gaps -- whether a forecast
    issued at *s* is evaluated depends on whether the next 120 hours contain an
    outage, information unavailable at *s* -- and Binance outages cluster on
    stress. So within an origin the surviving sample composition trends, the
    dropped targets are systematically the high-volatility ones, and beta1 would
    absorb that trend as though it were decay. Root §9.2 requires either a
    coverage covariate or a re-estimate on well-covered blocks; this is the
    second.

    Restricting usually leaves an **unbalanced** panel, and
    :func:`_balanced_matrix` refuses one by design: beta1's reduction to the mean
    of within-slopes holds only when every origin carries every block. ``None``
    comes back in that case, and that is the honest report -- the check could not
    be run, not that it passed. Only a restriction that removes whole origins
    leaves something estimable.

    Args:
        panel: :func:`amplification`'s output, carrying ``n_large``.
        min_coverage: Surviving windows as a fraction of :data:`BLOCK_HOURS`.
        B: Bootstrap draws, passed through to :func:`panel_beta1`.
        seed: Bootstrap seed, passed through.

    Returns:
        ``(full, restricted_or_None)``.
    """
    full = panel_beta1(panel, B=B, seed=seed)
    restricted = panel.filter((pl.col("n_large") / float(BLOCK_HOURS)) >= min_coverage)
    try:
        return full, panel_beta1(restricted, B=B, seed=seed)
    except ValueError:
        # Unbalanced after the restriction. Loosening the estimator to produce a
        # number here would answer a different question than the one asked.
        return full, None


def panel_beta1_covariate(
    panel: pl.DataFrame,
    value: str = "A",
    covariate: str = "coverage",
    B: int = 99_999,
    seed: int = 42,
) -> Beta1Result:
    """``A(i,b) = alpha_i + beta1 b + beta2 c(i,b) + eps``, clustered on origin.

    Root §9.2 requires **either** block coverage as a covariate **or** beta1
    re-estimated on well-covered blocks. Until `D80` neither had been produced:
    :func:`beta1_with_coverage` attempts the second and honestly returns ``None``
    because restricting unbalances the panel, so the requirement was reported as
    unmet by one route and never attempted by the other. This is the other route,
    and unlike the restriction it always runs --- the panel stays balanced because
    nothing is dropped.

    Why it matters, in the terms `D45` puts it: test-window survival is
    conditioned on **future** gaps, outages cluster on stress, so within an origin
    the surviving sample composition trends and beta1 would absorb that trend as
    though it were decay. Adding coverage as a regressor asks what is left of
    beta1 once the trend it could be absorbing is accounted for.

    Estimated by Frisch-Waugh inside the origin fixed effects: both the outcome
    and the block index are within-demeaned and then residualised on the
    within-demeaned coverage, after which beta1 is the simple slope. That is
    algebraically the two-regressor fit and it lets the same cluster-robust
    sandwich and the same restricted wild bootstrap carry over unchanged --- WCR,
    bootstrapping the cluster-robust *t*, one-sided at ``H1: beta1 < 0``, with
    Rademacher and Webb weights and the ``(1 + count) / (1 + B)`` floor (`D42`,
    `D53d`).

    Args:
        panel: Balanced long frame with ``origin``, ``block``, ``value`` and
            ``covariate``.
        value: Dependent variable column.
        covariate: The control. Coverage as a fraction, not a count, so
            ``beta2`` reads per unit of surviving-window share.
        B: Bootstrap replications.
        seed: Bootstrap seed, recorded so the p-value is regenerable (root §12).

    Returns:
        A :class:`Beta1Result` whose ``beta1`` is the coverage-adjusted slope.
        ``within_slopes`` are the per-origin adjusted slopes, so they still
        average to ``beta1`` on a balanced panel.

    Raises:
        ValueError: If the panel is unbalanced, or if coverage has no
            within-origin variation at all --- in which case there is nothing to
            adjust for and :func:`panel_beta1` is the estimator that applies.
    """
    a, blocks = _balanced_matrix(panel, value)
    c, _ = _balanced_matrix(panel, covariate)
    g, n_blocks = a.shape

    within = a - a.mean(axis=1, keepdims=True)
    cw = c - c.mean(axis=1, keepdims=True)
    xw = np.tile(blocks - blocks.mean(), (g, 1))

    scc = float((cw * cw).sum())
    if scc <= 0.0:
        raise ValueError(
            "coverage has no within-origin variation, so it cannot be a "
            "covariate here; panel_beta1 is the estimator that applies"
        )

    # Frisch-Waugh: residualise the regressor of interest and the outcome on the
    # control, both already swept of origin means. beta1 and the residuals of the
    # two-regressor fit are then exactly those of the simple fit on the residuals.
    xr = xw - (float((xw * cw).sum()) / scc) * cw
    ar = within - (float((within * cw).sum()) / scc) * cw
    sxx = float((xr * xr).sum())
    if sxx <= 0.0:
        raise ValueError("the block index is collinear with coverage within origin")

    beta = float((ar * xr).sum() / sxx)
    resid = ar - beta * xr
    score = (resid * xr).sum(axis=1)
    variance = float((score * score).sum()) / sxx**2
    se = math.sqrt(variance) if variance > 0 else float("nan")
    t_obs = beta / se if se == se and se > 0 else float("nan")

    # Restricted residuals: imposing beta1 = 0 leaves alpha_i and beta2, and ar is
    # already swept of both, so u_tilde is ar itself. Per-cluster inner products
    # against the fixed regressors are all a bootstrap draw needs.
    ux = (ar * xr).sum(axis=1)
    uc = (ar * cw).sum(axis=1)
    xx = (xr * xr).sum(axis=1)
    cx = (cw * xr).sum(axis=1)

    def _p(kind: str) -> float:
        rng = np.random.default_rng(seed)
        weights = _weights(kind, (B, g), rng)
        beta_star = (weights @ ux) / sxx
        delta_star = (weights @ uc) / scc
        score_star = (
            weights * ux[None, :]
            - delta_star[:, None] * cx[None, :]
            - beta_star[:, None] * xx[None, :]
        )
        var_star = np.square(score_star).sum(axis=1) / sxx**2
        ok = var_star > 0
        t_star = beta_star[ok] / np.sqrt(var_star[ok])
        below = int(np.sum(t_star <= t_obs))  # H1: beta1 < 0, left tail
        return (1.0 + below) / (1.0 + int(ok.sum()))

    return Beta1Result(
        beta1=beta,
        t_statistic=t_obs,
        cluster_se=se,
        p_rademacher=_p("rademacher"),
        p_webb=_p("webb"),
        n_clusters=g,
        n_observations=g * n_blocks,
        within_slopes=(ar * xr).sum(axis=1) / (xr * xr).sum(axis=1),
        B=B,
    )


<div style="background: linear-gradient(135deg, #0a1a12, #0f2a1c); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #52b78833;">
  <h2 style="color: #95d5b2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🪶 11 &middot; Baseline
  </h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 1.02em;">Ridge, DLinear, PatchTST, LSTM, dan dua komparator naif — masing-masing dengan K eksplisit.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Setiap baseline membawa K</strong> (<code>D40</code>). Baseline channel-independent dengan K yang tak dinyatakan tidak bisa bicara soal debat channel-independence yang jadi pilar Related Work.</li>
    <li style="margin-bottom: 6px;">K=8 milik DLinear dan PatchTST berarti <em>dilatih pada</em> delapan kanal lewat bobot bersama, <strong>bukan</strong> meramal target dari delapan kanal (<code>D56</code>). K=8 milik LSTM dan ridge berarti yang kedua. Nyatakan itu di mana pun angkanya muncul.</li>
    <li style="margin-bottom: 6px;">Ridge menjawab pertanyaan yang <code>D17</code> ajukan — apakah transformer dibutuhkan sama sekali? Terukur: <strong>tidak</strong>. Rata-rata <code>R&sup2;_oos</code> ridge &minus;0,00057 lawan &minus;0,0180 iTransformer-K8 dan &minus;0,0262 DLinear (<code>D60c</code>).</li>
    <li style="margin-bottom: 6px;">ARIMA tetap ditangguhkan <strong>dengan alasan tertulis</strong>: pada log-return crypto per jam, seleksi AIC mendarat di sekitar order (0,0,0) — yang <em>adalah</em> baseline naif (<code>D64</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 4px solid #52b788; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #95d5b2; margin: 0 0 6px 0; font-size: 1.22em;">📘 baselines.py</h3>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.94em;">Ridge, DLinear, PatchTST — masing-masing dengan K eksplisit (<code>D40</code>) dan objektif terbitannya sendiri (<code>D56</code>).</p>
</div>

<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & protokol baseline</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Setiap baseline membawa K eksplisit (<code>D40</code>), dan yang channel-independent membawa objektif all-channel terbitannya sendiri (<code>D56</code>).</p>
</div>

In [ ]:
"""Ridge, DLinear and PatchTST — the comparators root §7 calls mandatory (`D56`).

Until the 534-run grid finished on 2026-08-08 this module did not exist, and
neither did any other baseline class. Root §7 says DLinear and PatchTST are "not
optional", root §10.2 budgets 255 baseline runs, and
:func:`itransformer_btc.metrics.dm_nonnested` has been sitting ready for input
that was never produced — so §10.2's 789 was never executable and Table 6 had no
inputs. The consequence is not bookkeeping: with Naive-RW the only comparator,
"iTransformer has no edge" rests on one contrast, and §6.2/`D38` says the
hyperparameters were adopted unchanged and never tuned. A referee reads that null
as an immature configuration rather than a finding. These three models are the
minimum that answers the question deciding what the negative result means:
**did iTransformer fail, or did the whole LTSF class fail here?**

What is built, and what each one is for:

=========  ==========  =====================================================
Model      K           Question it answers
=========  ==========  =====================================================
Ridge      1, 4, 8, 12 `D17` — is a transformer needed at all? Linear and
                       genuinely multivariate, so it separates *does the
                       information help* from *does attention help*
DLinear    8           root §7 — the first thing an LTSF-literate reviewer
                       looks for. Linear, decomposition-based, ~4.7k weights
PatchTST   8           root §7 — SOTA and channel-independent, the other side
                       of the debate §13.1 makes a Related Work pillar
=========  ==========  =====================================================

**What "K = 8" means for a channel-independent model, stated because it is not
what it means elsewhere in this study.** DLinear and PatchTST forecast each
channel from that channel's own history; that is the architecture's claim, not a
shortcoming of this implementation. They are therefore trained with their
published **all-channel** objective and their weights are **shared across
channels**, which is the only route by which the other seven variates reach the
target's forecast at all. Trained on the target channel alone they would be K=1
wearing a K=8 label — the exact collapse `D40` was written to prevent — and the
paper's central architectural comparison would quietly become
univariate-versus-multivariate again. Both facts are recorded as fields in every
``meta/*.json`` (``loss_channels``, ``channel_independent``) so no reader has to
infer them. Ridge carries no such caveat: every one of its ``L x K`` inputs
enters the target's forecast, so its K label means what K means in §5.2.

**Capacity is held fixed rather than tuned.** PatchTST takes iTransformer's
``d_model``, ``d_ff``, ``e_layers``, ``n_heads`` and ``dropout`` and reuses
:class:`itransformer_btc.model.EncoderLayer` itself, so the two models differ in
**what a token is** — a patch of one variate against the whole lookback of each
variate — and in nothing else. That is the cleanest available form of the
contrast, and it extends §6.2/`D38`'s no-tuning posture to the baselines instead
of quietly exempting them.

**Scale space is identical to the ladder's** (root §6.3, §11). Every model reads
the same ``StandardScaler`` output fitted on the same 21-month sub-block. What
differs is internal normalisation, and it differs the way the published models
do: PatchTST normalises per window (RevIN — the same operation ``use_norm=True``
applies), DLinear and ridge do not, which is precisely the case §6.3 says the
outer scaler exists to serve.

**Not built here, and not silently dropped.** ARIMA, LSTM, naive-persist and
seasonal-naive are deferred from this minimal set. Naive-RW needs no run at all —
:func:`itransformer_btc.metrics.block_metrics` computes it from ``naive_rw_z`` on
exactly the rows the model was scored on. If the other four are eventually cut,
root §7 must be edited with a written reason rather than left standing over
models nobody built.

Upstream
--------
**Two of these are reimplemented from published architectures; three have no
upstream code at all.** That distinction is the answer to "where did this come
from", so it is drawn per model rather than left to a blanket acknowledgement.

- **DLinear**, ``SeriesDecomposition`` — A. Zeng, M. Chen, L. Zhang, and Q. Xu,
  "Are transformers effective for time series forecasting?," in *Proc. 37th
  AAAI Conf. Artif. Intell.*, 2023, pp. 11121-11128. arXiv:2205.13504.
  Official code: https://github.com/cure-lab/LTSF-Linear (Apache-2.0; accessed
  2026-09-03). Reimplemented; the published all-channel objective, shared
  weights and centred moving average are kept as published (`D40`, `D56`).
- **PatchTST** — Y. Nie, N. H. Nguyen, P. Sinthong, and J. Kalagnanam, "A time
  series is worth 64 words: Long-term forecasting with transformers," in *Proc.
  11th Int. Conf. Learn. Represent. (ICLR)*, 2023. arXiv:2211.14730. Official
  code: https://github.com/yuqinie98/PatchTST (Apache-2.0; accessed
  2026-09-03). Reimplemented on this study's own
  :class:`itransformer_btc.model.EncoderLayer`, so it differs from iTransformer
  in what a token is and in nothing else. Patch 16 / stride 8 as published.
- **LSTMForecaster** — ``torch.nn.LSTM``
  (https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html,
  BSD-3-Clause; accessed 2026-09-03) is called directly and only the forecasting
  head is written here. S. Hochreiter and J. Schmidhuber, "Long short-term
  memory," *Neural Computation*, vol. 9, no. 8, pp. 1735-1780, 1997.
- **RidgeForecaster** — **not scikit-learn.** The normal equations are solved
  here in ``float64``. A. E. Hoerl and R. W. Kennard, "Ridge regression: Biased
  estimation for nonorthogonal problems," *Technometrics*, vol. 12, no. 1,
  pp. 55-67, 1970.
- **NaiveForecaster**, Naive-RW — closed forms, no upstream code.
  R. J. Hyndman and G. Athanasopoulos, *Forecasting: Principles and Practice*,
  3rd ed. OTexts, 2021.

:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries every row in the same
form, with the licence and the full list of departures.
"""

class BaselineModule(nn.Module):
    """Shared plumbing for the two channel-independent baselines.

    ``forward`` returns all N channels because that is what the published
    objective supervises; ``forecast_target`` is the projection root §10.4's
    prediction file actually holds. Channel ``TARGET_INDEX`` is ``r`` at every
    rung — the ladder pins it there, see
    :data:`itransformer_btc.features.VARIATE_ORDER` — so this is one constant
    rather than a lookup.
    """

    def n_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, N) -> (B, H)`` on the target channel."""
        return self(x)[:, :, TARGET_INDEX]


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📏 Ridge — konfigurasi</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">α dipilih pada sub-blok validasi. <strong>Satu-satunya hiperparameter yang dipilih di mana pun dalam studi ini.</strong></p>
</div>

In [ ]:


# -- ridge -------------------------------------------------------------------


#: Root §11: ridge alpha is selected on the validation sub-block, and with ARIMA
#: outside the minimal set it is the **only** hyperparameter selected anywhere in
#: this study (`D38`). The solve is unnormalised — ``(X'X + a I) W = X'Y`` — so
#: the scale that matters is ``diag(X'X) ~ n``, about 1.4e4 at these origins; the
#: grid spans five orders below it and two above.
RIDGE_ALPHAS: tuple[float, ...] = (1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6)


@dataclass(frozen=True, slots=True)
class RidgeConfig:
    """L2-regularised linear map from the flattened window to the H-step target.

    `D17`: K=1 iTransformer controls for *architecture* — it answers "does
    cross-variate attention help?" It does not answer "is a transformer needed at
    all?" Ridge on the same K features separates *does the information help* from
    *does attention help*, at seconds per run, and closes a question a reviewer
    asks otherwise.

    ``k`` is a field here and nowhere else among the study's configs.
    iTransformer's parameter count is identical at every rung because K changes
    the token count and not a weight shape; ridge's weight matrix is
    ``(L*K, H)``, so K is part of its geometry and belongs in ``meta['config']``.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    k: int = 8
    alphas: tuple[float, ...] = RIDGE_ALPHAS
    #: Chosen by :meth:`fit` on the validation sub-block. ``None`` in an unfitted
    #: config and never in a written ``meta/*.json`` — root §12 cannot regenerate
    #: a number whose only free parameter went unrecorded.
    alpha: float | None = None

    def build(self) -> "RidgeForecaster":
        return RidgeForecaster(self)

    def loss_target(self) -> str:
        """``"target"``: ridge predicts the target channel and nothing else."""
        return "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["RidgeForecaster", "RidgeConfig", TrainOutcome]:
        """Solve the normal equations once, then pick alpha on validation.

        The Gram matrix and the right-hand side are built **once** and reused for
        every alpha, so the sweep costs one solve per candidate rather than a
        refit. In ``float64``: at ``L*K = 1152`` the design is conditioned badly
        enough that a ``float32`` Gram would make the smallest alphas report
        noise, and showing what an essentially unregularised linear map does is
        the whole reason the small alphas are in the grid.

        The intercept is fitted by centring and is **not** penalised. Shrinking
        it toward zero would shrink the forecast toward zero *in scaler space*,
        which is ``r = mu_g`` — the constant-drift model `D31` spent a section
        removing from the Naive-RW baseline.
        """
        device = device or pick_device()
        # A solve consumes no RNG. Seeded anyway, so a ridge run and an
        # iTransformer run of the same cell are reproducible under one rule
        # (root §16) rather than two.
        started = time.perf_counter()

        # Seeding and construction under one lock, as in ``train_one``: both draw
        # from the CPU generator, which every worker shares (`D68`).
        with SEED_LOCK:
            set_seed(spec.seed, device)
            model = self.build().to(device)
        x_tr = self._design(tensors.train.x, device)
        y_tr = torch.from_numpy(tensors.train.y).to(device).double()
        x_va = self._design(tensors.val.x, device)
        y_va = torch.from_numpy(tensors.val.y).to(device).double()

        x_mean, y_mean = x_tr.mean(0), y_tr.mean(0)
        x_tr -= x_mean
        y_tr -= y_mean
        gram = x_tr.T @ x_tr
        rhs = x_tr.T @ y_tr
        eye = torch.eye(gram.shape[0], dtype=gram.dtype, device=gram.device)

        best: tuple[float, float, Tensor] | None = None
        for alpha in self.alphas:
            weight = torch.linalg.solve(gram + alpha * eye, rhs)
            residual = (x_va - x_mean) @ weight + y_mean - y_va
            val_mse = float(residual.pow(2).mean())
            if best is None or val_mse < best[0]:
                best = (val_mse, float(alpha), weight)
        if best is None:
            raise ValueError("no ridge alpha to select; `alphas` is empty")

        val_mse, alpha, weight = best
        if len(self.alphas) > 1 and alpha in (self.alphas[0], self.alphas[-1]):
            # Not a failure. An alpha pinned at the top of the grid says the
            # least-squares fit is worthless and the best linear predictor is the
            # training mean, which is a finding. It is warned about because a
            # boundary selection is also what an unbracketed grid looks like, and
            # the two are indistinguishable from the number alone.
            warnings.warn(
                f"{spec.run_id}: ridge alpha {alpha:g} sits at the edge of "
                f"{self.alphas}; the grid may not bracket the optimum",
                stacklevel=2,
            )

        with torch.no_grad():
            model.weight.copy_(weight.to(torch.float32))
            model.bias.copy_((y_mean - x_mean @ weight).to(torch.float32))
        train_mse = float((x_tr @ weight - y_tr).pow(2).mean())

        return (
            model,
            replace(self, alpha=alpha),
            TrainOutcome(
                run_id=spec.run_id,
                # A solve, not a loop. Zero is the honest number, and it is what
                # tells a reader of Table 3 why this row has no epochs-to-stop.
                epochs_run=0,
                best_val_mse=val_mse,
                train_loss=train_mse,
                wall_time_s=time.perf_counter() - started,
                n_parameters=model.n_parameters(),
                device=str(device),
            ),
        )

    @staticmethod
    def _design(x: np.ndarray, device: torch.device) -> Tensor:
        """``(n, L, K) -> (n, L*K)`` in float64, on the device.

        Row-major, so a column is one (lag, variate) pair. Nothing depends on
        which ordering it is, only that this and
        :meth:`RidgeForecaster.forward` agree — which they do by both being
        ``reshape``.
        """
        return torch.from_numpy(x).to(device).reshape(len(x), -1).double()


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">➗ Ridge — forecaster</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">L2 pada fitur K yang sama. Menjawab pertanyaan <code>D17</code>: apakah transformer dibutuhkan sama sekali?</p>
</div>

In [ ]:


class RidgeForecaster(nn.Module):
    """``y_hat = vec(x) @ W + b``, fitted in closed form.

    ``W`` and ``b`` are **buffers**, not parameters: nothing here is trained by
    gradient descent, and registering them as parameters would put them in front
    of an optimiser that must never see them. That is why :meth:`n_parameters`
    counts them explicitly — the usual sum over ``self.parameters()`` would
    report zero, and root §12 would record a model with no coefficients.
    """

    def __init__(self, cfg: RidgeConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.register_buffer(
            "weight",
            torch.zeros(cfg.seq_len * cfg.k, cfg.pred_len, dtype=torch.float32),
        )
        self.register_buffer("bias", torch.zeros(cfg.pred_len, dtype=torch.float32))

    def forward(self, x: Tensor) -> Tensor:
        """``(B, L, K) -> (B, H)``."""
        return x.reshape(len(x), -1) @ self.weight + self.bias

    def forecast_target(self, x: Tensor) -> Tensor:
        return self(x)

    def n_parameters(self) -> int:
        return self.weight.numel() + self.bias.numel()


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📉 DLinear — konfigurasi</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Dekomposisi tren–musiman plus linear.</p>
</div>

In [ ]:


# -- DLinear -----------------------------------------------------------------


@dataclass(frozen=True, slots=True)
class DLinearConfig:
    """Trend-seasonal decomposition plus two linear maps (Zeng et al., 2023).

    Root §7 calls it mandatory for a reason worth stating: a missing DLinear is
    the first thing a reviewer familiar with the LTSF literature flags, because
    it is the model that showed a linear map beating several transformers on the
    standard benchmarks. Against a null result it does more than that — if ~4.7k
    weights also fail here, the failure belongs to the problem and not to
    attention.

    Weights are **shared across channels**, never per-channel. The published
    implementation offers both; only the shared form lets the all-channel
    objective carry information from the other seven variates into the target's
    forecast, which is what makes the K=8 label true (see this module's header).
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    #: Odd, so the decomposition's padding is symmetric. 25 is the published
    #: default and is not tuned here (`D38`, extended to the baselines).
    moving_avg: int = 25
    #: Recorded in ``meta/*.json`` rather than left to be inferred: see the
    #: header. A reader who does not know the objective cannot read
    #: ``best_val_mse``, which is an all-channel figure for this model and a
    #: target-channel one for the ladder.
    loss_channels: str = "all"
    channel_independent: bool = True

    def build(self) -> "DLinear":
        return DLinear(self)

    def loss_target(self) -> str:
        return "all" if self.loss_channels == "all" else "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["DLinear", "DLinearConfig", TrainOutcome]:
        """Root §6.2's schedule; nothing is selected, so the config returns as given."""
        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪚 DLinear — dekomposisi & model</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Moving average-nya <em>terpusat</em> — dan §8.3 selamat: rata-ratanya dihitung dari 96 bar jendela itu sendiri, seluruhnya mendahului jam ramalan pertama.</p>
</div>

In [ ]:


class SeriesDecomposition(nn.Module):
    """Moving-average trend and the residual seasonal component.

    **This is a rolling window inside a model, and root §5.3's ban is on rolling
    *features*. The distinction is not a technicality, so here is the argument.**
    The ban exists because a rolling feature computed over the full series can let
    a later bar reach an earlier feature value — the ``center=True`` leak class —
    and root §8.3's no-embargo justification rests on no feature having one. This
    average is computed at inference time from the 96 bars of the window itself,
    every one of which precedes the first forecast hour, and the padding
    replicates the window's own endpoints rather than reaching outside it. No
    test-period bar can therefore influence any training-set value, which is the
    property §8.3 actually needs, and it holds even though the average is centred
    **within** the window, as the published DLinear's is. Reproducing the
    published decomposition matters: a causal variant would be a different model,
    and the question this baseline exists to answer is about DLinear.
    """

    def __init__(self, kernel: int) -> None:
        super().__init__()
        self.kernel = kernel
        self.average = nn.AvgPool1d(kernel, stride=1, padding=0)

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        """``(B, L, N) -> (seasonal, trend)``, both ``(B, L, N)``."""
        front_pad = (self.kernel - 1) // 2
        padded = torch.cat(
            [
                x[:, :1, :].repeat(1, front_pad, 1),
                x,
                x[:, -1:, :].repeat(1, self.kernel - 1 - front_pad, 1),
            ],
            dim=1,
        )
        trend = self.average(padded.permute(0, 2, 1)).permute(0, 2, 1)
        return x - trend, trend


class DLinear(BaselineModule):
    """``(B, L, N) -> (B, H, N)``: decompose, map each part linearly, add.

    No instance normalisation, as published — which is exactly the case root §6.3
    says the outer ``StandardScaler`` exists to serve, so this model reads the
    same scaler space as every other and needs nothing of its own.
    """

    def __init__(self, cfg: DLinearConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.decomposition = SeriesDecomposition(cfg.moving_avg)
        self.seasonal = nn.Linear(cfg.seq_len, cfg.pred_len)
        self.trend = nn.Linear(cfg.seq_len, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        seasonal, trend = self.decomposition(x)
        out = self.seasonal(seasonal.permute(0, 2, 1)) + self.trend(
            trend.permute(0, 2, 1)
        )
        return out.permute(0, 2, 1)


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧩 PatchTST — konfigurasi</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Patch 16, stride 8.</p>
</div>

In [ ]:


# -- PatchTST ----------------------------------------------------------------


@dataclass(frozen=True, slots=True)
class PatchTSTConfig:
    """Patched, channel-independent transformer (Nie et al., 2023).

    The other side of the channel-independence debate root §13.1 makes a Related
    Work pillar, and the comparison `D40` says an LTSF-literate reviewer wants:
    iTransformer at K=8 against PatchTST at K=8, on the same information, the same
    windows and the same scale space.

    Capacity is iTransformer's, field for field, and the encoder block is
    literally :class:`itransformer_btc.model.EncoderLayer`. The two models
    therefore differ in **what a token is** — a patch of one variate here, one
    variate's entire lookback there — and in nothing else. Root §6.2/`D38`'s
    no-tuning rule applies unchanged.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    #: Root §7's committed geometry. ``(96 - 16) / 8 + 1 = 11`` patches, with no
    #: end-padding patch: the published option that adds one is a convenience for
    #: lookbacks the stride does not divide evenly, and 96 is not one of those.
    patch_len: int = 16
    stride: int = 8
    d_model: int = 128
    d_ff: int = 256
    e_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.1
    #: Reversible instance normalisation, as published. It is the **same**
    #: operation ``use_norm=True`` applies in :class:`ITransformer` — per window,
    #: per channel — so the two models are normalised alike and root §6.3's
    #: cross-model scale consistency holds.
    revin: bool = True
    loss_channels: str = "all"
    channel_independent: bool = True

    @property
    def n_patches(self) -> int:
        return (self.seq_len - self.patch_len) // self.stride + 1

    def build(self) -> "PatchTST":
        return PatchTST(self)

    def loss_target(self) -> str:
        return "all" if self.loss_channels == "all" else "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["PatchTST", "PatchTSTConfig", TrainOutcome]:
        """Root §6.2's schedule; nothing is selected, so the config returns as given."""
        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔷 PatchTST — model</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Memakai ulang blok encoder dan kapasitas iTransformer <strong>verbatim</strong>, jadi keduanya berbeda hanya pada <em>apa itu token</em>.</p>
</div>

In [ ]:


class PatchTST(BaselineModule):
    """``(B, L, N) -> (B, H, N)``, each channel processed as its own sequence."""

    def __init__(self, cfg: PatchTSTConfig) -> None:
        super().__init__()
        self.cfg = cfg
        if (cfg.seq_len - cfg.patch_len) % cfg.stride:
            raise ValueError(
                f"seq_len {cfg.seq_len} and patch_len {cfg.patch_len} leave "
                f"{(cfg.seq_len - cfg.patch_len) % cfg.stride} bars uncovered at "
                f"stride {cfg.stride}. Dropping the tail of every window would "
                f"make this model's lookback shorter than the ladder's, and the "
                f"comparison would no longer be on identical information."
            )
        # EncoderLayer reads d_model, d_ff, n_heads, dropout and
        # uniform_attention; its remaining fields are inert here and stay at
        # their defaults. Reusing the block rather than reimplementing it is what
        # makes "same capacity, different tokenisation" a fact and not a claim.
        block = ITransformerConfig(
            d_model=cfg.d_model,
            d_ff=cfg.d_ff,
            n_heads=cfg.n_heads,
            dropout=cfg.dropout,
        )
        self.embedding = nn.Linear(cfg.patch_len, cfg.d_model)
        self.position = nn.Parameter(torch.zeros(cfg.n_patches, cfg.d_model))
        nn.init.normal_(self.position, std=0.02)
        self.dropout = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList(EncoderLayer(block) for _ in range(cfg.e_layers))
        self.head = nn.Linear(cfg.n_patches * cfg.d_model, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        b, length, n = x.shape
        mean = std = None
        if self.cfg.revin:
            mean = x.mean(dim=1, keepdim=True)
            x = x - mean
            std = torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-5)
            x = x / std

        # (B, L, N) -> (B*N, L). Folding the channels into the batch **is**
        # channel independence: from here on nothing in the network sees two
        # variates at once, which is the property under test.
        series = x.permute(0, 2, 1).reshape(b * n, length)
        patches = series.unfold(1, self.cfg.patch_len, self.cfg.stride)
        h = self.dropout(self.embedding(patches) + self.position)
        for layer in self.layers:
            h = layer(h)
        out = self.head(h.reshape(b * n, -1)).reshape(b, n, self.cfg.pred_len)
        out = out.permute(0, 2, 1)

        if self.cfg.revin:
            out = out * std[:, 0, :].unsqueeze(1) + mean[:, 0, :].unsqueeze(1)
        return out


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔁 LSTM — konfigurasi</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Dua layer, hidden 128, dropout 0,1 — diadopsi dari root §7, tidak di-tune. <strong>Multivariat</strong>, bukan channel-independent: K=8-nya berarti apa yang ridge dan iTransformer maksud (<code>D64</code>).</p>
</div>

In [ ]:


# -- LSTM --------------------------------------------------------------------


@dataclass(frozen=True, slots=True)
class LSTMConfig:
    """Two layers, hidden 128, dropout 0.1 — root §7, adopted not tuned (`D38`).

    **Multivariate, not channel-independent, and the distinction is the point.**
    DLinear and PatchTST wear their K=8 label through an all-channel objective
    with shared weights: they are *trained on* eight channels but predict the
    target from its own history alone (`D56`). An LSTM reads all K channels of
    every timestep and emits the target directly, so its K=8 means what ridge's
    and iTransformer's mean. That makes it the only recurrent point of comparison
    on the same information set the ladder uses.

    It is here because it is the model the crypto forecasting literature this
    paper argues against reaches for first. Claiming "no deep model beats
    Naive-RW" while leaving the most-cited deep model untested is a hole a
    reviewer finds in one pass.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    k: int = 8
    hidden: int = 128
    layers: int = 2
    dropout: float = 0.1
    #: Target-channel, like the ladder (`D39`) and unlike the two
    #: channel-independent baselines. Logged so a reader of Table 3 can see that
    #: this model's ``best_val_mse`` *is* comparable to the ladder's.
    loss_channels: str = "target"
    channel_independent: bool = False

    def build(self) -> "LSTMForecaster":
        return LSTMForecaster(self)

    def loss_target(self) -> str:
        return "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["LSTMForecaster", "LSTMConfig", TrainOutcome]:
        """Root §6.2's schedule; nothing is selected, so the config returns as given."""
        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧬 LSTM — forecaster</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Hidden state terakhir langsung ke <code>Linear(hidden, H)</code>. Model deep paling sering disitir literatur crypto yang paper ini lawan.</p>
</div>

In [ ]:


class LSTMForecaster(nn.Module):
    """``(B, L, K) -> (B, H)`` from the last hidden state.

    No instance normalisation, as is standard for this baseline — root §6.3's
    outer ``StandardScaler`` is what serves that case, so this model reads the
    same scaler space as every other and needs nothing of its own.
    """

    def __init__(self, cfg: LSTMConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.lstm = nn.LSTM(
            input_size=cfg.k,
            hidden_size=cfg.hidden,
            num_layers=cfg.layers,
            batch_first=True,
            dropout=cfg.dropout if cfg.layers > 1 else 0.0,
        )
        self.head = nn.Linear(cfg.hidden, cfg.pred_len)

    def forward(self, x: Tensor) -> Tensor:
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

    def forecast_target(self, x: Tensor) -> Tensor:
        return self(x)

    def n_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪶 Dua komparator naif</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;"><code>persist</code> mengulang return terakhir; <code>seasonal</code> mengulang return satu siklus harian ke belakang. Nol parameter, K=1, dan berbeda dari Naive-RW yang meramalkan <code>y_raw = 0</code> (<code>D31</code>).</p>
</div>

In [ ]:


# -- the two naive comparators -----------------------------------------------


#: Hours in the seasonal cycle a daily pattern would repeat on.
SEASONAL_PERIOD: int = 24


@dataclass(frozen=True, slots=True)
class NaiveConfig:
    """``persist`` and ``seasonal`` — root §7's two secondary naive comparators.

    Neither trains and neither has a parameter, so both cost microseconds and
    exist purely to close a hole: root §7 listed them and `D56` recorded, in
    writing, that nobody had built them. Two rows marked *deferred* in a results
    table read as unfinished work, and these are the cheapest rows in the study.

    Both read the target channel and nothing else, so their honest K is **1**
    (`D40` requires the label; it does not require the label to be large). They
    are distinct from Naive-RW, which forecasts ``y_hat_raw = 0`` and needs no run
    at all (`D31`):

    - ``persist`` repeats the last observed return for all H steps. Root §7 calls
      it a weaker baseline than Naive-RW and that is the expected result; the
      point is to show it rather than assert it.
    - ``seasonal`` repeats the return one daily cycle back, step for step. It is
      the comparator for any claim that hourly crypto carries a daily pattern.
    """

    mode: str = "persist"
    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    k: int = 1
    loss_channels: str = "target"
    channel_independent: bool = False

    def build(self) -> "NaiveForecaster":
        return NaiveForecaster(self)

    def loss_target(self) -> str:
        return "target"

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["NaiveForecaster", "NaiveConfig", TrainOutcome]:
        """No fit. The two split losses are still measured and reported.

        ``epochs_run=0`` is the honest number and it is what tells a reader of
        Table 3 why these rows have no epochs-to-stop, exactly as for ridge.
        """
        device = device or pick_device()
        # Consumes no RNG. Seeded anyway so every arm is reproducible under one
        # rule rather than two (root §16).
        started = time.perf_counter()

        with SEED_LOCK:
            set_seed(spec.seed, device)
            model = self.build().to(device)

        def split_mse(split) -> float:
            x = torch.from_numpy(split.x).to(device)
            y = torch.from_numpy(split.y).to(device)
            with torch.no_grad():
                return float((model.forecast_target(x) - y).pow(2).mean())

        return (
            model,
            self,
            TrainOutcome(
                run_id=spec.run_id,
                epochs_run=0,
                best_val_mse=split_mse(tensors.val),
                train_loss=split_mse(tensors.train),
                wall_time_s=time.perf_counter() - started,
                n_parameters=0,
                device=str(device),
            ),
        )


class NaiveForecaster(nn.Module):
    """``(B, L, K) -> (B, H)`` by copying a past value of the target channel.

    Stateless: no parameters, no buffers, nothing to move to a device beyond the
    module shell. ``n_parameters`` returns 0 and that is the truthful figure —
    unlike ridge, where the coefficients are buffers and the usual sum would
    under-report (root §12).
    """

    def __init__(self, cfg: NaiveConfig) -> None:
        super().__init__()
        if cfg.mode not in ("persist", "seasonal"):
            raise ValueError(f"unknown naive mode {cfg.mode!r}")
        self.cfg = cfg

    def forward(self, x: Tensor) -> Tensor:
        target = x[:, :, TARGET_INDEX]
        if self.cfg.mode == "persist":
            return target[:, -1:].expand(-1, self.cfg.pred_len)
        length = target.shape[1]
        # Modulo, so a horizon longer than one cycle repeats the cycle instead of
        # indexing past the lookback. At the H=24 arms it is the identity.
        index = [
            length - SEASONAL_PERIOD + (h % SEASONAL_PERIOD)
            for h in range(self.cfg.pred_len)
        ]
        return target[:, index]

    def forecast_target(self, x: Tensor) -> Tensor:
        return self(x)

    def n_parameters(self) -> int:
        return 0


<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔗 Penyelarasan jendela baseline</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Timestamp yang dievaluasi diasersi sama sebelum RelMSE dihitung — fatal untuk setiap pasangan selain Naive-RW.</p>
</div>

In [ ]:


# -- the `D45` assertion -----------------------------------------------------


def assert_baseline_alignment(
    baseline_run_id: str, reference_run_id: str, roots: list[Path]
) -> None:
    """`D45` — a baseline may only be scored on its comparator's exact windows.

    Root §7: "Baselines are scored on exactly the same surviving windows." Unless
    that holds, RelMSE is a ratio across two samples rather than a ratio, and the
    two samples would differ systematically rather than randomly: test-window
    survival is conditioned on *future* gaps (root §4.3) and Binance outages
    cluster on stress, so the windows one model kept and the other dropped are
    disproportionately the high-volatility ones.

    Here the sets are equal by construction — both come from
    :func:`itransformer_btc.splits.window_starts` with the same origin, span and
    ``"origin"`` semantics — and that is exactly why the assertion is cheap, and
    why it is the only thing that would notice if it ever stopped being true.
    Root §4.3 names positional-index drift after a row drop as the
    highest-probability silent bug in this pipeline; this is its detector on the
    cross-model axis.

    Raises:
        ValueError: If the evaluated ``(block, timestamp)`` sets differ.
        FileNotFoundError: If either run is absent from ``roots``.
    """
    assert_same_windows(
        load_predictions(baseline_run_id, roots),
        load_predictions(reference_run_id, roots),
        f"{baseline_run_id} vs {reference_run_id}",
    )


<div style="background: linear-gradient(135deg, #2b0a00, #3d1000); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #ff7b5433;">
  <h2 style="color: #ffb4a2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    ⚔️ 12 &middot; Perbandingan model
  </h2>
  <p style="color: #ffd8c2; margin: 0; font-size: 1.02em;">Matriks pasangan, stepdown Romano&ndash;Wolf, dan Model Confidence Set (<code>D35</code>).</p>
  <ul style="color: #ffd8c2; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">SPA dan Reality Check menguji null <em>satu-lawan-banyak</em> dan tidak mengatakan apa pun tentang matriks semua-pasangan yang Tabel 6 muat. Romano&ndash;Wolf yang mengendalikan FWER lintas seluruh pasangan.</li>
    <li style="margin-bottom: 6px;">Terukur: stepdown itu <strong>menghapus setiap penolakan terhadap Naive-RW</strong> — 8 dari 11 model menolak mentah di &alpha; = 0,05, nol setelah adjustment, p &ge; 0,336 (<code>D62a</code>).</li>
    <li style="margin-bottom: 6px;">MCS di 90% maupun 75% memuat Naive-RW dan keempat rung ridge, dan <strong>tidak satu pun model deep</strong>.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 4px solid #ff7b54; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffb4a2; margin: 0 0 6px 0; font-size: 1.22em;">📘 comparisons.py</h3>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.94em;">Matriks pasangan, stepdown Romano–Wolf, dan Model Confidence Set (<code>D35</code>).</p>
</div>

<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Level MCS 90% dan 75%, dan B bootstrap bawaan.</p>
</div>

In [ ]:
"""Table 6: every pair, the right statistic for each, and honest multiplicity.

Root §7 calls this the comparison an LTSF-literate reviewer wants, and `D40`
gave every baseline an explicit K precisely so it could be built. `D56` supplied
the missing models; `D60g` found the missing *call* -- the session log contains
zero Diebold-Mariano, Romano-Wolf or Model Confidence Set lines. This module is
that call.

Four decisions here are load-bearing, and each is pinned by root §9.2 rather
than chosen locally.

**Which statistic.** The comparisons that carry the paper are **nested**: the
ladder is cumulative, so K=1's feature set is a strict subset of K=8's under one
architecture; Naive-RW is nested inside every model in §7; Ridge-K1 inside
Ridge-K8. Under the null of equal population predictive ability with nested
models and estimated parameters, the loss differential has a mean shifted away
from zero -- the larger model's extra estimation noise makes it look worse --
and the statistic is not asymptotically ``N(0,1)`` (Clark & McCracken 2001;
McCracken 2007). Standard DM is therefore systematically **undersized against
the alternative this study exists to establish**. Nested pairs take Clark-West;
non-nested pairs take DM. Which one ran is a **column of the output**, never an
inference the reader has to make.

**Which multiplicity correction.** White's Reality Check (2000) and Hansen's SPA
(2005) test a *one-against-many* null and return one p-value for that composite;
they say nothing about the all-pairs matrix this table is (`D35`). With ten
models the matrix holds 45 tests and at alpha = 0.05 expects ~2.3 spurious
rejections under a complete null. Romano-Wolf (2005) stepdown controls FWER
across all pairs and is bootstrap-based like the machinery already here. The
Model Confidence Set answers the question a reader actually has -- *which models
are indistinguishable from the best* -- which the pairwise matrix does not.

**What the bootstrap resamples.** Origins, not observations. Blocks within an
origin come from one trained model, and root §9.2 fixes the inferential unit at
the origin: G = 15, with effective independence bounded near 4 by the 79.2%
training-window overlap, which is stated wherever these p-values are. **One
resample of origins is applied to every pair at once** -- that shared draw is
what preserves the cross-pair dependence Romano-Wolf needs, and what makes this
a stepdown rather than 45 separate tests.

**How cells are combined.** Root §9.2 pins the DM sample per (origin, block) on
the overlapping hourly loss differential, requires the combining method be
*stated*, and forbids concatenating ``d_t`` across origins, because the model
changes at each origin and the DM null has no interpretation across that
boundary. Stated, then: the per-origin mean differential is the unit, the
cluster bootstrap over the 15 of them is the inference, and the per-(origin,
block) HLN statistic travels beside it as a diagnostic with its ``T`` and ``h``.

Upstream
--------
**Both procedures are written here on numpy. Neither existed in this package
until `D62a`, and no package supplying them is a dependency.**

- ``romano_wolf`` — J. P. Romano and M. Wolf, "Stepwise multiple testing as
  formalized data snooping," *Econometrica*, vol. 73, no. 4, pp. 1237-1282,
  2005. It controls FWER across the all-pairs matrix, which is what White's
  Reality Check (2000) and Hansen's SPA (2005) cannot do: those test a
  one-against-many null and say nothing about a pairwise family (`D35`).
- ``model_confidence_set``, ``mcs_table`` — P. R. Hansen, A. Lunde, and
  J. M. Nason, "The model confidence set," *Econometrica*, vol. 79, no. 2,
  pp. 453-497, 2011. Reported at 90% and 75% as a membership column in Table 6.

Romano-Wolf is reported in two columns after `D79` — the all-pairs family and
the declared claim family — because widening the model set from 12 to 15 took
it from 31 of 66 rejections to 0 of 105 while no effect moved.
:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries these rows in full.
"""

#: ``(model_tag, k)``. ``("naive", 0)`` denotes Naive-RW, which needs no run:
#: root §7 defines it as ``y_raw = 0``, mapped into scaler space as
#: ``y_z = -mu_g/sigma_g`` (`D31`), evaluated on exactly the rows its comparator
#: was scored on.
ModelKey = tuple[str, int]

#: Naive-RW's sentinel key.
NAIVE: Final[ModelKey] = ("naive", 0)

#: Models whose feature sets nest along K under one architecture. ``itru`` is
#: absent deliberately: it is the uniform-attention arm of ``itr`` at K=8 and
#: sees exactly the same eight variates, so it nests against nothing along K --
#: the arm varies *what attention selects*, not what the model can see (`D50`).
NESTS_ALONG_K: Final[frozenset[str]] = frozenset({"itr", "rdg"})

#: Bootstrap draws. The floor on any p-value is ``1/(1+B)`` (`D53d`): a literal
#: ``p = 0`` is not a probability, and the observed statistic belongs to its own
#: reference distribution (Davison & Hinkley 1997).
DEFAULT_B: Final = 9_999

#: Model Confidence Set levels root §9.2 asks for, as a membership column.
MCS_LEVELS: Final[tuple[float, ...]] = (0.10, 0.25)

#: The claim families the stepdown is *also* reported within (`D79`), in the
#: order Table 6 groups them.
#:
#: **Why a second column exists at all.** Romano-Wolf controls FWER over the
#: family it is given, and this table gives it the Cartesian product. Widening
#: the model list from twelve to fifteen took the matrix from 66 pairs to 105 and
#: the surviving rejections from 31 to **zero** -- not because any effect moved,
#: but because the two naive comparators generate the largest ``|t|`` in the
#: table (up to 8.5) and the shared bootstrap draw puts them into the max-``|t|``
#: null that every other hypothesis is judged against. Two closed-form baselines
#: that no claim in the paper rests on cost the paper every adjusted rejection it
#: had. That is a property of the declared family, not of the data.
#:
#: **What this column is and is not.** ``p_romano_wolf`` over all pairs stays the
#: headline: it is what root §9.2 says in as many words, and it is the
#: conservative number. ``p_romano_wolf_family`` steps down within one claim
#: family instead, which is the standard construction when the families are
#: declared in advance -- and this one was **not**, so it is reported as
#: *post-hoc* and labelled that way wherever it appears. Reporting both is the
#: only honest option left once the all-pairs column has been seen: suppressing
#: the narrower one hides how much of the collapse is the family's doing, and
#: promoting it to headline would be choosing the correction after seeing which
#: correction rejects.
FAMILY_ORDER: Final[tuple[str, ...]] = (
    "vs-naive", "ladder", "architecture", "other",
)


def pair_family(left: ModelKey, right: ModelKey) -> str:
    """Which claim of the paper a pair speaks to (`D79`).

    Four families, mutually exclusive and exhaustive by construction, each
    matching a sentence the manuscript actually makes:

    * ``vs-naive`` --- every pair containing Naive-RW. Root §13.2's headline
      disclosure, *no model beats Naive-RW*, is exactly this family.
    * ``ladder`` --- both sides the same model tag, so only K moves. RQ1's rungs
      and ridge's own rungs.
    * ``architecture`` --- both sides the same K, different tags. The
      channel-independence comparison root §13.1 makes a Related Work pillar.
    * ``other`` --- the remainder, which no claim rests on: a rung of one model
      against a different rung of another.

    Declared as a function of the *keys* alone, so no p-value can enter the
    definition.
    """
    if NAIVE in (left, right):
        return "vs-naive"
    if left[0] == right[0]:
        return "ladder"
    if left[1] == right[1]:
        return "architecture"
    return "other"


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪆 Nesting & label model</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Tangga bersifat kumulatif dan Naive-RW bersarang di setiap model — itu yang menentukan statistik mana yang sah per pasangan.</p>
</div>

In [ ]:


def nesting_order(left: ModelKey, right: ModelKey) -> tuple[ModelKey, ModelKey] | None:
    """``(small, large)`` if one information set nests inside the other, else None.

    **The orientation is not cosmetic and getting it wrong inverts the answer.**
    Clark-West is
    ``f = (y - y_small)^2 - (y - y_large)^2 + (y_small - y_large)^2``, and the
    adjustment term is symmetric while the first two are not. Swapping the roles
    therefore reports ``-(first two) + adjustment``, which is not a Clark-West
    statistic at all. Against Naive-RW the error is loud in exactly the wrong
    way: every model returns a large positive statistic and appears to beat the
    baseline, while its own sample MSE is worse -- a contradiction a reader would
    have to resolve by disbelieving one number or the other.

    Root §9.2 names the nested pairs: K=1 vs K=8 under one architecture, any
    model against Naive-RW, and Ridge-K1 against Ridge-K8. Cross-arm pairs are
    **not** on that list and are treated as non-nested here even where the
    information sets happen to nest, because the model classes differ and §9.2's
    enumeration is the pre-registered one.
    """
    model_left, k_left = left
    model_right, k_right = right
    if model_left == NAIVE[0]:
        return (left, right)
    if model_right == NAIVE[0]:
        return (right, left)
    if model_left != model_right:
        return None
    if model_left not in NESTS_ALONG_K or k_left == k_right:
        return None
    return (left, right) if k_left < k_right else (right, left)


def is_nested(left: ModelKey, right: ModelKey) -> bool:
    """Whether one model's information set is a strict subset of the other's."""
    return nesting_order(left, right) is not None


def label(key: ModelKey) -> str:
    """``itr-K8``, or ``Naive-RW`` for the sentinel."""
    return "Naive-RW" if key == NAIVE else f"{key[0]}-K{key[1]}"


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🗂️ Panel prediksi</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Prediksi seluruh model pada himpunan jendela bersama.</p>
</div>

In [ ]:


# -- the aligned prediction panel --------------------------------------------


@dataclass(frozen=True, slots=True)
class PredictionPanel:
    """Seed-averaged forecasts for every model at every origin, aligned row-wise.

    Every model in the study is scored on **identical** evaluated windows --
    verified on the artifact: at origin 1, ``itr``, ``ptst`` and ``rdg`` each hold
    88,992 rows over 3,708 timestamps, their timestamp sets are equal, and
    ``max |y_true_itr - y_true_ptst|`` is exactly 0. `D45` makes that an
    assertion rather than an assumption, because a differential across two
    samples is not a differential, and :func:`build_panel` refuses a mismatch.

    Forecasts are averaged **across seeds before** any differential is formed,
    which is `D42`'s order of operations one level down from the ratio metrics it
    was written for: average the primitive, then form the derived quantity. The
    alternative -- differencing per seed and averaging statistics -- requires
    pairing seed 42 at K=1 with seed 42 at K=8, which are independent training
    runs of different models.
    """

    keys: tuple[ModelKey, ...]
    origin_indices: tuple[int, ...]
    origins: tuple[str, ...]
    #: origin index -> ``(n_rows,)`` block labels, sorted with the arrays below.
    block: dict[int, np.ndarray]
    #: origin index -> ``(n_rows,)`` realised target, shared by every model.
    y_true: dict[int, np.ndarray]
    #: ``(key, origin index)`` -> ``(n_rows,)`` seed-averaged forecast.
    y_pred: dict[tuple[ModelKey, int], np.ndarray]
    #: Forecast steps per window, so a per-origin reduction can recover ``T``.
    pred_len: int


def _run_ids(
    key: ModelKey, origin_index: int, roots: list[Path], pred_len: int
) -> list[str]:
    """Every seed of one cell that is actually on disk, in seed order."""
    model, k = key
    stem = f"{model}_o{origin_index:02d}_K{k:02d}_H{pred_len:03d}_s"
    found: set[str] = set()
    for root in roots:
        for path in (root / "preds").glob(f"{stem}*.parquet"):
            found.add(path.stem)
    return sorted(found, key=lambda run_id: int(parse_run_id(run_id)["seed"]))


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🏗️ Ketersediaan & membangun panel</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Memuat setiap <code>preds/*.parquet</code> dan menegakkan keselarasan jendela lintas model.</p>
</div>

In [ ]:


def available_keys(
    keys: list[ModelKey],
    roots: list[Path],
    pred_len: int = PRED_LEN,
    origin_indices: tuple[int, ...] | None = None,
) -> tuple[list[ModelKey], list[ModelKey]]:
    """Split ``keys`` into those with a run at **every** origin, and the rest.

    Returns:
        ``(present, absent)``, both in the order given.

    :func:`build_panel` raises on a key missing at any origin, and that raise is
    right: a matrix short one origin for one model compares models over
    different origin sets, which is `D45` one level up. But an arm that has not
    run *anywhere* is a different situation --- the report should name it, the
    way an absent robustness arm is named, rather than refuse to generate until
    a GPU session finishes.

    Partial coverage is deliberately **not** rescued here. A key present at some
    origins and absent at others still reaches :func:`build_panel` and still
    raises: that is the defect, and quietly dropping it would hide it.
    """
    indices = origin_indices or tuple(o.index for o in ORIGINS)
    present: list[ModelKey] = []
    absent: list[ModelKey] = []
    for key in keys:
        if key == NAIVE or any(
            _run_ids(key, index, roots, pred_len) for index in indices
        ):
            present.append(key)
        else:
            absent.append(key)
    return present, absent


def build_panel(
    keys: list[ModelKey],
    roots: list[Path],
    pred_len: int = PRED_LEN,
    origin_indices: tuple[int, ...] | None = None,
) -> PredictionPanel:
    """Load, seed-average and align every model's forecasts.

    Args:
        keys: Models to compare. :data:`NAIVE` may appear and needs no run.
        roots: Artifact roots, working directory first.
        pred_len: Horizon. Table 6 is the headline H=24.
        origin_indices: Defaults to every origin in the walk-forward grid.

    Raises:
        FileNotFoundError: If a key has no run at some origin. A silently short
            matrix would compare models over different origin sets, which is the
            defect `D45` names one level up.
        ValueError: If two models were evaluated on different windows.
    """
    indices = origin_indices or tuple(o.index for o in ORIGINS)
    labels = tuple(o.label for o in ORIGINS if o.index in indices)

    block: dict[int, np.ndarray] = {}
    y_true: dict[int, np.ndarray] = {}
    y_pred: dict[tuple[ModelKey, int], np.ndarray] = {}
    signature: dict[int, tuple] = {}

    for origin_index in indices:
        for key in keys:
            if key == NAIVE:
                continue
            runs = _run_ids(key, origin_index, roots, pred_len)
            if not runs:
                raise FileNotFoundError(
                    f"{key} has no run at origin {origin_index} (H={pred_len}) in "
                    f"{[str(r) for r in roots]}"
                )
            stacked: list[np.ndarray] = []
            for run_id in runs:
                frame = load_predictions(run_id, roots).sort(
                    ["block", "timestamp", "step"]
                )
                sig = (
                    tuple(frame.get_column("block").to_list()),
                    tuple(frame.get_column("timestamp").to_list()),
                )
                if origin_index not in signature:
                    signature[origin_index] = sig
                    block[origin_index] = frame.get_column("block").to_numpy()
                    y_true[origin_index] = (
                        frame.get_column("y_true").to_numpy().astype(np.float64)
                    )
                elif sig != signature[origin_index]:
                    raise ValueError(
                        f"`D45`: {run_id} was evaluated on different windows than "
                        f"the first model at origin {origin_index}; a differential "
                        f"across two samples is not a differential"
                    )
                stacked.append(
                    frame.get_column("y_pred").to_numpy().astype(np.float64)
                )
            y_pred[(key, origin_index)] = np.mean(stacked, axis=0)

        if NAIVE in keys:
            # Root §7 / `D31`. Constant in scaler space, and the constant is
            # -mu_g/sigma_g rather than 0: zero there means r_hat = mu_g, the
            # training-window mean hourly return, so the "EMH baseline" would
            # silently be a constant-drift model. Read from a comparator's meta,
            # which is where the grid logged it per origin.
            donor = next(k for k in keys if k != NAIVE)
            meta = load_meta(_run_ids(donor, origin_index, roots, pred_len)[0], roots)
            y_pred[(NAIVE, origin_index)] = np.full(
                len(y_true[origin_index]), float(meta["naive_rw_z"])
            )

    return PredictionPanel(
        keys=tuple(keys),
        origin_indices=tuple(indices),
        origins=labels,
        block=block,
        y_true=y_true,
        y_pred=y_pred,
        pred_len=pred_len,
    )


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">➖ Diferensial rugi</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Diferensial per jendela, per (origin, blok) — T ≈ 720, h = 24, lag pemotongan 23.</p>
</div>

In [ ]:


def _per_window(values: np.ndarray, pred_len: int) -> np.ndarray:
    """Mean over the ``pred_len`` forecast steps of each window.

    The reduction is what keeps ``T`` counting **window starts** and ``h`` equal
    to the horizon, which is the sample root §9.2 pins: ``T ~ 720`` per block,
    truncation lag 23 at H=24. Left unreduced, ``T`` would be 17,280 and the HLN
    factor would be computed for a horizon the series does not have.
    """
    return values.reshape(-1, pred_len).mean(axis=1)


def differential(
    panel: PredictionPanel, left: ModelKey, right: ModelKey, origin_index: int
) -> np.ndarray:
    """Per-window loss differential for one pair at one origin, ``left - right``.

    Positive means ``right`` forecasts better. For a **nested** pair the
    Clark-West adjustment ``+(y_left - y_right)^2`` is added back, which is the
    whole content of `D29`: without it the larger model's extra estimation noise
    biases the differential against the alternative the study exists to test.

    Raises:
        ValueError: If the pair nests and ``left`` is not the restricted model.
            :func:`nesting_order` explains why silently accepting either order
            would report a quantity that is not a Clark-West statistic; refusing
            is cheaper than a table nobody can reconcile.
    """
    order = nesting_order(left, right)
    if order is not None and order[0] != left:
        raise ValueError(
            f"{label(left)} vs {label(right)} nests the other way: Clark-West "
            f"needs the restricted model first, so pass "
            f"({label(order[0])}, {label(order[1])}). See nesting_order."
        )
    y = panel.y_true[origin_index]
    a = panel.y_pred[(left, origin_index)]
    b = panel.y_pred[(right, origin_index)]
    d = np.square(y - a) - np.square(y - b)
    if order is not None:
        d = d + np.square(a - b)
    return _per_window(d, panel.pred_len)


def per_origin_differential(
    panel: PredictionPanel, left: ModelKey, right: ModelKey
) -> np.ndarray:
    """``(G,)`` mean differential per origin -- the unit inference runs on."""
    return np.array(
        [
            float(differential(panel, left, right, index).mean())
            for index in panel.origin_indices
        ]
    )


def per_origin_loss(panel: PredictionPanel, key: ModelKey) -> np.ndarray:
    """``(G,)`` mean squared error per origin, for the Model Confidence Set."""
    return np.array(
        [
            float(
                np.square(panel.y_true[index] - panel.y_pred[(key, index)]).mean()
            )
            for index in panel.origin_indices
        ]
    )


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🪜 Bootstrap klaster & Romano–Wolf</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Stepdown mengendalikan FWER lintas <em>seluruh</em> pasangan. Ia menghapus kedelapan penolakan mentah terhadap Naive-RW (<code>D35</code>, <code>D62a</code>).</p>
</div>

In [ ]:


# -- clustered inference over origins ----------------------------------------


def _studentised(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Column means and their cluster standard errors, ``G = matrix.shape[0]``."""
    g = matrix.shape[0]
    return matrix.mean(axis=0), matrix.std(axis=0, ddof=1) / math.sqrt(g)


def cluster_bootstrap_t(
    per_origin: np.ndarray, B: int = DEFAULT_B, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Observed and bootstrap studentised statistics, resampling **origins**.

    Args:
        per_origin: ``(G, P)`` -- one mean differential per origin, per pair.
        B: Bootstrap draws.
        seed: Generator seed.

    Returns:
        ``(t_obs, t_boot)`` of shapes ``(P,)`` and ``(B, P)``. The bootstrap
        statistics are centred on the observed mean, so they are draws from the
        null. A resample that happens to pick one origin ``G`` times has no
        dispersion; it contributes 0 rather than an infinity.
    """
    g = per_origin.shape[0]
    theta, se = _studentised(per_origin)
    with np.errstate(divide="ignore", invalid="ignore"):
        t_obs = np.where(se > 0, theta / se, 0.0)

    rng = np.random.default_rng(seed)
    draws = per_origin[rng.integers(0, g, size=(B, g))]
    theta_b = draws.mean(axis=1)
    se_b = draws.std(axis=1, ddof=1) / math.sqrt(g)
    with np.errstate(divide="ignore", invalid="ignore"):
        t_boot = np.where(se_b > 0, (theta_b - theta) / se_b, 0.0)
    return t_obs, t_boot


def romano_wolf(
    per_origin: np.ndarray, B: int = DEFAULT_B, seed: int = 42
) -> np.ndarray:
    """Stepdown FWER-controlled p-values across every pair (Romano & Wolf 2005).

    Two-sided, deliberately: the family is "is there **any** difference in
    predictive ability between these two models", and mixing one- and two-sided
    alternatives inside one stepdown would make the controlled family
    ill-defined. The directional Clark-West reading travels in the raw column
    beside it.

    Args:
        per_origin: ``(G, P)`` mean differential per origin, per pair.
        B: Bootstrap draws.
        seed: Generator seed.

    Returns:
        ``(P,)`` adjusted p-values, monotone in ``|t|``.
    """
    t_obs, t_boot = cluster_bootstrap_t(per_origin, B=B, seed=seed)
    order = list(np.argsort(-np.abs(t_obs)))
    adjusted = np.empty(per_origin.shape[1])
    remaining = list(order)
    running = 0.0
    for position in order:
        block_max = np.abs(t_boot[:, remaining]).max(axis=1)
        raw = (1 + int((block_max >= abs(t_obs[position])).sum())) / (1 + B)
        running = max(running, raw)  # stepdown monotonicity
        adjusted[position] = min(running, 1.0)
        remaining.remove(position)
    return adjusted


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🏅 Model Confidence Set</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Siapa yang tak terbedakan dari yang terbaik. Di 90% maupun 75%: Naive-RW dan keempat rung ridge, tanpa satu pun model deep.</p>
</div>

In [ ]:


def model_confidence_set(
    losses: np.ndarray, alpha: float, B: int = DEFAULT_B, seed: int = 42
) -> list[int]:
    """Hansen, Lunde & Nason (2011) MCS by the ``T_max`` statistic.

    Answers what a reader actually wants from Table 6 and the pairwise matrix
    does not: *which models are indistinguishable from the best*. Reported at
    90% and 75% as a membership column (root §9.2).

    Args:
        losses: ``(G, M)`` mean loss per origin per model.
        alpha: 0.10 for the 90% set, 0.25 for the 75% set.
        B: Bootstrap draws.
        seed: Generator seed.

    Returns:
        Column indices of the surviving models.
    """
    g, m = losses.shape
    alive = list(range(m))
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, g, size=(B, g))

    while len(alive) > 1:
        sub = losses[:, alive]
        deviation = sub - sub.mean(axis=1, keepdims=True)
        theta, se = _studentised(deviation)
        with np.errstate(divide="ignore", invalid="ignore"):
            t = np.where(se > 0, theta / se, 0.0)
        t_max = float(t.max())

        draws = deviation[idx]
        theta_b = draws.mean(axis=1)
        se_b = draws.std(axis=1, ddof=1) / math.sqrt(g)
        with np.errstate(divide="ignore", invalid="ignore"):
            t_b = np.where(se_b > 0, (theta_b - theta) / se_b, 0.0)
        p = (1 + int((t_b.max(axis=1) >= t_max).sum())) / (1 + B)

        if p >= alpha:
            break
        alive.pop(int(np.argmax(t)))  # eliminate the worst, then re-test
    return alive


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔎 Diagnostik per sel</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">T, h, dan apakah fallback Bartlett menyala — dilaporkan per pasangan.</p>
</div>

In [ ]:


# -- Table 6 -----------------------------------------------------------------


def _cell_diagnostics(
    panel: PredictionPanel, left: ModelKey, right: ModelKey, nested: bool
) -> dict[str, float | int | bool]:
    """Median HLN statistic over the (origin, block) cells, and how many reject.

    Root §9.2 computes the statistic **per (origin, block)** and states ``T``
    alongside every p-value. Those cell statistics do not carry the headline ---
    combining them would mean concatenating across a boundary the DM null does
    not survive --- but a table reporting only the origin-level number would hide
    how uniform the pair's ordering actually is across 90 cells.
    """
    stats: list[float] = []
    rejects = 0
    t_min = -1
    fallback = False
    name = f"{label(left)} vs {label(right)}"
    for index in panel.origin_indices:
        d = differential(panel, left, right, index)
        blocks = _per_window(
            panel.block[index].astype(np.float64), panel.pred_len
        ).round()
        for b in range(1, TEST_BLOCKS + 1):
            cell = d[blocks == float(b)]
            if len(cell) < 2:
                continue
            result = hln_test(cell, panel.pred_len, name=name, one_sided=nested)
            stats.append(result.statistic)
            rejects += int(result.p_value < 0.05)
            t_min = result.T if t_min < 0 else min(t_min, result.T)
            fallback = fallback or result.fallback_fired
    return {
        "s_star_median": float(np.median(stats)) if stats else float("nan"),
        "n_cells": len(stats),
        "n_cells_reject": rejects,
        "T_min": t_min,
        "fallback_fired": fallback,
    }


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧮 Matriks pasangan</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">66 pasangan, statistik disebut namanya per pasangan, T dicantumkan di samping tiap p.</p>
</div>

In [ ]:


def pair_matrix(
    panel: PredictionPanel, B: int = DEFAULT_B, seed: int = 42
) -> pl.DataFrame:
    """Table 6 --- every unordered pair, with its statistic, p-values and MCS flags.

    Returns:
        One row per pair: ``left, right, nested, statistic_name, t_cluster,
        p_raw, p_romano_wolf, family, p_romano_wolf_family, s_star_median,
        n_cells, n_cells_reject, T_min, h, fallback_fired, G``, plus MCS
        membership for both models of the row.

    ``p_raw`` is one-sided for a nested pair, where root §9.2's alternative is
    directional, and two-sided otherwise. ``p_romano_wolf`` is always two-sided
    --- see :func:`romano_wolf`.

    **Two adjusted columns, and which is the headline is not a free choice**
    (`D79`). ``p_romano_wolf`` controls FWER across all pairs, which is what root
    §9.2 pre-registers, and it stays the headline. ``p_romano_wolf_family`` steps
    down inside the claim family :func:`pair_family` assigns; the families were
    declared after the all-pairs column had been read, so that column is
    **post-hoc** and every rendering of it says so.
    """
    keys = list(panel.keys)
    # Oriented, not merely enumerated: a nested pair is emitted restricted-model
    # first, so ``differential`` computes the Clark-West statistic rather than its
    # sign-inverted lookalike. See :func:`nesting_order`.
    pairs = [
        nesting_order(a, b) or (a, b)
        for i, a in enumerate(keys)
        for b in keys[i + 1 :]
    ]

    per_origin = np.column_stack(
        [per_origin_differential(panel, a, b) for a, b in pairs]
    )
    t_obs, t_boot = cluster_bootstrap_t(per_origin, B=B, seed=seed)
    p_adjusted = romano_wolf(per_origin, B=B, seed=seed)

    # `D79`. The same stepdown, run again inside each claim family. Same draw
    # seed, so the two columns differ in the family and in nothing else --- which
    # is what makes the difference between them readable as the cost of the
    # Cartesian product rather than as bootstrap noise.
    families = [pair_family(a, b) for a, b in pairs]
    p_family = np.ones(len(pairs))
    for name in FAMILY_ORDER:
        members = [i for i, f in enumerate(families) if f == name]
        if not members:
            continue
        p_family[members] = romano_wolf(per_origin[:, members], B=B, seed=seed)

    losses = np.column_stack([per_origin_loss(panel, k) for k in keys])
    members = {
        alpha: {keys[i] for i in model_confidence_set(losses, alpha, B=B, seed=seed)}
        for alpha in MCS_LEVELS
    }

    rows = []
    for position, (left, right) in enumerate(pairs):
        nested = is_nested(left, right)
        t = float(t_obs[position])
        if nested:
            # Directional: the alternative is that the larger model helps, and a
            # positive differential is what "helps" means here.
            count = int((t_boot[:, position] >= t).sum())
        else:
            count = int((np.abs(t_boot[:, position]) >= abs(t)).sum())
        rows.append(
            {
                "left": label(left),
                "right": label(right),
                "nested": nested,
                "statistic_name": "Clark-West" if nested else "DM-HLN",
                "t_cluster": t,
                "p_raw": (1 + count) / (1 + B),
                "p_romano_wolf": float(p_adjusted[position]),
                "family": families[position],
                "p_romano_wolf_family": float(p_family[position]),
                **_cell_diagnostics(panel, left, right, nested),
                "h": panel.pred_len,
                "G": int(per_origin.shape[0]),
                "left_in_mcs_90": left in members[0.10],
                "right_in_mcs_90": right in members[0.10],
                "left_in_mcs_75": left in members[0.25],
                "right_in_mcs_75": right in members[0.25],
            }
        )
    return pl.DataFrame(rows)


<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b5488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #ffb4a2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel MCS</h4>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.9em;">Kolom keanggotaan untuk Tabel 4 dan Tabel 6.</p>
</div>

In [ ]:


def mcs_table(
    panel: PredictionPanel, B: int = DEFAULT_B, seed: int = 42
) -> pl.DataFrame:
    """Model Confidence Set membership per model, with its mean loss and rank.

    The ``se_across_origins`` column is `D30`'s rule made structural: this row is
    aggregated across origins, so its dispersion is the standard error across
    origins and never the seed standard deviation, which measures
    re-initialisation noise on one fixed dataset and is roughly an order of
    magnitude smaller.
    """
    keys = list(panel.keys)
    losses = np.column_stack([per_origin_loss(panel, k) for k in keys])
    members = {
        alpha: {keys[i] for i in model_confidence_set(losses, alpha, B=B, seed=seed)}
        for alpha in MCS_LEVELS
    }
    mean_loss = losses.mean(axis=0)
    rank = {int(position): r + 1 for r, position in enumerate(np.argsort(mean_loss))}
    return pl.DataFrame(
        [
            {
                "model": label(key),
                "mean_loss": float(mean_loss[i]),
                "se_across_origins": float(
                    losses[:, i].std(ddof=1) / math.sqrt(losses.shape[0])
                ),
                "rank": rank[i],
                "in_mcs_90": key in members[0.10],
                "in_mcs_75": key in members[0.25],
            }
            for i, key in enumerate(keys)
        ]
    ).sort("rank")


<div style="background: linear-gradient(135deg, #0d1b2a, #1b263b); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #00b4d833;">
  <h2 style="color: #90e0ef; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    💰 13 &middot; Evaluasi ekonomi
  </h2>
  <p style="color: #ade8f4; margin: 0; font-size: 1.02em;">Fase 00:00 UTC, non-overlapping, per segmen, DSR per origin (root &sect;13.5).</p>
  <ul style="color: #ade8f4; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Return lintas-gap terlarang.</strong> Posisi yang ditahan melewati blok downtime tidak punya return terealisasi yang terdefinisi — di blok 2018-02-08 sebuah &ldquo;trade 24 jam&rdquo; akan membukukan gerak 57 jam (<code>D46</code>).</li>
    <li style="margin-bottom: 6px;">Fase dikunci di 00:00 UTC: ada 24 penyelarasan partisi harian non-overlapping yang sah, dan memilih fase setelah melihat kurva ekuitas adalah parameter bebas atas klaim ekonomi paper ini.</li>
    <li style="margin-bottom: 6px;">P&amp;L positif di bawah <code>R&sup2;_oos</code> negatif bukan kontradiksi: MSE dan P&amp;L arah adalah objektif berbeda, dan sampel yang didominasi kenaikan BTC 2020&ndash;2026 membayar posisi yang mayoritas long untuk <em>drift</em>, bukan untuk ramalannya. Laporkan +20,6% bersama buy-and-hold +29,0% dan DSR 0,173, atau angka pertama terbaca sebagai skill.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 4px solid #00b4d8; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #90e0ef; margin: 0 0 6px 0; font-size: 1.22em;">📘 economics.py</h3>
  <p style="color: #ade8f4; margin: 0; font-size: 0.94em;">Evaluasi ekonomi root §13.5: fase 00:00 UTC, non-overlapping, per segmen, DSR per origin.</p>
</div>

<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header, biaya, band slippage</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Fee taker 0,04% per sisi, slippage 0,02/0,05/0,10% — band dipra-registrasi, ketiganya dilaporkan (root §13.5).</p>
</div>

In [ ]:
"""Root §13.5's economic evaluation -- Table 8 and Figure 7.

`D60g` found this never ran at all: no positions, no equity curve, no Deflated
Sharpe Ratio anywhere in the session log. `D46` had already fixed three
specifications in advance, each of which moves every number in Table 8, and each
is honoured here rather than re-decided:

1. **Phase.** Positions open at **00:00 UTC**. There are 24 admissible
   alignments of a non-overlapping daily partition, each with a different Sharpe,
   MDD and turnover; choosing the phase after seeing the equity curve is a free
   parameter on the paper's economic claim. :func:`metrics.non_overlapping_mask`
   already implements exactly this and is reused rather than reimplemented.
2. **Gap-spanning returns are forbidden.** A position held across a downtime
   block has no defined realised return, and the obvious ``log(C_{t+24}/C_t)`` is
   the cross-gap return §4.3 forbids everywhere else -- at the 2018-02-08 block a
   nominal 24-hour trade would book a 57-hour move. The pipeline already makes
   this unreachable: ``windows.enumerate_windows`` validates every retained
   window by ``t[s + L + H - 1] - t[s] == (L + H - 1)`` hours, so a surviving
   forecast's target bars are contiguous by construction. The check is therefore
   *vacuous*, and saying so is the point -- what is **not** vacuous is
   :attr:`StrategyResult.n_flat_days`, the calendar days inside a test block with
   no surviving 00:00 window because the window was rejected upstream. Outages
   cluster on stress, so the strategy is flat precisely across the
   large-drawdown periods and the reported MDD is optimistic by an amount only
   that count lets a reader bound.
3. **Costs.** A 0.04% taker fee per side, plus slippage at a pre-registered
   sensitivity band of 0.02% / 0.05% / 0.10% per side, with Table 8 reported at
   all three. Fixing the fee exactly while leaving slippage blank fixes the lever
   that costs nothing and leaves open the one that decides whether the strategy
   makes money.

**The comparator is buy-and-hold, and the reason belongs in the caption.** Root
§13.5 asks for a Sharpe test "against the naive strategy", but Naive-RW forecasts
``y_raw = 0`` and therefore holds a constant zero position: its return series has
zero variance and its Sharpe is undefined. Comparing against it is not
conservative, it is meaningless.

**Everything is computed on raw, drift-free log-returns, never scaler-space**
(`D31`). ``y_z = 0`` in scaler space means ``r_hat = mu_g``, the training-window
mean hourly return, so a sign rule read there is a sign rule on a constant-drift
model -- and ``mu_g/sigma_g`` **changes sign across origins**, so it is not a
tilt a reader could mentally subtract.

Upstream
--------
**Written here on numpy; no backtesting package is a dependency.**

``deflated_sharpe`` -- D. H. Bailey and M. Lopez de Prado, "The deflated Sharpe
ratio: Correcting for selection bias, backtest overfitting, and non-normality,"
*J. Portfolio Manage.*, vol. 40, no. 5, pp. 94-107, 2014. The published
definition is followed, but the *arguments* are this study's and `D46` explains
why: DSR counts candidates whose Sharpe was computed on the **same** return
series, so it is computed **per origin** from that origin's non-overlapping
24-hour strategy returns and their **per-period** Sharpe -- never the annualised
one, which would inflate it by ``sqrt(periods per year)``. ``N`` is the number
of configurations evaluated on that origin's own test span, not the 1,620-run
total; the total is reported separately as the development trial count.
:data:`itransformer_btc.config.SOURCE_PROVENANCE` carries this row in full.
"""

#: Binance spot taker fee, per side. Fixed; the band below is the sensitivity,
#: because slippage is the lever that decides whether the strategy makes money
#: and the project's own reference library anchors BTC effective spreads near
#: 0.30%.
TAKER_FEE_PER_SIDE: Final = 0.0004

#: Root §13.5's pre-registered slippage band, per side. Table 8 is reported at
#: all three, never at one.
SLIPPAGE_BAND: Final[tuple[float, ...]] = (0.0002, 0.0005, 0.0010)

#: Crypto trades every day, so annualising a daily Sharpe uses 365, not 252.
PERIODS_PER_YEAR: Final = 365

#: Euler-Mascheroni, for the Deflated Sharpe Ratio's expected-maximum threshold.
EULER_MASCHERONI: Final = 0.5772156649015329

#: Expected block length of the stationary bootstrap behind the MDD interval, in
#: daily periods. From ~180 observations a point maximum drawdown is
#: uninterpretable, and a block short enough to break the drawdown's own
#: persistence would understate the interval.
MDD_BLOCK_DAYS: Final = 5

#: Draws for the drawdown interval.
MDD_BOOTSTRAP_B: Final = 1_999


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📊 Posisi & return bersih</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Tanda ramalan kumulatif H-langkah pada return <strong>mentah bebas-drift</strong>, non-overlapping, fase 00:00 UTC.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class StrategyResult:
    """One (model, origin, slippage) cell of Table 8.

    ``sharpe_per_period`` is the figure the Deflated Sharpe Ratio consumes;
    feeding it the annualised one inflates it by ``sqrt(PERIODS_PER_YEAR)`` and
    is the easiest single way to report a strategy as significant when it is not.
    """

    n_periods: int
    n_flat_days: int
    mean_net: float
    sharpe_per_period: float
    sharpe_annualised: float
    sortino_annualised: float
    max_drawdown: float
    mdd_ci_low: float
    mdd_ci_high: float
    turnover_per_period: float
    net_log_return: float
    net_total_return: float


def positions(preds: pl.DataFrame, sigma_g: float, mu_g: float) -> pl.DataFrame:
    """Non-overlapping daily positions and their realised raw returns.

    The position is the **sign of the cumulative H-step forecast** on raw,
    drift-free log-returns. Because ``r_hat - mu_g = y_z * sigma_g`` and
    ``sigma_g > 0``, that sign equals the sign of the summed scaler-space
    forecast; the multiplication is kept anyway so the quantity in the code is
    the quantity root §13.5 names.

    The realised return is the actual market move and therefore **does** carry
    the drift: ``sum_h(y_true_z) * sigma_g + H * mu_g``.

    Args:
        preds: One run's predictions, in scaler space.
        sigma_g: Training-window standard deviation of the target.
        mu_g: Training-window mean of the target.

    Returns:
        ``timestamp, block, position, realised_raw, forecast_raw``, one row per
        surviving 00:00-UTC window start.
    """
    per_window = (
        preds.group_by("timestamp")
        .agg(
            pl.col("y_pred").sum().alias("_f_z"),
            pl.col("y_true").sum().alias("_a_z"),
            pl.len().alias("_n_steps"),
            pl.col("block").first().alias("block"),
        )
        .sort("timestamp")
    )
    keep = non_overlapping_mask(per_window.get_column("timestamp").to_numpy())
    return (
        per_window.filter(pl.Series(keep))
        .with_columns(
            (pl.col("_f_z") * sigma_g).alias("forecast_raw"),
            (pl.col("_f_z") * sigma_g).sign().alias("position"),
            (pl.col("_a_z") * sigma_g + pl.col("_n_steps") * mu_g).alias("realised_raw"),
        )
        .select(["timestamp", "block", "position", "realised_raw", "forecast_raw"])
    )


def net_returns(
    position: np.ndarray, realised: np.ndarray, slippage_per_side: float
) -> np.ndarray:
    """Per-period net log-return after fees and slippage.

    ``net_t = pos_t * r_t - |pos_t - pos_{t-1}| * (fee + slippage)``, with the
    first period charged as an opening trade from flat.

    The cost is deducted on the **log** scale, which is an approximation: a
    proportional cost is multiplicative in price space. At 0.06%-0.14% per round
    trip the difference is below 1e-6 per period, far under the dispersion of the
    returns themselves -- but it is an approximation, and saying so is cheaper
    than leaving a reader to infer it.
    """
    turnover = np.abs(np.diff(position, prepend=0.0))
    return position * realised - turnover * (TAKER_FEE_PER_SIDE + slippage_per_side)


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📉 Max drawdown & bootstrap-nya</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Dari ~180 observasi MDD tak terinterpretasi tanpa interval; bootstrap stasioner memasoknya.</p>
</div>

In [ ]:


def max_drawdown(net: np.ndarray) -> float:
    """Largest peak-to-trough fall of the cumulative curve, as a fraction."""
    if len(net) == 0:
        return float("nan")
    equity = np.exp(np.cumsum(net))
    peak = np.maximum.accumulate(equity)
    return float((1.0 - equity / peak).max())


def _stationary_bootstrap_mdd(
    net: np.ndarray, B: int = MDD_BOOTSTRAP_B, seed: int = 42
) -> tuple[float, float]:
    """Percentile interval for the maximum drawdown (Politis & Romano 1994).

    Geometric block lengths with mean :data:`MDD_BLOCK_DAYS`, so the resample
    preserves the short-run persistence a drawdown is made of. An i.i.d.
    bootstrap would shatter exactly the runs the statistic measures and return an
    interval that is both too narrow and too low.
    """
    n = len(net)
    if n < 2:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    p = 1.0 / MDD_BLOCK_DAYS
    starts = rng.integers(0, n, size=(B, n))
    jumps = rng.random((B, n)) < p
    draws = np.empty(B)
    for b in range(B):
        idx = np.empty(n, dtype=np.int64)
        i = int(starts[b, 0])
        for t in range(n):
            idx[t] = i
            i = int(starts[b, t]) if jumps[b, t] else (i + 1) % n
        draws[b] = max_drawdown(net[idx])
    return float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧾 Ringkasan strategi</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Sharpe, Sortino, MDD, turnover — masing-masing dengan interval.</p>
</div>

In [ ]:


def summarise(
    position: np.ndarray,
    realised: np.ndarray,
    slippage_per_side: float,
    n_flat_days: int,
    mdd_interval: bool = True,
    seed: int = 42,
) -> StrategyResult:
    """Every Table 8 figure for one return series."""
    net = net_returns(position, realised, slippage_per_side)
    n = len(net)
    if n < 2:
        raise ValueError(f"a strategy needs at least two periods, got {n}")

    mean = float(net.mean())
    sd = float(net.std(ddof=1))
    downside = net[net < 0.0]
    downside_sd = float(downside.std(ddof=1)) if len(downside) > 1 else 0.0
    sharpe = mean / sd if sd > 0 else float("nan")
    sortino = (
        mean / downside_sd * math.sqrt(PERIODS_PER_YEAR)
        if downside_sd > 0
        else float("nan")
    )
    low, high = (
        _stationary_bootstrap_mdd(net, seed=seed)
        if mdd_interval
        else (float("nan"), float("nan"))
    )
    total_log = float(net.sum())
    return StrategyResult(
        n_periods=n,
        n_flat_days=n_flat_days,
        mean_net=mean,
        sharpe_per_period=sharpe,
        sharpe_annualised=sharpe * math.sqrt(PERIODS_PER_YEAR),
        sortino_annualised=sortino,
        max_drawdown=max_drawdown(net),
        mdd_ci_low=low,
        mdd_ci_high=high,
        # Half the mean absolute position change: expected round trips per period.
        turnover_per_period=float(np.abs(np.diff(position, prepend=0.0)).mean() / 2.0),
        net_log_return=total_log,
        net_total_return=float(math.expm1(total_log)),
    )


def _flat_days(frame: pl.DataFrame) -> int:
    """Calendar days in the covered blocks with no surviving 00:00-UTC window.

    Positions exist only where a valid window exists, so the strategy is flat
    precisely across the outages -- which, since outages cluster on stress, are
    disproportionately the large-drawdown periods (`D45`, `D46`).
    """
    n_blocks = frame.get_column("block").n_unique()
    return int(n_blocks * BLOCK_DAYS - frame.height)


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🏃 Menjalankan strategi & buy-and-hold</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Periode holding yang melintasi break <strong>dilewati</strong>: return lintas-gap tidak terdefinisi (<code>D46</code>).</p>
</div>

In [ ]:


def run_strategy(
    preds: pl.DataFrame,
    meta: dict,
    slippage_per_side: float,
    mdd_interval: bool = True,
    seed: int = 42,
) -> StrategyResult:
    """The forecast-sign strategy for one run at one slippage level."""
    frame = positions(preds, float(meta["sigma_g"]), float(meta["mu_g"]))
    return summarise(
        frame.get_column("position").to_numpy().astype(np.float64),
        frame.get_column("realised_raw").to_numpy().astype(np.float64),
        slippage_per_side,
        _flat_days(frame),
        mdd_interval=mdd_interval,
        seed=seed,
    )


def buy_and_hold(
    preds: pl.DataFrame,
    meta: dict,
    slippage_per_side: float,
    mdd_interval: bool = True,
    seed: int = 42,
) -> StrategyResult:
    """Always long, on the same daily grid -- Table 8's comparator.

    Not Naive-RW: that strategy holds a constant **zero** position, so its return
    series has zero variance and its Sharpe is undefined. A comparison against it
    is not conservative, it is meaningless.
    """
    frame = positions(preds, float(meta["sigma_g"]), float(meta["mu_g"]))
    return summarise(
        np.ones(frame.height),
        frame.get_column("realised_raw").to_numpy().astype(np.float64),
        slippage_per_side,
        _flat_days(frame),
        mdd_interval=mdd_interval,
        seed=seed,
    )


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⚖️ Uji Jobson–Korkie–Memmel</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Selisih Sharpe terhadap strategi naif. p ≈ 0,53 untuk <code>itr-K8</code>.</p>
</div>

In [ ]:


def jobson_korkie_memmel(a: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    """Test of equal Sharpe ratios with Memmel's (2003) correction.

    Root §13.5 asks for a test or an interval on the Sharpe difference rather
    than a bare point. Returns ``(z, two_sided_p)`` for ``SR_a - SR_b``.
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if len(a) != len(b):
        raise ValueError(f"paired test needs equal lengths, got {len(a)} and {len(b)}")
    t = len(a)
    mu_a, mu_b = float(a.mean()), float(b.mean())
    sd_a, sd_b = float(a.std(ddof=1)), float(b.std(ddof=1))
    if sd_a <= 0 or sd_b <= 0:
        return float("nan"), float("nan")
    cov = float(np.cov(a, b, ddof=1)[0, 1])
    theta = (
        2.0 * sd_a**2 * sd_b**2
        - 2.0 * sd_a * sd_b * cov
        + 0.5 * mu_a**2 * sd_b**2
        + 0.5 * mu_b**2 * sd_a**2
        - (mu_a * mu_b / (sd_a * sd_b)) * cov**2
    ) / t
    numerator = sd_b * mu_a - sd_a * mu_b
    if theta <= 0:
        # theta vanishes **exactly** when the two series coincide: with
        # ``a == b`` the expression collapses to ``m^2 s^2 - m^2 s^2``. The
        # difference is then identically zero with zero variance, so the answer
        # is z = 0 rather than undefined -- two identical strategies have
        # identical Sharpe ratios. A vanishing theta beside a non-zero numerator
        # is a genuine breakdown and stays undefined.
        if abs(numerator) <= 1e-15:
            return 0.0, 1.0
        return float("nan"), float("nan")
    z = numerator / math.sqrt(theta)
    return z, 2.0 * (1.0 - normal_cdf(abs(z)))


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎈 Deflated Sharpe Ratio</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Dihitung <strong>per origin</strong> dari Sharpe per-periode, bukan yang disetahunkan — memberi yang disetahunkan menggelembungkannya √(periode per tahun).</p>
</div>

In [ ]:


def deflated_sharpe(
    sharpe_per_period: float,
    T: int,
    skew: float,
    kurtosis: float,
    n_trials: int,
    var_sharpe: float,
) -> float:
    """Bailey & Lopez de Prado's Deflated Sharpe Ratio, computed **per origin**.

    `D46` made this computable. The earlier prescription -- ``N`` = the whole
    development trial count, about 837 -- could not be executed, because it named
    neither ``V[SR]`` nor the skewness and kurtosis the statistic needs; and if it
    could, it would return about 0 by construction: at ``N = 837`` and ``T = 180``
    the threshold sits at ``SR0 + 1.645/sqrt(T-1) ~ SR0 + 0.123``, essentially
    unmeetable. That is a second guaranteed null beside `D23`'s, reading to a
    referee as either a failed strategy or a misapplied statistic with no way to
    tell which.

    ``N`` is therefore the number of distinct strategy configurations evaluated on
    **this origin's** test span, and ``var_sharpe`` the observed variance of their
    Sharpe ratios. The development total is reported separately in Limitations and
    is *not* ``N``: those runs span largely disjoint test periods, seeds, horizons
    and baselines that never competed for one backtest, whereas the DSR counts
    candidates selected from one return series.

    Args:
        sharpe_per_period: **Per period**, never annualised.
        T: Number of periods.
        skew: Sample skewness of the per-period returns.
        kurtosis: Sample kurtosis, not excess.
        n_trials: Configurations evaluated on this origin's span.
        var_sharpe: Variance of their per-period Sharpe ratios.

    Returns:
        The probability that the observed Sharpe exceeds what selection alone
        would have produced.
    """
    if n_trials < 2 or not var_sharpe > 0 or T < 2:
        return float("nan")
    threshold = math.sqrt(var_sharpe) * (
        (1.0 - EULER_MASCHERONI) * normal_quantile(1.0 - 1.0 / n_trials)
        + EULER_MASCHERONI * normal_quantile(1.0 - 1.0 / (n_trials * math.e))
    )
    denominator = (
        1.0
        - skew * sharpe_per_period
        + 0.25 * (kurtosis - 1.0) * sharpe_per_period**2
    )
    if not denominator > 0:
        return float("nan")
    return float(
        normal_cdf(
            (sharpe_per_period - threshold) * math.sqrt(T - 1) / math.sqrt(denominator)
        )
    )


def moments(net: np.ndarray) -> tuple[float, float]:
    """Sample skewness and non-excess kurtosis of a return series."""
    centred = np.asarray(net, dtype=np.float64) - np.mean(net)
    sd = centred.std(ddof=0)
    if sd <= 0:
        return float("nan"), float("nan")
    return float((centred**3).mean() / sd**3), float((centred**4).mean() / sd**4)


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔖 Run id per origin</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Konfigurasi yang bersaing pada rentang uji origin itu — N untuk DSR.</p>
</div>

In [ ]:


def _origin_run_ids(roots: list[Path], origin_index: int, pred_len: int) -> list[str]:
    """Every run at one origin and horizon covering the full six-block span.

    The falsification arm is excluded: it is scored on blocks 4-6 only, so its
    strategy runs on half the span and its Sharpe is not a candidate the others
    competed against.
    """
    found: set[str] = set()
    for root in roots:
        pattern = f"*_o{origin_index:02d}_K*_H{pred_len:03d}_s*.parquet"
        for path in (root / "preds").glob(pattern):
            if not path.stem.startswith("itrf_"):
                found.add(path.stem)
    return sorted(found)


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel ekonomi</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Tabel 8, ketiga tingkat slippage. Tiga angka dilaporkan bersama: strategi, buy-and-hold, DSR.</p>
</div>

In [ ]:


def economics_table(
    roots: list[Path],
    keys: list[tuple[str, int]],
    origin_indices: tuple[int, ...],
    slippages: tuple[float, ...] = SLIPPAGE_BAND,
    pred_len: int = PRED_LEN,
    seed: int = 42,
) -> pl.DataFrame:
    """Table 8 -- every (model, origin, slippage) cell, with comparator and DSR.

    Args:
        roots: Artifact roots, working directory first.
        keys: ``(model_tag, k)`` pairs to evaluate.
        origin_indices: Origins to cover.
        slippages: Per-side slippage levels; all are reported, never one.
        pred_len: Horizon.
        seed: Bootstrap seed for the drawdown interval.

    Returns:
        One row per cell: every :class:`StrategyResult` field, the buy-and-hold
        comparator, the Jobson-Korkie/Memmel test of the Sharpe difference, and
        the Deflated Sharpe Ratio beside the ``n_trials`` and ``var_sharpe`` it
        was computed from -- so a reader can redo it (root §12).
    """
    rows: list[dict] = []
    for origin_index in origin_indices:
        # The DSR's trial set: every configuration evaluated on THIS origin's span.
        trial_sharpes: list[float] = []
        for run_id in _origin_run_ids(roots, origin_index, pred_len):
            meta = load_meta(run_id, roots)
            frame = positions(
                load_predictions(run_id, roots),
                float(meta["sigma_g"]),
                float(meta["mu_g"]),
            )
            net = net_returns(
                frame.get_column("position").to_numpy().astype(np.float64),
                frame.get_column("realised_raw").to_numpy().astype(np.float64),
                SLIPPAGE_BAND[1],
            )
            sd = net.std(ddof=1)
            if sd > 0:
                trial_sharpes.append(float(net.mean() / sd))
        n_trials = len(trial_sharpes)
        var_sharpe = float(np.var(trial_sharpes, ddof=1)) if n_trials > 1 else float("nan")

        for model, k in keys:
            run_id = f"{model}_o{origin_index:02d}_K{k:02d}_H{pred_len:03d}_s42"
            meta = load_meta(run_id, roots)
            frame = positions(
                load_predictions(run_id, roots),
                float(meta["sigma_g"]),
                float(meta["mu_g"]),
            )
            position = frame.get_column("position").to_numpy().astype(np.float64)
            realised = frame.get_column("realised_raw").to_numpy().astype(np.float64)
            hold_position = np.ones(len(position))
            flat = _flat_days(frame)

            for slippage in slippages:
                result = summarise(position, realised, slippage, flat, seed=seed)
                hold = summarise(
                    hold_position, realised, slippage, flat,
                    mdd_interval=False, seed=seed,
                )
                net = net_returns(position, realised, slippage)
                z, p = jobson_korkie_memmel(
                    net, net_returns(hold_position, realised, slippage)
                )
                skew, kurtosis = moments(net)
                rows.append(
                    {
                        "model": f"{model}-K{k}",
                        "origin_index": origin_index,
                        "origin": str(meta["origin"]),
                        "slippage_per_side": slippage,
                        **asdict(result),
                        "hold_sharpe_annualised": hold.sharpe_annualised,
                        "hold_net_total_return": hold.net_total_return,
                        "jk_memmel_z": z,
                        "jk_memmel_p": p,
                        "dsr": deflated_sharpe(
                            result.sharpe_per_period, result.n_periods,
                            skew, kurtosis, n_trials, var_sharpe,
                        ),
                        "dsr_n_trials": n_trials,
                        "dsr_var_sharpe": var_sharpe,
                    }
                )
    return pl.DataFrame(rows)


#: Buy-and-hold's name in :func:`equity_curves`. Not a model tag: it runs no
#: model. Root §13.2 requires the economic result be reported beside it, and
#: :func:`economics_table` already carries it as the ``hold_*`` columns.
HOLD_LABEL: Final = "Buy & hold"


<div style="background: linear-gradient(90deg, #0d1b2a, #1b263b); border-left: 3px solid #00b4d888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📈 Kurva ekuitas</h4>
  <p style="color: #ade8f4; margin: 0; font-size: 0.9em;">Figure 7, sebelum dan sesudah biaya.</p>
</div>

In [ ]:


def equity_curves(
    roots: list[Path],
    keys: list[tuple[str, int]],
    origin_indices: tuple[int, ...],
    slippages: tuple[float, ...] = SLIPPAGE_BAND,
    pred_len: int = PRED_LEN,
) -> pl.DataFrame:
    """Figure 7's input -- cumulative net equity per (model, origin, slippage).

    A zero-cost curve is emitted alongside the three priced ones, because §13.5's
    figure is "before and after costs, at three slippage levels" and the
    before-costs line is what makes the cost band legible.
    """
    rows: list[pl.DataFrame] = []
    for origin_index in origin_indices:
        for model, k in keys:
            run_id = f"{model}_o{origin_index:02d}_K{k:02d}_H{pred_len:03d}_s42"
            meta = load_meta(run_id, roots)
            frame = positions(
                load_predictions(run_id, roots),
                float(meta["sigma_g"]),
                float(meta["mu_g"]),
            )
            position = frame.get_column("position").to_numpy().astype(np.float64)
            realised = frame.get_column("realised_raw").to_numpy().astype(np.float64)
            for slippage in (0.0, *slippages):
                net = net_returns(position, realised, slippage)
                if (model, k) == keys[0]:
                    # Once per (origin, slippage), not once per model. Root §13.2
                    # states the economic result AGAINST buy-and-hold -- +20.6%
                    # net against +29.0% -- so a figure without it lets the
                    # strategy's own curve read as skill. Same `hold_position`
                    # :func:`economics_table` uses for its ``hold_*`` columns, so
                    # the figure and the table cannot disagree.
                    hold = net_returns(np.ones(len(position)), realised, slippage)
                    rows.append(
                        pl.DataFrame(
                            {
                                "model": [HOLD_LABEL] * len(hold),
                                "origin": [str(meta["origin"])] * len(hold),
                                "origin_index": [origin_index] * len(hold),
                                "slippage_per_side": [slippage] * len(hold),
                                "period": np.arange(1, len(hold) + 1, dtype=np.int32),
                                "timestamp": frame.get_column("timestamp").to_numpy(),
                                "equity": np.exp(np.cumsum(hold)),
                            }
                        )
                    )
                rows.append(
                    pl.DataFrame(
                        {
                            "model": [f"{model}-K{k}"] * len(net),
                            "origin": [str(meta["origin"])] * len(net),
                            "origin_index": [origin_index] * len(net),
                            "slippage_per_side": [slippage] * len(net),
                            "period": np.arange(1, len(net) + 1, dtype=np.int32),
                            "timestamp": frame.get_column("timestamp").to_numpy(),
                            "equity": np.exp(np.cumsum(net)),
                        }
                    )
                )
    return pl.concat(rows)


<div style="background: linear-gradient(135deg, #2d0036, #4a0060); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #e0aaff33;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🗺️ 14 &middot; Peta attention
  </h2>
  <p style="color: #c77dff; margin: 0; font-size: 1.02em;">Peta per tercile volatilitas untuk Figure 5. Calm dan stress ditentukan data, bukan dipilih setelah melihat petanya.</p>
  <ul style="color: #c77dff; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Attention bukan penjelasan.</strong> Peta ini bukti deskriptif soal kebergantungan variat, divalidasi lewat stabilitas antar-seed, dan tidak pernah kausal (Jain &amp; Wallace 2019; Wiegreffe &amp; Pinter 2019).</li>
    <li style="margin-bottom: 6px;"><code>capture</code> adalah atribut <strong>runtime</strong>, tidak pernah field config — cabangnya tidak mengonsumsi RNG, jadi run yang ditangkap identik bit-per-bit dengan grid utama, dan arm ini sekaligus jadi pemeriksaan reproduksibilitas (<code>D62d</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 4px solid #e0aaff; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">📘 attention.py</h3>
  <p style="color: #c77dff; margin: 0; font-size: 0.94em;">Peta attention per tercile volatilitas untuk Figure 5. Cabang <code>capture</code> tidak mengonsumsi RNG (<code>D62d</code>).</p>
</div>

<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 3px solid #e0aaff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #c77dff; margin: 0; font-size: 0.9em;">Tercile dan ukuran batch penangkapan.</p>
</div>

In [ ]:
"""Figure 5's input: attention maps, binned by volatility regime.

`D60g`: attention weights were never persisted. ``preds/`` holds forecasts only,
so Figure 5 and §13.2's interpretability claim rested on ``A_attn`` alone --- a
scalar that says whether attention helps, never what it attends to. This module
captures the maps; ``runner``'s ``attention`` arm re-runs the K=8 cells to
produce them.

**The regimes are data-determined** (`D48`): calm is the bottom tercile and
stress the top tercile of realised volatility across all of that origin's test
blocks. Picking the windows after seeing the maps would make the paper's
interpretability claim a free parameter, which is the same defect §3 forbids for
tau and `D49` forbids for the equivalence margin.

**Volatility is measured on the lookback, not on the forecast period.** The
attention map is a deterministic function of the 96-bar input window, so the
conditioning variable has to be a property of that same window or the figure
plots a map against something it does not depend on. The measure is the standard
deviation of the target channel over the lookback, in scaler space, which is
what ``use_norm`` divides by before the embedding sees anything.

**Attention is not explanation.** Root §13.2 admits these maps only as
*descriptive evidence of variate reliance*, validated for stability across seeds
and paired with the uniform-attention ablation (Jain & Wallace 2019; Wiegreffe &
Pinter 2019) --- which is why the arm carries three seeds rather than one, and
why ``A_attn`` stays in the paper beside the figure. The debate is also scoped to
RNN-era NLP and its transfer to variate-level attention in long-term forecasting
is genuinely open; that is itself a limitation sentence, not a footnote.
"""

#: Tercile labels, coldest first. Root §13.4 names calm and stress; the middle
#: band is carried because a figure showing only the extremes invites a reader to
#: assume the relationship between them is monotone.
TERCILES: Final[tuple[str, str, str]] = ("calm", "mid", "stress")

#: Inference batch. Larger than training's 32 because nothing here backpropagates.
CAPTURE_BATCH: Final = 512


<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 3px solid #e0aaff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🌡️ Volatilitas lookback & tercile</h4>
  <p style="color: #c77dff; margin: 0; font-size: 0.9em;">Calm dan stress ditentukan <strong>data</strong>, bukan dipilih setelah melihat petanya (<code>D48</code>).</p>
</div>

In [ ]:


def lookback_volatility(split: SplitTensors, target_index: int = 0) -> np.ndarray:
    """Standard deviation of the target channel over each window's lookback.

    Scaler-space, so the figure's regimes are the regimes ``use_norm`` sees.
    """
    if len(split) == 0:
        return np.empty(0, dtype=np.float64)
    return split.x[:, :, target_index].std(axis=1).astype(np.float64)


def tercile_edges(volatility: np.ndarray) -> tuple[float, float]:
    """The 33.3rd and 66.7th percentiles, computed **across all test blocks**.

    Per-block edges would make "stress" mean a different thing in each block and
    the panels incomparable, which is the opposite of what the figure is for.
    """
    return float(np.quantile(volatility, 1 / 3)), float(np.quantile(volatility, 2 / 3))


<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 3px solid #e0aaff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎥 Penangkap batch</h4>
  <p style="color: #c77dff; margin: 0; font-size: 0.9em;">Atribut runtime, <strong>tidak pernah</strong> field config — cabangnya tidak mengonsumsi RNG, jadi run tertangkap identik bit-per-bit (<code>D62d</code>).</p>
</div>

In [ ]:


def _capture_batch(model: ITransformer, x: torch.Tensor) -> list[np.ndarray]:
    """One forward pass with capture on; per-layer ``(B, N, N)`` weight arrays.

    The flag is cleared in a ``finally`` so an exception cannot leave a model
    quietly accumulating detached tensors for the rest of a session.
    """
    for layer in model.layers:
        layer.attention.capture = True
    try:
        with torch.no_grad():
            model(x)
        return [
            layer.attention.last_weights.cpu().numpy().astype(np.float64)
            for layer in model.layers
        ]
    finally:
        for layer in model.layers:
            layer.attention.capture = False
            layer.attention.last_weights = None


<div style="background: linear-gradient(90deg, #2d0036, #4a0060); border-left: 3px solid #e0aaff88; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #e0aaff; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🗺️ Peta attention per tercile</h4>
  <p style="color: #c77dff; margin: 0; font-size: 0.9em;">Masukan Figure 5. Terukur: praktis uniform, dan kontras calm-vs-stress tenggelam di derau seed.</p>
</div>

In [ ]:


def tercile_maps(
    model: ITransformer,
    tensors: OriginTensors,
    device: torch.device,
    target_index: int = 0,
    batch: int = CAPTURE_BATCH,
) -> pl.DataFrame:
    """Mean attention map per (tercile, layer) over one origin's test blocks.

    Two passes, because the tercile edges are a property of the whole test span
    and a single streaming pass would have to guess them: the first collects the
    volatility of every test window, the second accumulates maps into the bins
    those volatilities define.

    Args:
        model: A trained iTransformer. Set to ``eval`` here, so dropout is off
            and the captured weights are the ones inference actually uses.
        tensors: The origin whose test blocks to sweep.
        device: Where the model lives.
        target_index: Channel the volatility is measured on. 0 is ``r``.
        batch: Inference batch size.

    Returns:
        ``tercile, layer, i, j, weight, n_windows, vol_low, vol_high``. The last
        two are the **two shared tercile edges**, repeated on every row rather
        than being that row's own bounds: there is one pair per origin by
        construction, and reading them as per-tercile limits makes three
        identical values look like a split that did not happen. One row
        per (tercile, layer, variate pair). At K=8 with two layers that is 384
        rows, so persisting it costs nothing beside ``preds/``.

    Raises:
        ValueError: If the origin has no test window at all, which would leave
            the terciles undefined rather than merely empty.
    """
    model.eval()
    splits = [s for s in tensors.test_blocks if len(s) > 0]
    if not splits:
        raise ValueError(f"origin {tensors.origin.label} has no test window to sweep")

    volatility = np.concatenate([lookback_volatility(s, target_index) for s in splits])
    low, high = tercile_edges(volatility)

    n_layers = len(model.layers)
    n_variates = tensors.k
    totals = np.zeros((len(TERCILES), n_layers, n_variates, n_variates))
    counts = np.zeros(len(TERCILES), dtype=np.int64)

    for split in splits:
        # Two edges, three bins: below low, between, at or above high.
        bins = np.digitize(lookback_volatility(split, target_index), [low, high])
        x_all = torch.from_numpy(split.x).to(device)
        for start in range(0, len(split), batch):
            weights = _capture_batch(model, x_all[start : start + batch])
            chunk_bins = bins[start : start + batch]
            for tercile in range(len(TERCILES)):
                mask = chunk_bins == tercile
                if not mask.any():
                    continue
                counts[tercile] += int(mask.sum())
                for layer, w in enumerate(weights):
                    totals[tercile, layer] += w[mask].sum(axis=0)

    rows: list[dict[str, float | int | str]] = []
    for tercile, name in enumerate(TERCILES):
        if counts[tercile] == 0:
            continue
        for layer in range(n_layers):
            mean_map = totals[tercile, layer] / counts[tercile]
            for i in range(n_variates):
                for j in range(n_variates):
                    rows.append(
                        {
                            "tercile": name,
                            "layer": layer,
                            "i": i,
                            "j": j,
                            "weight": float(mean_map[i, j]),
                            "n_windows": int(counts[tercile]),
                            "vol_low": low,
                            "vol_high": high,
                        }
                    )
    return pl.DataFrame(rows)


<div style="background: linear-gradient(135deg, #012a4a, #013a63); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #48cae433;">
  <h2 style="color: #90e0ef; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🚀 15 &middot; Manifes & eksekutor grid
  </h2>
  <p style="color: #caf0f8; margin: 0; font-size: 1.02em;">Manifes 1.620 run, penemuan resume lewat glob, penjaga anggaran sesi, dan eksekutor dua-GPU.</p>
  <ul style="color: #caf0f8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Ditemukan lewat glob, tidak pernah lewat slug Dataset yang dikodekan mati</strong>, jadi nama Kaggle Dataset bebas berubah. Run dianggap lengkap hanya bila kedua berkas ada <em>dan</em> <code>meta.status</code> berbunyi <code>complete</code>.</li>
    <li style="margin-bottom: 6px;"><strong>Paralelisme di tingkat run, tidak pernah tingkat batch.</strong> Satu worker per device dari satu antrean bersama; <code>set_seed(seed, device)</code> menyemai generator CPU dan <strong>hanya device itu</strong>, jadi satu run menghasilkan byte yang sama entah ia berjalan sendiri atau berdampingan (<code>D68</code>). <code>DataParallel</code> ditolak — pada batch 32 biaya scatter/gather melebihi hematnya; DDP lebih buruk lagi pada 969 run pendek.</li>
    <li style="margin-bottom: 6px;"><strong>Penjaga anggaran mengukur sesi, bukan worker</strong> (<code>D54f</code>). Dinding 12 jam Kaggle berjalan dari sel 0, jadi prelude dikurangi lebih dulu lewat <code>SESSION_T0</code>.</li>
    <li style="margin-bottom: 6px;">Jalankan lewat <em>Save Version &rarr; Save &amp; Run All</em>, jangan lewat editor: idle timeout 20 menit membunuh sesi interaktif, dan menabrak dinding 12 jam secara interaktif menghilangkan <code>/kaggle/working</code> seluruhnya.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #90e0ef; margin: 0 0 6px 0; font-size: 1.22em;">📘 runner.py</h3>
  <p style="color: #caf0f8; margin: 0; font-size: 0.94em;">Manifes 969 run, penemuan resume lewat glob, penjaga anggaran sesi, dan eksekutor grid.</p>
</div>

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header, arm, konstanta sesi</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Sepuluh arm, dan <code>SESSION_BUDGET_H = 11,0</code> dengan cadangan setengah jam.</p>
</div>

In [ ]:
"""The run manifest, resume, the budget guard, and the two-GPU launcher.

Root §10. This module is what a Kaggle notebook calls; the notebook itself
installs, discovers inputs, calls :func:`launch_workers`, and saves. Logic in a
notebook is a defect (``notebooks/CLAUDE.md``), and a run queue is logic.

**Two GPUs are two independent run *processes*, not two threads and not
``nn.DataParallel``.** Root §10.3 rejects DataParallel on cost grounds — at
batch 32 the scatter/gather transfer costs more than the split saves, and
parallelism belongs at the *run* level because the grid is many small runs
rather than one large one. Threads are rejected for a second, sharper reason:
``torch.manual_seed`` seeds **every** CUDA device, so two threads seeding
concurrently would clobber each other's generator mid-run and root §12's
reproducibility contract would be unenforceable. One process per GPU with
``CUDA_VISIBLE_DEVICES`` pinned gives each worker its own global RNG, its own
interpreter lock and crash isolation, at the cost of rebuilding the feature
frame once per worker — seconds against hours.

**Work is split statically, by group.** A *group* is one ``(arm, origin, K, H)``
cell, whose seeds share a tensor build; shards take groups round-robin. Root
§10.5's idempotence rule makes this safe with no coordination: a run is complete
only when both artifacts exist and ``status == "complete"``, so a worker that
finishes early drains whatever is still pending regardless of which shard owned
it.
"""

# Names, not the module. In the flattened notebook there is no ``baselines``
# module object to attribute off — every definition lands in one namespace — so
# ``baselines.RidgeConfig`` would be a NameError hours into a Kaggle session
# while passing every parse-level check here (root §15, `D58`).

#: Arm to the ``model`` component of ``run_id``. Distinct tags mean a changed
#: arm **orphans** prior outputs rather than silently reusing a mismatched
#: result (root §10.4).
ARM_MODEL_TAG: dict[str, str] = {
    "main": "itr",       # 15 origins x 4 K x 5 seeds, H=24
    "uniform": "itru",   # `D50` — attention forced uniform, K=8
    "fresh": "itrf",     # root §8.1 falsification arm, trained at o_i + 90 d
    "horizon": "itr",    # `D08`/`D48` — 4 named origins x 4 K x 4 H x 3 seeds
    "ridge": "rdg",      # `D17` — is a transformer needed at all? K = 1,4,8,12
    "dlinear": "dlin",   # root §7 — "not optional", K=8
    "patchtst": "ptst",  # root §7 — SOTA channel-independent, K=8
    "lstm": "lstm",      # root §7 — the RNN the crypto literature reaches for, K=8
    "persist": "npst",   # root §7 — y_hat = last observed return, K=1
    "seasonal": "nsea",  # root §7 — y_hat = return at t-24, K=1
    # `D70`'s four arms. All exploratory, all declared before running, all
    # reported whatever they show — root §13.2's commitment, which an arm
    # reported only when it agrees with the headline does not meet.
    "orthogonal": "itro",  # K=8, one or two per family — high effective rank
    "redundant": "itrr",   # K=8, F2 and F3 loaded whole — low effective rank
    "look048": "l048",     # L=48 at K=8 — half the pre-registered lookback
    "look192": "l192",     # L=192 at K=8 — double it
    "tuned": "itrt",       # K=8 at the config origin 1's validation preferred
    # `D62`'s three exploratory arms. Distinct tags, so none can collide with a
    # completed run_id and all 684 stay complete under resume.
    "attention": "itra",  # `D62d` — Figure 5's attention maps, K=8
    "longsched": "itrl",  # `D62c` — LR halved every 8, 60 epochs, patience 10
    "capacity": "itrc",   # `D62b` — root §6.2's own larger-d_ff run at K=12
}

#: The §7 comparators (`D56`). Two things key off this set, and both follow from
#: these arms being a *different model* rather than a different configuration of
#: the same one: the `D45` window-alignment assertion runs for them, and their
#: ``run_id`` prefix keeps them apart in every table.
BASELINE_ARMS: tuple[str, ...] = (
    "ridge", "dlinear", "patchtst", "lstm", "persist", "seasonal",
)

#: Every arm, in execution order. iTransformer first, so that a session cut short
#: leaves the ladder — which RQ1, RQ2 and RQ3 all read — complete before the
#: comparators, and so a baseline's alignment assertion finds its reference on
#: disk rather than reporting itself unchecked.
#: `D62`'s exploratory arms. Ordered **last** so a session cut short loses
#: robustness rather than anything RQ1-RQ3 reads, on the same reasoning that put
#: the baselines after the ladder.
ROBUSTNESS_ARMS: tuple[str, ...] = (
    "attention", "longsched", "capacity",
    # `D70`. Ordered with the rest of the robustness block, after the baselines,
    # so a session cut short loses a robustness arm rather than an RQ input.
    "orthogonal", "redundant", "look048", "look192", "tuned",
)

ALL_ARMS: tuple[str, ...] = (
    "main", "uniform", "fresh", "horizon", *BASELINE_ARMS, *ROBUSTNESS_ARMS,
)

#: Seeds for the horizon sweep. Three, not five: root §10.2 budgets 192 runs for
#: it, and root §10.3 says to cut the sweep before cutting seed counts if the
#: grid ever stops fitting, because `D30` and `D49` depend on the seed counts.
SWEEP_SEEDS: tuple[int, ...] = SEEDS
#: Was ``SEEDS[:3]``, and the reason was budget rather than design (`D70`): root
#: §10.3 sized the sweep against a single-GPU session. With both devices working
#: the third and fourth seed cost wall-clock the session has, and `D30`'s rule
#: cuts the other way once they are affordable — a number aggregated across
#: origins carries an SE across origins, and seed dispersion is the Monte-Carlo
#: diagnostic beside it. Three seeds is a thin diagnostic.

#: Seeds for the stochastic baselines. Three, matching root §10.2's baseline
#: budget of "3 stochastic x 3 seeds". `D49`'s five-seed rule is about the
#: **rungs of the ladder**, where the 8->12 contrast cannot be the one carrying
#: the fewest; a baseline is a single cell rather than a rung, and a fourth and
#: fifth seed there would buy precision on a number no hypothesis is stated about.
BASELINE_SEEDS: tuple[int, ...] = SEEDS
#: Also raised from three (`D70`), and the attention arm is why it matters most:
#: root §13.2 admits attention maps only when they are "validated for stability
#: across seeds", and the measured calm-to-stress shift (**+0.00056**) is smaller
#: than the between-seed standard deviation of a single weight (**0.00064**). A
#: claim that rests on that comparison should not rest on three draws.

#: Root §10.5. Checked at **run boundaries**, not epoch boundaries: runs are
#: short, epochs are shorter, and the checkpoint granularity is the run.
SESSION_BUDGET_H: float = 11.0
RESERVE_H: float = 0.5


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔲 Sel run</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Satu sel grid, dan <code>run_id</code> deterministiknya. Mengubah komponen mana pun <strong>meng-orphan</strong> keluaran lama, bukan diam-diam memakainya ulang.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class RunCell:
    """One (arm, origin, K, H, seed) cell of the grid."""

    arm: str
    origin_index: int
    k: int
    pred_len: int
    seed: int
    #: Lookback. Only the `D70` sweep moves it; every other arm takes root §6.2's
    #: 96. It is **not** in ``run_id`` — root §10.4 fixes that format — so the two
    #: sweep arms carry their own tags and cannot collide with the ladder.
    seq_len: int = SEQ_LEN
    #: Name in :data:`MATCHED_K_SUBSETS`, or "" for the rung's own columns.
    subset: str = ""

    @property
    def model_tag(self) -> str:
        return ARM_MODEL_TAG[self.arm]

    @property
    def spec(self) -> RunSpec:
        return RunSpec(
            model=self.model_tag,
            origin_index=self.origin_index,
            k=self.k,
            pred_len=self.pred_len,
            seed=self.seed,
        )

    @property
    def run_id(self) -> str:
        return self.spec.run_id

    @property
    def group(self) -> tuple[str, int, int, int]:
        """The **shard** key. Seeds inside a group stay with one worker."""
        return (self.arm, self.origin_index, self.k, self.pred_len)

    @property
    def tensor_key(self) -> tuple[bool, int, int, int]:
        """The **tensor-build** key, which is coarser than :attr:`group`.

        :func:`build_origin_tensors` reads the origin, K and H and nothing else,
        and only the falsification arm changes the origin object. So ridge at
        (origin 7, K=8, H=24) consumes byte-for-byte the tensors the main arm
        already built there, and keying the cache by arm would rebuild them —
        150 redundant builds across the baseline arms. Sharding still keys on
        :attr:`group`, because that partition must be a function of the arm.
        """
        return (
            self.arm == "fresh",
            self.origin_index,
            self.k,
            self.pred_len,
            # `D70`: a lookback and a column set change the tensors, so two cells
            # that differ in either must not share a cached build. Left out, the
            # L=192 arm would silently train on L=96 windows.
            self.seq_len,
            self.subset,
        )

    def columns(self) -> tuple[str, ...] | None:
        """The named column set this cell trains on, or ``None`` for its rung."""
        return MATCHED_K_SUBSETS[self.subset] if self.subset else None

    def origin(self) -> OriginLike:
        base = ORIGINS[self.origin_index - 1]
        return FalsificationOrigin(base) if self.arm == "fresh" else base

    def model_config(self, overrides: dict[str, Architecture] | None = None) -> Architecture:
        """The arm's configuration — hyperparameters fixed a priori in every case.

        **No per-rung tuning** (`D38`): holding capacity fixed is what makes the
        rungs comparable, and tuning per rung would confound the ladder with
        model selection. The only field an iTransformer arm may move is
        ``uniform_attention``, which *is* the arm.

        The rule extends to the §7 baselines rather than exempting them. Ridge's
        alpha is the single exception root §11 names, and it is not chosen here:
        :meth:`RidgeConfig.fit` selects it on the validation sub-block and
        returns the resolved config, which is what reaches ``meta['config']``.
        """
        if self.arm == "ridge":
            return RidgeConfig(pred_len=self.pred_len, k=self.k)
        if self.arm == "dlinear":
            return DLinearConfig(pred_len=self.pred_len)
        if self.arm == "patchtst":
            return PatchTSTConfig(pred_len=self.pred_len)
        if self.arm == "lstm":
            return LSTMConfig(pred_len=self.pred_len, k=self.k)
        if self.arm in ("persist", "seasonal"):
            return NaiveConfig(mode=self.arm, pred_len=self.pred_len, k=self.k)
        if self.arm == "tuned":
            # Selected once, on origin 1's validation, exactly where `D27` put the
            # Stage 5 gate and for the same reason: a selection that reads a test
            # block cannot coexist with root §11's "test blocks are opened once".
            # The notebook computes it in the prelude and hands it in; there is no
            # default, because a tuned arm silently running the untuned config
            # would answer the referee's question with the wrong number.
            if not overrides or "tuned" not in overrides:
                raise ValueError(
                    "the tuned arm needs its selected config passed in "
                    "(`configs={'tuned': ...}`); run tune_on_validation first"
                )
            return overrides["tuned"]
        if self.arm in ("look048", "look192", "orthogonal", "redundant"):
            return ITransformerConfig(seq_len=self.seq_len, pred_len=self.pred_len)
        if self.arm == "longsched":
            # `D62c`. The only thing that moves is the schedule, which is a
            # method, so meta['config'] is byte-identical to the main arm's and
            # meta['schedule'] is where the difference shows.
            return LongScheduleConfig(pred_len=self.pred_len)
        if self.arm == "capacity":
            # `D62b`. Root §6.2 pre-registers exactly this --- "one robustness run
            # at K=12 with larger d_ff, so a flat 8->12 rung cannot be read as an
            # under-tuning artefact" --- and it was never built. ``d_ff`` IS a
            # config field, so this run's meta records the widening, correctly:
            # it is the only thing differing from the rung it answers for.
            return ITransformerConfig(pred_len=self.pred_len, d_ff=512)
        return ITransformerConfig(
            pred_len=self.pred_len,
            uniform_attention=(self.arm == "uniform"),
        )

    def reference_run_id(self) -> str:
        """The iTransformer run this cell is compared against (`D45`).

        Same origin, same K, same horizon, first seed — the main-grid cell whose
        evaluated window set this run's must equal exactly before any RelMSE or
        DM statistic is formed across the two.
        """
        return RunSpec(
            ARM_MODEL_TAG["main"], self.origin_index, self.k, self.pred_len, SEEDS[0]
        ).run_id


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📜 Manifes 969 run</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">684 konfirmatori, 210 robustness <code>D62</code>, 75 baseline yang <code>D64</code> bangun. Irisan H=24 sweep dideduplikasi terhadap grid utama (<code>D53e</code>).</p>
</div>

In [ ]:


def manifest(arms: tuple[str, ...] = ALL_ARMS) -> list[RunCell]:
    """Every run in the study, deduplicated and ordered.

    Root §10.2's accounting, arm by arm:

    ==========  =====  =========================================================
    Arm         Runs   Composition
    ==========  =====  =========================================================
    main          300  15 origins x 4 K x 5 seeds (`D49` — 5 at *every* rung,
                       because the 8->12 rung is RQ1's designed contrast and
                       cannot carry the fewest)
    uniform        75  15 x K=8 x 5 seeds (`D50`)
    fresh          15  one fresh model per origin at ``o_i + 90 d``
    horizon       192  4 named origins x 4 K x 4 H x 3 seeds (`D08`, `D48`)
    ridge          60  15 x 4 K, deterministic (`D17`)
    dlinear        45  15 x K=8 x 3 seeds (root §7)
    patchtst       45  15 x K=8 x 3 seeds (root §7)
    lstm           45  15 x K=8 x 3 seeds (root §7, `D64`)
    persist        15  15 origins, deterministic (root §7, `D64`)
    seasonal       15  15 origins, deterministic (root §7, `D64`)
    ==========  =====  =========================================================

    **48 of those cells are literally the same run.** The sweep's ``H=24`` slice
    at seeds 42-44 carries the same ``run_id`` as the corresponding main-grid
    cells, so the iTransformer union is **534 unique runs**, not 582, and the
    whole manifest is **684**. Deduplicating is not a saving quietly banked: root
    §10.4 makes ``run_id`` the identity of a run, so executing one twice would
    mean two files racing for one path.

    **The three baseline arms are new, and their absence was `D56`.** Root §7
    calls DLinear and PatchTST "not optional" and §10.2 budgets 255 baseline
    runs, but no baseline class existed and this manifest never contained one —
    so §10.2's 789 was never executable, and the study's central architectural
    comparison had no data. 150 of that 255 are built: the deferred remainder is
    ARIMA, LSTM, naive-persist and seasonal-naive, listed in ``baselines.py``
    rather than left silently unbuilt. Naive-RW needs no run at all, being
    computed inside :func:`itransformer_btc.metrics.block_metrics` on exactly the
    rows its comparator was scored on.

    Ordering is by group, so the seeds of a cell reuse one tensor build, and
    groups are emitted arm by arm so a shard split stays balanced across the
    heavy ``H=168`` cells.
    """
    cells: list[RunCell] = []

    if "main" in arms:
        cells += [
            RunCell("main", o.index, k, PRED_LEN, s)
            for o in ORIGINS for k in K_LADDER for s in SEEDS
        ]
    if "uniform" in arms:
        cells += [
            RunCell("uniform", o.index, 8, PRED_LEN, s) for o in ORIGINS for s in SEEDS
        ]
    if "fresh" in arms:
        # One seed. The arm asks whether the aged-minus-fresh gap is zero, and
        # that contrast is between two models, not between five initialisations.
        cells += [RunCell("fresh", o.index, 8, PRED_LEN, SEEDS[0]) for o in ORIGINS]
    if "horizon" in arms:
        cells += [
            RunCell("horizon", i, k, h, s)
            for i in SWEEP_ORIGIN_INDICES
            for k in K_LADDER
            for h in HORIZONS
            for s in SWEEP_SEEDS
        ]
    if "ridge" in arms:
        # One seed. Ridge is a solve, not an optimisation: a second seed would
        # reproduce the first to the last bit. The seed component of ``run_id``
        # is carried only because root §10.4 fixes the format.
        cells += [
            RunCell("ridge", o.index, k, PRED_LEN, SEEDS[0])
            for o in ORIGINS for k in K_LADDER
        ]
    if "dlinear" in arms:
        cells += [
            RunCell("dlinear", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in BASELINE_SEEDS
        ]
    if "patchtst" in arms:
        cells += [
            RunCell("patchtst", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in BASELINE_SEEDS
        ]
    if "lstm" in arms:
        cells += [
            RunCell("lstm", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in BASELINE_SEEDS
        ]
    for naive in ("persist", "seasonal"):
        # One seed each. Neither has a parameter and neither consumes RNG, so a
        # second seed would reproduce the first to the last bit — the same
        # reasoning that gives ridge one seed.
        if naive in arms:
            cells += [
                RunCell(naive, o.index, 1, PRED_LEN, SEEDS[0]) for o in ORIGINS
            ]
    for arm, subset in (("orthogonal", "orthogonal"), ("redundant", "redundant")):
        # Same K, same seeds, same everything but the column set (`D70`). Five
        # seeds because this is RQ1's direct contrast and `D49`'s reasoning about
        # the ladder applies to it: the rung carrying the comparison cannot be the
        # one carrying the fewest draws.
        if arm in arms:
            cells += [
                RunCell(arm, o.index, 8, PRED_LEN, s, subset=subset)
                for o in ORIGINS for s in SEEDS
            ]
    for arm, seq_len in (("look048", 48), ("look192", 192)):
        # L is the one first-order hyperparameter root §6.2 never varied. 192 is
        # the ceiling: window cost per break is ``L + H - 1``, so at 336 the
        # pooled loss approaches root §4.3's tolerance and the early origins,
        # where every outage lives, would carry it worst.
        if arm in arms:
            cells += [
                RunCell(arm, o.index, 8, PRED_LEN, s, seq_len=seq_len)
                for o in ORIGINS for s in SWEEP_SEEDS
            ]
    if "tuned" in arms:
        # The config is selected once on origin 1's validation and handed to
        # ``execute``; these cells only carry it across the panel.
        cells += [
            RunCell("tuned", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in SEEDS
        ]
    if "attention" in arms:
        # Three seeds, because root §13.2 admits attention maps only when they are
        # "validated for stability across seeds" (Jain & Wallace 2019; Wiegreffe &
        # Pinter 2019). One seed makes that claim unfalsifiable; five would buy
        # precision on a descriptive figure. Same seeds as the main arm, so each
        # cell must reproduce its twin bit for bit --- which is the arm's second
        # product and the study's reproducibility statement.
        cells += [
            RunCell("attention", o.index, 8, PRED_LEN, s)
            for o in ORIGINS for s in BASELINE_SEEDS
        ]
    if "longsched" in arms:
        # K in {1, 8} only: the arm asks whether the null survives a longer
        # schedule, and that needs the control and the treatment, not the whole
        # ladder. Three seeds --- `D49`'s five-seed rule protects the 8->12
        # contrast, which this arm does not contain.
        cells += [
            RunCell("longsched", o.index, k, PRED_LEN, s)
            for o in ORIGINS for k in (1, 8) for s in BASELINE_SEEDS
        ]
    if "capacity" in arms:
        # Five seeds, matching the K=12 rung it is compared against.
        cells += [
            RunCell("capacity", o.index, 12, PRED_LEN, s) for o in ORIGINS for s in SEEDS
        ]

    seen: set[str] = set()
    unique: list[RunCell] = []
    for cell in cells:
        if cell.run_id not in seen:
            seen.add(cell.run_id)
            unique.append(cell)
    return unique


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔍 Penemuan & resume</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Ditemukan lewat glob, <strong>tidak pernah lewat slug Dataset yang dikodekan mati</strong>, jadi nama Dataset Kaggle bebas berubah.</p>
</div>

In [ ]:


def discover_roots(working: Path = ARTIFACTS) -> list[Path]:
    """Artifact roots to search, working directory first.

    Root §10.5: discover by **globbing** ``/kaggle/input/*/``, never a hard-coded
    dataset slug, so the Kaggle Dataset can be renamed without editing code. Any
    input directory holding a ``preds`` folder counts, whatever it is called, and
    one nesting level is searched because Kaggle wraps some dataset uploads in an
    extra folder.
    """
    roots = [Path(working)]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        roots += sorted(p for p in kaggle_input.iterdir() if (p / "preds").is_dir())
        roots += sorted(p.parent for p in kaggle_input.glob("*/*/preds") if p.is_dir())
    seen: set[str] = set()
    return [r for r in roots if not (str(r) in seen or seen.add(str(r)))]


def completed_run_ids(roots: list[Path], code_digest: str = "") -> set[str]:
    """Run ids complete under root §10.5: both artifacts present, status complete.

    A prediction file without its meta, or a meta whose status is anything else,
    is **not** complete and the run is redone from scratch. Intra-run
    checkpointing is deliberately omitted: at ~30 s per run measured (`D57`) it
    costs far more complexity than it saves.

    **A run from a different code vintage is not complete either** (`D85`). Root
    §10.4 says a changed component orphans prior outputs rather than silently
    reusing a mismatched result, but ``run_id`` encodes ``{model, origin, K, H,
    seed}`` and nothing about the configuration, so a fix *inside* an arm leaves
    the ids identical and resume happily keeps the stale predictions. `D76` is
    exactly that shape: the tuned arm's learning rate changed, the 75 ``itrt``
    ids did not, and a resumed session would have skipped every one of them and
    reported the old configuration under the new caption. Comparing
    ``meta['code_sha256']`` against the running package closes it, and it is also
    §12's rule stated as behaviour --- numbers produced under different digests
    are not comparable, so a grid must not silently become a mixture of two.

    The cost is bounded and the trade is the right way round: within one vintage
    every digest matches and resume behaves exactly as before, which is the case
    a partial session actually hits. Across vintages the grid re-runs, which is
    what §12 asks for and what the 1,620-run session did anyway in 8.96 h.

    **The filter is off by default, and only :func:`pending` turns it on.** This
    function has two callers wanting opposite things. Resume decides whether to
    *skip* a run and must be strict. The report merely *discovers* what exists,
    and `D62g` settles that case in the other direction: the vintage that matters
    for a number is the vintage of the runs that produced it, and the reporting
    code is a reader of those runs, not a producer of them. A strict default
    would have made the generator refuse to read a grid it is perfectly able to
    describe.

    Args:
        roots: Directories to search, each holding ``preds/`` and ``meta/``.
        code_digest: Vintage to require. Empty accepts any, which is what a
            reader wants; :func:`pending` passes the running package's.

    Returns:
        The run ids that may be skipped.
    """
    wanted = code_digest
    done: set[str] = set()
    for root in roots:
        meta_dir = Path(root) / "meta"
        if not meta_dir.is_dir():
            continue
        for meta_path in meta_dir.glob("*.json"):
            run_id = meta_path.stem
            if not (Path(root) / "preds" / f"{run_id}.parquet").exists():
                continue
            try:
                meta = json.loads(meta_path.read_text())
            except (json.JSONDecodeError, OSError):
                continue
            if meta.get("status") != "complete":
                continue
            if wanted and meta.get("code_sha256") not in (None, wanted):
                continue
            done.add(run_id)
    return done


def pending(cells: list[RunCell], roots: list[Path]) -> list[RunCell]:
    """Manifest minus what is already complete **at this code vintage** (`D85`).

    The vintage clause is the whole point. ``run_id`` encodes
    ``{model, origin, K, H, seed}`` and nothing about the configuration, so root
    §10.4's "a changed component orphans prior outputs" only covers a change that
    renames the arm. A fix *inside* an arm leaves every id identical, and `D76`
    was exactly that: the tuned arm's learning rate changed and its 75 ids did
    not, so a resumed session would have skipped all of them and reported the old
    configuration under the corrected caption.

    Passing the digest makes the skip decision agree with §12, which already says
    numbers from different digests are not comparable: a grid must not silently
    become a mixture of two. Within one vintage nothing changes, which is the
    case a partial session actually hits.
    """
    done = completed_run_ids(roots, code_digest=code_sha256())
    return [c for c in cells if c.run_id not in done]


def shard(cells: list[RunCell], index: int, count: int) -> list[RunCell]:
    """Round-robin by **group**, so a cell's seeds share one tensor build.

    Sharding by cell instead would send consecutive seeds to different workers
    and make both build the same tensors — correct, but paying the build cost
    twice for nothing.
    """
    groups: list[tuple[str, int, int, int]] = []
    for cell in cells:
        if cell.group not in groups:
            groups.append(cell.group)
    owned = {g for i, g in enumerate(groups) if i % count == index}
    return [c for c in cells if c.group in owned]


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">⏰ Penjaga anggaran sesi</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Diperiksa di batas run, bukan batas epoch. <code>SESSION_T0</code> distempel di sel 0 supaya prelude ikut terhitung (<code>D54f</code>).</p>
</div>

In [ ]:


class BudgetGuard:
    """Root §10.5's session budget, checked at run boundaries.

    Hitting Kaggle's own 12 h wall interactively loses ``/kaggle/working``
    entirely, so the guard stops early enough that Save Version still runs. It
    also refuses to *start* a run it does not expect to finish, using the
    observed mean wall time: stopping at 10.9 h and then beginning a 98 s run is
    the failure mode a naive elapsed-only check has.
    """

    def __init__(
        self, budget_h: float = SESSION_BUDGET_H, reserve_h: float = RESERVE_H
    ) -> None:
        self.deadline = time.perf_counter() + (budget_h - reserve_h) * 3600.0
        self.durations: list[float] = []

    def record(self, seconds: float) -> None:
        self.durations.append(seconds)

    @property
    def mean_run_s(self) -> float:
        return sum(self.durations) / len(self.durations) if self.durations else 120.0

    @property
    def remaining_s(self) -> float:
        return self.deadline - time.perf_counter()

    def may_start(self) -> bool:
        return self.remaining_s > self.mean_run_s


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🗄️ Cache tensor per origin</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Tensor origin dimuat sekali dan dipakai ulang lintas rung dan seed.</p>
</div>

In [ ]:


class _TensorCache:
    """Small LRU over ``(arm, origin, K, H)`` builds.

    Bounded because one build is up to 70.12 MB of training tensor plus its
    validation and test blocks (root §10.3 / `D25`); a few is comfortable in
    Kaggle's RAM, an unbounded cache across 154 groups is not.
    """

    def __init__(self, features: pl.DataFrame, size: int = 3) -> None:
        self.features = features
        self.size = size
        self._store: OrderedDict[tuple, OriginTensors] = OrderedDict()

    def get(self, cell: RunCell) -> OriginTensors:
        key = cell.tensor_key
        if key in self._store:
            self._store.move_to_end(key)
            return self._store[key]
        tensors = build_origin_tensors(
            self.features,
            cell.origin(),
            cell.k,
            seq_len=cell.seq_len,
            pred_len=cell.pred_len,
            columns=cell.columns(),
        )
        self._store[key] = tensors
        while len(self._store) > self.size:
            self._store.popitem(last=False)
        return tensors


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧾 Ringkasan eksekusi & penyelarasan</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Selesai, dilewati, gagal — dan asersi keselarasan sebelum satu pun perbandingan dibentuk.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class ExecutionSummary:
    """What one worker did, and what is left.

    ``remaining`` and ``estimated_sessions`` are printed on exit because a
    session that ends without saying how much is left forces the next one to
    re-derive it (``notebooks/CLAUDE.md``).
    """

    completed: int
    skipped: int
    failed: int
    #: Pending **in this shard**, not in the whole manifest. The notebook prints
    #: the global figure; a worker only knows its own queue.
    remaining: int
    wall_time_s: float
    mean_run_s: float

    @property
    def estimated_sessions(self) -> float:
        if self.remaining == 0:
            return 0.0
        usable = (SESSION_BUDGET_H - RESERVE_H) * 3600.0
        return self.remaining * self.mean_run_s / usable

    def __str__(self) -> str:
        return (
            f"completed {self.completed}  skipped {self.skipped}  "
            f"failed {self.failed}  remaining {self.remaining}\n"
            f"wall {self.wall_time_s / 3600:.2f} h  mean run {self.mean_run_s:.1f} s  "
            f"estimated sessions left {self.estimated_sessions:.2f}"
        )


def _assert_alignment(cell: RunCell, roots: list[Path], log) -> None:
    """`D45`, enforced when the file is written rather than when the table is built.

    Root §7 requires every baseline to be scored on **exactly** the surviving
    window set of the run it is compared against. The two sets are equal by
    construction — both come from :func:`window_starts` with the same origin,
    span and semantics — which is why this costs microseconds and why it is the
    only thing that would notice if that ever stopped holding.

    A missing comparator is **reported, never swallowed**. The check is then
    unrun, and an unrun check that prints nothing is indistinguishable from a
    passing one; :data:`ALL_ARMS` orders the ladder first precisely so this stays
    the rare case rather than the normal one.
    """
    reference = cell.reference_run_id()
    try:
        assert_baseline_alignment(cell.run_id, reference, roots)
    except FileNotFoundError:
        log(
            f"  {cell.run_id}: `D45` alignment UNCHECKED — comparator "
            f"{reference} is not on disk in {[str(r) for r in roots]}"
        )


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🚀 Eksekutor grid</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Berurutan pada satu <code>cuda:0</code>. Terukur pada manifes 894-run: 0 gagal, 7,79 jam, rata-rata 31,8 s/run.</p>
</div>

In [ ]:


def execute(
    cells: list[RunCell],
    features: pl.DataFrame,
    *,
    out_root: Path = ARTIFACTS,
    roots: list[Path] | None = None,
    guard: BudgetGuard | None = None,
    device=None,
    configs: dict[str, Architecture] | None = None,
    log=print,
) -> ExecutionSummary:
    """Run a shard to completion or to the budget, whichever comes first.

    A failing run is logged and skipped rather than aborting the shard: with 684
    runs, losing the rest of a session to one bad cell is worse than finishing
    the others and letting root §10.5's resume pick that cell up next session.
    A failing *invariant* is the opposite case and does end the shard — see
    :func:`_assert_alignment`.
    """
    guard = guard or BudgetGuard()
    roots = roots or discover_roots(out_root)
    device = device or pick_device()
    cache = _TensorCache(features)

    started = time.perf_counter()
    done = completed_run_ids(roots)
    completed = skipped = failed = 0
    queue = list(cells)

    for position, cell in enumerate(queue, start=1):
        if cell.run_id in done or is_complete(cell.run_id, out_root):
            skipped += 1
            continue
        if not guard.may_start():
            log(
                f"budget guard: {guard.remaining_s / 60:.1f} min left, mean run "
                f"{guard.mean_run_s:.0f} s — stopping cleanly so the version saves"
            )
            break

        began = time.perf_counter()
        try:
            tensors = cache.get(cell)
            # The config comes back **resolved**: identical for every
            # iTransformer arm (`D38` — nothing is tuned), and carrying the
            # chosen alpha for ridge, which is the one selection root §11 admits.
            # Writing the config that went in would lose it.
            model, cfg, outcome = cell.model_config(configs).fit(
                tensors, cell.spec, device=device
            )
            # Figure 5's maps, and only for the arm that exists to produce them
            # (`D62d`). Captured after training rather than during it, so nothing
            # about the optimisation changes and the arm reproduces its main-grid
            # twin bit for bit.
            maps = (
                tercile_maps(model, tensors, device) if cell.arm == "attention" else None
            )
            write_artifacts(
                model, tensors, cell.spec, cfg, outcome, device,
                root=out_root, attention=maps,
            )
        except Exception as exc:  # noqa: BLE001 - one bad cell must not end the shard
            failed += 1
            log(f"[{position}/{len(queue)}] {cell.run_id} FAILED: {exc!r}")
            continue

        # Outside the try, and deliberately fatal. A window-set mismatch between
        # a baseline and its comparator is the defect class root §11 calls
        # fatal — RelMSE across two samples is not a ratio — and continuing would
        # fill Table 6 with statistics that mean nothing. One bad *cell* must not
        # end a shard; one broken *invariant* must.
        if cell.arm in BASELINE_ARMS:
            _assert_alignment(cell, roots, log)

        elapsed = time.perf_counter() - began
        guard.record(elapsed)
        completed += 1
        log(
            f"[{position}/{len(queue)}] {cell.run_id}  "
            f"epochs={outcome.epochs_run}  val={outcome.best_val_mse:.6f}  "
            f"{elapsed:.1f}s  n_train={len(tensors.train)}"
        )

    return ExecutionSummary(
        completed=completed,
        skipped=skipped,
        failed=failed,
        remaining=len(pending(queue, discover_roots(out_root))),
        wall_time_s=time.perf_counter() - started,
        mean_run_s=guard.mean_run_s,
    )


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🖥️ Dua GPU, tingkat-run</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Satu worker per device dari satu antrean bersama. <strong>Tingkat-run, tidak pernah tingkat-batch</strong> — <code>DataParallel</code> ditolak root §10.3 dengan alasan terukur, dan DDP lebih buruk lagi untuk 969 run ~32 detik. Determinisme dijaga: seed ber-scope device, prolog di bawah satu lock (<code>D68</code>).</p>
</div>

In [ ]:



def visible_devices() -> list[torch.device]:
    """Every CUDA device the session can see, or ``[cpu]`` when there is none.

    Root §10.1 lists 2 x T4 and root §10.3 says one is enough — which was true of
    a 969-run manifest that fits an 11 h session. It stops being true the moment
    the grid grows, and the 894-run session left **half the hardware idle for 7.8
    hours** because nothing asked this question (`D68`).
    """
    if not torch.cuda.is_available():
        return [torch.device("cpu")]
    return [torch.device("cuda", i) for i in range(torch.cuda.device_count())]


def execute_parallel(
    cells: list[RunCell],
    features: pl.DataFrame,
    *,
    devices: list[torch.device] | None = None,
    out_root: Path = ARTIFACTS,
    roots: list[Path] | None = None,
    guard: BudgetGuard | None = None,
    configs: dict[str, Architecture] | None = None,
    log=print,
) -> ExecutionSummary:
    """:func:`execute`, one worker thread per device, off a shared queue.

    **Run level, never batch level** (root §10.3). ``nn.DataParallel`` is rejected
    there with a measured reason that has not changed: at batch 32 and ~280k
    parameters the scatter/gather costs more than splitting saves, and DDP is
    worse again for 969 runs of ~32 s each, where the process group is paid per
    run. What parallelises cleanly is the *grid*, because a run is already the
    unit of work and nothing crosses between two of them.

    Threads rather than processes, because §15's notebook carries the package as
    definition cells in one kernel namespace: a subprocess inherits none of it and
    ``python -m itransformer_btc.runner`` has no files to import. ``launch_workers``
    remains the path from a checkout. Threads are the path from the notebook, and
    the GIL is not the constraint here — every run spends its time inside CUDA
    kernels and tensor ops that release it.

    **Determinism is the property this must not cost**, and the whole design is
    that one guarantee (`D68`):

    - each worker owns one device and never touches another's;
    - :func:`set_seed` is device-scoped, so seeding ``cuda:0`` leaves ``cuda:1``'s
      generator alone;
    - the CPU generator is shared, so seeding and module construction happen under
      :data:`itransformer_btc.train.SEED_LOCK` — milliseconds against a ~32 s run;
    - each worker keeps its **own** tensor cache, so no build races another.

    Everything after the prologue draws from the device's own CUDA generator. A
    run therefore produces the same bytes whether it ran alone or beside another,
    which is what `D62d` demonstrated for the attention arm and what root §12
    requires of every number in the manuscript.

    The budget guard is shared and its methods are called under a lock, so two
    workers cannot both slip past a deadline that only one of them had room for.
    """
    devices = devices or visible_devices()
    if len(devices) < 2:
        return execute(
            cells, features,
            out_root=out_root, roots=roots, guard=guard,
            device=devices[0] if devices else None, configs=configs, log=log,
        )

    guard = guard or BudgetGuard()
    roots = roots or discover_roots(out_root)
    done = completed_run_ids(roots)

    queue = list(cells)
    cursor = 0
    completed = skipped = failed = 0
    state = threading.Lock()
    started = time.perf_counter()
    log(f"run-level parallelism across {[str(d) for d in devices]} (`D68`)")

    def take() -> tuple[int, RunCell] | None:
        nonlocal cursor
        with state:
            if cursor >= len(queue) or not guard.may_start():
                return None
            cursor += 1
            return cursor, queue[cursor - 1]

    def worker(device: torch.device) -> None:
        nonlocal completed, skipped, failed
        cache = _TensorCache(features, size=2)
        while (item := take()) is not None:
            position, cell = item
            if cell.run_id in done or is_complete(cell.run_id, out_root):
                with state:
                    skipped += 1
                continue

            began = time.perf_counter()
            try:
                tensors = cache.get(cell)
                model, cfg, outcome = cell.model_config(configs).fit(
                    tensors, cell.spec, device=device
                )
                maps = (
                    tercile_maps(model, tensors, device)
                    if cell.arm == "attention"
                    else None
                )
                write_artifacts(
                    model, tensors, cell.spec, cfg, outcome, device,
                    root=out_root, attention=maps,
                )
            except Exception as exc:  # noqa: BLE001 - one bad cell must not end the shard
                with state:
                    failed += 1
                log(f"[{position}/{len(queue)}] {device} {cell.run_id} FAILED: {exc!r}")
                continue

            if cell.arm in BASELINE_ARMS:
                _assert_alignment(cell, roots, log)

            elapsed = time.perf_counter() - began
            with state:
                guard.record(elapsed)
                completed += 1
            log(
                f"[{position}/{len(queue)}] {device} {cell.run_id}  "
                f"epochs={outcome.epochs_run}  val={outcome.best_val_mse:.6f}  "
                f"{elapsed:.1f}s  n_train={len(tensors.train)}"
            )

    threads = [
        threading.Thread(target=worker, args=(d,), name=f"grid-{d}", daemon=True)
        for d in devices
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    if cursor < len(queue):
        log(
            f"budget guard: stopped with {len(queue) - cursor} cells unstarted — "
            f"resume picks them up next session (root §10.5)"
        )

    return ExecutionSummary(
        completed=completed,
        skipped=skipped,
        failed=failed,
        remaining=len(pending(queue, discover_roots(out_root))),
        wall_time_s=time.perf_counter() - started,
        mean_run_s=guard.mean_run_s,
    )


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎛️ Grid tuning & pemilih konfigurasi</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Delapan belas konfigurasi, <strong>dideklarasikan sebelum dijalankan</strong>, diranking pada sub-blok <strong>validasi</strong> origin 1 — persis tempat <code>D27</code> menaruh gerbang Stage 5, dan karena alasan yang sama. Menjawab satu-satunya serangan yang §6.2 biarkan terbuka: <em>kalian tidak mencoba</em> (<code>D70</code>).</p>
</div>

In [ ]:



#: The grid the tuned arm searches. Declared here, before it runs, because a
#: search space chosen after seeing which config won is not a search (`D70`).
#:
#: Three knobs root §6.2 adopted from Liu et al. (2024) and never varied, each at
#: the published value and one step either side. ``d_ff`` is absent on purpose:
#: `D62b`'s capacity arm already swept it and made things worse at 13 of 15
#: origins on RelMSE, and repeating it here would spend the budget re-answering a
#: question that has an answer.
TUNING_GRID: tuple[dict[str, object], ...] = tuple(
    {"d_model": d, "e_layers": e, "lr": lr}
    for d in (64, 128, 256)
    for e in (2, 3)
    for lr in (1e-4, 3e-4, 1e-3)
)

#: Epochs each probe is given. Short on purpose: this ranks configurations, it
#: does not train them. The winner is then trained under root §6.2's full budget
#: --- 30 epochs, patience 5, LR halved every 4 --- **at the learning rate the
#: search selected**, which :class:`~itransformer_btc.model.TunedConfig` carries
#: (`D76`). Everything except ``lr`` is the main arm's, so the arm differs from
#: the ladder in exactly what the search chose and in nothing else.
TUNING_EPOCHS: int = 6


def tune_on_validation(
    features: pl.DataFrame,
    *,
    origin_index: int = 1,
    k: int = 8,
    device=None,
    log=print,
) -> tuple[TunedConfig, list[dict]]:
    """Pick one iTransformer config on one origin's **validation** sub-block.

    This exists to answer the one attack on the null that root §6.2 leaves open.
    Nothing in this study is tuned — every hyperparameter is adopted from
    Liu et al. (2024) and held identical at every rung, which is what makes the
    rungs comparable (`D38`). A referee reads that as *"you did not try"*, and the
    honest reply is a number rather than a paragraph: **if the configuration the
    validation set prefers is still worse than a random walk, the null is not an
    artefact of the defaults.**

    Three properties keep it inside the pre-registration:

    - **The winner is run as selected, learning rate included** (`D76`). Anything
      the search ranks on and the arm then does not apply makes the arm answer a
      different question from the one its caption claims.
    - **Validation only, one origin.** Exactly where `D27` put the Stage 5 gate,
      and for its reason: root §11 opens the test blocks once, after the design is
      frozen, so a selection that reads one cannot coexist with it.
    - **The grid is declared before it runs** (:data:`TUNING_GRID`). A space
      chosen after seeing the winner is not a search.
    - **The arm is exploratory and reported whatever it shows** (root §13.2). It
      does not enter RQ1's ladder comparison, and it gets its own row.

    The probes are deterministic given the data, so a resumed session recomputes
    the same winner rather than needing it persisted. Its trial count enters
    root §13.5's development trial total, which is stated rather than concealed.

    Returns:
        The winning config under root §6.2's full schedule, and the ranked table.
    """
    device = device or pick_device()
    origin = ORIGINS[origin_index - 1]
    tensors = build_origin_tensors(features, origin, k, pred_len=PRED_LEN)

    rows: list[dict] = []
    for index, point in enumerate(TUNING_GRID):
        cfg = ITransformerConfig(
            pred_len=PRED_LEN,
            d_model=int(point["d_model"]), e_layers=int(point["e_layers"]),
        )
        spec = RunSpec("tuned", origin_index, k, PRED_LEN, SEEDS[0])
        _, outcome = train_one(
            tensors, spec, cfg, device=device,
            max_epochs=TUNING_EPOCHS, patience=TUNING_EPOCHS,
            lr=float(point["lr"]),
        )
        rows.append({**point, "val_mse": outcome.best_val_mse})
        log(f"  probe {index + 1}/{len(TUNING_GRID)} {point} -> {outcome.best_val_mse:.6f}")

    rows.sort(key=lambda r: r["val_mse"])
    best = rows[0]
    log(f"tuned config selected on origin {origin_index} validation: {best}")
    # `D76`: **the learning rate travels with the winner.** It is a third of the
    # declared search space, and returning a bare ``ITransformerConfig`` --- which
    # has no ``lr`` --- selected it and then threw it away, so the arm ran the
    # winner's architecture under the default rate: a point this same search had
    # evaluated and had not ranked first. :class:`TunedConfig` carries it into
    # ``train_one`` through ``schedule()`` and into ``meta['config']`` through
    # ``asdict``, which is what makes the arm regenerable under root §12.
    return (
        TunedConfig(
            pred_len=PRED_LEN,
            d_model=int(best["d_model"]), e_layers=int(best["e_layers"]),
            lr=float(best["lr"]),
        ),
        rows,
    )


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🚧 Pilot Stage 5</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Berjalan di <strong>validasi</strong>, tidak pernah di uji (<code>D27</code>). Hasilnya <code>S* = +0,8759, p = 0,1906</code> — gerbang <strong>gagal</strong>, dan judul direposisi.</p>
</div>

In [ ]:


@dataclass(frozen=True, slots=True)
class PilotResult:
    """Root §8.5's Stage 5 gate. Note what it does **not** touch: the test blocks."""

    val_mse: dict[int, float]
    clark_west: object
    n_val: int
    passed: bool

    def __str__(self) -> str:
        rungs = "  ".join(f"K={k}: {v:.6f}" for k, v in sorted(self.val_mse.items()))
        verdict = (
            "PASS — K=8 beats K=1 on validation; keep the framing as written"
            if self.passed
            else "FAIL — reposition the title to the descriptive variant NOW, "
                 "not in week nine (root §8.5)"
        )
        return f"validation MSE  {rungs}\n{self.clark_west}\n{verdict}"


def stage5_pilot(
    features: pl.DataFrame,
    *,
    origin_index: int = 1,
    rungs: tuple[int, ...] = K_LADDER,
    seeds: tuple[int, ...] = SEEDS[:3],
    out_root: Path = ARTIFACTS,
    device=None,
    log=print,
) -> PilotResult:
    """Origin 1, 4 K x 3 seeds, scored on the **validation** sub-block (`D27`).

    §11's final item requires the test blocks be opened once, after the design is
    frozen; a gate that repositions the title on a test-block result cannot
    coexist with it. The validation sub-block is the leak-free instrument for a
    go/no-go on architecture.

    The twelve cells are ordinary main-grid ``run_id``s and their artifacts are
    written, so the pilot costs the grid nothing: §10.5's resume finds them
    complete and the main run skips them. That is deliberate and is *why* the
    gate must run on validation — with a test-block gate, the origin that decided
    the paper's framing would end up back inside the evidence for it.

    The gate statistic is **Clark-West, not DM** (`D29`): K=1's feature set is a
    strict subset of K=8's under the same architecture and sample, so the pair is
    nested and standard DM is undersized against exactly the alternative being
    tested. Predictions are averaged across seeds before the test, matching §9.1's
    order of operations.
    """
    # The *name*, not the module, for the reason given at the top of this file.
    # ``from itransformer_btc import metrics`` binds a module **object**, and the
    # flattened notebook has no such object — so ``metrics.clark_west_test`` was a
    # NameError six minutes into a Kaggle session while satisfying every check the
    # repository had (`D59`).

    device = device or pick_device()
    origin = ORIGINS[origin_index - 1]
    cache = _TensorCache(features, size=1)

    val_mse: dict[int, float] = {}
    val_pred: dict[int, "object"] = {}
    y_val = None

    import numpy as _np
    import torch as _torch

    for k in rungs:
        cell = RunCell("main", origin_index, k, PRED_LEN, seeds[0])
        tensors = cache.get(cell)
        y_val = tensors.val.y
        stacked = []
        for seed in seeds:
            spec = RunCell("main", origin_index, k, PRED_LEN, seed).spec
            model, outcome = train_one(tensors, spec, cell.model_config(), device=device)
            write_artifacts(model, tensors, spec, cell.model_config(), outcome,
                            device, root=out_root)
            stacked.append(
                predict(model, _torch.from_numpy(tensors.val.x).to(device))
            )
            log(f"pilot {spec.run_id}  val={outcome.best_val_mse:.6f}  "
                f"{outcome.wall_time_s:.1f}s")
        mean_pred = _np.mean(_np.stack(stacked), axis=0)
        val_pred[k] = mean_pred
        val_mse[k] = float(_np.mean((y_val - mean_pred) ** 2))

    small, large = min(rungs), 8
    # One loss value per forecast origin: the DM/CW series is indexed by the
    # moment the forecast was issued, not by the (origin, step) pair.
    cw = clark_west_test(
        y_val.mean(axis=1),
        val_pred[small].mean(axis=1),
        val_pred[large].mean(axis=1),
        h=PRED_LEN,
        name=f"Clark-West K={small} vs K={large} (validation)",
    )
    return PilotResult(
        val_mse=val_mse,
        clark_west=cw,
        n_val=len(y_val),
        passed=bool(cw.p_value < 0.05 and val_mse[large] < val_mse[small]),
    )


<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae488; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #90e0ef; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧊 Frame fitur</h4>
  <p style="color: #caf0f8; margin: 0; font-size: 0.9em;">Satu pemanggilan yang notebook pakai untuk membangun frame fitur.</p>
</div>

In [ ]:


def build_feature_frame(parquet: Path = DEFAULT_PARQUET) -> pl.DataFrame:
    """Load the immutable artifact and compute the twelve variates."""

    return build_features(usable_mask(load_bars(parquet)))








<div style="background: linear-gradient(135deg, #001a0d, #003317); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #52b78833;">
  <h2 style="color: #95d5b2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📋 16 &middot; Pelaporan
  </h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 1.02em;">Sembilan tabel dan enam figure, seluruhnya di-render dari paper_numbers.json — tidak pernah disalin tangan.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Ini bagian yang <code>D60g</code> soroti: grid menghasilkan angkanya dan meninggalkan setiap tabel dan figure tak tergenerate, empat di antaranya tanpa masukan sama sekali.</li>
    <li style="margin-bottom: 6px;"><code>paper/paper_numbers.json</code> menyebut berkas grid lewat sha256, jadi keduanya tidak bisa diam-diam berbeda (<code>D60f</code>, <code>D62a</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 4px solid #52b788; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #95d5b2; margin: 0 0 6px 0; font-size: 1.22em;">📘 report.py</h3>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.94em;">Sembilan tabel dan enam figure, seluruhnya di-render dari <code>paper_numbers.json</code> — tidak pernah disalin tangan.</p>
</div>

<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📄 Header & konstanta</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Nama model, kunci perbandingan, dan tag robustness.</p>
</div>

In [ ]:
"""Every table and figure the manuscript needs, generated rather than transcribed.

Root §12: *"Aggregation writes ``paper_numbers.json``, and every table and figure
is generated **from that file** rather than transcribed. A number that cannot be
regenerated is a documented failure, not a footnote."*

`D60g` recorded what the grid actually left behind: eight tables and seven
figures promised by §13.4, **none generated**, and four of them --- Table 6,
Table 8, Figure 5, Figure 7 --- with no inputs at all. `D56` had already fixed
the missing *models*; what stayed missing was the missing *call*. This module is
that call.

Two files, and the distinction is load-bearing. ``notebooks/outputs/artifacts/
paper_numbers.json`` is the **grid's** output and stays immutable evidence.
:func:`build_report` reads it, computes every analysis pass the grid never ran,
and returns the **manuscript's** single source --- which names the grid file by
digest, so the two cannot silently diverge. Tables and figures read only the
second. Note that repo-root ``artifacts/`` holds one stale 2026-08-06 CPU smoke
run and is not the results directory (`D60f`); passing it here is a caller error
and is rejected on the run count.

Three reporting rules from ``CLAUDE.md`` are enforced here rather than left to
the person writing the paper, because each has already been broken once:

* **Dispersion is bound to the aggregation level** (`D30`). Anything aggregated
  across origins carries the SE **across origins**; seed dispersion appears only
  as a Monte-Carlo diagnostic column. Measured on this grid the two differ by
  roughly seventy-fold --- seed std 0.001334 against an origin SE of 0.0959 ---
  so reporting the first as the second would understate the headline uncertainty
  by about that factor, reintroducing through a reporting convention exactly the
  overstated precision the wild cluster bootstrap was added to prevent.
* **Cross-origin comparisons are on RelMSE or R2_oos, never scaler-space MSE**
  (`D60i`). Two arms fitted at different origins carry different ``sigma_g``, so
  a raw MSE difference compares numbers in different units --- which is how the
  falsification arm shipped a figure that was 99.7% scaler drift and whose sign
  read backwards.
* **RQ3's wording is fixed and is not interchangeable** (`D60b`). *"The decay
  estimand is undefined under non-positive out-of-sample skill"*, **never** *"no
  decay detected within 180 days"*: the second is §3's right-censored phrasing
  and it asserts an edge the data does not contain.
"""

#: Every model Table 6 compares, in the order the table prints them. Twelve
#: models is 66 unordered pairs --- which is precisely why `D35` replaced SPA and
#: Reality Check here: those test a one-against-many null and say nothing about
#: an all-pairs matrix, where a complete null expects ~3 spurious rejections at
#: alpha = 0.05.
COMPARISON_KEYS: Final[tuple[ModelKey, ...]] = (
    ("itr", 1), ("itr", 4), ("itr", 8), ("itr", 12),
    ("itru", 8),
    ("rdg", 1), ("rdg", 4), ("rdg", 8), ("rdg", 12),
    ("dlin", 8), ("ptst", 8),
    # `D64`'s three arms. Root §7 gives each an explicit K and root §13.2 makes
    # "no deep model beats Naive-RW" a mandatory disclosure -- a claim that omits
    # the RNN the crypto literature reaches for first is a hole a reviewer finds
    # in one pass. LSTM's K=8 means what ridge's and iTransformer's mean, not what
    # the channel-independent baselines' does, and its loss is target-channel, so
    # it belongs in this matrix rather than beside it.
    ("lstm", 8),
    # K=1 and they are not Naive-RW: both read the target channel and nothing
    # else, which is what makes 1 the honest label (`D40`). Naive-RW stays the
    # separate sentinel that forecasts a zero raw log-return (`D31`).
    ("npst", 1), ("nsea", 1),
    NAIVE,
)

#: Models carried into the economic evaluation. A subset, because §13.5 asks for
#: an interval on every figure and a twelve-model Table 8 would be a wall of
#: numbers rather than a comparison. Ridge is in it because `D60c` made it the
#: finding: at its selected alpha it shrinks close enough to the training mean
#: that it nearly *is* the baseline, and it loses to Naive-RW by ~30x less than
#: any deep model.
ECONOMIC_KEYS: Final[tuple[ModelKey, ...]] = (
    ("itr", 1), ("itr", 8), ("rdg", 8), ("dlin", 8), ("ptst", 8),
    # `D77`. The LSTM is the only deep model the Model Confidence Set keeps, and
    # a Table 8 that omits it evaluates the economics of five models whose
    # statistical standing did not change while leaving out the one whose did.
    ("lstm", 8),
)

#: Arms whose runs may legitimately be absent: the `D62` and `D70` exploratory
#: arms, executed after the 684-run grid. A generator that crashed on their
#: absence would make the report un-runnable until a GPU session finished.
#:
#: They sit here rather than in :data:`COMPARISON_KEYS` because root §10.2 says
#: so in as many words -- *none enters RQ1's ladder comparison; each gets its own
#: row*. Mixing an exploratory arm into the ladder would make the rungs differ in
#: something other than K, which is the one thing the ladder holds fixed.
#:
#: ``orthogonal`` and ``redundant`` are the exception that proves it. They are the
#: matched-K pair, same K=8 and same target with PR the only thing that moves
#: (3.609 against 5.011, either side of the ladder's own 4.668), so they test
#: RQ1 **by contrast** where the ladder can only infer it through a panel at
#: ``corr(K, K_eff) = 0.828``. That makes them evidence rather than robustness --
#: and still not ladder rungs.
ROBUSTNESS_TAGS: Final[dict[str, str]] = {
    "itrl": "longsched",
    "itrc": "capacity",
    "itra": "attention",
    "itro": "orthogonal",
    "itrr": "redundant",
    "l048": "look048",
    "l192": "look192",
    "itrt": "tuned",
}

#: Paired contrasts reported beside the marginal tables (`D82`), as
#: ``(left, right, what it answers)``. Positive means the LEFT arm is worse.
#:
#: Every table in this report prints an arm's mean RelMSE with its standard error
#: **across** origins, and a reader who differences two such rows is comparing
#: marginal spreads for arms that were evaluated on the same fifteen origins
#: against the same naive baselines. The paired error is about half the marginal
#: one here, so overlapping bars in Table 4 and Table 9 carry no information about
#: whether two arms differ. These are the differences the paper's sentences
#: actually make, computed as differences.
#:
#: The first entry is the one that changes an RQ rather than its robustness: at
#: fixed K = 8, same target and same seeds, ``itrr`` against ``itro`` moves only
#: the participation ratio, so it is RQ1's K-versus-K_eff question asked **by
#: contrast** instead of inferred from a panel where the two move together at
#: ``corr(K, K_eff) = 0.828``.
PAIRED_CONTRASTS: Final[tuple[tuple[tuple[str, int | None], tuple[str, int | None], str], ...]] = (
    (("itrr", None), ("itro", None), "RQ1 matched-K: low PR against high PR at K=8"),
    (("itr", 1), ("itr", 8), "ladder: K=1 against K=8"),
    (("itr", 4), ("itr", 8), "ladder: K=4 against K=8"),
    (("itr", 8), ("itr", 12), "ladder: K=8 against K=12"),
    (("itru", None), ("itr", 8), "uniform attention against learned, at K=8"),
    (("lstm", None), ("itr", 8), "the RNN against the transformer, both K=8"),
    (("lstm", None), ("rdg", 8), "the RNN against ridge, both K=8"),
    (("ptst", None), ("itr", 8), "patch tokens against inverted tokens"),
    (("dlin", None), ("itr", 8), "the linear decomposition against the transformer"),
    (("itr", 8), ("rdg", 8), "the transformer against ridge on the same features"),
)

#: Figure 1's splits. Named rather than inline because its two panels must agree
#: exactly: a train bar that is one blue in the scheme and another in the
#: resolved origin reads as two different things. ``oos`` is a shading, not a
#: fourth split --- out-of-sample IS the six test blocks, and the span is drawn
#: so a reader stops looking for a region that does not exist.
SPLIT_COLOUR: Final[dict[str, str]] = {
    "train": "#1f4e79",
    "val": "#5b8db8",
    "purge": "#f4a259",
    "test": "#c1121f",
    "test_alt": "#e07a1f",
    "oos": "#6c757d",
}

#: Figure 4's encoding: hue is the model family, dash is the rung. Matplotlib's
#: default cycle holds ten colours against this matrix's fourteen series, so it
#: paints two models identically and the legend stops being readable against the
#: plot. Hue-by-family also puts `D60c` on the page rather than in a table: the
#: ridge lines hug 1.000 and the deep lines sit well above it.
FAMILY_COLOUR: Final[dict[str, str]] = {
    "itr": "#1f4e79", "itru": "#5b8db8", "itrf": "#9dc3e6",
    "rdg": "#2e7d32", "dlin": "#c1121f", "ptst": "#e07a1f",
    "lstm": "#7b2cbf", "npst": "#8d6e63", "nsea": "#455a64",
}

#: The two closed-form comparators. They sit near RelMSE 2.0 while every other
#: model is inside [1.000, 1.027], so Figure 4 zooms past them in its lower panel
#: and keeps them in its upper one (`D84`).
NAIVE_COMPARATORS: Final[frozenset[str]] = frozenset({"npst", "nsea"})

#: Dash by rung, so K is legible without a second colour axis.
RUNG_STYLE: Final[dict[int, str]] = {1: ":", 4: "-.", 8: "-", 12: "--"}

#: Marker by family, so the figure survives a greyscale print -- which an IEEE
#: submission may well become.
FAMILY_MARKER: Final[dict[str, str]] = {
    "itr": "o", "itru": "s", "itrf": "D",
    "rdg": "^", "dlin": "v", "ptst": "P",
    "lstm": "X", "npst": "*", "nsea": "h",
}

#: Figure 5's colour scale is centred on **uniform attention**, ``1/N``. Over
#: eight variates that is 0.125 and every measured weight lands within about
#: 0.01 of it, so a sequential map anchored at zero renders the whole panel one
#: flat colour and the reader concludes "attention is uniform" from the scale
#: rather than from the data. Uniform is also exactly the null the `D50`
#: uniform-attention arm implements, so deviation from it is the quantity of
#: interest and a diverging map centred there is the honest encoding.
ATTENTION_CMAP: Final = "RdBu_r"

#: Figure 5's regimes, fixed before any map was seen (`D48`): calm is the bottom
#: tercile of realised volatility across all test blocks, stress the top.
TERCILE_SHOWN: Final[tuple[str, str]] = ("calm", "stress")

#: The 684-run grid is the floor for a report. Below it the panel is unbalanced
#: and §9.1's estimators refuse it by design (`D54e`).
GRID_FLOOR: Final = 684

#: Fallback epoch cap for a run whose meta carries no schedule --- ridge and the
#: two naive comparators, which train nothing. Every arm that trains records its
#: own ``max_epochs``, and `D78` reads the cap from there rather than assuming
#: root §6.2's: the ``itrl`` arm runs to 60, so a hardcoded 30 counted its runs
#: as capped when they were not.
DEFAULT_MAX_EPOCHS: Final = 30

#: Human-readable model names, for tables and figure legends.
MODEL_NAMES: Final[dict[str, str]] = {
    "itr": "iTransformer",
    "itru": "iTransformer (uniform attn.)",
    "itrf": "iTransformer (fresh)",
    "itrl": "iTransformer (long schedule)",
    "itrc": "iTransformer (d_ff 512)",
    "itra": "iTransformer (attn. captured)",
    "rdg": "Ridge",
    "dlin": "DLinear",
    "ptst": "PatchTST",
    "lstm": "LSTM",
    "npst": "Naive-persist",
    "nsea": "Seasonal-naive",
    "itro": "iTransformer (orthogonal $K$=8)",
    "itrr": "iTransformer (redundant $K$=8)",
    "l048": "iTransformer ($L$=48)",
    "l192": "iTransformer ($L$=192)",
    "itrt": "iTransformer (tuned)",
}

_MISSING: Final = "---"


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔤 Format angka & LaTeX</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Satu jalur format, jadi tiap tabel <code>.tex</code> memakai presisi yang sama.</p>
</div>

In [ ]:


# -- formatting --------------------------------------------------------------


def fmt(value: float | int | None, digits: int = 4) -> str:
    """A number for LaTeX, or an em-dash. **Never** the string ``nan``.

    A NaN printed into a table is a defect wearing a value's clothes: it reads as
    a measurement and it is the absence of one. Anything non-finite comes back as
    an em-dash, so the gap is visible to a reader and to the test that forbids it.
    """
    if value is None:
        return _MISSING
    number = float(value)
    if not math.isfinite(number):
        return _MISSING
    if digits == 0:
        return f"{int(round(number)):,}"
    return f"{number:.{digits}f}"


def tex_escape(text: str) -> str:
    """Escape the characters that appear in this study's labels."""
    for old, new in (("_", r"\_"), ("%", r"\%"), ("&", r"\&"), ("#", r"\#")):
        text = text.replace(old, new)
    return text


def tabular(
    caption: str,
    tag: str,
    header: list[str],
    rows: list[list[str]],
    align: str,
    note: str = "",
) -> str:
    """One ``booktabs`` table float, ready to be included by the manuscript.

    ``booktabs`` and no vertical rules: the IEEE house style §1 targets, and the
    one that survives a two-column layout without a reader's help.
    """
    # The provenance line names the *command*, not the module. It is a string
    # literal in flattened notebook source, and the generator refuses to emit a
    # cell whose executable source still mentions the package by name (`D59`) ---
    # correctly, since it cannot tell a comment about the package from a
    # reference to it. Naming the command is more useful to a reader anyway.
    lines = [
        "% GENERATED --- do not hand-edit.",
        "% Regenerate: python tools/build_report.py",
        r"\begin{table}[!t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + tag + "}",
        r"\begin{tabular}{" + align + "}",
        r"\toprule",
        " & ".join(header) + r" \\",
        r"\midrule",
    ]
    lines += [" & ".join(row) + r" \\" for row in rows]
    lines += [r"\bottomrule", r"\end{tabular}"]
    if note:
        lines.append(r"\vspace{2pt}\par\footnotesize " + note)
    lines.append(r"\end{table}")
    return "\n".join(lines) + "\n"


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧰 Helper kecil</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Digest, standard error lintas origin, dan penanda signifikansi.</p>
</div>

In [ ]:


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def se_across(values: np.ndarray) -> float:
    """Standard error across origins --- `D30`'s only admissible ``+/-``."""
    values = np.asarray(values, dtype=np.float64)
    if len(values) < 2:
        return float("nan")
    return float(values.std(ddof=1) / math.sqrt(len(values)))


def _star(p: float | None, threshold: float = 0.05) -> str:
    """A significance mark, so a reader does not have to scan a p-value column."""
    if p is None or not math.isfinite(float(p)):
        return ""
    return r"$^{*}$" if float(p) < threshold else ""


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔐 Input laporan & provenance</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;"><code>paper/paper_numbers.json</code> menyebut berkas grid lewat sha256, jadi keduanya tidak bisa diam-diam berbeda (<code>D60f</code>, <code>D62a</code>).</p>
</div>

In [ ]:


# -- what a report is built from ---------------------------------------------


@dataclass(frozen=True, slots=True)
class ReportInputs:
    """Everything the tables and figures read, computed exactly once.

    The numbers dict is the manuscript's single source (root §12). The frames
    beside it are what the *figures* need and the JSON must not carry: Figure 2b
    alone is 3,044 points, and a JSON file that a human is expected to read
    should not be padded with a series only matplotlib consumes.
    """

    numbers: dict
    seed_avg: pl.DataFrame
    amplification: pl.DataFrame
    rolling_pr: pl.DataFrame
    rolling_r2: pl.DataFrame
    equity: pl.DataFrame
    attention: pl.DataFrame | None


# -- sections ----------------------------------------------------------------


def _provenance(artifacts: Path, grid: dict) -> dict:
    grid_path = artifacts / "paper_numbers.json"
    return {
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "grid_paper_numbers": str(grid_path),
        "grid_paper_numbers_sha256": _sha256(grid_path),
        "grid_generated_utc": grid.get("generated_utc"),
        "artifacts_root": str(artifacts),
    }


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🧱 Bagian dataset & arsitektur</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Di sinilah 0 dari 444 run menyentuh cap epoch terlihat (<code>D62c</code>).</p>
</div>

In [ ]:


def _dataset_section(bars: pl.DataFrame) -> dict:
    """Table 1 --- provenance, the measured gap profile, and per-origin budgets.

    Everything here is **measured from the artifact**, never transcribed from
    §4.1: `D33` exists because a report and its own gaps file disagreed by one
    bar, in the direction that flatters the data, and nothing downstream noticed.
    The ``H == L`` count in particular was assumed additive to the zero-volume
    count until `D51c` measured it: there are **3** unusable bars, and they are
    the *same* 3 bars, because no volume implies no trades implies a high and a
    low that never separate.
    """
    summary = break_summary(bars, DATA_START, DATA_END)
    budgets = budget_table(bars)
    return {
        "window": [DATA_START.isoformat(), DATA_END.isoformat()],
        "bars_expected": BARS_EXPECTED,
        "bars_actual": BARS_ACTUAL,
        "missing_bars": MISSING_BARS,
        "gap_blocks": GAP_BLOCKS,
        "measured": {
            "calendar_hours": summary.calendar_hours,
            "bars_present": summary.bars_present,
            "bars_usable": summary.bars_usable,
            "missing_bars": summary.missing_bars,
            "zero_volume_bars": summary.zero_volume_bars,
            "flat_bars": summary.flat_bars,
            "zero_trade_bars": summary.zero_trade_bars,
            "excluded_positions": summary.excluded_positions,
            "break_runs": summary.break_runs,
            "segments": summary.segments,
        },
        "per_origin": [
            {
                "origin": budget.label,
                "train_windows": budget.windows_measured,
                "closed_form": budget.windows_closed_form,
                "closed_form_agrees": budget.closed_form_agrees,
                "loss_pct": budget.loss_pct,
                "test_block_starts": list(budget.test_block_starts),
                "worst_block_starts": int(min(budget.test_block_starts)),
            }
            for budget in budgets
        ],
    }


def _architecture_section(run_ids: list[str], roots: list[Path]) -> dict:
    """Table 3 --- K, capacity and epochs-to-early-stop for every model.

    §6.2 requires epochs-to-stop logged per rung, and the reason is exact: it is
    how a reader tells a flat 8->12 rung from an under-trained one. It is also
    what `D62c` rests on: the iTransformer arms early-stop far below their cap,
    so the binding constraint is the learning-rate schedule and widening the cap
    alone would be a no-op.

    **`D78`: the cap is read from the arm's own schedule, and the count is
    reported rather than asserted.** Two defects lived in the hardcoded 30. The
    `itrl` arm runs to 60, so runs between 30 and 60 were being counted as capped
    when they were not. And the claim "no iTransformer run reaches the cap" was
    written as though permanent: on the 1,620-run grid five of 540 do, while
    **DLinear reaches it in 56 of 75 runs and PatchTST in 39 of 75**. Those two
    are budget-truncated, so any reading of the ordering that calls DLinear the
    worst model is confounded with DLinear being the most truncated --- which is
    why this number goes in the table instead of in a sentence.

    Every baseline carries an explicit K (`D40`). For the channel-independent
    pair that K means *trained on eight channels*, not *predicts the target from
    eight channels*, and their ``best_val_mse`` is an all-channel figure that is
    **not** comparable to the ladder's target-channel one (`D56`).
    """
    rows: dict[tuple[str, int, int], dict] = {}
    for run_id in run_ids:
        parts = parse_run_id(run_id)
        key = (str(parts["model"]), int(parts["k"]), int(parts["pred_len"]))
        meta = load_meta(run_id, roots)
        row = rows.setdefault(key, {
            "model": key[0], "k": key[1], "pred_len": key[2],
            "n_parameters": meta.get("n_parameters"),
            "epochs": [], "n_runs": 0,
            "config": meta.get("config", {}),
            "schedule": meta.get("schedule"),
        })
        row["epochs"].append(int(meta.get("epochs_run", 0)))
        row["n_runs"] += 1

    out = []
    for row in rows.values():
        epochs = np.asarray(row.pop("epochs"), dtype=np.float64)
        row["epochs_mean"] = float(epochs.mean())
        row["epochs_max"] = int(epochs.max())
        schedule = row.get("schedule") or {}
        cap = int(schedule.get("max_epochs", DEFAULT_MAX_EPOCHS))
        row["max_epochs"] = cap
        row["epochs_at_cap"] = int((epochs >= cap).sum())
        out.append(row)
    out.sort(key=lambda row: (row["model"], row["pred_len"], row["k"]))
    return {"cells": out}


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🔭 Bagian horizon & robustness</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Sweep horizon dibatasi ke empat origin bernama di <strong>setiap</strong> horizon supaya kolomnya berbagi sampel.</p>
</div>

In [ ]:


def _horizon_section(seed_avg: pl.DataFrame) -> dict:
    """Table 7 --- the horizon sweep, restricted to the four sweep origins.

    Restricted deliberately. The sweep ran at origins 1, 5, 10 and 15 (`D48`,
    named in advance so the choice could not follow the result), while H=24 also
    ran at all fifteen. Aggregating H=24 over fifteen origins and every other
    horizon over four would put a different sample in every column of one table,
    and the horizons would then differ in origin composition as well as in
    horizon --- the confound this study is built to avoid everywhere else.
    """
    sweep = seed_avg.filter(
        (pl.col("model") == "itr")
        & pl.col("origin_index").is_in(list(SWEEP_ORIGIN_INDICES))
    )
    per_origin = sweep.group_by(["origin", "k", "pred_len"]).agg(
        pl.col("rel_mse").mean().alias("rel_mse"),
        pl.col("r2_oos").mean().alias("r2_oos"),
        pl.col("mse_seed_std").mean().alias("seed_std"),
    )
    cells = []
    for (k, h), part in per_origin.group_by(["k", "pred_len"], maintain_order=True):
        r2 = part.get_column("r2_oos").to_numpy()
        cells.append({
            "k": int(k),
            "pred_len": int(h),
            "r2_oos": float(r2.mean()),
            "se_across_origins": se_across(r2),
            "rel_mse": float(part.get_column("rel_mse").to_numpy().mean()),
            "seed_std": float(part.get_column("seed_std").to_numpy().mean()),
            "n_origins": int(len(r2)),
        })
    cells.sort(key=lambda row: (row["pred_len"], row["k"]))
    return {"origins": list(SWEEP_ORIGIN_INDICES), "cells": cells}


#: Arms that carry exactly one rung, so their frame needs no ``k`` filter.
_SINGLE_RUNG: Final[frozenset[str]] = frozenset(
    {"itrc", "itra", "itro", "itrr", "l048", "l192", "itrt"}
)


def _contrasts_section(seed_avg: pl.DataFrame) -> dict:
    """:data:`PAIRED_CONTRASTS`, computed (`D82`).

    Post-hoc and carrying no multiplicity control of its own --- root §9.2's
    pre-registered machinery is what a confirmatory claim goes through, and this
    is here so the paper states its differences as differences instead of leaving
    a reader to subtract two marginal means and their marginal error bars.
    """
    present = set(seed_avg.get_column("model").unique().to_list())
    rows = []
    for left, right, question in PAIRED_CONTRASTS:
        if left[0] not in present or right[0] not in present:
            rows.append({
                "left": left[0], "right": right[0], "question": question,
                "status": "not run",
            })
            continue
        rows.append({**paired_contrast(seed_avg, left, right),
                     "question": question, "status": "run"})
    return {
        "note": (
            "Positive mean_diff means the LEFT arm is worse. Paired across the "
            "origins both arms were evaluated on, on RelMSE so no comparison "
            "crosses a sigma_g boundary (`D60i`), with the origin as the "
            "inferential unit (`D30`). Post-hoc and uncorrected for "
            "multiplicity (`D82`)."
        ),
        "rows": rows,
    }


def _robustness_section(seed_avg: pl.DataFrame, grid_r2: dict[int, float]) -> dict:
    """`D62b`, `D62c`, `D62d` --- the three arms, reported apart from RQ1-RQ3.

    An absent arm returns a ``status`` rather than raising: these run after the
    684-run grid, and a report that cannot be generated until a GPU session
    finishes is a report nobody regenerates.

    Whatever they return goes in the paper. A robustness arm reported only when
    it agrees with the headline is not a robustness arm, and §13.2 carries that
    as a disclosure.
    """
    out: dict = {}
    present = set(seed_avg.get_column("model").unique().to_list())
    for tag, arm in ROBUSTNESS_TAGS.items():
        if tag not in present:
            out[arm] = {
                "status": "not run",
                "model_tag": tag,
                "note": "exploratory `D62` arm; execute the manifest to populate it",
            }
            continue
        part = seed_avg.filter(
            (pl.col("model") == tag) & (pl.col("pred_len") == PRED_LEN)
        )
        per_origin = part.group_by(["origin", "k"]).agg(
            pl.col("r2_oos").mean().alias("r2_oos"),
            pl.col("rel_mse").mean().alias("rel_mse"),
        )
        cells = []
        for (k,), sub in per_origin.group_by(["k"], maintain_order=True):
            r2 = sub.get_column("r2_oos").to_numpy()
            cells.append({
                "k": int(k),
                "r2_oos": float(r2.mean()),
                "se_across_origins": se_across(r2),
                "n_origins": int(len(r2)),
                "grid_r2_oos_same_rung": grid_r2.get(int(k)),
                # `D82`. The arm against its own ladder rung, paired on the
                # origins both were evaluated on. The marginal SE beside it
                # answers a different question and is roughly twice as wide.
                "paired_vs_grid": paired_contrast(
                    seed_avg, (tag, None if tag in _SINGLE_RUNG else int(k)),
                    ("itr", int(k)),
                ),
            })
        cells.sort(key=lambda row: row["k"])
        out[arm] = {"status": "run", "model_tag": tag, "cells": cells}
    return out


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🗺️ Memuat peta attention</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Membaca 45 parquet attention arm <code>D62d</code>.</p>
</div>

In [ ]:


def _load_attention(artifacts: Path) -> pl.DataFrame | None:
    """Figure 5's maps, or ``None`` --- they arrive only with the `D62d` arm."""
    folder = artifacts / "attn"
    files = sorted(folder.glob("*.parquet")) if folder.exists() else []
    if not files:
        return None
    frames = []
    for path in files:
        parts = parse_run_id(path.stem)
        frames.append(
            pl.read_parquet(path).with_columns(
                pl.lit(path.stem).alias("run_id"),
                pl.lit(int(parts["origin_index"])).cast(pl.Int32).alias("origin_index"),
                pl.lit(int(parts["seed"])).cast(pl.Int32).alias("seed"),
            )
        )
    return pl.concat(frames)


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🏗️ build_report</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Pengumpul tunggal. Setiap angka manuskrip lahir di sini, lalu tabel dan figure di-render <em>dari</em> berkas itu — tidak pernah disalin tangan (root §12).</p>
</div>

In [ ]:


# -- the enriched paper_numbers.json ----------------------------------------


def build_report(
    artifacts: Path,
    bars: pl.DataFrame,
    features: pl.DataFrame,
    *,
    roots: list[Path] | None = None,
    bootstrap_b: int = 9_999,
    seed: int = 42,
    log=print,
) -> ReportInputs:
    """The manuscript's single source, plus the frames its figures read.

    Args:
        artifacts: Directory holding ``preds/``, ``meta/`` and the grid's own
            ``paper_numbers.json`` --- ``notebooks/outputs/artifacts/`` (`D60f`).
        bars: ``usable_mask(load_bars(...))`` --- Table 1's measured gap profile.
        features: The frame ``build_features`` returns. §4.5's efficiency tests
            and Figure 2b's rolling statistics read it; neither reads a
            prediction file.
        bootstrap_b: Draws for Romano-Wolf and the Model Confidence Set. The floor
            on any bootstrap p-value is ``1/(1+B)`` (`D53d`).

    Returns:
        A :class:`ReportInputs`. Every section of ``numbers`` names the span, the
        n and the dispersion measure it used; nothing in it is a bare point
        estimate a reader would have to trust.

    Raises:
        FileNotFoundError: If the grid's ``paper_numbers.json`` is absent.
        ValueError: If fewer than 684 runs are present --- a partial panel is a
            different estimand, not a noisier one (`D54e`).
    """
    grid_path = artifacts / "paper_numbers.json"
    if not grid_path.exists():
        raise FileNotFoundError(
            f"{grid_path} is absent. The grid's own aggregation writes it; this "
            f"function enriches it and never replaces it. Repo-root artifacts/ "
            f"holds one stale CPU smoke run and is not the results directory "
            f"(`D60f`)."
        )
    grid = json.loads(grid_path.read_text(encoding="utf-8"))

    # A resumed Kaggle session holds earlier runs under /kaggle/input/<slug>/ and
    # this session's under /kaggle/working/, so the caller passes what
    # ``discover_roots`` found. Locally the two coincide (root §10.5).
    roots = list(roots) if roots else [artifacts]
    run_ids = sorted(completed_run_ids(roots))
    log(f"report: {len(run_ids)} completed runs under {artifacts}")
    if len(run_ids) < GRID_FLOOR:
        raise ValueError(
            f"only {len(run_ids)} runs found, below the {GRID_FLOOR}-run grid. A "
            f"partial panel is a different estimand, not a noisier one (`D54e`). "
            f"Repo-root artifacts/ is a stale smoke run (`D60f`)."
        )

    raw = gather_grid(run_ids, roots)
    seed_avg = seed_average(raw)
    log(f"report: {raw.height} run-block rows -> {seed_avg.height} seed-averaged cells")

    dataset = _dataset_section(bars)
    architecture = _architecture_section(run_ids, roots)
    log(f"report: {len(architecture['cells'])} architecture cells")

    # §4.5 and Figure 2b read the feature frame; neither touches a prediction.
    efficiency = efficiency_table(features)
    roll_pr = rolling_pr(features, k=8)
    roll_r2 = rolling_ols_r2(features, k=8)
    log(f"report: efficiency {efficiency.height} spans, rolling {roll_pr.height} windows")

    # Table 6.
    # An arm that has not run anywhere is NAMED, not a crash (`D64`, `D70` --- the
    # `D64` baselines and the exploratory arms land after the 684-run grid, and a
    # report that cannot be generated until a GPU session finishes is a report
    # nobody regenerates). Partial coverage still raises inside `build_panel`:
    # that one is the `D45` defect and hiding it would be the wrong lesson.
    comparison_keys, absent_keys = available_keys(list(COMPARISON_KEYS), roots)
    for key in absent_keys:
        log(f"report: {label(key)} is in COMPARISON_KEYS but has no run --- "
            f"omitted from Table 4, Table 6 and the MCS, and named in "
            f"paper_numbers.json")
    panel = build_panel(comparison_keys, roots)
    pairs = pair_matrix(panel, B=bootstrap_b, seed=seed)
    mcs = mcs_table(panel, B=bootstrap_b, seed=seed)
    log(f"report: {pairs.height} pairs, MCS over {mcs.height} models")

    # Table 8 and Figure 7.
    origin_indices = tuple(origin.index for origin in ORIGINS)
    economics = economics_table(
        roots, list(ECONOMIC_KEYS), origin_indices, SLIPPAGE_BAND, seed=seed
    )
    equity = equity_curves(roots, list(ECONOMIC_KEYS), origin_indices, SLIPPAGE_BAND)
    log(f"report: {economics.height} economic cells, {equity.height} equity points")

    # DA, raw scale, the falsification arm's real number, and `D45`'s coverage check.
    ladder_ids = [
        run_id for run_id in run_ids
        if parse_run_id(run_id)["model"] == "itr"
        and int(parse_run_id(run_id)["pred_len"]) == PRED_LEN
    ]
    da = directional_accuracy_table(ladder_ids, roots)
    log(f"report: DA over {da.height} runs")

    raw_scale = raw_scale_table(seed_avg)
    falsification = falsification_relmse(seed_avg)
    amp = amplification(seed_avg)
    attn_amp = attention_amplification(seed_avg)
    beta_full, beta_covered = beta1_with_coverage(amp, seed=seed)
    # `D80`. Root §9.2 asks for a coverage covariate OR a restriction to
    # well-covered blocks. The restriction unbalances the panel and comes back
    # None, which is honest but leaves the requirement unmet; the covariate always
    # runs, because nothing is dropped.
    beta_covariate = panel_beta1_covariate(
        amp.with_columns(
            (pl.col("n_large") / float(BLOCK_HOURS)).alias("coverage")
        ),
        seed=seed,
    )

    grid_r2 = {int(row["k"]): float(row["R2_oos"]) for row in grid["rq1"]["rung_effects"]}

    per_rung_raw = (
        raw_scale.filter((pl.col("model") == "itr") & (pl.col("pred_len") == PRED_LEN))
        .group_by(["origin", "k"]).agg(pl.col("rmse_raw").mean().alias("rmse_raw"))
        .group_by("k").agg(
            pl.col("rmse_raw").mean().alias("rmse_raw"),
            pl.col("rmse_raw").std().alias("sd_across_origins"),
            pl.len().alias("n_origins"),
        ).sort("k")
    )

    da_by_rung = (
        da.group_by("k").agg(
            pl.col("da_h1").mean().alias("da_h1"),
            pl.col("p_h1").median().alias("p_h1_median"),
            pl.col("da_hH").mean().alias("da_hH"),
            pl.col("p_hH").median().alias("p_hH_median"),
            pl.col("da_cum").mean().alias("da_cum"),
            pl.col("p_cum").median().alias("p_cum_median"),
            pl.col("da_hH_overlapping").mean().alias("da_hH_overlapping"),
            pl.col("da_cum_overlapping").mean().alias("da_cum_overlapping"),
            pl.len().alias("n_runs"),
        ).sort("k")
    )

    # Table 4's per-model summary, on the scale-free metrics only (`D60i`).
    main = seed_avg.filter(pl.col("pred_len") == PRED_LEN)
    per_model = []
    for tag, k in comparison_keys:
        if tag == "naive":
            continue
        cell = main.filter((pl.col("model") == tag) & (pl.col("k") == k))
        by_origin = cell.group_by("origin").agg(
            pl.col("rel_mse").mean().alias("rel_mse"),
            pl.col("r2_oos").mean().alias("r2_oos"),
            pl.col("mse").mean().alias("mse"),
            pl.col("mse_seed_std").mean().alias("seed_std"),
        )
        r2 = by_origin.get_column("r2_oos").to_numpy()
        membership = {row["model"]: row for row in mcs.to_dicts()}
        name = label((tag, k))
        per_model.append({
            "model": name,
            "model_tag": tag,
            "k": k,
            "rel_mse": float(by_origin.get_column("rel_mse").to_numpy().mean()),
            "r2_oos": float(r2.mean()),
            "se_across_origins": se_across(r2),
            "seed_std": float(by_origin.get_column("seed_std").to_numpy().mean()),
            "n_origins": int(by_origin.height),
            "n_seeds": int(cell.get_column("n_seeds").max()),
            "in_mcs_90": bool(membership.get(name, {}).get("in_mcs_90", False)),
            "in_mcs_75": bool(membership.get(name, {}).get("in_mcs_75", False)),
        })

    gap = falsification.get_column("gap_rel_mse").to_numpy()
    by_origin_gap = (
        falsification.group_by("origin")
        .agg(pl.col("gap_rel_mse").mean().alias("gap"))
        .sort("origin")
    )
    origin_gap = by_origin_gap.get_column("gap").to_numpy()

    numbers = {
        "derived_from": _provenance(artifacts, grid),
        "input_parquet": grid.get("input_parquet"),
        "input_sha256": grid.get("input_sha256"),
        "input_sha256_source": grid.get("input_sha256_source"),
        "code_sha256": grid.get("code_sha256"),
        "runs_complete": len(run_ids),
        "runs_in_grid_file": grid.get("runs_complete"),
        # Carried through verbatim and unrecomputed: these are the confirmatory
        # answers, and re-deriving them here would create a second definition of
        # a number root §12 wants to have exactly one of.
        "keff": grid["keff"],
        "rq1": grid["rq1"],
        "rq2": grid["rq2"],
        "rq3": grid["rq3"],
        # Everything below is new; the grid computed none of it.
        "dataset": dataset,
        "architecture": architecture,
        "keff_rolling": {
            "window_days": 90,
            "descriptive_only": True,
            "note": (
                "Full-sample span: every origin's test block lies inside it, so "
                "root section 5.4 forbids this informing any design decision. "
                "The gate is gate_pr on the pre-first-origin span (`D02`)."
            ),
            "pr": {
                "n": roll_pr.height,
                "min": float(roll_pr.get_column("pr").min()),
                "max": float(roll_pr.get_column("pr").max()),
                "mean": float(roll_pr.get_column("pr").mean()),
                "sd": float(roll_pr.get_column("pr").std()),
            },
            "ols_r2": {
                "n": roll_r2.height,
                "min": float(roll_r2.get_column("r2").min()),
                "max": float(roll_r2.get_column("r2").max()),
                "mean": float(roll_r2.get_column("r2").mean()),
                "sd": float(roll_r2.get_column("r2").std()),
            },
        },
        "efficiency": efficiency.to_dicts(),
        "comparisons": {
            "models": [label(key) for key in comparison_keys],
            "absent": [label(key) for key in absent_keys],
            "B": bootstrap_b,
            "p_floor": 1.0 / (1 + bootstrap_b),
            "pairs": pairs.to_dicts(),
            "mcs": mcs.to_dicts(),
        },
        "main_results": {
            "note": (
                "Aggregated across origins, so the dispersion is the SE ACROSS "
                "ORIGINS and the seed std is a Monte-Carlo diagnostic beside it, "
                "never the error bar (`D30`)."
            ),
            "by_model": per_model,
        },
        "economics": {
            "taker_fee_per_side": 0.0004,
            "slippage_band": list(SLIPPAGE_BAND),
            "phase_utc_hour": 0,
            "cells": economics.to_dicts(),
        },
        "directional_accuracy": {
            "note": (
                "h=1 is tested on hourly spacing; h=H and the cumulative return "
                "are tested on NON-OVERLAPPING windows only. The overlapping "
                "figures are descriptive and carry no p-value: their targets "
                "overlap by 23 of 24 hours, so Pesaran-Timmermann's variance is "
                "far too small and the test over-rejects badly (`D21`)."
            ),
            "by_rung": da_by_rung.to_dicts(),
            "n_runs": da.height,
        },
        "horizons": _horizon_section(seed_avg),
        "falsification": {
            "metric": "RelMSE",
            "note": (
                "Reported on RelMSE, never on scaler-space MSE (`D60i`): the two "
                "arms are fitted 90 days apart and carry different sigma_g, so a "
                "raw MSE difference compares numbers in different units. The "
                "shipped -0.053341 was ~99.7% scaler drift with its sign reversed."
            ),
            "mean_gap_rel_mse": float(gap.mean()),
            "se_across_origins": se_across(origin_gap),
            "n_cells": int(len(gap)),
            "n_origins": int(by_origin_gap.height),
            "origins_favouring_aged": int((origin_gap < 0).sum()),
            "origins_within_5e5_of_zero": int((np.abs(origin_gap) < 5e-5).sum()),
            "by_origin": by_origin_gap.to_dicts(),
        },
        "attention_amplification": {
            "note": (
                "A_attn holds information fixed and varies only what attention "
                "selects, which K=1 versus K=8 cannot do (`D50`)."
            ),
            "mean": float(attn_amp.get_column("A_attn").mean()),
            "n_cells": int(attn_amp.height),
        },
        "raw_scale": {
            "note": "RMSE in raw log-return units beside MSE in scaler space (§9.1).",
            "by_rung": per_rung_raw.to_dicts(),
        },
        "coverage": {
            "note": (
                "`D45`: test-window survival is conditioned on FUTURE gaps, and "
                "outages cluster on stress, so within an origin the surviving "
                "sample composition trends and beta1 would absorb it."
            ),
            "min_coverage": 0.9,
            "covariate": {
                "beta1": beta_covariate.beta1,
                "t": beta_covariate.t_statistic,
                "cluster_se": beta_covariate.cluster_se,
                "headline_p": beta_covariate.headline_p,
                "G": beta_covariate.n_clusters,
                "N": beta_covariate.n_observations,
                "note": (
                    "A(i,b) = alpha_i + beta1 b + beta2 coverage(i,b) + eps, "
                    "clustered on origin, WCR one-sided (`D80`). This is root "
                    "§9.2's covariate route, and it is the one that runs: the "
                    "restriction below unbalances the panel and returns None."
                ),
            },
            "full": {
                "beta1": beta_full.beta1,
                "t": beta_full.t_statistic,
                "headline_p": beta_full.headline_p,
                "G": beta_full.n_clusters,
                "N": beta_full.n_observations,
            },
            "restricted": (
                None if beta_covered is None else {
                    "beta1": beta_covered.beta1,
                    "t": beta_covered.t_statistic,
                    "headline_p": beta_covered.headline_p,
                    "G": beta_covered.n_clusters,
                    "N": beta_covered.n_observations,
                }
            ),
            "restricted_unavailable_reason": (
                None if beta_covered is not None else
                "restricting to well-covered blocks unbalances the panel, and "
                "beta1's reduction to the mean of within-slopes holds only on a "
                "balanced one. That the check cannot run IS the `D45` finding."
            ),
        },
        "robustness": _robustness_section(seed_avg, grid_r2),
        "contrasts": _contrasts_section(seed_avg),
    }

    return ReportInputs(
        numbers=numbers,
        seed_avg=seed_avg,
        amplification=amp,
        rolling_pr=roll_pr,
        rolling_r2=roll_r2,
        equity=equity,
        attention=_load_attention(artifacts),
    )


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">💾 paper_numbers.json</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Menulis berkas yang seluruh deliverable baca.</p>
</div>

In [ ]:


def build_paper_numbers(
    artifacts: Path, bars: pl.DataFrame, features: pl.DataFrame, **kwargs
) -> dict:
    """:func:`build_report`'s numbers alone --- the manuscript's single source."""
    return build_report(artifacts, bars, features, **kwargs).numbers


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel 1 & 2</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Profil dataset dan gap; deskriptif plus ADF, VR, Hurst.</p>
</div>

In [ ]:


# -- tables ------------------------------------------------------------------


def _table1(numbers: dict) -> str:
    data = numbers["dataset"]
    measured = data["measured"]
    rows = [
        [
            row["origin"],
            fmt(row["train_windows"], 0),
            fmt(row["loss_pct"], 2),
            fmt(row["worst_block_starts"], 0),
            "yes" if row["closed_form_agrees"] else "no",
        ]
        for row in data["per_origin"]
    ]
    note = (
        f"BTCUSDT spot 1\\,h, Binance REST. Window {data['window'][0][:10]} to "
        f"{data['window'][1][:10]}, end exclusive. "
        f"{data['bars_expected']:,} expected, {data['bars_actual']:,} actual, "
        f"{data['missing_bars']} missing across {data['gap_blocks']} downtime "
        f"blocks. Unusable bars: {measured['zero_volume_bars']} zero-volume, "
        f"{measured['flat_bars']} with $H=L$, {measured['zero_trade_bars']} "
        f"zero-trade --- and they are the \\emph{{same}} bars (`D51c'), so the "
        f"total is {measured['excluded_positions']}, not their sum. "
        f"Windows are counted segment-wise; the closed form of section 4.3 is "
        f"kept only as an upper bound (`D51a')."
    )
    return tabular(
        "Data provenance and the per-origin window budget.",
        "tab:dataset",
        ["Origin", "Train windows", "Loss (\\%)", "Worst test block", "Closed form agrees"],
        rows,
        "lrrrc",
        note,
    )


def _table2(numbers: dict) -> str:
    rows = []
    for row in numbers["efficiency"]:
        rows.append([
            tex_escape(str(row["span"])),
            fmt(row["n"], 0),
            fmt(row["adf_stat"], 2) + _star(row["adf_p"]),
            fmt(row["hurst"], 3),
            fmt(row.get("vr_2"), 3) + _star(row.get("vr_p_2")),
            fmt(row.get("vr_4"), 3) + _star(row.get("vr_p_4")),
            fmt(row.get("vr_8"), 3) + _star(row.get("vr_p_8")),
            fmt(row.get("vr_16"), 3) + _star(row.get("vr_p_16")),
        ])
    note = (
        "Log-returns. ADF: stationarity. Hurst by rescaled range: $H\\approx0.5$ "
        "reads as no long memory. Variance ratio (Lo--MacKinlay): $VR\\approx1$ is "
        "consistent with a random walk, $VR<1$ is mean reversion. "
        "$^{*}$ marks $p<0.05$. We do \\emph{not} claim the market is efficient: "
        "the evidence here is mixed --- the variance ratio rejects the random walk "
        "at every lag while the Hurst exponent sits slightly above one half --- and "
        "that is the finding section 4.5 asks for."
    )
    return tabular(
        "Preliminary market-efficiency tests, full sample and per training sub-block.",
        "tab:efficiency",
        ["Span", "$n$", "ADF", "Hurst", "$VR_2$", "$VR_4$", "$VR_8$", "$VR_{16}$"],
        rows,
        "lrrrrrrr",
        note,
    )


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel 2b & 3</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Eigenspektrum dan PR per rung per origin; hiperparameter dan K seluruh model.</p>
</div>

In [ ]:


def _table2b(numbers: dict) -> str:
    keff = numbers["keff"]
    rows = [
        [
            fmt(row["k"], 0),
            fmt(row["PR_raw"], 3) + " $\\pm$ " + fmt(row.get("PR_raw_sd"), 3),
            fmt(row.get("PR_windownorm"), 3),
            fmt(row.get("stable_rank"), 3),
            fmt(row.get("crosslag_share"), 3),
        ]
        for row in keff["per_rung"]
    ]
    note = (
        r"Participation ratio per rung, measured \textbf{per origin on that "
        r"origin's own 21-month training sub-block} (`D44'), which is what keeps "
        r"RQ1's regressor from reading a single bar its outcome is measured on. "
        r"$\pm$ is the standard deviation across origins. "
        f"$corr(K, K_{{eff}}) = {fmt(keff['corr_k_keff'], 3)}$, against the "
        r"$\approx 0.97$ section 9.1 anticipated --- so the $K$-versus-$K_{eff}$ "
        r"horse race is \emph{more} identifiable than that section feared. "
        f"The Stage 3b gate measured {fmt(keff['gate_pr_k8_pre_first_origin'], 3)} "
        f"at $K=8$ on the pre-first-origin span, below the pre-registered floor "
        f"of {fmt(keff['gate_floor'], 1)}; `D48''s prescribed action is "
        r"disclosure, not a re-cut, and the grid proceeded unchanged. $K=12$ "
        r"carries a \emph{lower} PR than $K=8$, so the redundancy that rung was "
        r"designed to contain is stronger than section 5.2 expected, not weaker."
    )
    return tabular(
        "Effective dimensionality per rung.",
        "tab:keff",
        ["$K$", "PR (raw)", "PR (window-norm.)", "Stable rank", "Cross-lag share"],
        rows,
        "rrrrr",
        note,
    )


def _table3(numbers: dict) -> str:
    rows = []
    for cell in numbers["architecture"]["cells"]:
        if cell["pred_len"] != PRED_LEN:
            continue
        name = MODEL_NAMES.get(cell["model"], cell["model"])
        rows.append([
            tex_escape(name),
            fmt(cell["k"], 0),
            fmt(cell["n_parameters"], 0),
            fmt(cell["epochs_mean"], 2),
            fmt(cell["epochs_max"], 0),
            fmt(cell["epochs_at_cap"], 0),
            fmt(cell["n_runs"], 0),
        ])
    note = (
        "Every hyperparameter is adopted unchanged from Liu et al. (2024) except "
        "$d_{model}$, reduced from 512 to 128 against $\\sim$14{,}000 training "
        "windows; \\textbf{nothing is tuned per rung} (`D38'), which is what makes "
        "the rungs comparable. Ridge's $\\alpha$ is the only hyperparameter "
        "selected anywhere in this study, on the validation sub-block. "
        "Parameter count is identical across rungs \\emph{at a fixed horizon} "
        "(`D60h'). \\textbf{Epochs at cap} counts runs reaching that arm's own "
        "epoch budget, 60 for the long-schedule arm and 30 elsewhere (`D78'). The "
        "iTransformer arms early-stop far below it, so their binding constraint is "
        "the learning-rate schedule rather than the budget. DLinear and PatchTST "
        "do not: where an arm sits at its cap its loss is a truncated-training "
        "figure and has to be read as one before it is called the worst model. "
        "For DLinear and PatchTST, $K=8$ "
        "means \\emph{trained on eight channels} through their published "
        "all-channel objective, not \\emph{predicts the target from eight "
        "channels}, and their validation losses are all-channel figures not "
        "comparable to the ladder's (`D56')."
    )
    return tabular(
        "Architectures, capacity, and epochs to early stop at $H=24$.",
        "tab:hyperparameters",
        ["Model", "$K$", "Params", "Epochs (mean)", "Max", "At cap", "Runs"],
        rows,
        "lrrrrrr",
        note,
    )


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel 4 & 5</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Hasil utama dengan SE <strong>lintas origin</strong> (<code>D30</code>); <code>D(i,b)</code> per blok yang membawa kata undefined (<code>D60b</code>).</p>
</div>

In [ ]:


def _table4(numbers: dict) -> str:
    rows = []
    for row in numbers["main_results"]["by_model"]:
        membership = "90\\%, 75\\%" if row["in_mcs_75"] else (
            "90\\%" if row["in_mcs_90"] else _MISSING
        )
        rows.append([
            tex_escape(row["model"]),
            fmt(row["rel_mse"], 4),
            fmt(row["r2_oos"], 4) + " $\\pm$ " + fmt(row["se_across_origins"], 4),
            fmt(row["seed_std"], 6),
            membership,
            fmt(row["n_origins"], 0),
            fmt(row["n_seeds"], 0),
        ])
    note = (
        "$R^2_{oos} = 1 - \\text{RelMSE}$ against Naive-RW, which forecasts a zero "
        "raw log-return and is mapped into scaler space as $\\hat{y}_z = "
        "-\\mu_g/\\sigma_g$ (`D31'). \\textbf{Every model is worse than Naive-RW at "
        "every rung}; ridge is worse by roughly thirty times less than any deep "
        "model. The $\\pm$ is the standard error \\emph{across origins}; the seed "
        "column is Monte-Carlo noise on one fixed dataset and is a diagnostic, "
        "never the error bar (`D30'). MCS is the Model Confidence Set (Hansen, "
        "Lunde \\& Nason 2011) at the stated levels."
    )
    return tabular(
        "Main results at $H=24$, aggregated across all fifteen origins.",
        "tab:main",
        ["Model", "RelMSE", "$R^2_{oos}$", "Seed std", "In MCS", "Origins", "Seeds"],
        rows,
        "lrrrcrr",
        note,
    )


def _table5(numbers: dict) -> str:
    rq3 = numbers["rq3"]
    rows = []
    for row in rq3["b_star"]:
        headline = " (headline)" if abs(row["tau"] - rq3["tau_headline"]) < 1e-12 else ""
        rows.append([
            fmt(100 * row["tau"], 1) + "\\%" + headline,
            tex_escape(str(row["status"]).upper()),
            fmt(row.get("median_b_star"), 1),
            fmt(row.get("ci_low"), 1) + "--" + fmt(row.get("ci_high"), 1),
            fmt(row.get("events"), 0),
            fmt(row.get("censored"), 0),
            fmt(row.get("n_origins"), 0),
        ])
    excluded = ", ".join(rq3["excluded_origins"])
    note = (
        "$b^{*}(i) = \\min\\{b : D(i,b) > \\tau\\}$, right-censored at six blocks. "
        "\\textbf{The decay estimand is undefined under non-positive out-of-sample "
        "skill}: $D(i,b)$ is a proportion of skill lost and there is no skill to "
        "lose a proportion of. This is \\emph{not} the right-censored result "
        "``no decay detected within 180 days'', which would assert an edge the "
        "data does not contain (`D60b'). All fifteen origins are excluded on mean "
        f"$R^2_{{oos}} \\leq 0$ and are named rather than dropped: {excluded}. "
        "The log-rank test of H3 is unavailable because neither arm has a "
        "surviving origin, so H3 is \\textbf{untestable}, not rejected."
    )
    return tabular(
        "RQ3: retraining cadence at each pre-registered threshold.",
        "tab:decay",
        ["$\\tau$", "Status", "Median $b^{*}$", "CI", "Events", "Censored", "Origins"],
        rows,
        "llrrrrr",
        note,
    )


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel 6</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Matriks DM — statistik disebut namanya per pasangan, T dicantumkan, kolom Romano–Wolf dan MCS.</p>
</div>

In [ ]:


def _table6(numbers: dict) -> str:
    comparisons = numbers["comparisons"]
    rows = []
    for row in comparisons["pairs"]:
        rows.append([
            tex_escape(row["left"]),
            tex_escape(row["right"]),
            "CW" if row["statistic_name"].startswith("Clark") else "DM",
            fmt(row["t_cluster"], 3),
            fmt(row["p_raw"], 4),
            fmt(row["p_romano_wolf"], 4),
            row.get("family", _MISSING),
            fmt(row.get("p_romano_wolf_family"), 4),
            fmt(row["T_min"], 0),
        ])
    note = (
        "\\textbf{CW} is Clark--West (2007), used on every \\emph{nested} pair --- "
        "the ladder is cumulative and Naive-RW is nested inside every model, and "
        "there standard DM is not asymptotically $N(0,1)$ and is systematically "
        "undersized against the alternative this study exists to establish "
        "(`D29'). \\textbf{DM} is Diebold--Mariano with the "
        "Harvey--Leybourne--Newbold correction, referred to $t(T-1)$, on a "
        "rectangular long-run variance with lag $h-1 = 23$ --- never Bartlett, "
        "which would shrink $\\hat\\gamma_{23}$ by about 92\\% and understate the "
        "variance (`D34'). $t$ is clustered on the origin, $G = 15$. "
        f"$p_{{RW}}$ is the Romano--Wolf (2005) stepdown across all "
        f"{len(comparisons['pairs'])} pairs, $B = {comparisons['B']:,}$, floor "
        f"${fmt(comparisons['p_floor'], 6)}$ (`D53d'). White's Reality Check and "
        "Hansen's SPA are \\emph{not} used here: they test a one-against-many null "
        "and say nothing about an all-pairs matrix (`D35'). $T$ is the smallest "
        "per-cell sample; $h = 24$. "
        "The post-hoc family column is $p_{RW}^{fam}$ and it is NOT the headline "
        "(`D79'). It is the same stepdown run inside one claim family --- pairs "
        "against Naive-RW, rungs of one model, or one $K$ across architectures "
        "--- rather than across the Cartesian product. It exists because widening "
        "the model list from twelve to fifteen took the matrix from 66 pairs to "
        f"{len(comparisons['pairs'])} and the adjusted rejections from 31 to "
        "zero while no effect moved: the two naive comparators carry the largest "
        "$|t|$ in the table, and the shared bootstrap draw puts them into the "
        "max-$|t|$ null every other pair is judged against. The families were "
        "declared AFTER that was seen, so $p_{RW}$ over all pairs remains the "
        "pre-registered and reported result and this column is disclosed beside "
        "it rather than in place of it."
    )
    return tabular(
        "Pairwise forecast comparison with family-wise error control.",
        "tab:dm",
        ["Left", "Right", "Stat.", "$t$", "$p_{raw}$", "$p_{RW}$",
         "Family", "$p_{RW}^{fam}$", "$T_{min}$"],
        rows,
        "llcrrrlrr",
        note,
    )


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📋 Tabel 7, 8, 9, dan render</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Sweep horizon dan ekonomi, lalu seluruhnya ditulis sebagai <code>.tex</code>.</p>
</div>

In [ ]:


def _table7(numbers: dict) -> str:
    horizons = numbers["horizons"]
    rows = [
        [
            fmt(cell["pred_len"], 0),
            fmt(cell["k"], 0),
            fmt(cell["rel_mse"], 4),
            fmt(cell["r2_oos"], 4) + " $\\pm$ " + fmt(cell["se_across_origins"], 4),
            fmt(cell["seed_std"], 6),
            fmt(cell["n_origins"], 0),
        ]
        for cell in horizons["cells"]
    ]
    note = (
        "Origins " + ", ".join(str(i) for i in horizons["origins"]) + ", named in "
        "advance (`D48') so the choice could not follow the result. $H=24$ is "
        "restricted to the same four origins here even though it ran at all "
        "fifteen: aggregating one column over fifteen origins and the rest over "
        "four would make the horizons differ in origin composition as well as in "
        "horizon. $\\pm$ is the standard error across origins."
    )
    return tabular(
        "Horizon sweep.",
        "tab:horizons",
        ["$H$", "$K$", "RelMSE", "$R^2_{oos}$", "Seed std", "Origins"],
        rows,
        "rrrrrr",
        note,
    )


def _table8(numbers: dict) -> str:
    cells = pl.DataFrame(numbers["economics"]["cells"])
    rows = []
    for (model, slippage), part in cells.group_by(
        ["model", "slippage_per_side"], maintain_order=True
    ):
        sharpe = part.get_column("sharpe_annualised").to_numpy()
        mdd = part.get_column("max_drawdown").to_numpy()
        rows.append([
            tex_escape(str(model)),
            fmt(100 * float(slippage), 2) + "\\%",
            fmt(sharpe.mean(), 3) + " $\\pm$ " + fmt(se_across(sharpe), 3),
            fmt(part.get_column("sortino_annualised").to_numpy().mean(), 3),
            fmt(mdd.mean(), 3),
            fmt(part.get_column("mdd_ci_low").to_numpy().mean(), 3) + "--"
            + fmt(part.get_column("mdd_ci_high").to_numpy().mean(), 3),
            fmt(part.get_column("turnover_per_period").to_numpy().mean(), 3),
            fmt(part.get_column("net_total_return").to_numpy().mean(), 4),
            fmt(part.get_column("dsr").to_numpy().mean(), 4),
        ])
    rows.sort(key=lambda row: (row[0], row[1]))
    note = (
        "Position from the sign of the cumulative 24-step forecast on \\emph{raw, "
        "drift-free} log-returns, opened at \\textbf{00:00 UTC} and held 24 hours, "
        "non-overlapping --- the phase is fixed in advance because there are 24 "
        "admissible alignments and each gives a different Sharpe (`D46'). Taker "
        "fee 0.04\\% per side, plus the slippage shown. Sharpe and Sortino are "
        "annualised at 365 periods; the DSR is computed per origin from the "
        "\\emph{per-period} Sharpe, with $N$ the configurations evaluated on that "
        "origin's own test span --- not the 837-run development total, which is "
        "reported separately in Limitations and is a different quantity (`D46'). "
        "$\\pm$ is the standard error across origins; the MDD interval is a "
        "stationary bootstrap. The strategy is flat wherever no valid window "
        "survives, and outages cluster on stress, so the reported drawdown is "
        "optimistic by an amount the flat-day count bounds (`D45')."
    )
    return tabular(
        "Economic evaluation across the pre-registered slippage band.",
        "tab:economics",
        ["Model", "Slip.", "Sharpe (ann.)", "Sortino", "MDD", "MDD CI",
         "Turnover", "Net return", "DSR"],
        rows,
        "llrrrrrrr",
        note,
    )


def _table9(numbers: dict) -> str:
    """The exploratory arms, in their own table --- root §13.2's commitment.

    Their own table and never a column of Table 4, because none of them is a
    ladder rung: root §10.2 admits them as exploratory and requires each to get
    its own row, and folding one into the ladder would make the rungs differ in
    something other than K.

    Every arm that ran appears here **whatever it shows**. An arm reported only
    when it agrees with the headline is not a robustness arm, and root §13.2
    carries that as a disclosure a reader is entitled to before the numbers.
    An arm that has not run says so by name rather than being omitted.
    """
    # The matched-K pair is the one contrast here that changes an RQ's evidence
    # rather than its robustness, and it is a difference BETWEEN two arms, so no
    # arm-versus-grid column can carry it (`D82`).
    matched = next(
        (row for row in numbers.get("contrasts", {}).get("rows", [])
         if row.get("left") == "itrr" and row.get("right") == "itro"),
        None,
    )
    if matched and matched.get("mean_diff") is not None:
        matched_note = (
            " Measured, redundant minus orthogonal is "
            f"{fmt(matched['mean_diff'], 6)} of RelMSE "
            f"($t = {fmt(matched['t'], 2)}$, $p = {fmt(matched['p_two_sided'], 4)}$, "
            f"orthogonal ahead at {matched['n_origins'] - matched['left_better']} "
            f"of {matched['n_origins']} origins), against "
            "the whole $K=1$-to-$K=8$ ladder gain."
        )
    else:
        matched_note = ""

    rows = []
    for arm, state in numbers["robustness"].items():
        tag = state["model_tag"]
        name = MODEL_NAMES.get(tag, tag)
        if state["status"] != "run":
            rows.append([tex_escape(arm), tex_escape(name), _MISSING,
                         "not run", _MISSING, _MISSING, _MISSING,
                         _MISSING, _MISSING])
            continue
        for cell in state["cells"]:
            grid = cell.get("grid_r2_oos_same_rung")
            paired = cell.get("paired_vs_grid") or {}
            delta = (
                fmt(cell["r2_oos"] - grid, 4) if grid is not None else _MISSING
            )
            rows.append([
                tex_escape(arm),
                tex_escape(name),
                fmt(cell["k"], 0),
                fmt(cell["r2_oos"], 4) + " $\\pm$ "
                + fmt(cell["se_across_origins"], 4),
                fmt(grid, 4),
                delta,
                fmt(paired.get("mean_diff"), 5),
                fmt(paired.get("p_two_sided"), 4),
                fmt(cell["n_origins"], 0),
            ])
    note = (
        "\\textbf{Exploratory, declared before running, and reported whatever "
        "they show} (root §13.2). Not one of them is a rung of the $K$ ladder: "
        "adding one "
        "would make the rungs differ in something other than $K$, which is the "
        "one thing the ladder holds fixed. \\emph{longsched} answers "
        "``you under-trained'' --- and the epoch cap is not what binds an "
        "iTransformer run, which is why this arm widens the learning-rate "
        "schedule rather than the budget (`D62c'); Table 3 carries the measured "
        "at-cap count per arm rather than a claim about it (`D78'). \\emph{capacity} answers ``you under-capacitised'' at $d_{ff} = "
        "512$ (`D62b'). \\emph{attention} reproduces the main grid bit-for-bit "
        "and is what licenses reading Figure 5's maps as maps of the model whose "
        "numbers are reported (`D62d'). \\emph{orthogonal} and "
        "\\emph{redundant} are the matched-$K$ pair (`D70'): same $K = 8$, same "
        "target, same seeds, with the participation ratio the only thing that "
        "moves --- 5.011 against 3.609, either side of the ladder's own K=8 rung "
        "at 4.668. They test RQ1 \\textbf{by contrast} where the ladder can only "
        "infer it through a panel at $corr(K, K_{eff}) = 0.828$."
        + matched_note + " "
        "\\emph{look048} and \\emph{look192} halve and double $L = 96$, the one "
        "first-order hyperparameter root §6.2 never varied. \\emph{tuned} takes "
        "the configuration origin 1's \\emph{validation} preferred, selected "
        "where `D27' put the Stage 5 gate and never on a test block --- "
        "\\textbf{including its learning rate}, which the arm selected and then "
        "failed to apply until `D76'. $\\Delta$ is against the main grid at the "
        "same rung and $\\pm$ is the standard error \\emph{across} origins "
        "(`D30'); \\textbf{Paired} $\\Delta$RelMSE is the same comparison made "
        "\\emph{pairwise} on the fifteen origins both arms were scored on, with "
        "its two-sided $p$ from $t(G-1)$ (`D82'). The paired error is about half "
        "the marginal one, so overlapping $\\pm$ columns say nothing about "
        "whether two arms differ and the paired column is what the comparison "
        "rests on. It is post-hoc and uncorrected for multiplicity."
    )
    return tabular(
        "Exploratory arms, reported apart from RQ1--RQ3.",
        "tab:robustness",
        ["Arm", "Model", "$K$", "$R^2_{oos}$", "Grid same rung", "$\\Delta$",
         "Paired $\\Delta$RelMSE", "$p$", "Origins"],
        rows,
        "llrrrrrrr",
        note,
    )


def render_tables(numbers: dict, out_dir: Path) -> list[Path]:
    """Write every table as a standalone ``.tex`` float.

    Returns:
        The paths written, in table order.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    builders = {
        "table1_dataset.tex": _table1,
        "table2_efficiency.tex": _table2,
        "table2b_keff.tex": _table2b,
        "table3_architecture.tex": _table3,
        "table4_main.tex": _table4,
        "table5_decay.tex": _table5,
        "table6_dm.tex": _table6,
        "table7_horizons.tex": _table7,
        "table8_economics.tex": _table8,
        "table9_robustness.tex": _table9,
    }
    written = []
    for name, builder in builders.items():
        path = out_dir / name
        path.write_text(builder(numbers), encoding="utf-8")
        written.append(path)
    return written


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">🎨 Helper plot</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Impor matplotlib ditunda dan penyimpan PDF/PNG.</p>
</div>

In [ ]:


# -- figures -----------------------------------------------------------------


def _pyplot():
    """Matplotlib on a headless backend, imported late.

    Late, because ``report`` is imported by the notebook's definition cells and a
    Kaggle session that never renders a figure should not pay for the import.
    ``Agg`` because no display exists in either place this runs.
    """
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def _save(fig, out_dir: Path, stem: str) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for suffix in ("pdf", "png"):
        path = out_dir / f"{stem}.{suffix}"
        fig.savefig(path, bbox_inches="tight", dpi=200)
        paths.append(path)
    return paths


def _as_datetime(ms: np.ndarray) -> np.ndarray:
    return np.asarray(ms, dtype="int64").astype("datetime64[ms]")


def _figure1(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """The walk-forward scheme: fifteen origins above, one origin resolved below.

    Root §13.4 listed Figures 1 and 2 as *schematic, drawn by hand*, which meant
    in practice that they did not exist. They are drawn here instead, from
    :data:`config.ORIGINS` and the model's own config, for two reasons. A
    hand-drawn figure cannot be regenerated, and root §12 requires that of every
    number in the manuscript. And a hand-drawn one can be **silently wrong**:
    a rolling window that is not expanding, a 21/3 split, a purge at **both**
    boundaries and 5-month spacing are four details easy to draw plausibly and
    incorrectly, and the third is exactly `D24`.

    **Two panels, because one cannot carry both facts.** The upper panel is the
    scheme: fifteen origins over eight calendar years, where what is legible is
    the staircase --- a *rolling* 24-month window rather than an expanding one,
    5-month spacing coprime to 12 (`D26`), and `D28`'s training overlap, which
    is a required disclosure the shape states without a sentence.

    The lower panel exists because at that scale **the purge is invisible**.
    ``H = 24`` hours is 0.3% of one 30-day block and 0.03% of the training
    window, so on an eight-year axis it is thinner than the line drawn to
    represent it --- and a reader takes its absence for its absence. Resolving
    one origin in days makes the structure the design turns on legible: train,
    purge, validation, purge, then six test blocks. **The purge bands are drawn
    wider than 24 hours and the title says so**, which is the honest way to show
    a quantity too small to see; drawing it to scale and letting a reader
    conclude there is no purge would be the dishonest one.

    **Out-of-sample is shaded as its own span, and it is the six test blocks.**
    There is no fourth split: a forecaster standing at the origin has seen
    everything before it and nothing after, so every bar to the right of ``o``
    is out of sample by construction. Naming the span and shading it is what
    stops a reader looking for a separate region that does not exist --- and it
    is where the purge earns its place, because the purge is precisely what
    makes the claim true at the boundary rather than approximately true.

    The lower panel also carries the **falsification arm** (root §8.1), a model
    trained fresh at ``o + 90 days`` and scored on blocks 4-6 --- the only design
    in the study that identifies decay directly, and a per-origin element the
    scheme is incomplete without.
    """
    plt = _pyplot()
    fig, (top, low) = plt.subplots(
        2, 1, figsize=(7.6, 7.8), gridspec_kw={"height_ratios": [2.05, 1.0]}
    )

    def _year(moment) -> float:
        return moment.year + (moment.timetuple().tm_yday - 1) / 365.25

    for row, origin in enumerate(ORIGINS):
        y = len(ORIGINS) - row
        train = (_year(origin.train_start), _year(origin.train_sub_end))
        val = (_year(origin.val_start), _year(origin.val_end))
        top.barh(y, train[1] - train[0], left=train[0], height=0.62,
                 color=SPLIT_COLOUR["train"], zorder=2)
        top.barh(y, val[1] - val[0], left=val[0], height=0.62,
                 color=SPLIT_COLOUR["val"], zorder=2)
        for b, start, end in origin.blocks():
            top.barh(y, _year(end) - _year(start), left=_year(start), height=0.62,
                     color=SPLIT_COLOUR["test"] if b % 2 else SPLIT_COLOUR["test_alt"],
                     edgecolor="#ffffff", lw=0.4, zorder=2)
        for boundary in (val[0], val[1]):
            top.plot([boundary], [y + 0.44], marker="v", ms=3.2,
                     color=SPLIT_COLOUR["purge"], zorder=3)

    top.set_yticks(range(1, len(ORIGINS) + 1))
    top.set_yticklabels([o.label for o in reversed(ORIGINS)], fontsize=7.2)
    top.set_xlabel("calendar time (UTC)", fontsize=8.5)
    top.set_ylabel("origin", fontsize=8.5)
    top.set_title(
        "Fifteen origins: rolling 24-month window, 5-month spacing. Consecutive "
        "origins share 79.2% of their training data (D28)",
        fontsize=9.0,
    )
    top.grid(axis="x", alpha=0.25, lw=0.5)
    top.set_axisbelow(True)

    # -- one origin, resolved in days -----------------------------------------
    origin = ORIGINS[0]
    fresh = FalsificationOrigin(origin)

    def _day(moment) -> float:
        return (moment - origin.train_start).total_seconds() / 86400.0

    # H = 24 hours. Drawn to scale it is a fraction of a point wide here, so it
    # is drawn WIDE and the title says it is exaggerated (see the docstring).
    purge_drawn = 22.0
    oos_start, oos_end = _day(origin.test_start), _day(origin.test_end)
    low.axvspan(oos_start, oos_end, color=SPLIT_COLOUR["oos"], alpha=0.30,
                zorder=0, lw=0)
    low.axvline(oos_start, color="#000000", lw=1.1, ls="--", zorder=4)
    low.annotate(
        "", xy=(oos_end, 2.72), xytext=(oos_start, 2.72),
        arrowprops={"arrowstyle": "<->", "color": "#111111", "lw": 0.9},
    )
    low.text((oos_start + oos_end) / 2, 2.78,
             "OUT OF SAMPLE — 180 days, the six test blocks",
             ha="center", va="bottom", fontsize=7.4, fontweight="bold")
    low.text(oos_start - 10, 1.45, "origin $o$", ha="right", va="center",
             fontsize=7.4, style="italic")

    low.barh(2.0, _day(origin.train_sub_end) - _day(origin.train_start),
             left=_day(origin.train_start), height=0.52,
             color=SPLIT_COLOUR["train"], zorder=2)
    low.barh(2.0, _day(origin.val_end) - _day(origin.val_start),
             left=_day(origin.val_start), height=0.52,
             color=SPLIT_COLOUR["val"], zorder=2)
    for b, start, end in origin.blocks():
        low.barh(2.0, _day(end) - _day(start), left=_day(start), height=0.52,
                 color=SPLIT_COLOUR["test"] if b % 2 else SPLIT_COLOUR["test_alt"],
                 edgecolor="#ffffff", lw=0.5, zorder=2)
        low.text((_day(start) + _day(end)) / 2, 2.0, str(b), ha="center",
                 va="center", fontsize=6.5, color="#ffffff", zorder=3)

    # Drawn LAST and above both, because `val_start` IS `train_sub_end` and
    # `val_end` IS `test_start`: a purge drawn in sequence is painted over by the
    # split beginning at the same instant, which is exactly how it vanished.
    for boundary in (origin.train_sub_end, origin.val_end):
        low.barh(2.0, purge_drawn, left=_day(boundary) - purge_drawn / 2,
                 height=0.62, color=SPLIT_COLOUR["purge"], zorder=5,
                 hatch="////", edgecolor="#5a2d00", lw=0.6)

    # Root §8.1's falsification arm: fresh at o + 90 d, scored on blocks 4-6.
    low.barh(1.15, _day(fresh.train_sub_end) - _day(fresh.train_start),
             left=_day(fresh.train_start), height=0.34,
             color=SPLIT_COLOUR["train"], alpha=0.45, zorder=2)
    low.barh(1.15, _day(fresh.val_end) - _day(fresh.val_start),
             left=_day(fresh.val_start), height=0.34,
             color=SPLIT_COLOUR["val"], alpha=0.45, zorder=2)
    for b, start, end in origin.blocks():
        if b < 4:
            continue
        low.barh(1.15, _day(end) - _day(start), left=_day(start), height=0.34,
                 color=SPLIT_COLOUR["test"] if b % 2 else SPLIT_COLOUR["test_alt"],
                 alpha=0.55, edgecolor="#ffffff", lw=0.5, zorder=2)

    low.set_yticks([2.0, 1.15])
    low.set_yticklabels(["aged model\ntrained at $o$",
                         "fresh model\n$o$ + 90 d, blocks 4-6"], fontsize=7.2)
    low.set_ylim(0.80, 3.05)
    low.set_xlabel(f"days since train_start, origin {origin.label}", fontsize=8.5)
    low.set_title(
        "One origin resolved: train, purge, validation, purge, then the "
        f"out-of-sample span. Purge drawn {purge_drawn:.0f}× wide — "
        "it is $H$ = 24 hours",
        fontsize=9.0,
    )
    low.grid(axis="x", alpha=0.25, lw=0.5)
    low.set_axisbelow(True)

    handles = [
        plt.Line2D([], [], lw=6, color=SPLIT_COLOUR["train"],
                   label="train, 21 months (scaler fit here and nowhere else)"),
        plt.Line2D([], [], lw=6, color=SPLIT_COLOUR["val"],
                   label="validation, final 3 months"),
        plt.Line2D([], [], lw=6, color=SPLIT_COLOUR["purge"],
                   label="$H$-step purge, both boundaries (D24)"),
        plt.Line2D([], [], lw=6, color=SPLIT_COLOUR["test"],
                   label="test blocks 1–6, 30 days each"),
        plt.Line2D([], [], lw=6, color=SPLIT_COLOUR["oos"], alpha=0.30,
                   label="out-of-sample span (= the six test blocks)"),
    ]
    low.legend(handles=handles, fontsize=7, frameon=False, ncol=2,
               loc="lower center", bbox_to_anchor=(0.5, -0.70))
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure1_walkforward")
    plt.close(fig)
    return paths


def _figure2(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Architecture and inverted tokenization, with the tensor shape at each step.

    The shapes are read from the live :class:`ITransformerConfig`, not typed in,
    so the figure cannot drift from the model the grid ran --- which is the whole
    reason it is generated rather than drawn (see :func:`_figure1`).

    Two things the caption must carry, and the figure therefore shows. The token
    is a **variate over its whole lookback**, so attention runs over ``N <= 12``
    tokens and not over ``L = 96`` timesteps --- which is why ``d_model`` is 128
    rather than the reference 512 (`D25`), and why **no causal mask appears**:
    the axis attention runs over is contemporaneous, and causality is enforced
    upstream in the features and the windowing. And the loss is taken on the
    **target channel only** (`D39`); under the reference implementation's
    all-channel default K=12 becomes a 12-task problem and K=1 a 1-task one, so
    auxiliary supervision would vary with the study's own independent variable.
    """
    plt = _pyplot()
    config = ITransformerConfig()
    n = 8
    steps = [
        ("input window", f"(B, L={config.seq_len}, N={n})", "#e9ecef"),
        ("transpose — the inversion", f"(B, N={n}, L={config.seq_len})", "#ffd6a5"),
        (f"InvertedEmbedding: Linear({config.seq_len} -> {config.d_model})",
         f"(B, N={n}, d={config.d_model})", "#5b8db8"),
        (f"Encoder x {config.e_layers}: MHA over the {n} VARIATE tokens, "
         f"{config.n_heads} heads\nLayerNorm, FFN({config.d_model} -> "
         f"{config.d_ff} -> {config.d_model}), LayerNorm",
         f"(B, N={n}, d={config.d_model})", "#1f4e79"),
        (f"Projection: Linear({config.d_model} -> H={config.pred_len})",
         f"(B, N={n}, H={config.pred_len})", "#5b8db8"),
        ("transpose, select the target channel",
         f"(B, H={config.pred_len}, 1)", "#2e7d32"),
    ]

    fig, ax = plt.subplots(figsize=(7.0, 5.6))
    height, gap = 0.62, 0.34
    for row, (text, shape, colour) in enumerate(steps):
        y = (len(steps) - row) * (height + gap)
        ax.add_patch(plt.Rectangle((0.04, y), 0.64, height, facecolor=colour,
                                   edgecolor="#333333", lw=0.7))
        ax.text(0.36, y + height / 2, text, ha="center", va="center", fontsize=7.4,
                color="#ffffff" if colour in ("#1f4e79", "#2e7d32") else "#111111")
        ax.text(0.71, y + height / 2, shape, ha="left", va="center", fontsize=7.4,
                family="monospace")
        if row + 1 < len(steps):
            ax.annotate("", xy=(0.36, y - gap + 0.02), xytext=(0.36, y),
                        arrowprops={"arrowstyle": "-|>", "color": "#333333",
                                    "lw": 0.9})

    ax.text(
        0.04, 0.56,
        "Attention runs over N variates, never over L timesteps, so there is no "
        "causal mask:\ncausality is enforced upstream, in the features and the "
        "windowing (root section 6.1).\nLoss is MSE on the TARGET CHANNEL ONLY "
        "at every rung, so auxiliary supervision\ndoes not vary with K, which is "
        "the study's own independent variable (D39).",
        fontsize=7.0, va="top", color="#333333",
    )
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0.0, (len(steps) + 1) * (height + gap))
    ax.axis("off")
    ax.set_title("Inverted tokenization: one token per variate, not per timestep",
                 fontsize=9.5)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure2_architecture")
    plt.close(fig)
    return paths


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📈 Figure 2b, 3, 4</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Figure 3 memikul seluruh paper: <strong>satu</strong> seri, lima belas garis per-origin, dengan MDE digambar di samping fit-nya (<code>D36</code>).</p>
</div>

In [ ]:


def _figure2b(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Rolling PR and rolling OLS $R^2$ --- H2's premise, before any model runs."""
    plt = _pyplot()
    fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True)

    pr = inputs.rolling_pr
    axes[0].plot(_as_datetime(pr.get_column("window_end_ms").to_numpy()),
                 pr.get_column("pr").to_numpy(), lw=1.0, color="#22577a")
    axes[0].set_ylabel("PR at $K=8$")
    axes[0].grid(alpha=0.25, lw=0.5)

    r2 = inputs.rolling_r2
    axes[1].plot(_as_datetime(r2.get_column("window_end_ms").to_numpy()),
                 r2.get_column("r2").to_numpy(), lw=1.0, color="#c1121f")
    axes[1].set_ylabel("in-window $R^2$")
    axes[1].set_xlabel("window end (UTC)")
    axes[1].grid(alpha=0.25, lw=0.5)

    fig.suptitle("90-day rolling participation ratio and OLS fit (descriptive only)",
                 fontsize=10)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure2b_rolling")
    plt.close(fig)
    return paths


def _figure3(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Figure 3 --- ``A(i,b)``: fifteen per-origin lines plus the fitted slope.

    **One series, not four** (`D36`). ``A(i,b)`` is defined only for the K=1
    versus K=8 pair and §3 fixes RQ2 on that pair, ``never K=12``; plotting an
    ``A`` for four rungs would require silently inventing a different estimand
    from the one $\\beta_1$ is regressed on. The per-origin form is also the
    better figure: it displays the actual identification, which is within-origin
    slopes. **This figure carries the entire paper.**
    """
    plt = _pyplot()
    amp = inputs.amplification
    numbers = inputs.numbers
    fig, ax = plt.subplots(figsize=(7.0, 4.2))

    for (origin,), part in amp.group_by(["origin"], maintain_order=True):
        part = part.sort("block")
        ax.plot(part.get_column("block").to_numpy(), part.get_column("A").to_numpy(),
                lw=0.8, alpha=0.55, marker="o", ms=2.5, color="#4a6fa5")

    beta1 = float(numbers["rq2"]["beta1"])
    blocks = np.arange(1, 7, dtype=float)
    intercept = float(amp.get_column("A").mean()) - beta1 * blocks.mean()
    ax.plot(blocks, intercept + beta1 * blocks, lw=2.4, color="#c1121f",
            label=f"fitted $\\beta_1$ = {beta1:+.6f}")

    mde = float(numbers["rq2"]["minimum_detectable_beta1"])
    ax.plot(blocks, intercept + mde * blocks, lw=1.6, ls="--", color="#333333",
            label=f"MDE at 80% power = {mde:+.6f}")

    ax.axhline(0.0, lw=0.8, color="#888888")
    ax.set_xlabel("test block $b$ (30 days each)")
    ax.set_ylabel("$A(i,b) = [MSE_{K1} - MSE_{K8}] / MSE_{K1}$")
    ax.set_title("Multivariate gap against model age, one line per origin", fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25, lw=0.5)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure3_decay")
    plt.close(fig)
    return paths


def _figure4(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """RelMSE per test block, every model, averaged over origins.

    **Colour is the model family and dash is the rung**, not matplotlib's default
    cycle. The cycle holds ten colours and this matrix carries fourteen series
    once the `D64` baselines land, so the default silently paints ``itr-K1`` and
    ``ptst-K8`` the same blue -- a legend that cannot be read against the plot.
    Encoding family in hue also puts the figure's actual finding on the page:
    every ridge line hugs 1.000 and every deep line sits well above it, which is
    `D60c` seen rather than tabulated.

    **Two panels, and the split is not decoration** (`D84`). The two naive
    comparators land at RelMSE close to 2.0 --- forecasting the previous return
    when returns are serially uncorrelated costs exactly ``2 sigma^2`` --- while
    every model the paper argues about lives inside ``[1.000, 1.027]``. On one
    pair of axes that is a 40:1 range and the twelve informative series collapse
    into a single indistinguishable band on the floor, so the figure that exists
    to show `D60c`'s ordering showed nothing. Adding those two models to
    :data:`COMPARISON_KEYS` under `D74` is what did it, quietly, the same way the
    same two models took the Romano-Wolf matrix to zero rejections (`D79`).

    The upper panel keeps them, because dropping a model to make a figure legible
    is the wrong repair and their distance from 1.0 is itself the white-noise
    check. The lower panel is the same data on the range the argument happens in.
    """
    plt = _pyplot()
    main = inputs.seed_avg.filter(pl.col("pred_len") == PRED_LEN)
    fig, axes = plt.subplots(
        2, 1, figsize=(7.2, 6.0), sharex=True,
        gridspec_kw={"height_ratios": [1.0, 2.0]},
    )

    series = []
    for tag, k in COMPARISON_KEYS:
        if tag == "naive":
            continue
        cell = main.filter((pl.col("model") == tag) & (pl.col("k") == k))
        if cell.height == 0:
            continue
        by_block = cell.group_by("block").agg(
            pl.col("rel_mse").mean().alias("rel_mse")
        ).sort("block")
        series.append((
            tag, k,
            by_block.get_column("block").to_numpy(),
            by_block.get_column("rel_mse").to_numpy(),
        ))

    for axis in axes:
        for tag, k, blocks, values in series:
            axis.plot(blocks, values,
                      marker=FAMILY_MARKER.get(tag, "o"), ms=3.5, lw=1.2,
                      color=FAMILY_COLOUR.get(tag, "#666666"),
                      ls=RUNG_STYLE.get(k, "-"),
                      label=label((tag, k)))
        axis.axhline(1.0, lw=1.2, color="#000000", ls="--", label="Naive-RW")
        axis.grid(alpha=0.25, lw=0.5)

    # The zoom bound is read off the data, never fixed: a hardcoded 1.03 would
    # silently crop an arm a later manifest adds.
    zoom = [v for tag, k, _, values in series if tag not in NAIVE_COMPARATORS
            for v in values]
    margin = 0.1 * (max(zoom) - 1.0)
    axes[1].set_ylim(min(min(zoom), 1.0) - margin, max(zoom) + margin)
    axes[0].set_title(
        "RelMSE per block; everything above the dashed line loses to Naive-RW",
        fontsize=10,
    )
    axes[0].set_ylabel("full range")
    axes[1].set_ylabel("RelMSE (lower is better), zoomed")
    axes[1].set_xlabel("test block $b$ (30 days each)")
    axes[0].legend(fontsize=6.5, ncol=2, frameon=False, loc="center left",
                   bbox_to_anchor=(1.01, 0.0))
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure4_relmse")
    plt.close(fig)
    return paths


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📈 Figure 5 & 6</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Peta attention per tercile dan sensitivitas horizon.</p>
</div>

In [ ]:


def _figure5(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Attention maps, calm against stress --- terciles fixed in advance (`D48`).

    Returns an empty list when the `D62d` arm has not run: an empty axes labelled
    as a figure is worse than a named absence, because only one of the two tells
    a reader that nothing was measured.

    **The scale is centred on uniform, and that is not a cosmetic choice.** Over
    N variates a row of softmax weights sums to one, so ``1/N`` is what "attends
    to nothing in particular" looks like; at N=8 that is 0.125 and every measured
    weight lands within about 0.01 of it. Rendered on a sequential map anchored
    at zero, the whole panel comes out one flat colour and a reader takes
    "attention is uniform" from the colour bar rather than from the data ---
    asserting the `D50` null by choice of scale. Centring a diverging map on
    ``1/N`` and giving every panel the **same** symmetric limits shows the
    deviation, which is the quantity Figure 5 exists to display, and keeps the
    calm and stress panels comparable to each other rather than each to itself.

    Axes carry **variate names**, not indices. "Which variate does the model
    lean on under stress" is the question the figure answers, and it cannot be
    answered from a tick labelled ``6``.
    """
    if inputs.attention is None:
        return []
    plt = _pyplot()
    maps = inputs.attention
    layers = sorted(set(maps.get_column("layer").to_list()))

    panels: dict[tuple[int, str], np.ndarray] = {}
    for layer in layers:
        for tercile in TERCILE_SHOWN:
            part = maps.filter(
                (pl.col("layer") == layer) & (pl.col("tercile") == tercile)
            ).group_by(["i", "j"]).agg(pl.col("weight").mean()).sort(["i", "j"])
            size = int(part.get_column("i").max()) + 1
            panels[(layer, tercile)] = (
                part.get_column("weight").to_numpy().reshape(size, size)
            )

    size = next(iter(panels.values())).shape[0]
    uniform = 1.0 / size
    # One symmetric limit across every panel, so a colour means the same thing in
    # all four. Taken from the data rather than fixed, because the deviation's
    # magnitude is itself a finding and clipping it would hide it.
    spread = max(float(np.abs(grid - uniform).max()) for grid in panels.values())
    names = ladder_columns(size) if size in K_LADDER else [
        str(i) for i in range(size)
    ]

    fig, axes = plt.subplots(
        len(layers), 2, figsize=(7.6, 3.6 * len(layers)), squeeze=False
    )
    image = None
    for row, layer in enumerate(layers):
        for col, tercile in enumerate(TERCILE_SHOWN):
            axis = axes[row][col]
            image = axis.imshow(
                panels[(layer, tercile)], cmap=ATTENTION_CMAP,
                vmin=uniform - spread, vmax=uniform + spread,
            )
            axis.set_title(f"layer {layer}, {tercile}", fontsize=9)
            axis.set_xticks(range(size))
            axis.set_yticks(range(size))
            # Only the bottom row carries x labels: rotated variate names under an
            # upper panel overprint the title of the panel beneath it.
            bottom = row == len(layers) - 1
            axis.set_xticklabels(
                names if bottom else [""] * size, rotation=90, fontsize=6.5
            )
            axis.set_yticklabels(names if col == 0 else [""] * size, fontsize=6.5)
            if col == 0:
                axis.set_ylabel("query variate", fontsize=8)
            if bottom:
                axis.set_xlabel("attended variate", fontsize=8)

    bar = fig.colorbar(image, ax=axes, fraction=0.030, pad=0.02)
    bar.set_label(f"attention weight (uniform = {uniform:.3f})", fontsize=8)
    bar.ax.axhline(uniform, color="#000000", lw=1.0)
    fig.suptitle(
        "Variate attention, calm versus stress terciles of realised volatility",
        fontsize=10,
    )
    paths = _save(fig, out_dir, "figure5_attention")
    plt.close(fig)
    return paths


def _figure6(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Horizon sensitivity: $R^2_{oos}$ against $H$, one line per rung."""
    plt = _pyplot()
    cells = pl.DataFrame(inputs.numbers["horizons"]["cells"])
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for (k,), part in cells.group_by(["k"], maintain_order=True):
        part = part.sort("pred_len")
        ax.errorbar(
            part.get_column("pred_len").to_numpy(),
            part.get_column("r2_oos").to_numpy(),
            yerr=part.get_column("se_across_origins").to_numpy(),
            marker="o", ms=4, lw=1.2, capsize=3, label=f"$K={int(k)}$",
        )
    ax.axhline(0.0, lw=1.2, color="#000000", ls="--", label="Naive-RW")
    ax.set_xscale("log")
    ax.set_xticks([1, 3, 24, 168])
    ax.set_xticklabels(["1", "3", "24", "168"])
    ax.set_xlabel("forecast horizon $H$ (hours)")
    ax.set_ylabel("$R^2_{oos}$")
    ax.set_title("Horizon sweep at four named origins", fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25, lw=0.5)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure6_horizons")
    plt.close(fig)
    return paths


<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b78888; border-radius: 6px; padding: 11px 18px;">
  <h4 style="color: #95d5b2; margin: 0 0 5px 0; font-size: 1.02em; font-weight: 600;">📈 Figure 7 & render</h4>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.9em;">Kurva ekuitas di tiga tingkat slippage, lalu seluruhnya ditulis ke disk.</p>
</div>

In [ ]:


def _figure7(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Equity curves before costs and at all three pre-registered slippage levels.

    **The series is a wealth multiple, not a cumulative log return.**
    ``economics.equity_curves`` emits ``exp(cumsum(net))``, so it starts at 1.0
    and break-even is the line at 1.0 --- the axis label said log return and the
    reference line sat at 0.0, which is a mislabelled unit in the one figure the
    economic claim rests on. A reader who took the label at face value would read
    a 20% gain as a 1.2 log return, an order of magnitude out.

    **Buy-and-hold is drawn, because it is the comparator the claim is stated
    against.** Root §13.2 requires the economic result be reported beside it ---
    +20.6% net against +29.0% --- and a figure that omits the number the headline
    is measured against lets the strategy's own curve read as skill. It comes
    from the same `economics.buy_and_hold` position the Table 8 ``hold_*``
    columns use, so figure and table cannot disagree.

    **The leftmost panel is before costs and is labelled so.** Root §13.5's band
    is 0.02/0.05/0.10% per side; a zero-slippage panel is outside the
    pre-registration and is shown because §13.4 asks for "before and after
    costs", which is what makes the size of the cost band legible.

    **Every curve stops at the shortest origin, and that truncation is the
    figure's correctness** (`D83`). Origins do not carry the same number of
    tradable days --- a holding period spanning an outage has no defined realised
    return and is skipped (`D46`), so the count runs from 146 to 180. Averaging
    each day over whatever origins still have data made the mean jump wherever an
    origin ran out, and the largest of those jumps rendered as a **near-vertical
    fall of about seven points at day 146** that a reader takes for a crash. It
    was not a crash: it was the first origin leaving the average. Worse, the
    origins that leave earliest are the ones with the most outages and outages
    cluster on stress, so the untruncated tail was an average over the *easy*
    origins and drifted optimistic exactly where it looked dramatic --- `D45`'s
    future-conditioned exclusion, surfacing in the figure the economic claim
    rests on. Truncating to the common horizon keeps the denominator at fifteen
    for every plotted day.
    """
    plt = _pyplot()
    equity = inputs.equity
    slippages = sorted(set(equity.get_column("slippage_per_side").to_list()))
    # The longest day index every (model, origin) series reaches.
    common = int(
        equity.group_by(["model", "origin", "slippage_per_side"])
        .agg(pl.col("period").max().alias("last"))
        .get_column("last").min()
    )
    equity = equity.filter(pl.col("period") <= common)
    fig, axes = plt.subplots(1, len(slippages), figsize=(3.6 * len(slippages), 3.8),
                             sharey=True, squeeze=False)
    for col, slippage in enumerate(slippages):
        axis = axes[0][col]
        part = equity.filter(pl.col("slippage_per_side") == slippage)
        for (model,), sub in part.group_by(["model"], maintain_order=True):
            by_period = sub.group_by("period").agg(
                pl.col("equity").mean().alias("equity")
            ).sort("period")
            tag = str(model).split("-")[0]
            hold = tag == HOLD_LABEL.split("-")[0]
            axis.plot(by_period.get_column("period").to_numpy(),
                      by_period.get_column("equity").to_numpy(),
                      lw=2.0 if hold else 1.2,
                      color="#000000" if hold else FAMILY_COLOUR.get(tag, "#666666"),
                      ls="--" if hold else "-",
                      label=str(model))
        # Break-even. The series is exp(cumsum(net)), so 1.0 is flat, not 0.0.
        axis.axhline(1.0, lw=0.8, color="#888888")
        priced = (
            "before costs (not in the §13.5 band)" if float(slippage) == 0.0
            else f"slippage {100 * float(slippage):.2f}% per side"
        )
        axis.set_title(priced, fontsize=8.5)
        axis.set_xlabel("trading day since origin", fontsize=8.5)
        axis.grid(alpha=0.25, lw=0.5)
    axes[0][0].set_ylabel("net equity multiple (1.0 = break-even)")
    # Framed and opaque: the curves reach the panel corners at every slippage
    # level, and a frameless legend printed the break-even line straight through
    # the comparator's own label.
    axes[0][-1].legend(fontsize=7, frameon=True, framealpha=0.9,
                       edgecolor="#cccccc", loc="lower left")
    fig.suptitle(
        "Equity curves, averaged over the fifteen origins, taker fee 0.04% per side\n"
        f"truncated at day {common}, the shortest origin, so every plotted day "
        "averages all fifteen (`D83`)",
        fontsize=10,
    )
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure7_equity")
    plt.close(fig)
    return paths


def render_figures(inputs: ReportInputs, out_dir: Path, log=print) -> list[Path]:
    """Write every figure as ``.pdf`` and ``.png``.

    Figure 5 is skipped, **by name**, when the `D62d` attention arm has not run:
    an empty axes labelled as a figure reads as a measurement of nothing, and a
    named absence reads as what it is.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    written: list[Path] = []
    for name, builder in (
        ("figure1", _figure1),
        ("figure2", _figure2),
        ("figure2b", _figure2b),
        ("figure3", _figure3),
        ("figure4", _figure4),
        ("figure5", _figure5),
        ("figure6", _figure6),
        ("figure7", _figure7),
    ):
        paths = builder(inputs, out_dir)
        if not paths:
            log(f"report: {name} SKIPPED --- its input has not been produced yet")
            continue
        written.extend(paths)
    return written


<div style="background: linear-gradient(135deg, #1b0033, #2d0052); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #bf5af233;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🔐 17 &middot; Provenance kode & input
  </h2>
  <p style="color: #cbb2e8; margin: 0; font-size: 1.02em;">Seluruh sel definisi sudah lewat; sekarang catat kode apa yang barusan didefinisikan, sebelum apa pun yang mahal berjalan.</p>
  <ul style="color: #cbb2e8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Root &sect;12 meminta tiap run menyebut kode yang menghasilkannya, dan menyebut git sha sebagai caranya. Di Kaggle tidak ada repositori git, dan sel-sel di atas adalah definisi bukan berkas — jadi tidak ada pula yang bisa di-hash dari disk. <code>code_sha256</code> yang memikulnya (<code>D54b</code>).</li>
    <li style="margin-bottom: 6px;">Sentinelnya satu per modul: sel yang terlewat meninggalkan <em>lubang</em>, bukan berkas basi, dan lubang itu baru muncul berjam-jam kemudian di tengah grid. Murah di sini, tak terbatas di sana.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 4px solid #9e9e9e; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #d0d0d0; margin: 0 0 6px 0; font-size: 1.22em;">🧾 Inventaris modul</h3>
  <p style="color: #a8a8a8; margin: 0; font-size: 0.94em;">Delapan belas nama yang sel-sel di atas definisikan, dicatat supaya sel provenance dapat menyebut apa yang sebenarnya berjalan.</p>
</div>

In [ ]:
MODULE_NAMES = [
    'config.py',
    '__init__.py',
    'segments.py',
    'windows.py',
    'budget.py',
    'features.py',
    'efficiency.py',
    'splits.py',
    'keff.py',
    'model.py',
    'train.py',
    'metrics.py',
    'baselines.py',
    'comparisons.py',
    'economics.py',
    'attention.py',
    'runner.py',
    'report.py',
]


<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 4px solid #bf5af2; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">🔐 Provenance kode & input</h3>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.94em;">Root §12 meminta tiap run menyebut kode yang menghasilkannya. <code>git_sha</code> berbunyi unknown di Kaggle, jadi <code>code_sha256</code> yang memikulnya (<code>D54b</code>).</p>
</div>

In [ ]:
# Root section 12 asks a run to name the code that produced it, and names the git
# sha as the way. There is no git repository on Kaggle and — the cells above
# being definitions rather than files — nothing on disk to hash either. So
# the digest is taken from src/itransformer_btc/ when this notebook is generated
# and pinned here (D54b). It is the SAME number a local checkout of the same
# source reports, which is the point: a run from the notebook and a run from the
# repository must not look like different code vintages.
CODE_SHA256_OVERRIDE = "0e207716dfdcaeb34aae9db479d7fb461871247683dc6567858adc4c4827135e"

# These cells are EXECUTED, not written, so a skipped one leaves a hole rather
# than a stale file — and the hole surfaces hours later, inside the grid. One
# sentinel per module, in MODULE_ORDER: cheap here, unbounded there.
_sentinels = (
    "ORIGINS", "__all__", "build_segments", "count_windows", "budget_table",
    "build_features", "efficiency_table", "build_origin_tensors", "keff_table",
    "ITransformer", "code_sha256", "seed_average", "PatchTST", "pair_matrix",
    "economics_table", "tercile_maps", "manifest", "build_report",
)
_missing = [name for name in _sentinels if name not in globals()]
assert not _missing, (
    f"{len(_missing)} of {len(_sentinels)} definition cells have not run: "
    f"{_missing}. Run the Definitions cells above in order, top to bottom."
)
assert "itransformer_btc" not in sys.modules, (
    "an installed or on-path itransformer_btc package was imported. This notebook "
    "must run its OWN definitions, or every number it produces is traceable to "
    "code that is not in the cells above — exactly the dependency this format "
    "exists to remove."
)

print(f"modules defined in-kernel: {len(MODULE_NAMES)}  {MODULE_NAMES}")
print(f"code_sha256 {code_sha256()}")
print("\nThat digest goes into every meta/*.json. There is no git repository on "
      "Kaggle, so it is what the traceability contract has to name the code with "
      "— and it is the better half of the pair anyway: it identifies the code "
      "that ran, not the commit someone was standing on with a dirty tree.")


<div style="background: linear-gradient(135deg, #002200, #003300); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #7ae58233;">
  <h2 style="color: #95d5b2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🛠️ 18 &middot; Invarian pra-terbang — Stage 4
  </h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 1.02em;">Tiga pemeriksaan yang harus lolos sebelum grid. Ketiganya gagal saat pertama kali dijalankan.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><code>MSE(c&middot;x)/c&sup2; == MSE(x)</code>, <strong>bukan</strong> <code>MSE(c&middot;x) == MSE(x)</code> (<code>D03</code>) — targetnya adalah kanal dari array yang sama, jadi ia ikut terskala dan loss-nya terskala oleh c&sup2;. Versi spesifikasi sumber tidak mungkin lolos.</li>
    <li style="margin-bottom: 6px;">Overfit satu batch dengan <strong><code>dropout=0.0</code></strong> (<code>D52d</code>). Dengan 0,1 yang terkonfigurasi masih menyala, loss-nya mentok di sekitar 7e-2 dan pembaca yang menuruti instruksinya secara harfiah menyimpulkan plumbing rusak padahal tidak.</li>
    <li style="margin-bottom: 6px;"><strong>Naive-RW dihitung lebih dulu</strong>, sebelum satu model pun latih, dan ia <code>&#375;<sub>z</sub> = &minus;&mu;<sub>g</sub>/&sigma;<sub>g</sub></code> — tidak pernah 0 (<code>D31</code>), yang diam-diam akan menjadi model constant-drift yang memakai nama baseline EMH.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 4px solid #7ae582; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #95d5b2; margin: 0 0 6px 0; font-size: 1.22em;">🛠️ Jalankan Stage 4 — tiga invarian pra-terbang</h3>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.94em;">Invariansi skala <code>MSE(c·x)/c² == MSE(x)</code>, overfit satu batch pada <code>dropout=0.0</code>, lalu Naive-RW dihitung lebih dulu sebelum satu model pun latih. Ketiganya gagal saat pertama kali dijalankan.</p>
</div>

In [151]:
device = pick_device()
print(f"device {device}")

set_seed(42)
probe = ITransformer(ITransformerConfig()).to(device).eval()
base, scaled = scale_invariance_check(
    probe,
    torch.randn(64, 96, 8, device=device),
    torch.randn(64, 24, device=device),
    c=100.0,
)
rel = abs(base - scaled) / base
print(f"use_norm invariance: {base:.8f} vs {scaled:.8f}  rel={rel:.2e}")
assert rel < 1e-3, "FATAL (D03): use_norm inactive, or the scaler no longer cancels"
print(f"parameters {probe.n_parameters():,} — identical at every rung by construction")

set_seed(42)
plumb = ITransformer(ITransformerConfig(dropout=0.0)).to(device).train()
xs, ys = torch.randn(8, 96, 8, device=device), torch.randn(8, 24, device=device)
opt = torch.optim.Adam(plumb.parameters(), lr=1e-3)
for _ in range(200):
    opt.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(plumb(xs), ys)
    loss.backward()
    opt.step()
print(f"single-batch overfit (dropout=0.0): {loss.item():.3e}")
assert loss.item() < 1e-3, "plumbing broken"

print("\nNaive-RW in scaler space, per origin (D31 / D52b):")
naive = pl.DataFrame([
    {"origin": o.label, **{
        "mu_g": (t := build_origin_tensors(features, o, 1)).scaler.mean[0],
        "sigma_g": t.scaler.std[0],
        "mu_over_sigma": t.scaler.target_mu_over_sigma,
        "naive_rw_z": t.naive_rw_z,
        "n_train": len(t.train),
    }}
    for o in ORIGINS
])
print(naive)
print(f"mu_g/sigma_g spans {naive['mu_over_sigma'].min():+.5f} … "
      f"{naive['mu_over_sigma'].max():+.5f} and CHANGES SIGN, so the tilt is not a "
      f"constant a reader could subtract. It tracks the same bull/bear cycle H2 "
      f"invokes as its own mechanism, which is why it is confounded with the "
      f"effect of interest and does not wash out.")
naive.write_parquet(ARTIFACTS / "naive_rw_by_origin.parquet")


device cuda:0
use_norm invariance: 1.42240524 vs 1.42240143  rel=2.68e-06
parameters 280,472 — identical at every rung by construction
single-batch overfit (dropout=0.0): 1.055e-10

Naive-RW in scaler space, per origin (D31 / D52b):
shape: (15, 6)
┌─────────┬───────────┬──────────┬───────────────┬────────────┬─────────┐
│ origin  ┆ mu_g      ┆ sigma_g  ┆ mu_over_sigma ┆ naive_rw_z ┆ n_train │
│ ---     ┆ ---       ┆ ---      ┆ ---           ┆ ---        ┆ ---     │
│ str     ┆ f64       ┆ f64      ┆ f64           ┆ f64        ┆ i64     │
╞═════════╪═══════════╪══════════╪═══════════════╪════════════╪═════════╡
│ 2020-01 ┆ -0.000043 ┆ 0.009151 ┆ -0.00475      ┆ 0.00475    ┆ 13924   │
│ 2020-06 ┆ 0.000008  ┆ 0.007075 ┆ 0.001109      ┆ -0.001109  ┆ 13689   │
│ 2020-11 ┆ 0.000037  ┆ 0.00812  ┆ 0.004597      ┆ -0.004597  ┆ 13729   │
│ 2021-04 ┆ 0.000127  ┆ 0.007915 ┆ 0.016039      ┆ -0.016039  ┆ 13704   │
│ 2021-09 ┆ 0.000088  ┆ 0.00862  ┆ 0.010165      ┆ -0.010165  ┆ 13547   │
│ …       ┆ 

<div style="background: linear-gradient(135deg, #2b0a00, #3d1000); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #ff7b5433;">
  <h2 style="color: #ffb4a2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🛡️ 19 &middot; Gerbang Stage 5 — validasi saja
  </h2>
  <p style="color: #ffd8c2; margin: 0; font-size: 1.02em;">Origin 1, 4 K &times; 3 seed, dinilai pada sub-blok validasi. Blok uji tetap tertutup.</p>
  <ul style="color: #ffd8c2; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Root &sect;11 menuntut blok uji dibuka sekali, setelah desain dibekukan, jadi gerbang yang mereposisi judul berdasarkan hasil blok uji tidak bisa hidup berdampingan dengannya (<code>D27</code>).</li>
    <li style="margin-bottom: 6px;">Statistiknya <strong>Clark&ndash;West, bukan DM</strong> (<code>D29</code>): himpunan fitur K=1 adalah subset tegas K=8 di bawah arsitektur dan sampel yang sama.</li>
    <li style="margin-bottom: 6px;">Gerbangnya <strong>K=1 lawan K=8, tidak pernah K=12</strong> — K=12 dibangun untuk redundan, dan menggerbangkan padanya akan membunuh paper yang layak untuk alasan yang salah.</li>
    <li style="margin-bottom: 6px;"><strong>Terukur: gerbang GAGAL.</strong> <code>S* = +0,8759, p = 0,1906</code> satu sisi. Judul direposisi ke varian deskriptif pada 2026-08-20, dan arm K=16 karenanya tidak dijalankan — klausa 1 gagal (<code>D60a</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 4px solid #ff7b54; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #ffb4a2; margin: 0 0 6px 0; font-size: 1.22em;">🛡️ Jalankan gerbang Stage 5 — pada validasi, bukan uji</h3>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.94em;">Origin 1, 4 K × 3 seed, dinilai pada sub-blok validasi supaya blok uji tetap tertutup sampai desain beku (<code>D27</code>). Statistiknya Clark–West, karena pasangannya nested (<code>D29</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 8px;"><span style="color:#ffd166; font-weight:600; font-size:0.84em;">MENULIS</span> <span style="color:#dcdcdc; font-size:0.84em;">🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧾 <code>artifacts/meta/*.json</code></span></div>
</div>

In [152]:
pilot = stage5_pilot(features, out_root=ARTIFACTS, device=device)
print(pilot)

if not pilot.passed:
    print("\n*** Reposition the title to the descriptive variant NOW, not in week "
          "nine (root section 8.5). ***")
print("\nDisclose the pilot in section 13.2 as a SELECTION EVENT, stated separately "
      "from the DSR trial count — the DSR does not correct for selection over a "
      "paper's conclusion.")


pilot itr_o01_K01_H024_s42  val=0.469304  35.8s
pilot itr_o01_K01_H024_s43  val=0.469613  25.2s
pilot itr_o01_K01_H024_s44  val=0.469509  32.9s
pilot itr_o01_K04_H024_s42  val=0.469239  21.9s
pilot itr_o01_K04_H024_s43  val=0.469220  21.9s
pilot itr_o01_K04_H024_s44  val=0.469377  27.5s
pilot itr_o01_K08_H024_s42  val=0.468420  22.3s
pilot itr_o01_K08_H024_s43  val=0.467739  22.2s
pilot itr_o01_K08_H024_s44  val=0.468300  28.0s
pilot itr_o01_K12_H024_s42  val=0.468643  22.3s
pilot itr_o01_K12_H024_s43  val=0.468447  22.3s
pilot itr_o01_K12_H024_s44  val=0.468845  27.7s
validation MSE  K=1: 0.467551  K=4: 0.466160  K=8: 0.465441  K=12: 0.465767
Clark-West K=1 vs K=8 (validation): S*=+0.8759  p=0.1906 (one-sided)  T=1845  h=24
FAIL — reposition the title to the descriptive variant NOW, not in week nine (root §8.5)

*** Reposition the title to the descriptive variant NOW, not in week nine (root section 8.5). ***

Disclose the pilot in section 13.2 as a SELECTION EVENT, stated separately f

<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 4px solid #c77dff; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #e0aaff; margin: 0 0 6px 0; font-size: 1.22em;">🎛️ Pemilihan konfigurasi pada validasi</h3>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.94em;">Delapan belas konfigurasi diranking pada sub-blok validasi origin 1, lalu pemenangnya dilatih penuh di lima belas origin sebagai arm <code>itrt</code>. Grid-nya dideklarasikan di <code>TUNING_GRID</code> sebelum dijalankan; ruang pencarian yang dipilih setelah melihat pemenangnya bukan pencarian. Pemenang dijalankan <strong>apa adanya, termasuk learning rate-nya</strong> (<code>D76</code>). Arm ini <strong>eksploratori</strong>, tidak masuk perbandingan tangga RQ1, dan jumlah percobaannya masuk hitungan development trial root §13.5 (<code>D70</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 8px;"><span style="color:#ffd166; font-weight:600; font-size:0.84em;">MENULIS</span> <span style="color:#dcdcdc; font-size:0.84em;">🧾 <code>artifacts/meta/tuning_selection.json</code></span></div>
</div>

In [153]:
TUNED_CONFIG, TUNING_TABLE = tune_on_validation(
    features, origin_index=1, k=8, device=device,
    log=lambda msg: print(msg, flush=True),
)
GRID_CONFIGS = {"tuned": TUNED_CONFIG}

(ARTIFACTS / "meta").mkdir(parents=True, exist_ok=True)
(ARTIFACTS / "meta" / "tuning_selection.json").write_text(
    json.dumps({"grid": list(TUNING_GRID), "ranked": TUNING_TABLE,
                "selected": {"d_model": TUNED_CONFIG.d_model,
                             "e_layers": TUNED_CONFIG.e_layers,
                             "lr": TUNED_CONFIG.lr}}, indent=1),
    encoding="utf-8",
)
# `lr` is in `selected` because it is in the search space and in the run. It was
# in neither before `D76`: the winner's rate was ranked on and then dropped, so
# the recorded decision and the executed arm disagreed.
assert TUNED_CONFIG.lr == TUNING_TABLE[0]["lr"], "tuned arm must run the selected lr"
print(f"tuned arm will run {TUNED_CONFIG.d_model=} {TUNED_CONFIG.e_layers=} "
      f"{TUNED_CONFIG.lr=}")


  probe 1/18 {'d_model': 64, 'e_layers': 2, 'lr': 0.0001} -> 0.467123
  probe 2/18 {'d_model': 64, 'e_layers': 2, 'lr': 0.0003} -> 0.467239
  probe 3/18 {'d_model': 64, 'e_layers': 2, 'lr': 0.001} -> 0.466733
  probe 4/18 {'d_model': 64, 'e_layers': 3, 'lr': 0.0001} -> 0.466477
  probe 5/18 {'d_model': 64, 'e_layers': 3, 'lr': 0.0003} -> 0.467468
  probe 6/18 {'d_model': 64, 'e_layers': 3, 'lr': 0.001} -> 0.468404
  probe 7/18 {'d_model': 128, 'e_layers': 2, 'lr': 0.0001} -> 0.468420
  probe 8/18 {'d_model': 128, 'e_layers': 2, 'lr': 0.0003} -> 0.468706
  probe 9/18 {'d_model': 128, 'e_layers': 2, 'lr': 0.001} -> 0.467696
  probe 10/18 {'d_model': 128, 'e_layers': 3, 'lr': 0.0001} -> 0.467961
  probe 11/18 {'d_model': 128, 'e_layers': 3, 'lr': 0.0003} -> 0.468949
  probe 12/18 {'d_model': 128, 'e_layers': 3, 'lr': 0.001} -> 0.468082
  probe 13/18 {'d_model': 256, 'e_layers': 2, 'lr': 0.0001} -> 0.471792
  probe 14/18 {'d_model': 256, 'e_layers': 2, 'lr': 0.0003} -> 0.471276
  probe 15/

<div style="background: linear-gradient(135deg, #001233, #001845); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #4cc9f033;">
  <h2 style="color: #8ecae6; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    ⚡ 20 &middot; Grid
  </h2>
  <p style="color: #a9d6e5; margin: 0; font-size: 1.02em;">Manifes penuh dijalankan di kernel ini. Resume otomatis, penjaga anggaran di batas run.</p>
  <ul style="color: #a9d6e5; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;"><strong>Granularitas resume adalah satu run</strong>, ~32 detik. Sesi yang terpotong di run 200 kehilangan paling banyak satu run yang sedang jalan; sesi berikutnya mengurangi yang sudah selesai lalu melanjutkan. Checkpointing intra-run sengaja ditiadakan.</li>
    <li style="margin-bottom: 6px;">Terukur pada manifes 894-run: <strong>0 gagal, 7,79 jam, rata-rata 31,8 detik per run</strong> — di satu device. Dengan dua worker (<code>D68</code>), sisa 726 run &asymp; 6,4 GPU-jam &asymp; 3,2 jam wall.</li>
    <li style="margin-bottom: 6px;">Baseline berjalan <em>setelah</em> tangga, jadi sesi yang pendek kehilangan komparator, bukan masukan RQ1&ndash;RQ3 — dan asersi keselarasan jendela <code>D45</code> tiap baseline menemukan pembandingnya sudah di disk.</li>
    <li style="margin-bottom: 6px;"><strong>Evaluasi digerbangi kelengkapan grid.</strong> Panel parsial adalah panel tak seimbang, dan estimator root &sect;9.1 menolaknya secara desain. &beta;&#8321; setengah-panel adalah estimand yang berbeda, bukan yang lebih berderau (<code>D54e</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #90e0ef; margin: 0 0 6px 0; font-size: 1.22em;">🚀 Jalankan grid — 1.620 run, dua T4</h3>
  <p style="color: #caf0f8; margin: 0; font-size: 0.94em;">Satu worker per device dari satu antrean bersama; seeding di-scope per device sehingga sebuah run menghasilkan byte yang sama entah ia berjalan sendiri atau berdampingan (<code>D68</code>). Resume lewat glob, tanpa slug yang di-hardcode (<code>D54</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 8px;"><span style="color:#ffd166; font-weight:600; font-size:0.84em;">MENULIS</span> <span style="color:#dcdcdc; font-size:0.84em;">🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧾 <code>artifacts/meta/*.json</code> &nbsp; 🧊 <code>artifacts/attn/*.parquet</code></span></div>
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>data/raw/BTCUSDT_1h.parquet</code></span></div>
</div>

In [154]:

# The pre-flight probe and the pilot allocated on this same device, and the grid
# is about to. Hand the memory back before it starts rather than carrying two
# dead models through 684 runs.
for _name in ("probe", "plumb", "xs", "ys", "opt", "loss"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ALL = manifest()
roots = discover_roots(ARTIFACTS)
todo = pending(ALL, roots)

by_arm = {}
for cell in ALL:
    by_arm[cell.arm] = by_arm.get(cell.arm, 0) + 1
already_done = len(ALL) - len(todo)
print(f"manifest {len(ALL)} unique runs {by_arm}")
print(f"roots searched: {[str(r) for r in roots]}")
print(f"already complete: {already_done}   pending: {len(todo)}")
print("Completeness is per run_id: both preds/ and meta/ present and "
      "meta.status == 'complete'. A run interrupted mid-training leaves no meta, "
      "so it is redone — losing at most one run, ~30 s (root §10.5).")

# The budget is what is LEFT of the session, not a fresh 11 h.
SESSION_BUDGET_H = 11.0
elapsed_h = (time.perf_counter() - SESSION_T0) / 3600.0
budget_h = max(0.25, SESSION_BUDGET_H - elapsed_h)
print(f"\nprelude took {elapsed_h * 60:.0f} min -> grid gets {budget_h:.2f} h "
      f"of the {SESSION_BUDGET_H:.1f} h session budget, guard reserves 0.5 h more")

# In-kernel and sequential. The definitions live in THIS namespace and nowhere
# else, so a subprocess could not reach them — and at ~30 s per run measured
# (D57) the 534 iTransformer cells are ~4.5 h, which fits without a second
# worker. The 150 baseline cells (D56) are not measured on a T4; they run last,
# so an overrun costs the comparators and never RQ1-RQ3's inputs. The second GPU
# idles; that is the price of the format, and it is stated rather than hidden.
# Both GPUs, one worker each, off a shared queue (`D68`). The note that used to
# stand here said threads were not the fix because torch.manual_seed seeds EVERY
# CUDA device — true of the seeding as it was, and that is what changed:
# set_seed now takes the device and seeds only that one, and the CPU generator is
# shared under a single lock across seeding and module construction. Parallelism
# stays at the RUN level; root §10.3 rejects nn.DataParallel with a measured
# reason, and DDP pays a process group per ~32 s run.
DEVICES = visible_devices()
print(f"devices for the grid: {[str(d) for d in DEVICES]}")

t0 = time.perf_counter()
summary = execute_parallel(
    todo, features,
    devices=DEVICES,
    configs=GRID_CONFIGS,
    out_root=ARTIFACTS,
    roots=roots,
    guard=BudgetGuard(budget_h, 0.5),
    log=lambda msg: print(msg, flush=True),
)
print(summary)
print(f"grid finished after {(time.perf_counter() - t0) / 3600:.2f} h")

left = pending(ALL, discover_roots(ARTIFACTS))
GRID_COMPLETE = not left
print(f"remaining after this session: {len(left)} of {len(ALL)}")
if left:
    # The evaluation below is GATED on this. A partial grid is an unbalanced
    # panel, and §9.1's estimators refuse one by design — `amplification` raises
    # rather than silently comparing K=1 at eleven origins against K=8 at ten.
    # Letting that exception reach Kaggle would mark the version failed at the
    # exact moment its output is the only thing worth keeping.
    print("\nEvaluation is SKIPPED this session — the panel is incomplete, and a")
    print("half-panel beta1 is not a smaller answer but a different estimand.")
    print("Nothing is lost: preds/ and meta/ are on disk and resume is by run_id.")
    print("\n  1. Save Version now (its output IS the session's work)")
    print("  2. Attach that output as the next session's input Dataset")
    print("  3. Run this notebook again — completed runs are skipped automatically")

    ran = len(ALL) - len(left) - already_done
    if ran > 0:
        mean_s = (time.perf_counter() - t0) / ran
        print(f"\n  {ran} runs this session at ~{mean_s:.0f}s each -> about "
              f"{len(left) * mean_s / 3600:.1f} h of wall still to do, "
              f"{len(left) * mean_s / 3600 / 10.5:.1f} more sessions")


manifest 1620 unique runs {'main': 300, 'uniform': 75, 'fresh': 15, 'horizon': 240, 'ridge': 60, 'dlinear': 75, 'patchtst': 75, 'lstm': 75, 'persist': 15, 'seasonal': 15, 'orthogonal': 75, 'redundant': 75, 'look048': 75, 'look192': 75, 'tuned': 75, 'attention': 75, 'longsched': 150, 'capacity': 75}
roots searched: ['artifacts']
already complete: 12   pending: 1608
Completeness is per run_id: both preds/ and meta/ present and meta.status == 'complete'. A run interrupted mid-training leaves no meta, so it is redone — losing at most one run, ~30 s (root §10.5).

prelude took 13 min -> grid gets 10.78 h of the 11.0 h session budget, guard reserves 0.5 h more
devices for the grid: ['cuda:0', 'cuda:1']
run-level parallelism across ['cuda:0', 'cuda:1'] (`D68`)
[2/1608] cuda:1 itr_o01_K01_H024_s46  epochs=11  val=0.469391  42.2s  n_train=13924
[1/1608] cuda:0 itr_o01_K01_H024_s45  epochs=15  val=0.470319  56.9s  n_train=13924
[3/1608] cuda:1 itr_o01_K04_H024_s45  epochs=10  val=0.470059  40.0s

/tmp/ipykernel_31/2205983436.py:99: UserWarning: rdg_o02_K04_H024_s42: ridge alpha 1e+06 sits at the edge of (0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0); the grid may not bracket the optimum
  model, cfg, outcome = cell.model_config(configs).fit(


[625/1608] cuda:1 rdg_o02_K08_H024_s42  epochs=0  val=3.729250  0.3s  n_train=13689


/tmp/ipykernel_31/2205983436.py:99: UserWarning: rdg_o02_K08_H024_s42: ridge alpha 1e+06 sits at the edge of (0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0); the grid may not bracket the optimum
  model, cfg, outcome = cell.model_config(configs).fit(


[626/1608] cuda:1 rdg_o02_K12_H024_s42  epochs=0  val=3.729008  0.6s  n_train=13689
[627/1608] cuda:1 rdg_o03_K01_H024_s42  epochs=0  val=0.331964  0.1s  n_train=13729


...

[677/1608] cuda:1 rdg_o15_K08_H024_s42  epochs=0  val=0.504514  0.4s  n_train=15217


/tmp/ipykernel_31/2205983436.py:99: UserWarning: rdg_o15_K08_H024_s42: ridge alpha 1e+06 sits at the edge of (0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0); the grid may not bracket the optimum
  model, cfg, outcome = cell.model_config(configs).fit(


[678/1608] cuda:0 rdg_o15_K12_H024_s42  epochs=0  val=0.504548  0.6s  n_train=15217


/tmp/ipykernel_31/2205983436.py:99: UserWarning: rdg_o15_K12_H024_s42: ridge alpha 1e+06 sits at the edge of (0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0); the grid may not bracket the optimum
  model, cfg, outcome = cell.model_config(configs).fit(


[679/1608] cuda:1 dlin_o01_K08_H024_s42  epochs=30  val=0.584577  32.0s  n_train=13924
[680/1608] cuda:0 dlin_o01_K08_H024_s43  epochs=30  val=0.584593  32.0s  n_train=13924
[681/1608] cuda:1 dlin_o01_K08_H024_s44  epochs=30  val=0.584522  32.1s  n_train=13924
[682/1608] cuda:0 dlin_o01_K08_H024_s45  epochs=30  val=0.584622  32.1s  n_train=13924
[684/1608] cuda:0 dlin_o02_K08_H024_s42  epochs=30  val=1.723213  31.5s  n_train=13689
[683/1608] cuda:1 dlin_o01_K08_H024_s46  epochs=30  val=0.584560  32.0s  n_train=13924
[685/1608] cuda:0 dlin_o02_K08_H024_s43  epochs=30  val=1.723716  31.6s  n_train=13689
[686/1608] cuda:1 dlin_o02_K08_H024_s44  epochs=30  val=1.723012  31.8s  n_train=13689
[687/1608] cuda:0 dlin_o02_K08_H024_s45  epochs=30  val=1.723075  32.2s  n_train=13689
[688/1608] cuda:1 dlin_o02_K08_H024_s46  epochs=30  val=1.723857  32.3s  n_train=13689
[689/1608] cuda:0 dlin_o03_K08_H024_s42  epochs=30  val=0.488274  32.0s  n_train=13729
[690/1608] cuda:1 dlin_o03_K08_H024_s43  ep

<div style="background: linear-gradient(135deg, #2d0036, #4a0060); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #e0aaff33;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📈 21 &middot; Evaluasi — RQ1, RQ2, RQ3
  </h2>
  <p style="color: #c77dff; margin: 0; font-size: 1.02em;">Setiap angka di bawah bermuara pada berkas prediksi yang dipersist dan satu config hash, atau ia tidak masuk manuskrip.</p>
  <ul style="color: #c77dff; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Metrik rasio dibentuk dari <strong>MSE yang sudah dirata-rata seed</strong>, tidak pernah dari rata-rata rasio per-seed (<code>D42</code>): keduanya berbeda oleh Jensen, dan yang kedua menuntut memasangkan seed 42 di K=1 dengan seed 42 di K=8 — dua run latih independen dari model berbeda, di mana salah satu dari 5! urutan memberi jawaban berbeda.</li>
    <li style="margin-bottom: 6px;"><strong>Unit inferensinya adalah origin.</strong> Dispersi seed adalah diagnostik derau Monte-Carlo, tidak pernah ketidakpastian atas estimasi yang teragregasi lintas origin (<code>D30</code>).</li>
    <li style="margin-bottom: 6px;">Perbandingan lintas-origin apa pun dilakukan pada RelMSE atau <code>R&sup2;_oos</code>, <strong>tidak pernah</strong> pada MSE ruang-scaler: dua origin membawa &sigma;<sub>g</sub> berbeda, dan angka mentahnya 99,7% adalah drift scaler (<code>D60i</code>).</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #e94560; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #f5a623; margin: 0 0 6px 0; font-size: 1.22em;">1️⃣ RQ1 — K nominal atau K_eff?</h3>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.94em;">ΔMSE per rung, TOST terhadap margin pra-registrasi, dan uji-J non-nested yang memacu kedua penjelasan (<code>D32</code>, <code>D49</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧾 <code>artifacts/meta/*.json</code></span></div>
</div>

In [155]:
if not GRID_COMPLETE:
    print("RQ1: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    done = sorted(completed_run_ids(discover_roots(ARTIFACTS)))
    grid = gather_grid(done, discover_roots(ARTIFACTS))
    seed_avg = seed_average(grid)
    print(f"gathered {len(done)} runs -> {grid.height} run-block rows -> "
          f"{seed_avg.height} seed-averaged cells")

    main = seed_avg.filter((pl.col("model") == "itr") & (pl.col("pred_len") == 24))

    # Root section 9.2 / D30: any number aggregated across origins carries the SE
    # ACROSS ORIGINS, never the seed std. Seed dispersion measures re-initialisation
    # noise on one fixed dataset; origin dispersion measures the sampling variability
    # of the estimand, and in walk-forward crypto evaluation the second is typically
    # an order of magnitude larger. Reporting the first as "+/-" on an aggregated row
    # understates the headline uncertainty by roughly that factor — reintroducing,
    # through the reporting convention, the overstated precision the wild cluster
    # bootstrap was added to prevent.
    per_origin = main.group_by(["origin", "k"]).agg(
        pl.col("mse").mean().alias("mse"), pl.col("r2_oos").mean().alias("r2_oos")
    )
    rung = (
        per_origin.group_by("k")
        .agg(
            pl.col("mse").mean().alias("MSE"),
            (pl.col("mse").std() / pl.col("mse").count().sqrt()).alias("SE_across_origins"),
            pl.col("r2_oos").mean().alias("R2_oos"),
            pl.col("mse").count().alias("n_origins"),
        )
        .sort("k")
    )
    print("\n--- RQ1: free rung effects ---")
    print(rung)
    print(f"seed std, a Monte-Carlo diagnostic only: {main['mse_seed_std'].mean():.6f} "
          f"mean across cells at n={main['n_seeds'][0]} seeds per cell")

    wide = {int(k): per_origin.filter(pl.col("k") == k).sort("origin")["mse"].to_numpy()
            for k in (1, 4, 8, 12)}
    d_4_8, d_8_12 = wide[4] - wide[8], wide[8] - wide[12]
    margin = 0.25 * abs(float(d_4_8.mean()))
    print(f"\ndelta MSE 4->8 = {d_4_8.mean():+.6f}   8->12 = {d_8_12.mean():+.6f}")
    print("D49's margin is 0.25 x delta(4->8), fixed in advance: a non-significant "
          "delta is a failure to reject, not evidence of equivalence, and choosing the "
          "margin after seeing the rung is the p-hacking section 3 forbids for tau.")
    print(tost_equivalence(d_8_12, margin))

    # D32: RQ1 is a NON-NESTED comparison, not an OLS on three points. Four rungs give
    # three deltas, and stacking 360 rows creates no information about a slope that
    # varies only between rungs. K_eff measured per origin is what makes the regressor
    # vary at all — and it is leak-free because the span is training-only.
    keff_join = keff_tbl.select(["origin", "k", "pr_raw"]).rename({"pr_raw": "k_eff"})
    race = main.join(keff_join, on=["origin", "k"], how="inner")
    groups = race["origin_index"].to_numpy() * 100 + race["block"].to_numpy()
    t_ab, p_ab = j_test(race["mse"].to_numpy(),
                        race["k"].to_numpy().astype(float),
                        race["k_eff"].to_numpy(), groups)
    t_ba, p_ba = j_test(race["mse"].to_numpy(), race["k_eff"].to_numpy(),
                        race["k"].to_numpy().astype(float), groups)
    print(f"\nJ-test  K augmented by K_eff: t={t_ab:+.3f} p={p_ab:.4f}   |   "
          f"K_eff augmented by K: t={t_ba:+.3f} p={p_ba:.4f}")
    print("Both reject -> neither explanation alone suffices. Neither rejects -> the "
          "data cannot separate them, which at corr(K, K_eff) near 1 is the outcome to "
          "expect and to report plainly rather than to spin.")


gathered 1620 runs -> 9675 run-block rows -> 2403 seed-averaged cells

--- RQ1: free rung effects ---
shape: (4, 5)
┌─────┬──────────┬───────────────────┬───────────┬───────────┐
│ k   ┆ MSE      ┆ SE_across_origins ┆ R2_oos    ┆ n_origins │
│ --- ┆ ---      ┆ ---               ┆ ---       ┆ ---       │
│ i32 ┆ f32      ┆ f64               ┆ f32       ┆ u32       │
╞═════╪══════════╪═══════════════════╪═══════════╪═══════════╡
│ 1   ┆ 0.862582 ┆ 0.095893          ┆ -0.020455 ┆ 15        │
│ 4   ┆ 0.86129  ┆ 0.095855          ┆ -0.018682 ┆ 15        │
│ 8   ┆ 0.860653 ┆ 0.095709          ┆ -0.017993 ┆ 15        │
│ 12  ┆ 0.861091 ┆ 0.095772          ┆ -0.01857  ┆ 15        │
└─────┴──────────┴───────────────────┴───────────┴───────────┘
seed std, a Monte-Carlo diagnostic only: 0.001334 mean across cells at n=5 seeds per cell

delta MSE 4->8 = +0.000636   8->12 = -0.000437
D49's margin is 0.25 x delta(4->8), fixed in advance: a non-significant delta is a failure to reject, not evidence o

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #e94560; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #f5a623; margin: 0 0 6px 0; font-size: 1.22em;">2️⃣ RQ2 — apakah gap menyempit seiring umur model?</h3>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.94em;">β₁ dengan origin fixed effects, klaster per origin, WCR B = 99.999 — dan MDE dicetak di sampingnya, karena null tanpa daya tidak berarti apa-apa (root §9.2 syarat 6).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧾 <code>artifacts/meta/*.json</code></span></div>
</div>

In [156]:
if not GRID_COMPLETE:
    print("RQ2: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    print("--- RQ2: does the multivariate gap narrow with model age? ---")
    amp = amplification(seed_avg, k_small=1, k_large=8)
    print(amp.select(["origin", "block", "mse_small", "mse_large", "A"]).head(12))

    beta = panel_beta1(amp, value="A", B=99_999, seed=42)
    print(f"\n{beta}")
    mde = minimum_detectable_beta1(beta.within_slopes)
    print(f"\nminimum detectable beta1 at 80% power, alpha=0.05: {mde:+.6f}")
    print(f"observed {beta.beta1:+.6f} is "
          f"{'INSIDE (undetectable)' if abs(beta.beta1) < abs(mde) else 'outside'} it")
    print("If the MDE exceeds the plausible magnitude of A, RQ2 must be repositioned as "
          "descriptive BEFORE the grid: a non-significant beta1 is otherwise "
          "indistinguishable from a design that could never have detected decay.")

    # D28: consecutive origins share 79.2% of their training data, so the clusters are
    # NOT independent draws and the bootstrapped p is anticonservative by an
    # unquantified amount. Windows become disjoint only at stride 5.
    print("\ntraining-window overlap: 79.2% at stride 1, 58.3% at 2, 37.5% at 3, "
          "16.7% at 4. Disjoint only at stride 5, which leaves G=3.")
    for offset in range(5):
        triple = [ORIGINS[i].label for i in range(offset, len(ORIGINS), 5)]
        subset = amp.filter(pl.col("origin").is_in(triple))
        if subset.height == len(triple) * 6:
            sub = panel_beta1(subset, value="A", B=9_999, seed=42)
            print(f"  {triple}: beta1={sub.beta1:+.6f}  p={sub.headline_p:.4f}  (G=3)")
    print("At G=3 these will very likely be inconclusive, and THAT IS THE FINDING — it "
          "bounds what the full-panel p-value can honestly claim.")

    # D50: K=1 vs K=8 differs in information AND in whether attention is active, at the
    # same time. This holds information fixed and varies only what attention selects.
    try:
        attn = attention_amplification(seed_avg, k=8)
        print(f"\nA_attn (uniform-attention control, D50):")
        print(panel_beta1(attn, value="A_attn", B=99_999, seed=42))
        print("Reporting both decompositions answers information-versus-attention "
              "directly, at runs Figure 5 needs anyway.")
    except (ValueError, KeyError) as exc:
        print(f"\nuniform-attention arm not complete yet: {exc}")

    # The falsification arm: the only design that identifies decay directly.
    aged = main.filter((pl.col("k") == 8) & (pl.col("block") >= 4)).select(
        ["origin_index", "block", "mse"]).rename({"mse": "mse_aged"})
    fresh = seed_avg.filter(pl.col("model") == "itrf").select(
        ["origin_index", "block", "mse"]).rename({"mse": "mse_fresh"})
    falsify = aged.join(fresh, on=["origin_index", "block"], how="inner")
    if falsify.height:
        gap = (falsify["mse_aged"] - falsify["mse_fresh"]).to_numpy()
        print(f"\nfalsification arm: mean(aged - fresh) = {gap.mean():+.6f} over "
              f"{len(gap)} (origin, block) cells")
        print("Positive means the fresh model really is better, so decay is age. Near "
              "zero while beta1 < 0 means beta1 is CALENDAR, not age, and RQ2's "
              "headline is an artefact.")
    else:
        print("\nfalsification arm not complete yet")


--- RQ2: does the multivariate gap narrow with model age? ---
shape: (12, 5)
┌─────────┬───────┬───────────┬───────────┬───────────┐
│ origin  ┆ block ┆ mse_small ┆ mse_large ┆ A         │
│ ---     ┆ ---   ┆ ---       ┆ ---       ┆ ---       │
│ str     ┆ i8    ┆ f32       ┆ f32       ┆ f32       │
╞═════════╪═══════╪═══════════╪═══════════╪═══════════╡
│ 2020-01 ┆ 1     ┆ 0.364145  ┆ 0.363549  ┆ 0.001637  │
│ 2020-01 ┆ 2     ┆ 0.329692  ┆ 0.327483  ┆ 0.0067    │
│ 2020-01 ┆ 3     ┆ 5.299982  ┆ 5.302567  ┆ -0.000488 │
│ 2020-01 ┆ 4     ┆ 0.633412  ┆ 0.634531  ┆ -0.001767 │
│ 2020-01 ┆ 5     ┆ 0.8336    ┆ 0.827101  ┆ 0.007796  │
│ …       ┆ …     ┆ …         ┆ …         ┆ …         │
│ 2020-06 ┆ 2     ┆ 0.535258  ┆ 0.533722  ┆ 0.00287   │
│ 2020-06 ┆ 3     ┆ 0.369908  ┆ 0.370113  ┆ -0.000552 │
│ 2020-06 ┆ 4     ┆ 0.533631  ┆ 0.534584  ┆ -0.001786 │
│ 2020-06 ┆ 5     ┆ 0.416878  ┆ 0.418755  ┆ -0.004502 │
│ 2020-06 ┆ 6     ┆ 1.123276  ┆ 1.114366  ┆ 0.007932  │
└─────────┴───────┴────────

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #e94560; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #f5a623; margin: 0 0 6px 0; font-size: 1.22em;">3️⃣ RQ3 — cadence retraining optimal?</h3>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.94em;">Bercabang ke <strong>undefined</strong>, tidak pernah ke no decay detected: frasa kedua adalah bentuk right-censored dan ia menyiratkan edge yang data ini tidak punya (<code>D55</code>, <code>D60b</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧾 <code>artifacts/meta/*.json</code></span></div>
</div>

In [157]:
if not GRID_COMPLETE:
    print("RQ3: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    print("--- RQ3: what retraining cadence? ---")
    dec = decay(seed_avg, k=8)
    print(dec.table)
    if dec.excluded_origins:
        print(f"\nEXCLUDED, mean R2_oos <= 0 so there is no edge to lose a proportion "
              f"of: {list(dec.excluded_origins)}")
        print("Named, never silently dropped. Root section 10.3's first measured run "
              "returned R2_oos = -0.0183, so this guard may be the common case rather "
              "than the edge case section 9.1 assumed.")

    print("\ntau sensitivity — headline 5%, pre-registered before the curve was seen:")
    b_star_rows = []

    if not dec.table.height:
        # Every origin failed the R2_oos > 0 guard, so D(i,b) has no denominator
        # anywhere and b* has nothing to estimate (D55). This is NOT censoring: a
        # censored origin has an edge that never decays past tau within 180 days,
        # whereas here there is no edge to lose a proportion of. Reporting the two
        # in one wording would claim skill the grid never found.
        for tau in TAU_SENSITIVITY:
            flag = "   <-- HEADLINE" if abs(tau - TAU_HEADLINE) < 1e-9 else ""
            print(f"  tau={tau:>6.1%}  UNDEFINED — no origin has positive mean skill{flag}")
            b_star_rows.append({"tau": tau, "status": "undefined", "median_b_star": None,
                                "ci_low": None, "ci_high": None, "events": 0,
                                "censored": 0, "n_origins": 0})
        print(f"\nRQ3 RETURNS NO ANSWER, and that is the finding. D(i,b) is a proportion "
              f"of skill lost; all {len(dec.excluded_origins)} origins have mean "
              f"R2_oos <= 0, so the proportion is undefined rather than large or small.")
        print("Root section 9.1's guard was written for an edge case and is here the "
              "ONLY case. Report it as 'the decay estimand is undefined under "
              "non-positive out-of-sample skill' — never as 'no decay detected within "
              "180 days', which is the right-censored wording and asserts an edge.")
    else:
        for tau in TAU_SENSITIVITY:
            bs = dec.b_star(tau)
            km = kaplan_meier(bs["b_star"].to_numpy(), bs["event"].to_numpy())
            lo, hi = km.median_interval
            median = "censored >6" if km.median == float("inf") else f"{km.median:.0f}"
            interval = "censored" if lo == float("inf") else f"[{lo:.0f}, {hi:.0f}]"
            flag = "   <-- HEADLINE" if abs(tau - TAU_HEADLINE) < 1e-9 else ""
            print(f"  tau={tau:>6.1%}  crossings {km.n_events}/"
                  f"{km.n_events + km.n_censored}  median b* {median}  CI {interval}{flag}")
            b_star_rows.append({"tau": tau, "status": "estimated",
                                "median_b_star": km.median, "ci_low": lo,
                                "ci_high": hi, "events": km.n_events,
                                "censored": km.n_censored, "n_origins": bs.height})

        print("\nb* resolves only to 30-day granularity and only out to 180 days. If no "
              "block crosses tau, the honest answer is 'no decay detected within 180 days' "
              "— a right-censored result, not a missing one. Say it in those words, and put "
              "the INTERVAL in the abstract, never a bare integer.")

    # H3: larger K decays faster. Needs surviving origins AND at least one crossing:
    # with zero events in both arms the log-rank variance is zero and the statistic
    # is 0/0, which prints as nan and reads like a computed result. Section 12 calls
    # a number that cannot be regenerated a documented failure, so say why instead.
    a = dec.b_star(TAU_HEADLINE)
    b = decay(seed_avg, k=1).b_star(TAU_HEADLINE)
    events = (int(a["event"].sum()) if a.height else 0,
              int(b["event"].sum()) if b.height else 0)
    if a.height and b.height and sum(events):
        chi2, p = logrank(a["b_star"].to_numpy(), a["event"].to_numpy(),
                          b["b_star"].to_numpy(), b["event"].to_numpy())
        print(f"\nlog-rank K=8 vs K=1 at tau=5%: chi2={chi2:.3f}  p={p:.4f}  "
              f"(H3; crossings K=8 {events[0]}, K=1 {events[1]})")
    elif not (a.height and b.height):
        print(f"\nlog-rank K=8 vs K=1 UNAVAILABLE: surviving origins K=8 {a.height}, "
              f"K=1 {b.height}. H3 compares decay RATES, so it needs an edge in both "
              "arms; with none, H3 is untestable rather than rejected.")
    else:
        print(f"\nlog-rank K=8 vs K=1 UNAVAILABLE: zero crossings in both arms "
              f"({a.height} and {b.height} origins, all censored at 6). The statistic "
              "is 0/0 here, not a large p-value — H3 is untestable, and reporting a "
              "nan as though it were computed would be the same defect as D55.")


--- RQ3: what retraining cadence? ---
shape: (0, 7)
┌────────┬──────────────┬───────┬───────────┬────────┬────────┬─────┐
│ origin ┆ origin_index ┆ block ┆ n_windows ┆ r2_oos ┆ r2_ref ┆ D   │
│ ---    ┆ ---          ┆ ---   ┆ ---       ┆ ---    ┆ ---    ┆ --- │
│ str    ┆ i32          ┆ i8    ┆ u32       ┆ f32    ┆ f32    ┆ f32 │
╞════════╪══════════════╪═══════╪═══════════╪════════╪════════╪═════╡
└────────┴──────────────┴───────┴───────────┴────────┴────────┴─────┘

EXCLUDED, mean R2_oos <= 0 so there is no edge to lose a proportion of: ['2020-01', '2020-06', '2020-11', '2021-04', '2021-09', '2022-02', '2022-07', '2022-12', '2023-05', '2023-10', '2024-03', '2024-08', '2025-01', '2025-06', '2025-11']
Named, never silently dropped. Root section 10.3's first measured run returned R2_oos = -0.0183, so this guard may be the common case rather than the edge case section 9.1 assumed.

tau sensitivity — headline 5%, pre-registered before the curve was seen:
  tau=  2.5%  UNDEFINED — no origi

<div style="background: linear-gradient(135deg, #1b0033, #2d0052); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #bf5af233;">
  <h2 style="color: #e0aaff; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    💾 22 &middot; Simpan angka
  </h2>
  <p style="color: #cbb2e8; margin: 0; font-size: 1.02em;">Setiap tabel dan figure digenerate DARI paper_numbers.json, tidak pernah disalin tangan.</p>
  <ul style="color: #cbb2e8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Angka yang dihasilkan di bawah hash artefak input yang berbeda tidak sebanding dan tidak boleh berbagi satu tabel, jadi digest parquet ikut bersamanya — begitu pula <code>code_sha256</code>, yang mengidentifikasi kode di luar repo.</li>
    <li style="margin-bottom: 6px;">Angka yang tidak bisa diregenerasi adalah kegagalan yang terdokumentasi, bukan catatan kaki.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 4px solid #e94560; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #f5a623; margin: 0 0 6px 0; font-size: 1.22em;">🧊 Simpan panel metrik — enam parquet plus paper_numbers.json</h3>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.94em;">Inilah sel yang <strong>menulis hasil metrik dalam bentuk parquet</strong>. Setiap panel diturunkan dari <code>preds/</code> mentah, bukan dari angka yang disalin: root §12 menuntut tiap angka bisa diregenerasi.</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 8px;"><span style="color:#ffd166; font-weight:600; font-size:0.84em;">MENULIS</span> <span style="color:#dcdcdc; font-size:0.84em;">🧾 <code>artifacts/paper_numbers.json</code> &nbsp; 🧊 <code>artifacts/run_block_metrics.parquet</code> &nbsp; 🧊 <code>artifacts/seed_averaged_cells.parquet</code> &nbsp; 🧊 <code>artifacts/amplification_panel.parquet</code> &nbsp; 🧊 <code>artifacts/decay_panel.parquet</code></span></div>
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧾 <code>artifacts/meta/*.json</code></span></div>
</div>

In [158]:
if not GRID_COMPLETE:
    print("paper_numbers.json: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    # ``_digest_source``, never ``_provenance``: in a flattened notebook every module
    # shares one kernel namespace, and ``report._provenance`` is a *function*
    # ``build_report`` calls one cell later. Binding that name to a string here
    # killed the last cell of the 894-run session with ``TypeError: 'str' object is
    # not callable`` --- after the grid had run for 7.8 hours and every artifact was
    # already safe on disk. `D59` from the opposite direction: not an import the
    # flattening dropped, but a package-private name an evaluation cell shadowed (`D65`).
    _digest, _digest_source = _input_sha256(PARQUET)

    paper_numbers = {
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "input_parquet": str(PARQUET),
        "input_sha256": _digest,
        "input_sha256_source": _digest_source,
        "code_sha256": code_sha256(),
        "runs_complete": len(done),
        "runs_in_manifest": len(ALL),
        "keff": {
            "gate_pr_k8_pre_first_origin": gate,
            "gate_floor": GATE_PR_FLOOR,
            "corr_k_keff": corr_k_keff(keff_tbl),
            "per_rung": rung_view.to_dicts(),
        },
        "rq1": {
            "rung_effects": rung.to_dicts(),
            "delta_4_to_8": float(d_4_8.mean()),
            "delta_8_to_12": float(d_8_12.mean()),
            "tost_margin": margin,
            "tost": str(tost_equivalence(d_8_12, margin)),
            "j_test_k_augmented_by_keff": {"t": t_ab, "p": p_ab},
            "j_test_keff_augmented_by_k": {"t": t_ba, "p": p_ba},
        },
        "rq2": {
            "beta1": beta.beta1, "t": beta.t_statistic, "cluster_se": beta.cluster_se,
            "p_rademacher": beta.p_rademacher, "p_webb": beta.p_webb,
            "headline_p": beta.headline_p, "G": beta.n_clusters,
            "N": beta.n_observations, "B": beta.B,
            "minimum_detectable_beta1": mde,
            "within_slopes": beta.within_slopes.tolist(),
            "effective_independent_training_sets": 4,
            "consecutive_origin_overlap_pct": 79.2,
        },
        "rq3": {
            "tau_headline": TAU_HEADLINE,
            "b_star": b_star_rows,
            "excluded_origins": list(dec.excluded_origins),
        },
    }

    out = ARTIFACTS / "paper_numbers.json"
    out.write_text(json.dumps(paper_numbers, indent=2, default=float))
    seed_avg.write_parquet(ARTIFACTS / "seed_averaged_cells.parquet")
    grid.write_parquet(ARTIFACTS / "run_block_metrics.parquet")
    amp.write_parquet(ARTIFACTS / "amplification_panel.parquet")
    dec.table.write_parquet(ARTIFACTS / "decay_panel.parquet")

    print(f"wrote {out}")
    for path in sorted(ARTIFACTS.glob("*.parquet")):
        print(f"  {path.name}  {path.stat().st_size / 1e6:.2f} MB")
    print(f"\npreds {len(list((ARTIFACTS / 'preds').glob('*.parquet')))} files  |  "
          f"meta {len(list((ARTIFACTS / 'meta').glob('*.json')))} files")
    print(f"remaining runs: {len(pending(ALL, discover_roots(ARTIFACTS)))}")
    print("\nSave Version now, then attach this output as the next session's input "
          "Dataset. Nothing else needs doing by hand.")


wrote artifacts/paper_numbers.json
  amplification_panel.parquet  0.00 MB
  decay_panel.parquet  0.00 MB
  keff_table.parquet  0.01 MB
  naive_rw_by_origin.parquet  0.00 MB
  run_block_metrics.parquet  0.14 MB
  seed_averaged_cells.parquet  0.05 MB

preds 1620 files  |  meta 1621 files
remaining runs: 0

Save Version now, then attach this output as the next session's input Dataset. Nothing else needs doing by hand.


<div style="background: linear-gradient(135deg, #001a0d, #003317); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #52b78833;">
  <h2 style="color: #95d5b2; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    📊 23 &middot; Tabel & figure
  </h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 1.02em;">Semua yang root &sect;13.4 janjikan, di-render dari paper_numbers.json dan tidak pernah disalin tangan.</p>
  <ul style="color: #b7e4c7; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Tiga dari empat deliverable yang dulu tanpa masukan — matriks DM, evaluasi ekonomi, kurva ekuitas — dihitung di sini dari berkas prediksi yang sudah ada di disk.</li>
    <li style="margin-bottom: 6px;"><strong>Figure 5 pengecualian</strong>: bobot attention tidak pernah dipersist oleh grid asli, jadi ia butuh arm <code>attention</code> dan dilewati <em>dengan disebut namanya</em> sampai arm itu jalan — sumbu kosong berlabel figure terbaca sebagai pengukuran atas ketiadaan.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 4px solid #52b788; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #95d5b2; margin: 0 0 6px 0; font-size: 1.22em;">🖼️ Render deliverable — sepuluh tabel dan delapan figure</h3>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.94em;">Inilah sel yang <strong>memproduksi seluruh figure</strong> dan seluruh <code>.tex</code>. Semuanya di-render <em>dari</em> <code>paper_numbers.json</code>, tidak pernah ditranskripsi tangan (<code>D62a</code>).</p>
</div>

<div style="background:#0e0e12; border-left:4px solid #6c757d; border-radius:0 8px 8px 0; padding:10px 22px; margin-top:-6px;">
<div style="margin-top: 8px;"><span style="color:#ffd166; font-weight:600; font-size:0.84em;">MENULIS</span> <span style="color:#dcdcdc; font-size:0.84em;">🧾 <code>paper/paper_numbers.json</code> &nbsp; 📄 <code>paper/tables/*.tex</code> &nbsp; 🖼️ <code>paper/figures/*.pdf</code> &nbsp; 🖼️ <code>paper/figures/*.png</code> &nbsp; 🧊 <code>paper/panels/*.parquet</code></span></div>
<div style="margin-top: 4px;"><span style="color:#9ec5fe; font-weight:600; font-size:0.84em;">MEMBACA</span> <span style="color:#b8b8b8; font-size:0.84em;">🧾 <code>artifacts/paper_numbers.json</code> &nbsp; 🧊 <code>artifacts/preds/*.parquet</code> &nbsp; 🧊 <code>artifacts/attn/*.parquet</code></span></div>
</div>

In [159]:
if not GRID_COMPLETE:
    print("tables and figures: SKIPPED — the grid is incomplete, so the panel is "
          "unbalanced and §9.1's estimators refuse it by design. "
          "Resume in the next session; nothing is recomputed.")
else:
    PAPER = WORK / "paper"

    # The manuscript's single source. The grid's own paper_numbers.json written above
    # stays immutable evidence; this reads it, adds every analysis pass the grid never
    # ran — section 4.5's efficiency tests, the DM/Romano-Wolf/MCS matrix, the
    # economic evaluation, directional accuracy, the horizon aggregation, the raw
    # metric scale, D60i's RelMSE falsification gap and D45's coverage check — and
    # names the grid file by digest so the two cannot silently diverge (root §12).
    report_inputs = build_report(
        ARTIFACTS,
        bars,
        features,
        roots=discover_roots(ARTIFACTS),
        bootstrap_b=9_999,
        seed=42,
        log=lambda message: print(message, flush=True),
    )

    PAPER.mkdir(parents=True, exist_ok=True)
    numbers_path = PAPER / "paper_numbers.json"
    numbers_path.write_text(json.dumps(report_inputs.numbers, indent=2, default=float))
    print(f"\nwrote {numbers_path}")

    print("\ntables:")
    for path in render_tables(report_inputs.numbers, PAPER / "tables"):
        print(f"  {path.name}")

    print("\nfigures:")
    for path in render_figures(report_inputs, PAPER / "figures", log=print):
        print(f"  {path.name}")

    # The frames a figure reads, persisted beside the numbers so a plot can be redone
    # without re-running the whole aggregation. Figure 2b alone is ~3,000 points and
    # has no business inside a JSON file a human is expected to read.
    panels = PAPER / "panels"
    panels.mkdir(parents=True, exist_ok=True)
    for _name, _frame in (
        ("seed_averaged_cells", report_inputs.seed_avg),
        ("amplification_panel", report_inputs.amplification),
        ("rolling_pr", report_inputs.rolling_pr),
        ("rolling_ols_r2", report_inputs.rolling_r2),
        ("equity_curves", report_inputs.equity),
    ):
        _frame.write_parquet(panels / f"{_name}.parquet")
    if report_inputs.attention is not None:
        report_inputs.attention.write_parquet(panels / "attention_maps.parquet")
    print(f"\nwrote panels to {panels}")

    _robustness = report_inputs.numbers["robustness"]
    print("\nD62 robustness arms:")
    for _arm, _state in _robustness.items():
        print(f"  {_arm:12s} {_state['status']}")
    print(
        "\nAn arm reading 'not run' is a pre-registered exploratory arm awaiting its "
        "runs, not a failure. Whatever it returns goes in the paper: an arm reported "
        "only when it agrees with the headline is not a robustness arm (root §13.2)."
    )


report: 1620 completed runs under artifacts
report: 9675 run-block rows -> 2403 seed-averaged cells
report: 36 architecture cells
report: efficiency 16 spans, rolling 3044 windows
report: 105 pairs, MCS over 15 models
report: 270 economic cells, 72968 equity points
report: DA over 300 runs

wrote /kaggle/working/paper/paper_numbers.json

tables:
  table1_dataset.tex
  table2_efficiency.tex
  table2b_keff.tex
  table3_architecture.tex
  table4_main.tex
  table5_decay.tex
  table6_dm.tex
  table7_horizons.tex
  table8_economics.tex
  table9_robustness.tex

figures:
  figure1_walkforward.pdf
  figure1_walkforward.png
  figure2_architecture.pdf
  figure2_architecture.png
  figure2b_rolling.pdf
  figure2b_rolling.png
  figure3_decay.pdf
  figure3_decay.png
  figure4_relmse.pdf
  figure4_relmse.png
  figure5_attention.pdf
  figure5_attention.png
  figure6_horizons.pdf
  figure6_horizons.png
  figure7_equity.pdf
  figure7_equity.png

wrote panels to /kaggle/working/paper/panels

D62 robustnes

<div style="background: linear-gradient(135deg, #101010, #1c1c1c); border-radius: 14px; padding: 26px 32px; margin-bottom: 8px; border: 1px solid #9e9e9e33;">
  <h2 style="color: #d0d0d0; margin: 0 0 8px 0; font-size: 1.6em; font-weight: 700; letter-spacing: 0.5px;">
    🔁 24 &middot; Sinkron balik ke src/ — nonaktif
  </h2>
  <p style="color: #a8a8a8; margin: 0; font-size: 1.02em;">Notebook adalah tempat mengetik; <code>src/</code> adalah proyeksi yang diuji. Sel ini menutup arah baliknya, dan ia dikomentari penuh.</p>
  <ul style="color: #a8a8a8; margin: 10px 0 0 0; padding-left: 20px; font-size: 0.92em;">
    <li style="margin-bottom: 6px;">Sampai <code>D88</code>, mengedit sebuah sel berarti mengetik ulang perubahannya ke <code>src/</code> — dua salinan yang harus setuju tanpa ada yang memeriksa, yaitu <code>D54a</code> dan <code>D69</code> dengan kostum baru.</li>
    <li style="margin-bottom: 6px;"><strong>Dikomentari penuh dan tetap begitu.</strong> Di Kaggle tidak ada <code>tools/</code> maupun <code>src/</code>; sel aktif di sana akan gagal atau menulis sampah ke direktori kerja sesi. Aktivasi adalah keputusan sadar di checkout lokal.</li>
    <li style="margin-bottom: 6px;">Impor baru tidak bisa datang dari sel: impor hidup di sel Library yang digenerate <em>dari</em> modul (<code>D66</code>). Skripnya menolak dan menyebut <code>src/</code> sebagai tempatnya, bukan menebak.</li>
  </ul>
</div>

<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 4px solid #9e9e9e; border-radius: 8px; padding: 16px 22px;">
  <h3 style="color: #d0d0d0; margin: 0 0 6px 0; font-size: 1.22em;">🔁 Sinkron balik ke src/ — nonaktif, aktifkan sendiri</h3>
  <p style="color: #a8a8a8; margin: 0; font-size: 0.94em;">Notebook ini adalah tempat kamu mengetik; <code>src/</code> adalah proyeksi yang diuji. Sel di bawah menulis balik <code>src/itransformer_btc/</code> dari sel-sel di atas, dan ia <strong>dikomentari penuh</strong> — di Kaggle tidak ada <code>src/</code> maupun <code>tools/</code>, dan sel aktif di sana akan gagal atau menulis sampah. Hapus <code>#</code> hanya di checkout lokal, setelah menyimpan notebook (<code>D88</code>).</p>
</div>

In [ ]:
# ============================================================================
# SINKRON BALIK — NONAKTIF. Hapus '# ' pada empat baris terakhir untuk memakai.
# ============================================================================
#
# Apa yang dilakukannya
#   Mengumpulkan setiap sel definisi di notebook ini lewat metadata `itbtc`,
#   menggabungkannya per modul, menyisipkan kembali blok impor yang dibuang
#   flattening, lalu menulis src/itransformer_btc/. Setelah menulis ia
#   mem-flatten ulang hasilnya dan menuntut byte-identik dengan sel-sel tadi;
#   kalau tidak, berkasnya dikembalikan dan tidak ada yang berubah.
#
# Kapan dipakai
#   Setelah kamu menyunting sel definisi di notebook dan ingin perubahan itu
#   sampai ke paket yang diuji pytest dan di-hash oleh code_sha256.
#
# Syarat
#   1. SIMPAN notebook lebih dulu. Skrip membaca berkas .ipynb di disk,
#      bukan kernel yang sedang jalan.
#   2. Jalankan dari checkout lokal. Di Kaggle tidak ada tools/ maupun src/.
#   3. Impor baru TIDAK bisa ditambahkan dari sel. Impor hidup di sel Library
#      yang digenerate *dari* modul (`D66`), jadi sel tidak punya baris impor
#      untuk disunting. Skrip menolak dan menyebut src/ sebagai tempatnya.
#
# Sesudahnya
#   python tools/build_notebook.py --check
#   lalu commit src/ dan notebook bersama-sama — keduanya satu perubahan.
#
# import subprocess, sys
# _sync = subprocess.run([sys.executable, "tools/notebook_to_src.py",
#                         "notebooks/iTransformer.ipynb"],
#                        capture_output=True, text=True)
# print(_sync.stdout or _sync.stderr)
